# 00 — YouTube Automation: Kalıcı Master Notebook (Orkestratör)

Bu, normal Colab kullanımında **açman gereken tek notebook**. Notebook 01–06'yı ayrıca yüklemene veya
çalıştırmana gerek yoktur — altı fazın hepsi burada, aynı `local_pipeline` kanonik modülleri üzerinden
gerçekten çalışır.

**Mimari sınır:** Ana üretim hattı yalnızca verilen kaynak videoyu `analiz → script → ses → render`
zinciriyle Shorts'a dönüştürür. YouTube keşfi ve viral tahmin bağımsız araştırma yan koludur;
kaynak videoyu, seçilen hot moment'i veya render kararını otomatik olarak değiştirmez.

## Maliyet politikası (ücretli sağlayıcı fan-out'u yok)

Her ücretli faz **tek bir seçili sağlayıcı/model** çağırır. Fallback kapalıyken bir istek en fazla
**bir** ücretli çağrı yapar; fallback açıkken adaylar **sırayla** denenir ve **ilk geçerli sonuçta
durulur** — bir sonraki aday yalnızca öncekinin gerçekten başarısız olduğu durumda çağrılır.
Karşılaştırma/benchmark amaçlı çok-modelli fan-out **sadece** standalone Notebook 02/03'te,
açık karşılaştırma sarmalayıcıları (`call_*_script`, `call_*_tts`) ile yapılır.

Denenen her sağlayıcı/model ve başarısızlık nedeni artefaktın `provider_attempts` alanına yazılır.

## Onay politikası (üretimden SONRA, içeriğe bağlı)

Üretim fonksiyonları **onay argümanı almaz** ve asla `approved` üretemez. Yeni üretilen içerik her zaman
`review_required` (gTTS için `development_placeholder`) olur — çünkü henüz kimse okumadı/dinlemedi/izlemedi.

Her faz için sıra: **üret → incele (görüntüle/dinle/izle) → ayrı onay hücresi**. Onay hücresi, inceleme
hücresinin yazdırdığı **SHA-256** değerini istemenizi gerektirir; bu, içeriği gerçekten gördüğünüzün
kaydıdır. İçerik sonradan değişirse (yeniden üretim, yeniden render) onay **otomatik olarak geçersiz**
olur (`approval.check_approval`). Hiçbir yerde otomatik `production_ready` yoktur ve
**bu notebook hiçbir şeyi YouTube'a yüklemez veya yayınlamaz.**

## Tek dosya Colab çalışması

`local_pipeline` modülleri bu notebook'a gömülüdür (base64 + SHA-256 doğrulamalı). Yükleme sırası:
repo → Drive/üst dizin kopyası → `PROJECT_REPO_URL` klonu → gömülü hash-doğrulanmış yedek.

## Bağımlılık farkındalığı

Bir faz kapalıysa, sonraki açık faz gerekli artefaktı diskte arar ve şemasını doğrular; bulamazsa
**sadece o fazı** net bir hatayla durdurur. Her faz iki durum bildirir:
`execution_status` (completed/skipped/failed) ve `artifact_status`
(review_required/approved/insufficient_data/failed/development_placeholder).

In [ ]:
!pip -q install requests pandas numpy google-api-python-client faster-whisper scenedetect librosa openai anthropic google-genai elevenlabs gTTS imageio-ffmpeg > /dev/null
print('Kurulum tamam.')

## Ana üretim hattı ve bağımsız araştırma anahtarları

In [ ]:
# --- ANA URETIM HATTI ---------------------------------------------------------
# Varsayilan: kaynak video -> moment secimi -> orijinal ses + gomulu altyazi.
# 'narrated_recap': istege bagli Gemini script -> TTS -> anlatimli render.
# NOT: PIPELINE_MODE artik ZORUNLU Faz 0 isleme planindan TURETILIR
# (direct_clip_series -> original_audio_subtitles, narrated_theory -> narrated_recap).
# Asagidaki deger yalnizca plan yokken gecerli olan guvenli baslangictir.
PIPELINE_MODE = 'original_audio_subtitles'  # 'original_audio_subtitles' | 'narrated_recap'
RUN_VIDEO_ANALYSIS = False
RUN_SCRIPT_GENERATION = False
RUN_TTS = False
RUN_RENDER = False

# --- BAGIMSIZ ARASTIRMA YAN KOLU: ana pipeline'a kaynak/karar BESLEMEZ -------
RUN_YOUTUBE_DATA = False
RUN_VIRAL_PREDICTION = False

# Her faz icin IKI ayri durum tutulur:
#   execution_status: completed | skipped | failed   (faz hucresi calisti mi?)
#   artifact_status : review_required | approved | insufficient_data | failed |
#                     development_placeholder | None
# Temiz calisip 'insufficient_data' artefakti yazan bir faz sadece "ok" gorunmez.
EXECUTION_STATUS_VALUES = {'completed', 'skipped', 'failed'}
ARTIFACT_STATUS_VALUES = {
    'review_required', 'approved', 'insufficient_data', 'failed',
    'development_placeholder', 'experimental', 'production_ready', None,
}
phase_report = {}


def record_phase(name, execution_status, artifact_status=None, detail=''):
    assert execution_status in EXECUTION_STATUS_VALUES, f'gecersiz execution_status: {execution_status!r}'
    assert artifact_status in ARTIFACT_STATUS_VALUES, f'gecersiz artifact_status: {artifact_status!r}'
    phase_report[name] = {
        'execution_status': execution_status,
        'artifact_status': artifact_status,
        'detail': detail,
    }
    suffix = f' | artifact_status={artifact_status}' if artifact_status is not None else ''
    print(f'[faz:{name}] execution_status={execution_status}{suffix}' + (f' | {detail}' if detail else ''))


## Google model router yapilandirmasi (tek merkez)

Her Google cagrisi `local_pipeline/google_ai.py` uzerinden gider. Notebook icinde **hicbir**
`genai.Client(...)` / sabit `model='...'` cagrisi yoktur.

**Kota proje bazlidir, anahtar bazli DEGILDIR** (ai.google.dev/gemini-api/docs/rate-limits).
Bu yuzden ikinci bir hesap/proje anahtari eklemek limitleri artirmaz; router yalnizca **ayni proje
icinde model degistirir**. `ALLOW_ACCOUNT_QUOTA_ROTATION` politika geregi `False`'tir ve
`google_ai.assert_no_account_quota_rotation()` True yapilirsa calismayi durdurur.

**Ucretsiz katman veri politikasi:** gonderdigin icerik Google tarafindan urun gelistirmede
kullanilabilir ve insan gozden gecirenler okuyabilir. Bu yuzden `ALLOW_RAW_MEDIA_TO_FREE_TIER`
varsayilan `False`: ham telifli video/ses **yuklenmez**, yalnizca yerel olarak uretilmis
transcript pencereleri / zaman damgalari / sahne-ses skorlari gonderilir.


In [ ]:
# --- Google model router yapilandirmasi (BURADAN AC/KAPA) --------------------
# Manuel credential profili: kota dolunca otomatik hesap gecisi YOKTUR.
# Profiller farkli proje/kanal ortamlari icindir; kota sinirini asmak icin profil degistirme.
# Profil degistirmek icin yalnizca asagidaki iki etiketi elle degistirip hucreleri yeniden calistir.
GOOGLE_CREDENTIAL_PROFILE = 'primary'
YOUTUBE_CREDENTIAL_PROFILE = 'primary'
GOOGLE_CREDENTIAL_PROFILES = {
    'primary': ('GeminiKey', 'GOOGLE_API_KEY'),
    'secondary': ('GeminiKeySecondary',),
}
YOUTUBE_CREDENTIAL_PROFILES = {
    'primary': ('YoutubeKEY', 'YOUTUBE_DATA_API_KEY'),
    'secondary': ('YoutubeKEYSecondary',),
}

GOOGLE_ROUTE_MODE = 'auto'              # 'auto' | 'explicit'
GOOGLE_TASK_MODEL_OVERRIDES = {}        # ornek: {'script_generation': ['gemini-3.5-flash-lite']}
ENABLE_GOOGLE_MODEL_FALLBACK = True     # False: gorev basina TEK Google modeli denenir
ENABLE_CROSS_PROVIDER_FALLBACK = True   # script/TTS/embedding: Groq/Cloudflare/Azure (kimlik varsa)
MAX_GOOGLE_MODEL_ATTEMPTS = 4
MAX_PROVIDER_ATTEMPTS = 3
RUN_GOOGLE_CONNECTION_TEST = False      # True: SADECE secili birincil modellere minik bir ping
ENABLE_GOOGLE_SEARCH_GROUNDING = False  # Gemini 2.5 arama-temelli arastirma
RUN_ANTIGRAVITY_RESEARCH = False        # deneysel ajan raporu (varsayilan KAPALI)
ALLOW_RAW_MEDIA_TO_FREE_TIER = False    # ham telifli medya ucretsiz katmana YUKLENMEZ

# POLITIKA -- ASLA True yapma. Birden fazla Google hesabi/projesi arasinda anahtar
# dondurerek RPD/RPM/TPM limitlerini asmak kota ihlalidir. Kota proje bazlidir;
# ikinci bir anahtar limiti artirmaz. Router yalnizca AYNI proje icinde MODEL degistirir.
ALLOW_ACCOUNT_QUOTA_ROTATION = False

# --- Her Google yetenegi icin BAGIMSIZ komut anahtari -----------------------
RUN_GOOGLE_HOTMOMENT_REVIEW = False     # Faz 1 adaylarinin nihai LLM incelemesi
RUN_GOOGLE_SCRIPT_GENERATION = True     # Faz 2 script uretimi (Google = kanonik varsayilan)
RUN_GOOGLE_BULK_CLASSIFICATION = False  # baslik/aciklama/hashtag, trend satiri siniflama
RUN_GOOGLE_GROUNDED_RESEARCH = False    # arama-temelli guncellik dogrulamasi
RUN_GOOGLE_TTS = True                   # Ana pipeline seslendirmesi: Gemini TTS (preview -> review_required)
RUN_GOOGLE_EMBEDDINGS = False           # duplikasyon/novelty/kumeleme

# Gemini TTS ayarlari (RUN_GOOGLE_TTS ve TTS_PROVIDER='gemini' ile kullanilir)
GEMINI_TTS_VOICE = 'Kore'               # bkz. google_ai.GEMINI_TTS_VOICES
GEMINI_TTS_LANGUAGE_CODE = 'tr-TR'
GEMINI_TTS_NARRATION_DIRECTION = 'Gergin, merak uyandiran bir tonda, net ve akici oku:'

# Antigravity deneysel arastirma icin ZORUNLU token butcesi (sessiz tuketim yasak)
ANTIGRAVITY_MAX_TOTAL_TOKENS = 8000


## Kurulum — Drive bir kez bağlanır, yollar bir kez tanımlanır, Secrets bir kez okunur

In [ ]:
import os, sys, json
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')
PROJECT_ROOT = Path.cwd()

USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/AVRUPA YAKASI/YoutubeOtomationv0"

# Colab Secrets erisim hatalari (panelde 'Notebook erisimi' kapali, secret hic
# eklenmemis, vb.) ACIKLAYICI olmali ama DEGERI asla gostermemeli.
_SECRET_ACCESS_NOTES = []


def get_secret(name):
    """Colab Secrets -> os.environ sirasiyla dener. Deger asla yazdirilmaz/loglanmaz."""
    try:
        from google.colab import userdata
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception as exc:
            _SECRET_ACCESS_NOTES.append(
                f"{name}: Colab Secrets okunamadi ({type(exc).__name__}). "
                "Sol paneldeki anahtar simgesi -> ilgili secret -> 'Notebook erisimi' ACIK olmali."
            )
    except Exception:
        pass
    return os.environ.get(name)


def resolve_alias(*names):
    """Ilk dolu alias'i dondurur: (deger, kullanilan_ad). Deger asla yazdirilmaz."""
    for name in names:
        value = get_secret(name)
        if value:
            return value, name
    return None, None


def resolve_profile_alias(profile, profiles, kind):
    """Yalnizca ELLE secilen profili cozer; baska profile otomatik gecmez."""
    if profile not in profiles:
        raise ValueError(f'Bilinmeyen {kind} profili: {profile!r}; mevcut={sorted(profiles)}')
    names = profiles[profile]
    names = (names,) if isinstance(names, str) else tuple(names)
    return resolve_alias(*names)


# Her servis icin tam olarak bir MANUEL profil secilir. Kota/429 baska profile gecis yaptirmaz.
YOUTUBE_DATA_API_KEY, YOUTUBE_SECRET_ALIAS = resolve_profile_alias(
    YOUTUBE_CREDENTIAL_PROFILE, YOUTUBE_CREDENTIAL_PROFILES, 'YouTube')
GOOGLE_API_KEY, GOOGLE_SECRET_ALIAS = resolve_profile_alias(
    GOOGLE_CREDENTIAL_PROFILE, GOOGLE_CREDENTIAL_PROFILES, 'Gemini')
TMDB_API_KEY, _ = resolve_alias('TMDB_API_KEY')

# Opsiyonel / varsayilan olarak KULLANILMAZ (capraz-saglayici fallback, TTS alternatifleri):
OPENAI_API_KEY, _ = resolve_alias('OPENAI_API_KEY')
ANTHROPIC_API_KEY, _ = resolve_alias('ANTHROPIC_API_KEY')
ELEVENLABS_API_KEY, _ = resolve_alias('ELEVENLABS_API_KEY')


def describe(value, alias=None):
    if not value:
        return 'missing'
    return f'configured ({alias})' if alias else 'configured'


print('--- Zorunlu/onerilen (Google-agirlikli ucretsiz strateji) ---')
print(f'Gemini profil={GOOGLE_CREDENTIAL_PROFILE}:', describe(GOOGLE_API_KEY, GOOGLE_SECRET_ALIAS))
print(f'YouTube profil={YOUTUBE_CREDENTIAL_PROFILE}:', describe(YOUTUBE_DATA_API_KEY, YOUTUBE_SECRET_ALIAS))
print('TMDB                                :', describe(TMDB_API_KEY))
print('--- Opsiyonel (varsayilan kapali) ---')
print('OpenAI     :', describe(OPENAI_API_KEY))
print('Anthropic  :', describe(ANTHROPIC_API_KEY))
print('ElevenLabs :', describe(ELEVENLABS_API_KEY))

if _SECRET_ACCESS_NOTES:
    print()
    print('!!! Colab Secrets erisim uyarilari (deger GOSTERILMEZ) !!!')
    for _note in _SECRET_ACCESS_NOTES:
        print(' -', _note)
if not GOOGLE_API_KEY:
    print()
    print("Gemini anahtari YOK. Duzeltme: Colab sol panel -> anahtar (Secrets) simgesi ->")
    print("  '+ Yeni gizli anahtar' -> Ad: 'GeminiKey' -> Deger: https://aistudio.google.com/apikey")
    print("  -> 'Notebook erisimi' anahtarini AC. Yerelde: ortam degiskeni GeminiKey veya GOOGLE_API_KEY.")


## `local_pipeline` modullerini yukle (tum standalone notebooklarla ayni tek kaynak)

In [ ]:
# @title Yerel pipeline yukleyici (otomatik - acmaniz gerekmez)
import sys, os, tempfile, subprocess, base64, hashlib, zlib
from pathlib import Path

PROJECT_REPO_URL = ''  # opsiyonel: gercek bir git repo URL'si verirsen modul(ler) oradan klonlanir

_LOCAL_PIPELINE_MODULE_NAMES = ['video_analysis', 'script_generation', 'tts', 'render_utils', 'youtube_data', 'viral_prediction', 'approval', 'google_ai', 'provider_adapters', 'narration_presets', 'source_strategy']

# Her modul icin: (base64 kaynak, SHA-256 hash). Hash, gomulu payload'in
# bozulmadigini/kesilmedigini dogrulamak icin kullanilir -- eslesmezse acik
# bir hata verilir, asla sessizce bozuk kod calistirilmaz.
_LOCAL_PIPELINE_PAYLOADS = {
    'video_analysis': {
        'sha256': 'eacc5e1ad15c78328ebcffddf527e948bbfe3ee494aea31c5fc1bc2ad1d164b3',
        'encoding': 'zlib+b85',
        'data': (
            'c-p;vYm?iymf&~&3S6aXn~Eh_?ZlZoa+EEP?Mz&g*eO4fz1!QRP$JY~M3EYT?6%y~RrAaC%hvv{Jr4i`KvLbgle?)T-Aw`q2L}h|'
            'g@eQ4aQIHvDqdw$T&IaFCUG8TN0ln^SZs@GvM=^BuS6m%xvtV87g=$Yu7%phrA)-?IGCL;spHagTtYYQHBAr4d?n&M5wqFySkzU$'
            'lFPWRihYc*mit&$vgBsd!Qf3%$yHJ8#Oz$`q&z5*R!S5%x!BiPl}^?GDNW)^ij~aQ+kIT_()>z%d@Z7Nu|LEm{-xx#EaPnO>o!#f'
            'S&Ds;$V{m8vs7cD)-soRGDQgw5Vzuh$i$iW@yDN_MN!I9L5Bo}InvM3c(AG~QEjD&*HsNI<Ls&^(`vg%6dP=mm`uDe04|JCCuuP`'
            '$apt6q-#W($3=Kqu?BEFJ(XS|77o=B8JyFmD4|J=(6jVfimfcAP{m+XR9lg2swn}WGJ+~U0iHs|dnwj1eUaus=NM+X5&JmJEBFt='
            'CiNjp*FewlAk7nbknq2}%8rwyOg9y_SQm#>Dp;BiKusiN4SbR#g$492;slnb6o-11rD`h`OzT)}q2n_oOOa<sQA%tS>xmMl3^pl1'
            'K?>Xe4WSpmJCyP|l{aFYr3d6boJc9Fy3B#m(tNU)`=UGwSpQh4s;prpN;w_8fJx(mMV1aD0lW!a)uG5$ikuDfqp@&GK;9*jS51Fc'
            'z|d78l2q-cgW+&E*p$UyESH-afiIUL-5-jw0%QR?@vIsQ^s`c$-&DOil*L*qWu8>&UNWphTy3*-WdQvQfAS0DV{B&Ln7_x=`E!w5'
            '@Y`4+2=gZ*ZIQ_9{oyELCGx{y@czT|4<Fwze|i4n$5-#gLfj4n{4<oFU>~G7<2W0R#V~>O%ZdX&SRS%?Ew@FM$P!=i5-dyk?{y0M'
            'dpKqY@d4&{jem<F@KU|mr0W#B<6_{$O$<v)uc2X56XP#&>5l;74lw)p^UrVJeR%b9`Tp(4cQ0Noe|i1#)!XH(|MB6~oA<BZzHzXf'
            '?hjuhpy|H2=AS$Mc_UYQ`X#<j`Nh}Q41JKu4X$^QUCX5^>T)evCPnTa%Oov90=Q3baTdejNKC$?m6<aNC=&@7@nD)CVDnQ4aRYlE'
            '#F$_LAv?taxjDzJ2E2qj19-m9Vap(>=kVwgZZg;ozx~!4`?uc!Vy?u^R_4G(2WTmisdx*+mN%*5@pE}2RYl<PBA>|pp*m7ySbx|V'
            'j98k7@L{$99(Jvg8Au|8`@x4+;%0^hD5}amhy+!EWRLcm3lS$Xw-HOMwccwWh+(^eAi-7)%WPDf76?Sxak#}<BTNlh>1VoWF9H)D'
            '|FPs+(27K@rX$-F0Ku-bJ*J>~5Z`FjKqJ&*q~!PT2K3AKMV%+F%Cab<&G6siV;=7`1y^-e!&KuWor_xo^KR%3XNOWC45AVISfXMP'
            '-!8<gYtUcftd`xQ6{=?#rlcK;`AAa%*0hr@@O(6INYXS%qQDQACJ&^?E{j!B);?Vtz=$$|v`Se4Y8Q4C;scbs^b4@3ixco0A?4B*'
            '+cisQ-yqx3E0Q{&rde?#p(jGTSEzIV@z?D;F{3t@F7cWPH8g9s+d`1+`MpSUSOa_%-HXR)bT9Ee=GhEgLJMF6P|RZ2&8<`M5;nBl'
            '!A6A*TYrx8D$U{&7t0`hH?-jOqG=}E$Dg8Rx??c9j*BebuacO^H5XApHIfa>9GcP#6b)uYV3LC`#CsHkpr}y+Om2!Y0b8XmS+anZ'
            'LU{%z%~gtqPVSg{#?o2KESnBq{q*(&5N`SF+jlP=)u6V;dWYnvU!XbuzLHfLADiD;$L-S0{$$0JNWfVCVins>OlohFUTrG^EZ0C9'
            '{0$5YZcf$?{t~YW5RzmBnC=-UMSQgd@)<P<b_3}_qzVle=U4pODk};`rHro-2#m5VLD3s<={C-JbXCwii>%)B&!i?0D!z^t{&JOm'
            '&PWRa?6suEvC3itsghtZ><E!vOe4wPLB(%LSywSm)+$c<3COmtmS}f@5uYr{UX3=J{Xt$WS7|;+b88H&d*g`VF*D*4w4wf+x;1}$'
            '93B1x996;|l<T?JfEihh!4@sqCukm$T7Cocv1<864gSD|?L2PaJaYSw1nOuoc72pVW`M01YfW12;|yd#B7T@Y`R41#&!4{_%Tp^p'
            'O%$|T&%fFI@EL61yb!-W|3#4TR>6u>a{`894w~m`U8dwqJZ6*M>Ob)gx4jbY>Kr!^ZF#aKpj(}z1)i{&uV9f%G$fe%LNnCV=uV&J'
            'Mi=-`S=jiiVOsi6M;$Py;6|bAw%?(I=0DWpLs^4KAV0xIUKTt0vul_?(3y1yvfO%~c$%YOJmA#{?d-`Ga9pmy3g?|P0Vgy=E0#DL'
            'sX1);Z8+I%viOR&$)hWoA1eBFg4+ccIr5oe<aSg_x3PRR!k(3W>hYM{Zyz6Uh}rylKBKYhgLpps?;dW~8`{TUNy!B*ew5xkelq&<'
            '$z1=nrN6vB>p5A#-~)UDR;2;JSyC)2(Bj;HAI*=`Ps`1oem~aNUq6{00zf1H;&@H1<Le?#mbtuwS%KxublS%|x%@24f+td3H^jtY'
            'h%ydPq)WhmV@Ic=Uj(b$ct}&2Wn|WU49fOEo)&ZlD_gTl1-sATF*FXK^OThqIkbQA3@%ox?&bnx##k}wDit6{74!ucTH7%Mk9XU}'
            '<g3|iHovq64YTy0&y|M%LTVvX<Snl8y0Vr!{{|4{g`@R5r{DFO9@hJ%I{zk<dX!+N-@hhy+W+uH0gWh78m`k_T&DM~2`MQfUMYTr'
            '>WR^1%|js`vgk-vp^c_My5l;4ubZx$$Lm8q?9)sbLaHm+2FF9w0aV6c0SwP_ndX~f4z_!d!MpF{Ov#beO&=_VN;Jr^l;2RCYjVRO'
            'DNqLlPKjt!XW0@d3yUVNC@8_?y`d0~jMrQ90zI|cGb@<R#^OwZvev&M4Ez~iUoC+RftIT?p6FR#EMZi+-Y)s6I77Qkoz2CC2d@D~'
            '4d4g1k5_4yR>!lu%RAqVBR$9yTk46KP=*e@{e;WHdy0c_3-3+TESp)U1^*a(Yq4<uVCooJEzy%03tScePVlA;r$nBRYSsj{Tn0v2'
            '9bd@>nipg6SK~b)vUm!1O`M<;0+5;WEfjnKIt?sbc&8S(Lof%S7Z~1XsT<v}SHrn@I-A{%1(I=z><?7lA5;KtU4Y?DUEp5|eigMM'
            'W>^qwjX5lsrHE?gboOZg6VE`iMjlPL3Bp1qw*HE|n&LCqWnid)6NQEgJS6wbfJ65@Xy*;MEQ=d*^)9XL6M$%B01?1%Z~BmL0G$Xa'
            'Vn~c6UR(ywJ79w^r^KhY1#rQH?mq>NHP7aQ_I?J@lh@LFVKJJ<2aI<{x2=9dyC+~FOLWm9GDo8Z5#XS&Mr}DV)H|rFAH3@8h3}yY'
            'HPd4d**ls#JF;sK=QYx3Hl4wkUk8Bh8gJBH(r$wE5(Ag5j}|?2z%Jcn7$Ubp>yYt+R*rD%0PVc(nDx*XgB|_-23;VCP4!9<&6@Xj'
            'Rx7%yA`I$ylAa-mfZj$j4M_z>R}y@0_=(`g&Rh6>IKnlCA9g##pCr$^AuhX^r^RASkDg&Y5M$!6Za%vkoSd~*>U>7BqyrNbxxS@2'
            'p=lI1Y-Z@qp<0x87gc@8<c0Pg>Br?|6ThuNyZ(H9kC0yyED>MDcnl(%VW!S7XL^7$oX=UDfCU65@foW;qpo8b86C^W@j*4Sj~EF-'
            'c&VwUCTWH|n9-4h+E3siUZDmnVC%6{(%EThPpnM$=NiM8`o&nh;0c%qtanX6aPCnC-&#7(!}jPFXqfOa1b;ULj))92%j+KamLTqk'
            'fg<bVdwIlgcf6%zvPA9xLJbC>f%cd3PDCpm(V7d5@>m%BM#fTS6Mx}QFKP^I()3z$Atjf4CN{KpX&-eonZ`-7G}uJ$B%>~f(+)m%'
            'n-SA2f<8dcW%D{3nf2NTWF{~TRD_lT8rF_Vtjc2$XXY)&yVi7I;2Me(e3YNo@=%Fa^aBi!6QGO$Fd#WLh5+@x_XCL+ksZ${1^WO+'
            'zBV=R6>S(Y7wqgWwLCC}o;KBFyi=Z?UloTXFa(}EEX4T?3?wkX5&#gR1=a+P@a^>Ji3rpd00DPhVThBVJw1Oyq9sY!m6akt(gCMq'
            '%tzpLgC&>A1WyeqFf^jzGLu13$J;Y9`M|!;57Rt}%Q8NK(OvFak7$N$hf`dYjc;rS#^ZE4y}Z-Ge~LV>j)G5SV9%I?+XP;Uyb#q*'
            'AqW~BeWK3+1P-ZVWs;U5;2M{Oig_UcA61eqNe`!ZxEr<XW>qf83uT@HOr0JZ8Pka@JdKVEfY(rY(fm5Pn6V*9L~^^MVGZ_gK^Aia'
            'gaT7LzmgI3ivZ?`hgW4EV4eq30rzg4W`UKOCZCY)2=Eosl%QVB=acixW+za8I+tBIyGmU^?@RBz0{qJ;Ua6?*FcDyS?=OaCEr*vQ'
            '@!g`e42%r4L~LAi1Jl=iWsS22xY6Ktd@`%Vup|e>HE!SuqdykkjPATiAZ<Mq#A2bhnTN}&#O#F@RzyYT&;VyOZ&SgQ6g&e&k$yG^'
            '@9}ryVff1(#l)$XKuA2WxJHFyh{b=vTH&%P+E}^)>=W`HIh0I|v*HGrC%xL*wdJN$fE*!z`43^NkyUlnZGs@gx8ht*&Y!fbJ3s;b'
            'd16?NB+AOC-u(3&1~n3oMbtnZ*@*lB<M5oNuw3C%#$_s1)DD|#<HP)YGUzUfYguN%N#Ic~t+U3~6w1BJd%Cm0ZqU(P+4dUS(_ZTe'
            'KMW}#vsQ#)Cdyp-X8^EL-%MxsA*RiCP}@8;r}u37)zf>rZ(u90;`Oc%z&|}D)zCIL$D}SAtMe7D8<>fdM3ms!eggbrk_ISk22Az+'
            'o>Oab^3TcEODml)@Sdi^=L>qd_|#0p0wXKU*I5lZKy1=dReY+??(sf8^6AAMx)L}_!NIxR#3kvhSmZJ;F|UcTl=NbPu)f)1QVt~#'
            '!4wb@0wDlKf+CEm_!S4JC_DwofbP)%8&EKRjJ{dA3ZN^tt(8pJpr&jh40%w*fer<;gaAoZO$;QllG_-gA|(@PU00OBL+wEv!tXrE'
            'QmnH=QS#TesLbI{oQuLHQd|{9f~h}3XQp|fEW@xELxN2Y=8X{rEuPi_!JJ_(nea|O6eZfHbj}d^x+AT*vk%dQWBdyb%D~}!>fxWh'
            '2h-4-{lf9)K{rjz+t&7RY(^d5;Rs~jDMDO@qMwnw;EXn6Ua81Ka6-{~XqaMvKg{0DwlxV8U4SV(`bST}jMX;*y^9+@0K_lCwfxds'
            'D6J1o&xQpqT()R>A#pDCiZ{X+9STlAxTS?|sP$#kW`l;C%;xggiAZF(owd!I(Kzm?Z3i2x^su?G{X<|*Ga#>h^X<Z?p0U(2(1f@)'
            '|262}EnEs;b|QY%gzuQphz~g06O|cA6QF4~vPaDVU9%vNozo+jW;F)J;_z8((t-WA#*FFFp^iZrXX#Zg6U(P9V1wXjz5;CI70zd9'
            '+j?AaWZ8+h_YY$kFq{nI-KBY*joU^g>@c*{>R5`2x5xetV)52&Lv9>ulnxGhIZg4wRMK1DlXWi2j~+&*5Q`oneIZ`T4bJx(a~iLK'
            '^<uLB2;avnY#23?lxPBw`M}mFX<fE!G&HtIrs6fUtyQJdg2`&AtVyEH2Gr5UwlL-h>);m005!a4<3QPt1|1{>HglegJq61h_TWrV'
            '>hQvexDF2m_|X^*9rFAqn*5U_EjfKV&L%{l$@RH=B<thkIb@;qcZ1zS7qRYMgumBE>j7^Id40V%r#pwkI?uz7-sYolwEMtm>GB`6'
            'bN}R4UNo4;4VD-^SO0}U$S$1rjfe%8P7-(2WZrZ;HOA2-5A(DS)8{Ajx;C7_(ss+dNL9c8!CT1pH@+fyi6Q+YjA_kIctZarH|p<p'
            'pze2_P9m&Lu_Y1J?f2wFzcgQx<FJI;;^A|50Pw4)csOAW3w%HA1tupy?B<2N=pmvMFGr-<3H&IN_k2cRCIp6|Os=1tA=7Tb;czn?'
            'ol7F-D8jvXqT|JM`D8Q|FN*yt&A~R%ayh{zt&W<3C*TS-k0?S1+fJ5d!+8mqwq3JPP|Iv^K6uuS!ab9pXl4m0hZ0-6xpFrW(<8y*'
            'oK4PWGe%P2algt^Ty)LGI&<!-lrl*{Z*bl@xP1p87>E3DSLiWdV9aKBjBztJEdexCr5)q;8H{^7DaC;znBQ*fYZPRP$6Dx(-BjxK'
            'Tb>x^dc+DfaYHW7M*zb=N9_)cV1)?Q%K}#D&D#&+3aO75ll)68cpFGc`*x|{QmUnjM!pwHoV3)lZbFwvMi?ir35>F3kE$;ePhk2q'
            'qD7z6J;EQeb62$3;1))#ym2TF=p^%WB7w2C3jk`I6C2{EC?}ZaUaz@tugJAAm3?O!@N7^YbwrgKyQj;M^SnrX#O>xNbSR53-vFLF'
            'KBoB2nG<dmdhPhp;#)Hz3#iF-=*Ymh;KS}q7pu;h7*x2qJ>~V1D{#K0oNj<(7>_Q87tbgEGoF0@KmR%TzyJH}^3jm<wrq~JpP7ND'
            'xowK!s|zpF;L^`dKaF>*t;zJjGj{T+`51kS$sV1#9@NMkzwLbcROTa!eXMpc8IISZ|JMXy#D6o7?P85W-sZy$jFmUwnjzfOPi=4-'
            'Hl7IQF!+2rYan|&`j6&UIMJgA=2tX<qIQ0TWqS;~<9WRQzH1?&W!K}5oCvTkYdTXVIfAPjj;{?2t%|&s;;~2Eup^l?O+Qud9a-@$'
            '#sr%+C3eFn9TALs#^E(*9Xixj1fD@1FufjcRLH2|ld<^c*=Q69PCQJq(SdNiSRV)TFdM_a?ci=1OCjVphq&Ure}R{cZg(fLmoaze'
            'UC-z{QlJNOZ5S-AVE`}*6D2E+O*eic@g<3OZ=s)%%ECS3WzGyt`X#H_ojm$d$zPpu$Q9!jf4R;{K>BeXt;SuGb<<{@D+Ay|0+Mrk'
            'Bz19f0e`bdzrls`cK#*D*PsQy^5=QySXj#UO#_w$JA5{7ZwRuLJ2u#Cdj9kT8FYhw#(~$fargSeB0hB;J37zI6ZvUuPbqTPlJ%a9'
            'p2(wx6J-f)Doa&YrZq$-3SZDTm&jaTijKvTdr8|!_={mUA}HZS3yoGUdQ43jG1|JkV7%&IwqP{epfS_!M~HEg>EXFQuv0c*X_nZ~'
            'p;C%Ij?#DUgL4M#;g~AAvlk$r=&iZ&ga5Xvdm!Kh-nJN8Rwf=3jDv@jy#U=-9b<%Nrrz{ej+nv6c+C3G4!N|TWw$TFQ|3|Y3^wA!'
            'XV*S^qz|MWs(>V#&KR&0N$}GJnqmWVd67cx9xHp-<=m5R%`oUmkf+00^Ujhgy`3<}0L`{R$04m|3X28!AlpL8d<5#QiDP<=eQSzN'
            'B3CN%sb=Ag8Xv*DV)&z$<NckDoSt&MM=isT0V4SvB0Dz`9lNg~iJx~6go(W1kcqIG*cK48_u#yp9Bk+iaIy2e*;oC&r-$MYbz4tl'
            '-?jrzZD1`AXWSReS@b+_!)yn17VSsvZe8J=Mb|^WtH1OP3DF@Z$%$5g!vyG!p7(WX`R|VG)u*y<3o>`_gVji-Zb&9AxerblG;jd<'
            '5kQz&E3|nJ&^B(pAazi*4`8?>2xR!McdaPwL0vksg4r5b+%_G~LZx?hd2#02EN7Q@U_f+$A`ZS6lxOJl!y>M}ADCwz^+MBD(!JmU'
            'YYN~EPizRc;)r$v8!;SaV|lyezGN(|^jz`EQSzRMQ|nHzn{(|z@#E(P2G-9wl}Q7R#1#-o(WRQ6%eygM84~~WWOndkWfV_r@-sO6'
            '=ngFQH5OpeDIat($_f5aDjk+G;rmu+rYFwqOhlN|Z649Zyt9!_hAbr|LoO`(`f8j0y$L+9i?n=GXW7sCb>tj~GoI$6t@EhW=An0M'
            'tD=gt4&mvbJtfaOnvJ>fSk!r1ErvUw=n#)il)_6+o8j#7Ej73kMV@}n<X-BE6R4*Xx&vhi@0jk+sQRw;;@mN0Tq>J0Wle_1u$!9R'
            '%Uk|=afTY<>~cPPlH5_DmnKz#8w_2TB#gY6j~&9v*wG!^qJGkP=g9BI4GAuK{u!4&gWJR>&=`$dH$#J4$qdU}K@JJfYUtiCo;;bu'
            'M$@-MPh26cGw=fW4sif2xG>yz8`wiF!2i-&yw1D854~~PbL(``s=d<7N-Zp#kaQMWo*@uk;!Bnj%{|p~x~FP;ej%Qt?5)!rD{sNp'
            '#Dj_rme$$Iq&XCz3^r)OA`7;hExN|{K4S+U2>VRvFcfTN78JUqPD>E*Rz58YZn4o_mp~W{)>#Zf&y#Uhe!nRu#X%G+CCh6`m6W(V'
            'kvZd{>SIV&g`(mJ!&gPUhIx~ywO*XUSwu=^8*-o;%yb<l(ixcjZhP?5xCd(EOoA_FXZfX>=<orc5LpdLw<SdwwAT5?Y8yX!`VAMP'
            'A$Gx5(3&Zdq*v@A>+)PvZmv}VsUb{Kj99r2%N5`hBWYkk_w#}8DAoB6*#NI@M(k0}b;~JM(TdJzPrjy@$CIx?Vys|(KDRP5)B1om'
            'MMUtttYiItD?jmXkx4anB>Ylew9)f+g+JB#nv<sWi2y76zM|rRNCg1`GvT6G+j<}8Rzu5mioOaSoFdy>ZU_DX*<)3)q#cdo7;BtO'
            'lX`z}T(=M(Onz1<G-9<*Q*&n)J#`BI1h;LK#H+h_g=!HE$0)$&L(GAbn;1}7pf$uvZd1l08rIcj^4Fn9#$c&6zM~GhuIe<ijwqjm'
            '0s*2y*F>1G`gBqk>oQ$YUs4{ySmS}h2wty_Y{BWLzqdNE73TI@BJ(SIqi$&4QK>g$DB9cY4fX7pamAhggUz9ZzHHdOH}e8`7VANw'
            'cpyfY7#QNWkR()t?>7NzB;CYolLn1B&E^r>VC2Lg`%uM0go~N>&JnNzpE*xAxFb!n5*0!8h-xj))e+E}f~*DsOeus#CS!_J#3BRK'
            'OR7V}S<Y0+Y)u6KIrE5X;bD4lohvDG%xzL>zLqdcV|OQ=dBzH);XY<ng3+dHFbHtNy6G%SZq(X+_N>+ea`LiWtJ0x@jw%HOwZ>TP'
            'XDkur0=u)!PK#A)jp^`QaN*Rgnq@kjz-ai>+n-*&`LKNV>X+BAetq?F`7eLNI%uN65)hxC8EFA_QMymDW+jk#2b;D`!J^c-c^Ba9'
            'W<iDrY*SVft-Ziq_3gLl!|}ifjUD7jHbaY3=)ZsU<Es}~=F80E`I~<ehvIWuVd5zmy;ZyyDn4Sp5e!;`z^A-*06S<*3I%jD2;U;c'
            '#?s(m$!KoqfSRE(%0qs*aF=(eZ;H{8u}c$NnKTbaJ~7Fl4q4Zf7q{LRcTO#`XMtC}8SdpZ9>`%sU6hO?sR2eKuSA(E)0{kKBo;L$'
            '($M^{fIz48)5QdI#xkhGytu)Nlosd(^>+c;mw1W0pLn)bTnO}`7hg-6B$y$yM+Io<5<)YZv9+WQrl*({cTpy-3XYF?ijpz*hUcAU'
            'uG3mExd`Wc;h}Ij=y3jQl}UYi@0i}c1I)V9b?oe$^fP!^N3JAj-<mvZGZcz<&&4LwtgHBAC)|#%eo;W**Jwd_P^rg&_(ts{g*u`D'
            '=Hibo5^e#=7NLAYwB^6ZjH(X!YW%@FJU|s|4GMG!a)~TkHY(FC1L-31m+~&$_QUVv&*GfSy!10TKaUL%u-2*+D~$@+2zb<pP6HMP'
            'Ec+@HI_IHUSJ63Skq_tM+&SMg#sU?I!F=A(&KUZstyPWr4NhU(@a<B^L+8RUv(`x9@y3*IgZ2WL2@*_R0rr5Sy|N_r0CuP(Dt6)Q'
            'ss85BGDsOY-JfBE@#=Z7L0b(*-4L2glg|G3dQ6y3(Ta400NO43VDzC7zHiNdj^SE})6=l6U2MOxeta^U#_!?03VV4nao!F_v?LwH'
            '8#@Ym_yPiN%X$y`T3Z?B_OK6~gbarlJ9R6j9qH#@Q)?zt12zY{8pKu80RKW=ZK_{SK?z2t0QR)5)+pHvtbdL&x&@7W8cj|2(Y>Wn'
            'hW(x|fcpb(@u&x;79%kbjqnPfVpn@h4yBQodr!M-$c|yu(~aBE)Nb!QEYRg@M!oIk2eSas!uOONA%tffloe9$`x=^R{%c3n&bc;P'
            'i>1s>ZtM4S`^WZA*D*fjG@tfin=Ei(2B$=fPAKL*jwcfb)GUgMDi{pV_XI;pae8tUXA<w0pyL;BnyQ@Ep}X)338xCPyzzWIvH<zY'
            'a650|qNT7zE*1Qng8BP7t9K%iS81l>)}}tYcUBiF$Z@C@$5!)=dz8`%Ej0Fb_}Jgi58GGcfH&Q8W@w3TQkf-e;PU+Dt}(`MwNc*i'
            'h{uk+66Ta`(<Z8}rF?cv(C*HJ+^KZe*xD_(T36$y$G$<1!3(WYJi3pt5aPL$?(E^zM#=9E>#VCe@6%&ZhF*Xj)L&W}vu*SABH{HG'
            'EtHto`i3?Fi^jAT)IAe+_7wLwfs`a1jS84!4OV|tm;1uXSG{Aqu?w)e^&l=2TGz!6<gut0ueJ(=^Un8ltT?sF$|!w$5evIf>yQ%d'
            '(YnO)!v^f=`M@!|B)lDy1bsi7wYzc;JL*3?E~YV}e3>Nj8jViua{Oini$5J!TvC90@rNvfp8az|Leie<9=n&wtuP<b7JRBOpk`by'
            '?X~PBMyUQj1*RStwrfO8mv$OGB#zfVjCYXSxjX#vY@D<&5_4a0?IC06_*j$qpp1$-7Q5l-A$rIO47Fuum-#+wRoZM}$}s@~YvM&R'
            '&tyq<6c{CdU>G%8s(QcY8}O$WGnA<z0F64WIBo}TmJg-wlyL7PIo3ImDwE1puszW%AE%C}*54nUf}Z!rqB^2YJxkRQVj8-~+r4xS'
            'oJpd1f57SQ!i5D6C>*T%SL9{P5(@0<;X@)=51p|MQ2A$edds-Sa<0fSGB)Z>e{wVAP1=@DpK9ITy{FO;mm>Br_Kr<*xb|pT-&wHv'
            'p_CSQ|F;!vVg2^CWP053rG*CoG&84fLjk=QTmLMueG>RDNN;@*7^*2$efW(9>M<4X*(wCPBF=8&BMZ5+;Dc0WoIR0py9TR?K6qpE'
            'QXa-!jX>9PuZoE=HXn!Ir)mmy_GvZ$tls)4T_LP-J8CuV2*!<D;k966QH?MghNq26<F@Y01#Tww8;c0fi}g1&M(x7L-A9FH@+c!0'
            'x6a6%vo+%H&^VqGo^&a^GmNI(T-?&&e<_1cCLxAVe~#97r*gDL_XgS@j<>m0j2QzUEzGf<YyEa%+{)2y5N_uL|C#tQ^geXmGLD79'
            'v&&_{JEoByoG3&Yz(p2*^@0=?^qEp2Ty7e+8F<<0Bs+YVZOPOPx1*gd3vbTsy^BwLc^)J0U6<>-1Nq$_uIrBH^UeR}yF|@R4@Zh%'
            '>62%ex&=)e=S9O;^r=XY2At+%jz~CHKz|L<MWIWD;6==i$4;+CyI{@f=03MOHKR;81gigVVX$sN0Q_55l(BWb;Ie@Ut_J**w!W^@'
            '7p@LD5`(|NqcEn};unoX7sFvG&er2zWV+ZHfOkC(dip9A`uGpJTDzLobU1Mb)qW{pSz~6JjcrFACF;W>2ELr4DzNcN6<G~jU2HZ='
            'qGN$M)&8PA>qvudOXA0Y+$Qj|$tqSd@u7FDM8^Qp%{E<cS=Eyg=BkKXRreyGYe45mEh=7i4N+|P`=Rj-=eUoa`F1**wp=XNpnz@|'
            '7xu7c<~i0Rlo)fwG7h_GkmTQeWa4_iUdZ3QGKuBAS{9X4wvEN&#jw?gGEnJMp!I6xf7y*efZhA@)%VXo{`g_}>#NuQ`ojn3b83Iq'
            '1=D}P<x=}m&pHpjE`0xY$x8m@4Br~)G=!<d%!#6+eP%NBd-65izSJK)S@wT<rWd&B+Vy1``|`xMXa=Xtv-VcbZxto#uRb(*u=8T7'
            '61lnm`kN}=ZIFJDrl@<P0nPo&;6pnt-{J?*XA?@nA7lRARVF7oo74GFG?$3C38#nj0L~^jEK$xSX}JCmf1bwusPt&wd;$(d0<rH^'
            'UFt0EvGYM9`~>vaskX0pC6xV&jV>Y11-f;a8Rdq2#`eCBa)S|^FFtk)j6cMa8Z;$VQ>Akyb2eY@{|Yx{aVx>)&zjP&;@!tL`m>hL'
            '-#q{EZ|`5fe<tq#@;3wH?_A%=a>8|PaLFuAK%5n(Q%0@AUSt3J5F=eTn?4BctX<DX!Ly<`V2XP1iER3AIVT~T%1(Zc<`?2O!@2bJ'
            'T&yB%$~2kWch2sfawhxevCh38=@0Chu{5{?X4J#OEhR={{X48`tJ706`gSuLj?WCEf!dbT4LJuHZN0-{ON9oAtv<e{Z}5T(CRmpe'
            'E{HEkKl-+}^T}_j;s>^3nq^q=fj*o${WI=9>8}V|ack16oJXUu%|kCgIfmuJ8{@IHesFs<mx!D){U<xz#Mi@tdxeUZ5d&eAA+C>3'
            '^|x^G_JsLRnG-I&_nVo0pUZSVmC3(%nmgW?20M@?0Ep&$<dGeCM`9a)rYzZx)>^AT`_0h*6xrmqt;y~<Da)Cy|A8_uBBnpR)%rYJ'
            'P@5<GPTQhAY@;Idx4?Ww8GIA0OHq!4XgML*`fk=2^igj5h_+J@E#*31o4ATIa0jrWWxAfw&Up;GV4};gu{z<cw2jRc2Xh^-K=4OD'
            '*`DH3QMOhad#_rSJ5}M%lr2@MVVS~nB)^~wSfn5hIH#Pk)-^V$t`ipDeElBg3%&5>M3LLdbYC>ywOb-FzK+uj(Oqb@s<0?H0L=Nm'
            'iQsllK+VW8n6G51BPACoM_~-o0<Z$4h$7c0|3G`hS#ecMw5L(vP#W}99|W~AG2w*vC&zR&rmK^i=7|SLXCMFe{M~EqQ}1X^F}FM^'
            '@}w5guE=XJ%z$X>kXC}uGr-?SL1zEp6#d&3C1%9JZaNDLmNOM665}ygt`{itJ8rp0`Xe?wsrC|WkRyE`Jw3vlj`0ByUB|m;ob{MN'
            '@0})Y8#+_*f1Tk~Z6Mzf>4!xGbIv6{BB1G6=jLYZNV32nlU9fYN5t!98*@waAMX9`(!w_;-LKjBWtp-5D(dB>|5+DxfnU|Q<a1dX'
            'l@!RuW&7?b6%%3&KNj8}e65Q}+-3GUH`<(TcX~}|@<~1Hp&4y5BS7tfI%%B53Z*$qU@4}B-RaNNjGC{TjP(~%n%)=s>lc?}?;(By'
            '<8shh48p6D*VYYV?;|BZb6b{*o-`GyMy3d`YojzK6R?fcUm5_(3QL}@!52`E&u7YtBvZ%?L1<uf(+8ISg|=FZ)geC}3fwA<1IxGg'
            'hG_1@_$2@;4?=R?8a60SO!-5qC``VAamqp6*08-mvRb)8`dqxVMgC*)WM*V?bEA?zvy86K;wgzQuzE2GfJ%7+xHVl?Vj~GH9hkK#'
            'ZESO`FeN8hdSAN7N;T(WI=d#|T4GsO*8k)S*CRDDt>5S2sx2)Z_^zuKqphIG6=U1I9(UsI#kxBu_nn))<hy;CaCW-dBz@AP6YNBs'
            'pSp1C4T98lD_EYW0{Mmb9(MpT=%GO0HYw8$@J_{k2UZ)fB{Q4y@M>GsI-88*CH{?DjlI|aq^(y82$*pL#w6_l;5hv;7d8kuOzZ#('
            '3Y=QzM%DmZsm3AVby+CmWAfcr=IfsI=K4=h&AxXC@77HZNa^-)%6~RJdf8U@_R5BeFF~OEb<|Zqsa>5p{MMv8U5|O(RSu^w(#pG<'
            '{_jRJ7B1&BXfL_~<R?oSExO6#T9evB={zp%_yY0}oHe{D=G3kQVQ_R$uLUb);06%Nsfs}<9b>dRz(~dmJ9I5$)UDTa;6Rb3e;d|~'
            'XzqIJ&tkJ-;u!k(mJR8x7;;ZxkftyMD2piY&YU_Cb^@b^RT|;xw@;H!obz|xC+<GvUp&ZLa!ivy#DeBPW)Z|^9WTc7AQ$)N1uUG`'
            'gYHy-#l7z4;>0mUE9SfikB*G!qzwPI5@_8`z{7ZI9Wra!(iG~ChJgE(tR`enOpGeU^8BY&Yac(6fi&EtpJX!OrW3pE5$dS#1SvS('
            '@ybfI)Ky1KR_}Pv`;@O+|G3r7JM;4^##+JN(ak8A4pL(Yjnj*D=Zd`MbfU{IqXUYr7yW(hM{R?yXd82ql5H`%ABy*l+It~-t^a`2'
            '*`VMbFJFpB3o(2qetrJ#&FeS+>Ynk68@ysnE_{S1N~W-TMOj&+VbAf6cVVI<AvP$@(0|dvshqCw<iln`GnxUJTMQASnCek=k@EK#'
            'H>Gb-#`)0=7jRf0(o<}r1|tp;;AbRNXee;h8ckg?^Gmq_hd@xlr(!KRj1dD3GQ9UgW?0sL*}|!^8JW}p@1}!$Am``bVK<$`NmM*}'
            'l>jkSpq-8wItDj_E)LAa8P+VlJM%MX(Z{wIb-eL=?uS@CgiA#Ek+@0S4La*hh&yuzIHRBkSqzk^>E*ZhHn&NYFshf*dWf!^dpQ;l'
            'PjI3;bz^&t&O7-V`rmy|lTY2Xrp#pLtvZseYau?$?F6UrvIsYycWQuenH^$Y+yb(9m+d<vxc;ekbHvXz{C~*6M5+'
        ),
    },
    'script_generation': {
        'sha256': '4a71343c730bf7563993694455a1890d2105b3984ffc2c4a0561a5c6cf728822',
        'encoding': 'zlib+b85',
        'data': (
            'c-q}PYjfK+w%_$Du<8d%U5Rnprgz3!?Tj{2Q{Osv)>gXH@p@!Pge<NpQp<-O)z|<1o(BK|BxO6<?&IFf#Fj|l05~{roC9~e-7{6@'
            'auF+WeEeSI%PiU!qidC@Ocqg^h&a8DmLgxvOod{x69?bSxzW686hzxyvgpVp6bA?MU0N39Ld|7aq#Jra-^jdB88-_$os+asi!{9f'
            'VB$upZ7!lB7wKIhHf3ByqivSnMxn}H@YhWms#q*kvRrRub`vGn;^UF%XJwMW=T5F}VIHwK-Ku1A^kR|}>nz<y%NOs`^cvPh-E)xx'
            'L|g#<#;L}tC=iwu>vAI#SbLhOPO(;-!D|7(1WaSIEyOZS@**pjxRRmx@AK1>QKs@OJcTAQ%K^ra0B%)gr!JN<jyrdm+`>~_Up4<+'
            'z$+0)w@R#4rbM2KMOv&yqOp=Qwndt#PPEy^YNL{ZHzR_XC>e@$CBkwWM@va7SY`4$f|b(T^ptqx5ZQ<x0j*OR0+KVaEf;Z=uNBNA'
            '*%fPO8whT+&7!2pUqH|N1x`E43v47NAC6$3c0s4x?RHjx6)~T$V3RX7p9`2`nibFt=1;TCJ01P(Z+V)S&qcIR+&7ek!f&Rh`97pJ'
            'zv8kx=d;P|<N5sO$??bOxfqN4j(~q%^$DgG5kBJXP;^6>cAReU!F(IbrCO&kARk|4>NZk$^GyA{jDU2zLxuqR0Xu}>MHjGHu2#`9'
            '!tO8&i4Rv2FiEeWVaU4<P-M6R5D%SBCsZr3D5E%>m$}O3OeFnW#cEloaK1@#HisgNmc<Zhr$FdPgNFjh2-eaC+JP~$(JE64NNFjz'
            'dUh@U3P6YLc&)PQC=oz78=&|bQSN~3q6}65^DSg{tzaD`^W;2N%R?ZT0y=(wuou3WU@VjCQeNw^{zsg^xX=@S4aDe=`1Fu5*zNua'
            'SDzD$s84cP2t<V8mK>nLtm!RCl%b$K2Odicjhj3mx<Fw3fN%x;ER&lTd0J*mHP2x+1unrnP)SH%VJD#Tivk+67#ajfln5*@H^5#y'
            'n04Q-cGaCE3|ZXQw&<dVk9(I7u+>icdb0qK+V9n;)f<Y|b~28x6L?lN91I>t&xWs8`4n{13}EMj!)HvNu?Ap!y}?5uW)^#gqP;_f'
            'U)_nv7|>7J(ZHev=uKVk3^%_yJ(*2UX7l68$-9q}chmEscsu=J^6_{!ufOULG@zRm3a@dqE6t@&p(w)kK^%y`T3kT02EdDb9WSZd'
            'S7HWKYX{0om{tVh7~~o#zaNgy&Sxa(fd`Sind7SVpu|+XIuud^1I$D*isEj%rha!>RA4g%Fioye3U`HEfocQ>nU$d0@-@kLt!>02'
            'PM0?@1r3n6E#A@Kpaw|~kN}77tR&YmUX5019H)2rP_P8!_XhkLG>irZ14ug&t@;CcmnjhLiN{CxdY>+O%;<B(4sef0eUls%Q7CUB'
            'k;^p*kn~rO>`ACVZ{D}arw3@3L^~BnH)0WG;+M3Xf#o31*FZ_Ro>3%r^4Gu{W|k&ExF{NdnBS)%d~pRWJUnL+3!*YZ#o~#}p|~!W'
            'aY-X0HpH^r#43+=u-gln$I>6{O>_fu2w^8-m^u;?xgjL<t2axS0X!ObP<rQ+x6?OM($FHL>ECyGeO+MSAP7+7;B1_)DhGv#q~{?G'
            '8vFso`X3)2{+j$X={Yk!uW)|&*W_NyL(96Xy0r18J^^*+eKZzS$l8kJJbti(9=Rb&lM(k3t4JBSuZ6wL!CG0RfYKz*HXxn=Sio#O'
            't6HHBR%l9G;R3IS6|UMBclBDJC9)INj@rs%O#ng3gs50TFz#>;8>C;H%PLNB24F@>!CD`>C1lX#@{XF&Hsc545(0N8VI^QKAV}iN'
            'Kui{-yg+c2iMEMvWeihC39u8AUynv;sS($q`9{E^<Q5ehsW}*l1SAfp5mx}V1A|oh&FDwN=7GLbw=rs2y(J|2m>>o?qi%N)z}sls'
            'XBix=sL!F@euj9#T`mC@ez-XN?%;rBTaoRY#s>M&4rob_d9jb;E(dzjO9iXx|D<-)EJK{nc3Y)C{*0KS&x7`H(3CJ!ubv|<RNww6'
            '4w@_4Hz|N=U%66?`)P8gl^Kr(?G7*o?#Rdi;|#?CTA#$m*2?NyM$mSU<%bUl40jn|kh^6406=0Y?t5BgYWTp$Ad1dtRykW#eHfCr'
            '1E>RC1SHaRS^^z{)eVq)Y=8C;G&ny47z&zZ&>loJ=o2#SfgW;bm4Gf_g)A4D@dczIy1TVv{veqHY7F=jSakzvhW?u<Cs1a}){64z'
            'gv=Pg39L6*fgta9@4JI)gOX6c%lPF3pzinLrELH^ts#uX=p_z|UH&lUr-z<RifdC274^F`Z_t3(Agar_ggugB<P6V53p2oN4amL0'
            '&AnjYFw*6~orw{*$i1k5b$6H)KBzeF!Zf_J9QV{r4HChz1AQjcO%4j}R_!D>cDXZSO`trE%TiPNg&H4(0z9h*B$uqe>R#OY)cSDA'
            '`zm)Ccxe#`U-9E)HaR~6%>n>`?tYXTu_WCU#W&)cg9C9RK?{Ph0ZkB3hw$w3ciBU8g2bww&R1fB-qI&xG4gI-z8@9o4bT-#YpL?r'
            'nuK#h!vz?c1ri+~2v#yZpB=rQ%%<l63K@O5R?8csKS;dya(09f9+Kl0?I@s9(8*bb(v}5q(D|xE#u>nk*((JH83DaHeg9!{c7&*S'
            '1M5FNeaDId9Q~w<GQcn(_oD^ID-=NTVhNx=wgKrY4pWG2w1vjNt(!70#L4vM=@}Sp)Rz?xsDLoVu6YW;R$wZNv@FnJH_WG19?OM1'
            '`UDG$Lc&(=4+Y#Uu2Y3tHA~CuHNd-K?^4SwyX|>@a3x-dtE^0#5g2$k1dflEYnVOYSX+vnuV_+i8{!V637C&Eyp_ok0}KKnl(<rq'
            'I^fO+ozoA~lgZKi{LR_XhuQr7>D%cs`rjA*?)A19y$Ze`At!=$@-jF$7>e)V=U^zRw|R<=`73()?E${M>`YE(Kc1a_IC|3tb{Weu'
            'RHL<wZpzVjqgNL6D+5}$047gVK@Gm81}|SRcv$4S)6;jy(?d1~(5o&YK<*iAiV_nY02(6GXZR=srU%*!#0R2cq9vRvki}90^#DL7'
            'VL-JtG61R|sId9?04O~MxfG%Y0K)a`!}}LA_<#1{t=<US$!m&0&1Dq)xl9Xrte|;zg^t1UCLfaE3BgzgdB7BYb*(m05{<qIUSTBZ'
            'ima5g>HE{4r*Cm}Fc3`y-u(@nHZb>daUlMVu!(D;g6yCOgI2(g0FmvWwFDl^%jHt3P%B-cZGH=Uh8_qAA04q<GM=D&zghup+Wc(#'
            '?&y4W_KS_%0V1v=q-B7+1rh@K%2OR104qDEuvLx%pbVn|jjBP=;r(io0|_D^CdR_IKu@b!f|E3){o$X3VZGI>&uKM^ftBk`fr#{6'
            'KAfHYeDrpDHaF75#-7rxN@N7qcN>eC=#q&&LbE+ouUj%+p;;R$DA2^tC@N{NL&21hU=FYiTa!{H+9agyYimHdB;BUK`UEE7a40=w'
            'v-r*A<n-j|&E(j^r+{Q1JCUTgjU*TpR*<GZEg6Vyz}Q!YSt{;$N0??NiE^e^O{hKm6j5}>NdBN>eDV6M#}+$?gtrP7i-2>aOJZDP'
            'WR`WOvmc>@0jae@76B;|K2{)a7m_?^SWjT~_=m~y@t-DdeyTS6^o$4Yn~{g^UJmW%8z)LHhP=&q4xHCFb0E`?49KF~f;&itAb2RU'
            '(7!INHE9e(L|~sF4Oy?wqr`jyHXPuJ02G>lg^O6NKndK3Gf=d^m3>gLatoSgw2HDE`V!ng{9rIa+sYAPBpb6UIz~Pk5i}MH($&)2'
            '40g8!ivm142_^?{bsjA!x&Z<aI3DoBUk%a;(vrf$)>;LF9dsb-(Pf$!*5(A$RigPj7JImmydg6i>i{8S-L*298rGR-yREtl49RTN'
            'P9;WH=khw1JFq`;?cviXU>J%1Lfyc`6}szrnt`9u-!s9%!=Uo^2d+1wCwKvF7}E|I_Gx^pH0=OdvYk&oL*5NH68l_&BFGf7SD)~B'
            '0K~I}t-%h^6v@!*#WNS8ve0e-%_h*HS!g^ar)8ti!s*rM&ZnK;|L@I&39=Ct<&I0(zRff>@^t{2D61OFLJt^ULThx5c)NWcL%Xur'
            's`EG|R^6Um=R?m$KoWvKc~8*c1^tf1F1-oFdv&`kG5C9-a1JzEXQr36n9)9i)FWF>sx8zz!~?JNZf+#QA-F|py!S|g|H)f0Nzq)G'
            'Tk%d@fad>&!wOA)xNJM9APGNxi^1s8+m9cPLF>(?Z>tah!6igAt#}>L`jF-k`Dh171v0*qJ97xayoRQa4yHHRE)ahKl|+DiY#^{%'
            'Rsxokv?|T6elNqBs4yNdCTP?}R{!tOZ#)AEKj)|rwC<*bgCVK?QI0Sp5{US8TuqxH#;pOgkpvf@kv8}n+glgM;LEDzyQo;BgCy~|'
            'X%$_UXe=&cJa;3NO_n08fd)%bXIF5b1&57fQK-!paf{$E;;lm%OMnKDA{qNJwhD(}!)<AAw}_<{%HklLJqfYF$8i;hbWR;=V#5fC'
            'W_KNFume^+{*^d4lww<>F^`KUyu2noDu_VzX&PyCXH_Pi#_;r+MA%|3^-1Z&3|*4jVlzN#I0#(1gj23@Ynx!YxU6DCEm%v`r=s6?'
            'aQ=kl1q1tA1CAf*Juj5h7^oqP+h@r9!VyS1mswj%2K<}sf;dgCIl?SO7_C-{;tyu5jxlGn>^pppAVQbsveKp$xM{WVFpSn>7Bp6D'
            'gh_2O0Skft-VXam9t=8$e)nxV2}KG*KDm+pP5?~q2Rg2~c^2w^E^!a3qoh>c3+g=Ab~As$mGLLWY12%nw*wDO3mU1_@W~Mj?sA0('
            'TxAjgNnsEl%p0KJi^>wH<QeH$OG9=ZH;Rs}?JH2qPI&s_r60+})&K~H^>Nj}6E+EC81|t{1%hX4R%?gx%vNrr^*;*rIlDmG3h1+x'
            'a>+~Eu7<`A4KG+yvrnA7iOOB_4xZj%pp29XC-sOW_&y`_Jz_;ECYozW6Hpw~M&dEVE%j-O@v!cDc9FfI3DypKsKMrC9>PcTNs{G`'
            '58r!M=MH;p+_HXvo%A6ow8VN37$c&X?h8zmF`+bZ#W=bmYk@UC<1pYw(RySNkt|<OJg1+c+f|^74bu`|&<g1&2~aHZGF!=|-YE*R'
            'U__`)__UbaHk+<gWmRca`$C`5v(<9`_9p@pMwvpL4|^#o9ts8mc%0z+$1&O(%Nt}$$GkVRJO<N~7c{!B$l#&n7@;_T#+cj_;E%o='
            'Z_c;yisu*LCI`+Y2TKemE8pm6zr>Yl8<m66tMc*1eK$)fsM19eL%&@g3&SIa=pHV8+Y{{yjHAM?jSqtV8d_qSGck?tyTxuxaPg>N'
            '&YRVJ7}lHw{HF=YU#}Pe3YzB@EoW0OFrXpDuq4QUKrb!O%W!0ZClEBm$JF6{*LH<<*w1*%j6{rUtDh&pZpa)ni@FYXKtuiQp^_&!'
            'M^<6mfWCApiIk%t?$gsiADYt<LJs8ka(-}dfd2>LYb!X!?@jZH6dvsYqNVff)!H-n`e6?d3(EuS0w4D+yy>`fjle)O>QE8%CL)DL'
            'F0p?7b=v{_;Nf|vLC@aIRTo7O@1Q;rYHs8~aPZ(tcpLHh7YKP@jM$NXBS~UJf{`FCLPLJiL^XbOaPWjL>PVl@mo>kwY~V3aHegc0'
            'f2cq(Twe)<AWF97b2#7W0@U<WpZ_q{_d>@1*SKGw^!yW8e?ND{bRag?aU+>TD=>ZZS3Ms(iSqMjO}u4))~v{5rMB}`GPcoE&%V@0'
            'Av(<CWV??>VsiYK$uH;TJXR-aBxw=>mh|~ymEx4?^Av4G8JjQQczU2AtGVXeeh>tME5bJ)wa5fbtW2{gALPm9GA+@NzeY<0#NP|_'
            '$I{hmC}@L0$jLm9R+RC9Dd}wPVNhknJVU^4WsGz|gp`DoX;(zQf>a<NP7*pW-c_8!oaW|8DX^!0R~-K#fEsBg2DFi(;!G$1pxvIW'
            'gHfw?xtFwvMl2_Nrm%xMq;Lu_TTm_MTN>F{4SE>&3?cNcptU|p(I>p%#?O8{I@f*z20(_!VCM8M&ddkgMSut7`P>VBvE84GJFu76'
            'm}jZOPdO(tNMmiUjAf7Et!0+xxjA<BgHfC_Ym#IwemgD7kYBcFLQV;SHr1glCIfhk>*fSF2AXBap;nP&m%+yk+*X`r&iwMRNmz7-'
            '05Nh=O%qe8Efbp$6(r5<{A<VuMkd~2q7Gw<OR{?AHwN$HmlY(--R8qk=yHX*H;6d^C4tiMqWO$V&;bS5{dt$-@LkNlY^q|@lzFRL'
            '#g(X*;1epYF)S|WJ7^w&zWT1-kb@G~vYzLsHPH~0&jx;9qf4v79?_o8K>u2O+0=`^vN1SxpI#1q?P-o)o3M;3HBk6F_IC*v8q7UH'
            ')(NnEol4gP{HQuM__4kAAgX$A)Gv>xcb4dD&I$NBmD32)6!kNv9OHp%^sBS-R_rlA_YLJQIP3ZXbnG_)xL0T`freHWUk@Ks;7^&<'
            'o=V5e6If0qY@P+8RfO>A5&)$`l_zJ>5HzM}=DVS9MLY`>)lE=65o+#8S!mxhjga@G=SDV<W1msNX-z+TJU+G=P0O{660I0NCUJD5'
            '!18DsabRln>0~BK5)<hjQbf{I*#yR_?ZJ@4N_R58;aBx^99&5~t)}f6rP}Bh8#(%J^zF;VDAK|rFVgLn5>^Z6bc1q~@;ngdq?f_4'
            '0tGuF<5H3lo>k|tXj93hN&e-`HmarI1VeQjr6t=JI?D|jaCYrysOEp3&LuVLP`v&0k1NsErxlDA<=T8SY|*lvM@fC$`n2lwSt`0J'
            '=6Y0!Jzt%-s=LbA(g1Y4_mcc2s{rd3fTIeV%F0oVAf1Q*CQYdE^t8^audSy2Jaids@gT$BQsaHT*4wvaqtPCtn|%c@ldOnVAX7%U'
            'T!AD8=R%R~Z@daE4f-ILumr%wi7}H?yYC<as*1#>3Bi-GhmW;Y*5A6So!@O^iK4u<pLyr@GE`;Ry4JJ#-tBD<?eyFFPS;<%eT~?`'
            'D%tvc{dcXsH2Li5XNb$`Nin1OMir7OdHs&Hmu|nRx`e#%`9gtyRJHzl8a)PwRM8V)xQ&N#mS1fg<&TDpH^&;l`ViHNnuVv6c+8$@'
            'DIhh)_w{#<D%OsVhr$T8p>Q~{QKhP@o}U#=<sNe`*fQnPx~3GzPHDT$3#WQ`t{Y+M{1f3N_ut=33FZjQ8>{^1FXykc(drjIgVrdr'
            '?(-n((~K<;B=vVHhIF#Q^Wb;_b8gIlmO||3fvfCT%Ix_92vk^RtD&CPr@csvt=})0!gI^gJ?ZVFd)3ICD$WULhV|;$D-cJv7!rxz'
            '1j{*X9OSu8RH!m;UYLS)%97DH34kV&nifW6gld8?30j$2Vj529Uf$prY^~&tvgNlrvgn@czL}hyoX&JLuhR4~?wJYnFlreUPK66b'
            '+W;n7RvjCExRtSx*>y=p-r@%k8f$20B9x&)xs%)0tKFf{8){F16Tg#mk%l`;Tf+Mha0)wBU{yuBNonFmDx6uaDd8YzOC1wFbn*mv'
            '-4t+O^B!=trM&ZcOwueue^8a^vb8T7-!xl&$o=DF^v$d9xlmf!HBwC|UHSq2OFHP*m7`&Goq#KgBFZpZMU>rC8Tk!)rS-AA)6Y;y'
            'u?@7RsZ3<N1G9JzzQNXM-Yt`xB)v=Ay8D{6omV!pYhlO)Yl!QY9q{u@SJl+6G-!6x+Na*G{c?hC<LZO%V9*>+UtRFTV5Zd*Lz!m&'
            'Kugx&?eLd%y=@JR*7m-k!KN{ougb@{W4l|;*=gA9tV-R9TH%AkJNnm2>pL^j1JuF-1jeSS>tshbFSpwmD;_|0z1Bf#7(W7*vmtc{'
            '9uLD&{Y0Rm*m3dk8v5rL{pM`%ZZ!g!|0M9P@vs0|b*qFyQ?ZWmOoHN9Ds%ZUKJ+F3{>t|D7p(|^70hPmLqf$ZW=wE-At^nK(c{qy'
            'ixiYjD95j{N=4s1!ocpZq{80cKw%H93x=+^j*=BBWy*~S+#nryr1L@k=&Z4>!YEYgKWa0znfjXaIB$4#UC)Ff0&~7MS+6e1aE$gh'
            'd-~QppcEUrw9sm!S@LLo>qRe(E&_)FdYOBH0+54!b2U&<7Rx~p<teA;^xggq6Ayqio_RaOM(x~CosMSXGbCDc-}i-Noo`;#2Yt83'
            'lx{Joa+a$rt!ynOW@qVSF~M>Mb+0iOoP)Qos>EFi{E><_)pSnG3zebhM+qiVBe8OgRF!Y2f!*Q3BMS<yl`;w)1%k|CRC1IAZ|Vhg'
            'hA_&#ix!U&(_E43`u2zn5OSgO9=H?f+PTr-)WZ><_D-flPD>v)U(FXgcw3Vq_9c-^?IlJy^GEs^Ud-1L3<R&?gU#r!WO5Zq;uTea'
            'rsh>stxb*3dP&zjv|0R!Hb2)*s|W!UQh;e2y%t9&=aUoo`DS`NeLp=v5`UWhbbR#7=@|xk@s5gJ7SY8fGK@B%bgwpYmfvDT3A};_'
            'CKAO_)~J3_uqpo@1b`i`aiFc=@U^*B#F*LY)0V1vxQgWyFawxWFM!^F^PMPqw<TH6Qncg8(fs^JZY&48m_n}BLTp#gVX)x|54K_9'
            '%+XML`oKYF<Lbb&uClTDTi#<G2Xmiwp5b~ud1cx2?<RX}xW}(8Gdj(%tv%eS35xvz21kN8KCN|$<|HSc>Fw&>ZYz5--mPiVD1v&9'
            '`^y>lR>C$;w4PqH-fQnzPCtA8>D>Oz8$PCcM&Cl_H~K)~GtTt%xrLeV7ar^3F@y)Ad9wG+?)y*k9w(D$9~<fgQ{KowwC_0ShGsUi'
            '{q}Kw_Xh|5gT;bG10dNmq=ny`FoyE@`Eus7LOk^|)0V?hlf`p-GUbvqyr_pVuJlz77>D4ZN)*64;_?R5ENp<NPp&}3F@(EyigkD?'
            'B2&i^OlZd53^Nis_$2Ekg%rxzf1^hr{u1d6t{#y9V^gk&?uxG0^A_q9?&4&)exe#koX^gV-pr1F5$DrCf1IAojwZ+O1+CDt>Fnd#'
            '$+^Aff{tM+cg)SL#rU$AXn)EnXK_wpIj&>9Q}+PYLaHCciasVPD!xIAY9DMJ2d(vvZk0jjlTtGV)e8B-Qhh5+?$@e2HR`&cm5k|@'
            '>sgkv@&;`_7v*xbUfgoB=SCUtKACE<&b2Ot5<a5;+Bu!K_rsbZ4C8fbs+^d>ouSM_UA%_22wvo2t+(9dEjEI7TvXDjBGz7%au9fF'
            'jEzX)YkS#Q6)CjWo^f@ijf8RyB3_t?wJJ^}=+)PAQwxLTo<JHSTcrZs&for|PsnqctW*Dz5B#*fTCEFu_PF^hw2*=pmJPY)h3{na'
            '5B!Z8w%C!eh0e&v1jxXPCPQ+Ai>cdk7p{-ny?kg?emW*xGx@sK3WnW8O?$C6#t^H$#<AQiLMb*dJ2TNX%Y5HdbcJ1Q-A~+V)sWq*'
            'ZmX$!-}dyu#ht*roxr@O((`T$TrNi!*fEw_Ri9s*u(^e_X~?&r*6(`JHp$CQr*RVHd!|m~M`Eg;2U{~c_xiF;^VG)4DmQIzyjcH1'
            'OGM3tuRlL@^3bWJs{jg~yx-Wr*_d5{)+>m=?6_xKe=z8@hTG|yL!W=#I5LOQ^%aZ0Pv`Hmvr@l&qblp5g*q`jUUwH44o?ezH(C&Z'
            'jrO7EfeDea7W6bxM)%o29@-;4&Sr1qv^;UL>+Jv&nER8GROm-zkNtq+W5GW@Hr(Uk`j{X4Uws(ufq`)Y5jIQ7@#>u`e6<*0;m=oH'
            'E%b@c$Z9J-#21s3Un(Wf_aufYaIYn)`=E9FdwbSxIU*GO8VI)M2B=(XHY#5Dg%jxN^%%Vf<XYW8<vIUD|Jtc~5Mb!sHRVMD9<&PQ'
            '3;aBGumd#<e_eUzjM`d1)<{e>4ZZ9+R|>gTHe6szIVUSp?Tk_Arkpho!1~*O4UOAmURKFum+lT$aM9$&Cr@}@k8d&T8f_R4THR^!'
            '#j&YD{dxf_gPT4>EePR99ESTmh@o@0f`8?X9rCw8O{Z=!Eb->2>QWlfzpu^zk&88cmpu%~b`W^M+-6Ca*I<OH7Syz$Y?%B;Zoz|j'
            'E!NQzgP}`#o#8do+bE_xMC(Gs2Cm0$)xKT2t{8mc3Wq(V{aB6X0;0@B!w%j$><w+&l5hAA^)e)L<70Z4*s7Bn#J-ob>t*Xcm8MI1'
            'z^Vq1E0lEVCK3$Ye}neo3yXZN?E!bOx`nC+<;o@5*4iEN{%jpi;9vY;0!px57#!$75_4kxKWJ2tR|&mUl&gMsMYZ8vK;;JZ8di-R'
            '@uH<lhAske>SQ`?x$s_PTdiVK=OFJ-(r;dw!7}o~klN+YH*0i+(2Ex~Z8aM+ti=$YOkwUkD^-8|SirAsHm!Nft+BpG)0dx4W8N14'
            '9PV*Id_SzQX`o>*JSlLwf5-fGPs$yCU-wMUz5W<)LFFXaPFvcTxfbNUEvrFKqn3uuCO1EG=R0`H_@@kQefgEmC4NqlSUvUEorm{o'
            'P~-hCu$ZgObSt3D6N@D0nN7+z4{S<!OTPG?eDHtJ7Jt+pf5eJ*Qvk;<Rw>bg>eE_yyESQJ+@AUB`9v+io^%EG(Kx72|1x7#X?&WF'
            'scH81<JphX6S^dYOa@dtcw1GayDMeWV2RGY%I|&r;qopMdrK(4HH)5~K`-Zu!*wlp`SUoT>i1xMrE|Gb1?Il3${Z%f_oveRVHg7B'
            'a#nju8U}!^sg>VF1(+Uui*Q<Kcnf9!Ov7Tgm3eOdFi8)Cf;3jB@M=Q5jg|a8dN`*EP_d;<93!%CQVQDEqg8t|?U^hr7qE<-E6&lf'
            '(@-f1QSY%|9s0NlY%OJf_Rfu2UPV!LE)&WvU^nkn4GqL`vlZ1D)sI(zQtdQ+G3D1Af&GhwQ|!jwb#%R+n=}|oHmkYZ-ht0!r$Mz_'
            'bL>&izB3@ded`g`xb>;ukvAh8HxK@FiF=xv04e=#ht@y&fPoyGru_qLE<+5OERW9Gqb!fkr(Z+S9XOm{z2+x%wk2UuH6r24K><e<'
            'R`2;q9Y}r?fn*B3mgboQ&fg!SaVv44uMSrf^ETAAuRa7XHXC@Bh2O&zD>N&xyrOE;simyi`}-)Z1-ie5N^cf5S5tD8diB(*bsI5j'
            '0R2zU+Kj3%*4l3Of1=jzJQ+b)b&GN%KpO6<@IfrIs(MVMaafA}Jr>j(oD3d7!6a&fu{9fuxl!~7oN_7TO#yx5(9KENgIB!!_J08m'
            '5~w}'
        ),
    },
    'tts': {
        'sha256': '7df5642ff7d3b8594b5b8eaf3a9970a364b6bd4206eb43cacd645254fe56d94b',
        'encoding': 'zlib+b85',
        'data': (
            'c-qxlTT|mmvVQlks8K&)ov;Nxy|XbvhtCP7+p|k711!+JXM!MvY?W;-vgFYP+QZ?$pD(kjq|(I=eR(*C2e758tjw&;FE3R%j&mW)'
            'Jeb8&6!KFsF4A$nk#fEgak`A=B3}iW48?3K4&L}hk^A>)G?!kq*(Ng)B;nxTz~822QO=|vltsD@iYQI|b&wY_Q^UN$;504dEKTn~'
            'MBGWa$wgG;B7I22x{Qlxyvfr0D3qC)$z;A-2iaYeEX9Z8x1v}{Vcw4CK^zYrvS723Sx&3->s-ciUdT|fmfmCCf9Gi;=4n!7!MqT&'
            'GKz_K9NkN?k{PC?VwM)GL8AG}m7F3?q=?p=SgvJKuq4F11W8eD;xq`sbS5_CEROP(Bo?;C3aolU4ccT;QsnzkUA|8|MtMPl#L@XN'
            '<lcG%$8iRWEL{uVUqES@^nF?|%?b<yH@s#(80crKAYaAN%zP!W%;$Vn7ExS%D`(KsT;{oXQbcR1mI{MH(l@h^`99(?e?n$8ZW9!='
            'm>G7CuY<wm)zQ_5OaJ51$%pBsn25)Lz>gz8LA;2y6gwl~gb+VYH}t^Y#KByyQc^j+%H(|{AN)-IQARK&XQU`#6S1xIT{uyamy1O-'
            'k7zpB3+Uk@h+_2`1H*F8c9qE>+=9f@U~qMH>Hl{2etP0xO#gO#d3Eu-#a;}4k~o;<&Y^g8)@dkXKML_hzw+7cVr*R*#Qyz{v<#b%'
            '9`)nv^})f_eEpZJ`2PJLKSy`}I{M9d(yXT&nFLXXGQ4B_db|oNTJ#VWn{Jg5L>*f5U*Gw2SX=IwIn6up4&EFX(whh)I+Xe8I8I^z'
            '!~}#NgmNK*GK^9`EHmYExttFd3n&Y&IE#`IZ>THA|0xzQ-9r42IK^Hc@<P!<z}kd0Ts^0sOu{e|9?Z|ku%O!7#JsFbhSl)v#+uIf'
            'o}`somh$e8^HuubV_&mK=De+G28@^(u6azlpg&~Y_?s8mkU75@Rcp+H4Lb{fW>XfEtE?pJ1zxF-JkB+O5tp74Vh$yWA12~pw{%6e'
            '?Yzv=G6{#QwIOlkvFi_+a9)Jq!oAjlI5*c$CCiOF5^oeA^3z;y3NhtBc-PN>+DC#lzcyKd+2K4UXMi^5S+ps9lJ$^OGZI+cC;g0s'
            'F%kX3*e`$OeGj8~p%w=2+$xf{B4S9Y*Yng1Z^}<j@MJts*H|K=sF2S74RjSl;9eo~f~i3Rc;$+Rs91?xkC^j|)K`JS1MUI~i=p9X'
            'Qgb@dyXM{wYE=-_gP{lkgX2)TK)5NSn6Jv@F82sBX|RzSrby5<G~eCsHWmjMMBwJpuc2GLV2{B&E%9@Vn-7)%=7Sc*+Maq9Rttf|'
            'u-+@@Vdl6Z$i<>&lYU@DN>C9cioNE09^*Y^(9V#@)yu@d@npVjiYmsCzV@3mFYtaCgBP1su`ga992~gfKL9^(4%)W)=$MdOBPN3>'
            '<X{Ehajgg&I4g~Hgm4sEkv1H*(0T(wf*2({^kHHV+={T1p5c`ili&>x^|z;9CnNph*<n{6g-3^C;nY?RzguQe9*J3y2lU}j;4Y>='
            'IZw7@DzyJ5r_`UjA>dF!^Rv#Xc<|2f=jgb{JIUgB>aE9W_KZa%Uczvgr-^0;f1`v@)V;MG5TEl6at~<}1KToKlZz}6j1?^3epipA'
            'RcnXb&3tWcN(laKB<b}+qTCjsVRql!le*jcdB4(W#LIl@|E@zRqZCNTUg%Jq&9FbIPsker*b0fDa}*QHls=SS?e@8@_xzek&{&jl'
            'ycL`^F?aV-!1cm8tt#G~93yK_O=><uJ|q%ZJfztjvx&r?eZb~Xxg+$rUQ_Xr1W2lfu(LEqBFps8kFNgiQjHLB_9cpxmQ-mJFDpOE'
            'd09}Ldn<0+Qfm5&WiL&VPf8KWFa(*f$Fz>bI?8kMTg4cysK%%bVlHDD*gNHU-V^)<by*N1#;O1uL_}po1rZ5NWmR5g3ncJjualMU'
            'VLYTkBCrDGd4}SISQw9)R!-WeYOWyfOKbL9h>C(sP@a$6?uDqL6W502?6yuv?KM_KB^G5-9IS&dI&4fA+hBtiNl5S40Tk;z>Fh*p'
            'ERCc5mkTcMVHM3+4a=<<YfQjSk|}eQ%5pTOc=8j~hC|n8zOkG={BsL@);D*EuLYl#ahdQVF?_Vwdvdi3w6@!JO6Eq-S+4fWe%W5i'
            'T8^rkX+=%;+N)G7zAVf<Q^Rl_V{4bHlCgqiRgrY0<PoYK;n<?bRn~gbMC?wW8oj8fsSBIW6gOr&%-TI0@XRK(V4FbAtT1_OE&1dE'
            ')lDe`QrCw+9UR;|>Fs!7IPDFHs`$0y;ST!(gsqz)l7q%~h>)nozkh(-q2%7!xTLW7x%3i1wJd|BN-3dXryz~dG#{v4q}g^t%IL<#'
            '%cJ+xcT=&WQ3DVcOA3-4fEI|K#3;NF2nMt-#EKer0TcUMnl6!gKFFDnNw`U)L?xgUL91wrfD%;kR71S#y+yg!<c7O;`-mlBERpjh'
            '+(KI*(8Qj&wM7~#hauUKv_e@y8H1f7RUw*&dc^(hTdq+l^Z*|dPK_Z=a3_VPL;*Q~;e^z>!)77QY9<G&&w#-HJ=0rwtSv|vw7I^c'
            '?i&p=tvB8YSXN`Icm)Oj5y04ge*H!aVX<>+L6#ep<09S%dCoaWy*$yJh>j|oI7Dil7`t+bWl{sBC_{UuD`br7z<y+}<K@Bqw<OrT'
            'h(L?kfWQyN+5nvYM;O36YT;|J+|Olp4|9NX1%QPV3XkF_b8T371-JZeKcIGQ+U%-jPP|k9fs$g?u;;9UoZcI}!uWrd2+zfKyofSX'
            'KU(dF!YZNryQ!YgMS*FFlGDGULiX(Z>iF#R=mgm@d~g9fn+5YbPv_(n`0R5EJR&$515U;aNYx<$0TRezG<GeHInf_4fFVm{;dQeZ'
            'wx)oM&l)($(wx0kLwb*oF~A)+q{-|@qD<llph3IlRk*0|X%oOf(}g*hFl(XSLr%tvX<?ycjD_LE+wl^ALK}Nq=t_8XI{i4k5Hl%E'
            'XB<R^xisGaiaf$UIzPWS`*`%r$<(J0$M2^Xm#+7nZDjz`??1ew&GM<yUadqmaH}I!*Rd*qyMy=BUynYVTs3A=tg_|}FzBQI>(R-{'
            'FGuhG;a|SHI6lXO>3?6*3My#*Qlh9>B@$^4l5TMqt#g?`%QJaLCp2NSL^zGeno$77<u=Vcag+q&4r;%HHIERVklo*5N(e_UlQI#G'
            'K7Gic{4k(xxC@vFf?+~#7?w*66zMX<T1@g>-cd4&;R(W8l2KeEf1L++|0Ui@m`p^^4e1<ow#4%i8w>*CUMkj!MgV{R6RU-CiCV%L'
            'xH|ar5fz`W&8ZThBO2~Ev%xn@V>(fI@Qv#rnNL%AN^p-DOUCP)SvltblwMzB`c0JxoMm`P@ivs>u-wGaoQ<>I@Y>OoQ4>T^B|~mN'
            'P6Mpe#qqx=>NokB{2EY{C{}QosunGf(gQ%wPNx-Zogq;58B1MmF=_S%?zWg|v^K*9N9W2NsYaMd<zNCPMP60;9}(1vh2@7_jHL-a'
            'Jgi_ENI@WdRNy--7`h8Rqa^-Y>68wZHmrvYw`se&=G2_t?eZHHI?Pce)pth8pxWmzrv9p?P=^1R1+4IcqF#!RL0rly1)ye`vv4TR'
            'Bx|{a29T5??(xp%!7>iEP)Cj`?9r_HL-r(w>~D!o@SMzPxQhiBMVVE#sEmqDXd<p}Is)06thwh29v$%};n0k&O6->Lw`L)op;^{e'
            '?SwJ2U-cf7ZETE|nVuFwhjS%KhX<7v@yqMO*EbD+?w(Bhh#nDTPs7U34>?b-+`F6n>nC{;6lJcSb<&(2KV^=^5q2tWeMQWmhIMo|'
            '>Cc~b{QRLZ1CUEm!M^r?Qq%j(la~?=3KVvm0y2`BX^ntLe?jvLsW^Yx4FJ|Kr`bgDAxgOC4U5oOzupRJuppB`dS<{&U47{3WQqOu'
            '>44a;N(e6FTn&;}1ci$-QF+3<qtnx~E1fxjbKca+o{9N;8o~<HEIAcaCip8r%9bT%4+wL1?qQ&0FQ?=wUs=gc5I+Rlo}7ot4U=@1'
            'hFiL2z#5k%E(#>9zm<ji7WlkWpwC*V$noU89hD@fQw-uNU$PP!LUe|3<e}?SrD&>zv*cf~>*orlsik_$xu9QVrd2wm=hQCc-YQe$'
            '7M;+tDhs*%`_cHvpZ;nRH+2q3Z5#f&D%ao_Bw*E871C_mu^>;&43&oVMY@*(!_?O^ynGda_|Pug3DzCQ^(6O9hE7>5#y>l*>#gJ`'
            'BrXVaLl=1QR|Shm42+J<9fsy5i9^A0byPp{XIoSBSPz4uQwceA$tCX@OWr}7r?_v-dm0$Pt$o8ZG)XX3*bq?GI7TlXpL&{p4fl7H'
            '#ZxWUbL*pTDBby>n*oO;i9Tj9OU~0W4#~u{KZU4^bz@w2BJz=~T(v<XZqTD}Fn}Y}T@$y>fK`fW<(WcMe{Gkat#Y#qsZQ69s4VR~'
            '9JNPQYHNIN5#I&(n6~Rxi-LWbq!wy-nOwb5&Qnim*3UG!xlkv7*i%&u;ksLWY)xJQsVJk<_HHgTc}e;(G<TZ3vY5MGl&9*R#?ald'
            'x4tm~M}?hXWfU-y>znp8h6>+ckfVb4ur@Jkym6wO!vP&~8oTwDvKZFZ<%-Ef_d1N*=}EOSsr{yiy6dV{Yp8Y`n!zAmm(^A6`Bp=8'
            'UHS6uw$^Pc3V@pwiSkx0LHSM}U+sHdJNX&5JVNHkF2g8WOO=ntVf0oUpI#oF;`h7h$@I7B<+1o>`j3<2-_I_nCe9<MJ%PtYDDzBF'
            'mL|bgI8EBYHd7^H5Ju}T7>S+1Lr^S$GPnoR$znTDR>N4^5j&{W_->7}z@;*b>-wp6eo&!~HBLRU2U%FuGPLTAWt>f*vItwxZR#(@'
            'tz9#o!>@0{t~PExcQbB>ytaON573)>?G70c()Chl;$uCaiIc4b?3W$VeF)ll)>!E?u==f#`W0~cQYba&BUzd7rGYBhH>FIYe4_3y'
            'u%xH7v2MKi>bgl!$)nSG`@Pt8OF>sAW<dL#s)u^#m}W#BQopayV~98AtID%db+ZSA89GN)!^mOuf_=lf1=Vld0v?-0n^LVqAwPj#'
            '<iy;&F-;gcw&%PVTeVde9fnm0mjaEdR!|3oDVH$?Woiks|7Aco5LDB1R)(;0QzpHmR*xyzOnezrsn9NSP=;j4+p`D>-R0HA@w=;&'
            '-^JzhpC6{DSI0*u_=2*b(%WrsYYnABb$>ig(~Y>8UVXSYy)=@kj(1+p`K-7irrX*P7ZsU=A4nZT`St`VoCS5!tp+)hkRj4gS<Jmk'
            'EUR<Y*-qU;Yjw@qS}<KMbvY?10`iG^0ks`f4OAb^?JL`Us~Tm91EvAuKz3B5d@sH_Hn;CdaD$}R)OkZRW1&~yPO0c93j6AcN<`;#'
            '<#I)hmK-zMw}M<=Nnb<B>2#&uO_ivd#2po^N?_Dw6nCUczXkM~gF0&2o?9&T;T-gm@|BiKt!Be&RhoulrDbdhj7+r<vHZ4odmg|y'
            'u&in?rCD2uqto}hT$OJCc;dLI&m9^D7T6aX%+i&$sw+YoS8JcL$3EX7c&p@A19c)>4c=Azd*Vu;@!GwMx^JOa>-Xs}jOY<p;F=5d'
            'uM^a?o4ywY?xRJcm8!$Q^`6Qo_HO7(MBPu|dtEhffw6TEO5KSN!ywW1w-Uc95yiTM>k)9FxYUgxwkIaVIfL755a_^T9~R@QYrpf>'
            'clbTrz*~|c*B3tcu7;ngE!XL0NzT*ActDnzX$L>$Da~gpb+B3yj$5@)yM(PwzI`Ul8A3B(0~@=JZASVk!sqLgR`Fo<OD3HiZOCwA'
            'oh2*Qx-(4Dr1Kobk=tIalB8d6Do^eCzZFvoAPuofBvsiBX>+>D)kxM{cVe}526o$1b;+x{UY&KUZZ|_XO?Yhnt4k-v8Cn0W(#UP<'
            '+dB3s$2DJRr0zL=80m$$A+DD6BJfHOO|2pQQU!R*R&E+Mm0lJ*npYtzX3Pmdb*9+p^n=yjux@8lqh0aUIxN$BYA)heLs<>x7@*0a'
            'x`Sm6n=}tLyVv*UizWE&{=3q(J?*A5O?O=<z5gE4+fvti)TBFVI<|qSeVb`-y?L{vO%LX;>;FXpHGWW4lsDa3ZsQ43$v%Y{qx=G8'
            '=_$`C%DS%qc_kToI~BChL&lFJ^PW+{-p$k3>8tv-<6psEpvzo-iN=)cJg2Za@#+=GcH>>8&Tbpy@!msl?>r|Q%*3>n^%<OEvn#N{'
            'O9K&&D?<%avl_OFSWn+%_m7j_VWNuC)m0vXxhJ<X`;{&rM6{-h34GdoNU6u7%uIgS&UsV?f-!u}q-U47=k)wZVy`Gl-qyx6doMi!'
            't6wd4DcxwJm{2>Z(*?6R6Z7G@egCT6gw3mPU3r@O+>WZ<G^Z$LJB?YW?>Bm9n~v7u@Cy$Ktc9C14HMlRbgYB<py12trgO%9PCVh8'
            '>aBjKI^Q2Qwt4c{`1(Wk<h~W<9Q5NzT}o9K;OU-qlbYSi)`9YsI7>J^4**+~GH?+@8Fo(vq$?>?l~yO`-NwYA(I9EUQ~O42177&%'
            '=WZ}={PY-WqaV|_>p0jAOBRoZFSq39uQI+6ZN45BEXKbb7rrAVxP$Ma!NHgMzNHTv00v*D<<9MmFH-AwP^UZiU#3|4ss'
        ),
    },
    'render_utils': {
        'sha256': '92f838cddfc00ca9563f6a9a9e198d7c6fc5c6514382d719e4c719101c3b0289',
        'encoding': 'zlib+b85',
        'data': (
            'c-rlK+m_olmgswZ1yWfvloCpH@g;7#l`O}0IzH{#S+>)eUgbJ6B~c}|C{n{qmE{_(HGgnk=5x;1%nzJjIQs&C00^qej(d78?e5qV'
            '32fXpHufEdhlj7?x{Ri2EPnW5y@@YG5ohzb7;KVFoF-W;mT|g?i&B)!sEFrcx)sOwCtPTv3JsIZHk*px)iRka50VVul_JVS{LgiA'
            '8KrSnh2n9ZM$<t86*pKrtHdndY(>5h)iNnXnqMTdK2(~E%eY7u+ew<wqI9^~9xz;ytT%a4m0WzC&ui%PJSpN?m2Sl%FW^HgsFECn'
            'DA)Na7FAqU{o%pEiyUfZ7cinC7WF30qj@Qy+NPeSNx7sRZ>h*oJgY>W0oWqOLJV|~q*P`uu8IVJ9+<CdYA_P;CWYxM>&+%jV7~IY'
            '+SHZUMAdS5aCmrluqg7im`oORRTuGOB1Z*av%HF`B+tr&1Nm+lmCH1ls-J%;^GrP#v3e?(b(N(0S3TVn`7AC=^`=_l7+_G`@;s{I'
            'Dp|*>UG+N@`1glAi@6+j7}{3Feuke12e02edh^Tc$)6rQ|K;gxF&5Vc0{$Gv?*T)gc2-5{VJHsg@nxLm8+<X@q|q#1<|)h$ek$V2'
            'B)$Ut0mJ}wIt&>?w1Hk<;%{-7WM#crB(nsoL-!(lxrktv=`$3Z*E5``2@Y%vAa>BRU%r3y?9KD1lV5*)_WI|iuO>geeDd@;^)aZD'
            'Y#W3kDA%wy_^^)WNxjCWG%7CQ!R4I+EXFhs0MO${KfihQ^2OxMk57MkI{Eq4%O9RSH^v&w<3&`bRWOn-=q2<Cv0ws-TSCX+<4M?j'
            'UF0y3ELz9#K6n!UCHhk>UPoCOxMj-Z1D5&b)c6p=P%q=jEKl=d5|t&w_{SfQ;m;57#{@BrN>8~TesIgJqxX}myqIH=yUzO=Y;t^m'
            '=h*sCTqN1#65gMlnjZki<6;KQ!j~VSw2X~!X<Su6A<9iO1KfiGvf_^a2`*qhCi!9l=m|Ji-ZY=5UGw=<RLtTutxNIxap!~{|IH`#'
            '^ufnY=$oS6Kc7PrWK{q{&5C408w2KVy+Iu9=@L}xNffX1OuT{iqhcFpp?I~7;%r+)=DR;7Wn64a^VLt=GAfb?c8m@1$Gk3Mv*52l'
            'YvUp>L-C}DE`XR-u)3l6BlP)gP>Sb3J>jth=UIW>B2<9(qjU?6VNYV%pj$uyUApfsG2%%J7hXReyoxVss64EyMH?-a(doVWSmXH4'
            '@!h-8^!~lm(+AVzd-v|neiz-Foy7O=%onG>o8OJ^J&3+Jd2sKW`1IuT!SP8HExx(`V7{229N(YL7soBob&(?6WmRp;(a}*6T@5di'
            'YFSTVo{AZi0HQXW<?Ex1JikceBdk;&t)nD6$`>g-$PtvYqg#w7O=fWhNNP^P%O9Q(tM`?!*k|mVr<Exjfk2~yDyI=`F+4)BJ=i=k'
            'z){d7kiqSSsGzDbd<1m6kCO0wDgGF5;J9V><UJhHvjjPj^#%|e4%nk2Pt$2MTZx?Z(oj6At9%V@&cq*Hz7Yi+=*UE5`IUqRlUEY!'
            'vCe_7fCIQnsx%(p;Y56uNuh+KY$BS%aX0jm!U=5f4{`d*C(L86-a!lRKlt0z!f!q|Ej$^X?xPDlBZEI734;~6fhH{O&L7;F-?=+K'
            'S=^s}bNA%rcX#5u(QI*d3PfS?yL)#|;(O8kZ>IO7i0H=r_&EAzKK<r2x*H!a?%e&{G=b6KhzIcx_rAZnSXTdV@5$em7RXM0G8wQ2'
            ')wwbsxB0z&+7WyBheyE7z5D^Vx>qk>Jbf*>yFWht^RF*oJwfiS*K9?WNSG^ugK9}f9K8c^gAO%(U8yq<e_Y8^PqrB(c$_f|{5)@t'
            'Li$OkFg_LX^d%Sx%z3ehkwrb|9~>M2rwMF+nWwlzBR<%n59_#x+%vW~=iehyRt1b9h=5W7C|bSu)dc_2--5mvd<){ztQs-PjglV$'
            '2ccyiL(}UdOG+ejvN27oNFLCnlA1)ZNg)XeH3(7n3YvpP-Ckl9?qPojpOa0mKTPwhxajq%da^Jee4$1&(#T+a2U$rZnea2~Ba62v'
            'B4B5br>NtnMUfXaON4(Ghu<e4v94oSO+9!yv)8)KFN)nmap<YMjxT2@;n8e?><5)f5Sik6&)>cN&W6(0zvODqLlisVEP{#D%>#z0'
            'CE)pFo+t*cyBTkm7a-#Rp`TAA63DeMQGymCuqPk?z!^7N(&}YU%ATg)D&8VlCox++!vfr}uVBrL!tLG}+#LRSf$ZNwdtLyH?3}Z-'
            '?fGiS6`@!JvJazc31!#tllJ^|0aZdDEJBv^^iL$v3Be5H4gfJSGmmX&4*<z(7St@oO&Wz{ADl9^5_?Nx0r(+oaS8AVH+l#Dohj(&'
            'a{70Y*vK=GPLRo>PSYNeS)v<&Lp)3K<N|sRb0T#UL#`vy=n75{;G5!%Y9fJ~0^I>#YZ)OSZzdYiXtu1g72-t##HW`=>*+ii$&y3l'
            'o_hE1-Me$YABriUtG!6vjbXh(^+k^$_01)a-<R=we(EWL3dablT`YP!!Q-RHk1#i|&^&)YNh3g)$vi3A#Pg?Ug_Kj%NB|+C%P2{S'
            '=OfKfx;3;}Fv?PFk}QKA_WH+11DLeo0Rj7-4>`eS8O$N@oX;{*0&4?oBta+D8LGTuI)&o*z_`JYgi4Yq6@Q=;nqd8a2O(*sL{f@*'
            'ewCq?3)N8B3|?hbP5_=1aEK0&%Y{!;S4Lw6ZkH+FT+SwaLgfo8!Rk3`#9(D()U`#}M^x>>%Mw<8M#?p40nU?Tv4{&)@!`A-)tu44'
            'YR1OGobfY_yXRC&aWPO#ei&ZFRnOt=^vQwefqFYK=qc|<t|fT}A~MXGmAJIct1_>FQ%AO<+`<lB6JDTB2w`bdfF7YTtk>X%{e8Gz'
            '!QZd}aQx+%SXWpvI9ez9ihgp1IiO{dk;V@|HsFu+K$yV4-^4Q{AiU9<`8c!q)^qnPHNX#(lBTT3+ST?pH_GZ`L5#<4+c!)so--yS'
            'XOig{(sO1_H|>xFj|Kd$W*o-(7ifM`<awngeZsw-^oNp&0TaP9E&rjKifzP>@q{&ij+*)NbE@ZR22Ch8PN7-&c5}#j+ac<}Pp#^q'
            'xpvJV=<vdw<Klg?pK}h80WgnOr+715o9ue~EbD|84qKgRi6&tqn)I7~FhbyYt3YeSAlm5Tc%*t7q!~Pn<l%atKf^AD@4%<KTPzq}'
            'tKRH}icU6cK~-$+k<-a$w=#sEeAZbT5(TSs)~4rmHNJa~4nRQH4Tt~^uVHm@NLGc(bX&z`PnG9`(3m8-U~LT4a2oXcVh{S+?D4BQ'
            '!wxfp93&ac63z}EYx1~)<K&{K;hc`jL`)-82$so*!0uWznj$9Ef6t4|-Z{SYJexACG1aHXH%v8gvY&{hw_lz3{R}lsPw5||?$9R%'
            '8@{!>wJG3W?k$2}{`~0Gvk}jEnv_viiWTfMoNl!uXt@CS!T`5(7NFTgia(er^w!(SV1GS*ZS@h*tyxw&$x4?n*h^od0dUppAKZXw'
            '9%1TDypCuFYTyJvBt;s*QbhAa^sen~Mk{o_>%(l|EDSxO+2IdSu}TvH7l|@10bRvKJOj>X8*P#l;SL>HObNTQgM-eU<~=U=?KRKP'
            '7lAQSlrq#NTfv(GIDNE`VrSKo8;z<Z|GcL#KkOXE!(xF(;mH)h6-lhuowNtodMSEM5s<YOuu9VyfAojuFA(6iVj2^xr5UyyMk)UN'
            'U;mA#dL3QFNiJJt>ub48Hl(PPFR2^bkZUvW5IQapaiIm|Kw}~y2P_Q>X5>`$Br7Xm%dLI^GR*SznwrDT4aH-!;Fe-tgP>EFv0_G`'
            '0<J{_i!BY(3n*C%QWgW-3k(SI?BbnlZIHa{Tih}nV)Ysgv4rY&AJ_)3bt&>#v*+YMR#elCvOOQj)E!`s=@Up?AYIv2k;199jNiv{'
            'Ynpv%!=i6Cs&y>2lC4S*tC+(8*15nPgOXiah=fr-rLwSUWIg9qB6bZW!+1T7=V&Lw6-z+qX)u6cUKG)0iSX2H_JD>DweGzf{~lp|'
            '3n=(F&w%Pxyh>#nk(nPB3!1r#in&m)kgMbVmjo~pzXHw8uRw?*S&3E70yW^7=sh0ccpkrfJL>BtGqf$t1gZltc>5NQlA}gif%BzD'
            '>JRa)IH9KT*V)NAGK|5<P<lzA^!fN)d@TUjD1d+I&FSdeTpfHTmttBcDbf2X0YP+P%vuknNTLHF?38)69HCekie;Q!EGzkAn%2dn'
            'OfJ?Dy?`Fz+3GBzA^`rySrpL)Isk|sY{Qg25RNLKgek7;v_RPtmW4HK1Tu&2;_jR+BcL9*Wzfw5;H+VN&W=Zy=PbZRLXAuO`u<1d'
            'U;zShj8gO&RhXZ%(8+@h;Iu@8!K}zP0ja-mfCW%$#EOdo=}=%xsi<;7aT^!Jtlg4<>_cpENF4=4O5^c0&F^k>&2zgO0n*LlNnRuu'
            'K*v&GYB$8bBcUEA*$mY=ad_we#AaKd)F=-fZJF6^&lqpn`%RhvH+43>n3Tor{A|JhK6HDg3jEba8(!d8#)OdL>jpV@&;w7m7QSu`'
            'PWN_b9&?AFvUi|33+ITTX|x^pejQx(k4{Iu9|ueLw`(z4n&-vgv{5um-3nvqHU9HOu@k*%yaMtNXCq;t9q&kcG5d#oYqLl79Lc@0'
            ')0a_cj21tS#5B)S5HRc63bx?{_*^)3=JK_dUv5-xP=_ZASX5vqkxZab@Di96)RfD2a6nU`%>MAH)MK5zM|uHx8J&xs8jYBx$wttS'
            'xYyEGi<C|?paQTD+N1BYba8e<uyjZJOjgb!pd{x*g8(3x;h95s4*`B4L>}(=+MR$M1J+R8xxdAhY=9dxj>r#B9&Fv;8+hZdCO<@K'
            '`)A0{%;R%>fMxJ*3#_0E;$K9((V7knWI0CPvl5-Z#>d0sdn4Xz*9|7^Mp0D(MvvheP;VIA_|Eb185V&3)NW9IP(6y^FCZmA{>$;{'
            'XkPP3$8bcW&UZA2odZ)pA>WvUKAi$|V>yhy>Q4SgP5dWJx;1??@4t00+Gbd1lGQU<b9v%N#YH(n;j%h|-yl1G6(VbUL91n(Ula$D'
            'S13Zt?o7&fCU>;>7iQ+i^bhZE?R+?V3@89RQ#}8fUo`u^hD|SbJZ;CpRt%PaIY8R73<%;NMQ!$FoQfEUCh$7g>7jUq)U>pusK(}i'
            'N^R8X*XW{JRzO$_(W_u4W)bwEk8DPkXM+!Mk)tlQlB)M#mq3WWfBEJ|@lMUcJJAD@luygJ0BX<Bhch6!70CI-tG#Pa$vfyHGcgyy'
            'EE01}oH+0|SDAP>Xn@14$I%)&epDSW+X?$+9?u5=C(q{Ppv(u02pITMBIbsSi6Xj)P=2AIt}=uxIUm)Lp?HHV6M&9DaLj6yh1RgA'
            'U`~*fhEgV~$ngaP#5GLKNG$~jMitENGQR@32SghQJ0S{4{;m~{i+EM&08If?UdevN*#*oLG|qjS#tZa@;dMz<HQz9sPyvn`RCD4-'
            'P0^*f1E>W%%d$`e<AyZ^;pqYmav<peVia37k{FJy6~Tw<RSb<TqszEt`v_WJO4*MVZ>JUnTOlk4M8m0RFD-uYy#4)H95<wUwnnq*'
            'nJFFx1M)dFQJa1%?8YC2;PCJjV)*$%qYuhwC`mNC2@tslZG;NPCp_!YxPW}WVOVH}I2$rqL;OMo#qsDSjm#oYlD5+CfoKK$MO-`J'
            'Nby4Nc(<JmfN&p=&P{ccq^Vd++E9MeG?{%5^p?T`=!QNzp?X=4m|WVEO6vzL6W!{ySChNEK_#O-??02?k!Bu-i8@B}<FlWihLYH{'
            '-tXt7^ojuwRWHypYkbm|?qrgBq4^FLC?ne)84qz=u`~BbQcj0bCbaq4%ARf|xOly(w%AOC!nPE@0G|MrfDEjmPMqcS#gc^L1a=bc'
            'N}N#I>vL!=%~8K0g|si_L=gD``UpZ-el?U9Fs8o~=#6p4;EJCgNR!t>OlVZ5{;;J7HUSNz4Mq#}XmKR2Nv*5&W9Zanj<B~`{?8!8'
            'e*gUG@$re($R;XF*(;jnf((+<Retz}o_gry6IWyIGqBC%xMSMzxGMB3Dk`Z|v4(@%42q&iGNFe6(`OHO6~hWmLUD<r0`!AIDwL1R'
            '20+t!-wVb;%MJill`~8)ApQXM@}#D1p?|K+z@x84AB-fc<C~T8>jZ?5Qg0yYrdQ|P$U5E4HE+IDhYzA*DS{VAj{@`HSR@6UcO+a{'
            '(7qJ!jG?_lq~{*;0g7`W0ng)dg9igbLxu4}I(ZymFoh-HTv2SPT!~iaI!h|4S&Xsg((Mu+@rf+gL)#&NS)nzcHqn<*P1=z-nE++P'
            '|Axo*q$v_6*$a~DdULWxpY%h)za=z$ft?x$WfttEE^B}oUBu%>AOm`?5#(riyx1LG6Eu1jWhPenJOzB0Shc`&I5n2vMA+dhh57Bd'
            '#GOu-u`GENC6!S^ZH~X>I4RObb5BizBy(eHx1){=klK7j_CDC>jgMHziS|*anRAf&&oJ-$80+zo3)EhiW;OP&j66v9u-AM>HKRfU'
            'uR5EQT%Z)T%z~CZLxC%$1Fflb+yq%tNBuUOM(<=jfxOz4f+l}0@bIbLiU0Nb<qINy9QTBfJp4PZr=tg7L}|GdGM0-tJ9a+#_kaC2'
            '8Cq46W{_F`2%T=Udp4@$l`@_eet=@zTx;f`i*-6jw$j$;Gj0KMjVyV-poYko1q01ggqf=uFsE-H5-LqP9KmVL#;jq3U>-(K13NRC'
            'K;Guk<QW5N`kNQL0`xmy;6E5OMz|pL(&r6w?&l`M152;CpP%g5eRAY#`?);hskPrEnq!BD6r9&ng`tBAi+M&alSr8})T+0IT|YX1'
            '53<i7UM$eQLhCPNC$}C8j$lK=JGqrY!f9z}V)=~J)F^hSGlxwP&_Txp(<dGbZBj9`1Jahz(?eD*DI$5=CT0&RN>M9HucB>9Vr<z+'
            'wYWhlDgP-$BmtyoN95&D)Oq<vAi|J5S<=>m^&g6ts6NwDBRtoc4qqT=tzMS1XY&)uHje`4YY;GwiN#tO_lbQ>`-~H%9GBMj(Q05c'
            '=>|HHCgep2tj-$NLeB`158IA^9Vk>EiL>hfQE)b?wi|XJ0_h=VC+ay_L6fZd7wOf-I%MPBE)>JzP;G-iZWut#K*<DbC*{IBD)t5`'
            '0S2j}%(yAgh8(j|5Bj7HigN9#6Gz;xqMC>gMSKyz=c5*l%*%QmWlfj}8-x%?pq(ZHQU|tz96q3N9^n(IB99KV!zy`(9;pF{*)pE3'
            'qyP<w4IEp7o{}<d7r07SAZCiDH_#0PjSp2#&mK0Um*-&~v4uTaC)>QZJ-IXGLRTV8elro%EQdjZScs5<fD#GGrI(zqhh-cUvt_Sn'
            '5dUrYRqyTm>;9<uyZ7Db?GS%`*EddG1W60#h4-3w%$HDY1pl4=@3-0cSKs{!%a8bn1B&(gw+Hozl+`UzjT+h-tkQ#!!>Z9~^S#*v'
            '`Tln7mx~>FxsA4o$XE52@W<f5PVT7_K`t3{7bHW<o*UgWmFJ~*ZhGJS{)@NsKCII9X?Snff3Kf<-{A+C83W!BO)Hz;vEk7}n;C$j'
            'i;iwYmOJV9nI%<i6R(Q%qN__P;IDsrbA34hN({b{@X@~})srSPeT*B6IJ#{sR43T=oo&LACAzyUq-x*;-&K?-5@mH2!-JC4#!d~M'
            'm(mm>?8cA);7k(0zW7Sqxqo~tzNXog(DChke8<A7Y^pCiuYG2k(?mXnT}Of0h273#LzsVTF&=w3>#7F6{fxcKQ9Gj9lAp0gDc)hp'
            'cR;w24^hX;6{@MVb9pFo^eMT*{bqTfFp+J0p$zrm3v;)~C)_%De`oVPLN8+VQ>mS`{|6CBht&$a+ITj?0Tz%#T`7#z7@`XO!zy&0'
            'F$dGvDrHS1kSif}CIW;hOUGeGa!bHrAXy?GoeY)2k^tdYLezxO7vL){F5<$1Z}Rb89=@dl(atZO@{?TPu$i`QtyD`3MT(G!G;f8{'
            '#8HP$5k}7rDclZh_3DibvDrU+B_%q$48ko@I&6I2OkWcH(sU~E>nvc?dW}mUTnz64#faW_e8!DGJoZSdH8GiGi~$m&j6fvcDt1Vj'
            'P$Fr3$9YBfPnS6*hCAYYfGZTw18F#qM8L2DE(uMbYbKE48m-d!340_917-*FxIzJz5d!w#EUn42i9N`ae8hY2Dq@yxbi?-ROP-pz'
            'QpSEAiNW?F<RE9rVlnMtIR`htn2`vK>>i%PGVV``3b^Ue89YZnEVXad_H>SN+NFT0WX57`9g?QTKEvDOsFORDf(k{nLBA<~1W%co'
            'awtG=ZDy6{pCfcIEyqr(mT8)s*MY=Wo?02?v6Cc{PIL+=pnY**06q)I(UaX=1JqvI{{QWwAcB#S-BXyxL$A;((Zj$;io+J)isJ*5'
            '_4c-H*q7+D3U|7;b0?sS0`o9D<vJM$f8qn*`D8;dV#{WEf$9zehOzy!GMT2a@r~zLi7;*GyM8-lM6PZFL~f{&9A((-nUPvn84+Sp'
            'X2fQFiZN5~Igu^OCCQ*Ot!M5qa2-*0XqXW0#hHUsSWZ44+{&4HPkS^Ot%9wk-CxvN>h7YR<?b%rUjBAjXMR{kbuQU8n1RoW1&vIA'
            'D8Rn^jT-Fi+?2cw`Jp69nydrrEo{D$>l^$oMeRt&`rMGbJ{c5q&*uj>E-bo*THWpwow#w8v7Xh(zC~XwkPR|kg#xLd_4VxdoCm?;'
            'L_h5KETxNn8v`2LqOL81>#N=S>!lHKdjl5suB@mj7Qc{;F0J?6gT5ys=vG*62{NV-vshOaVrg~ibt6JS8UCn+gniabqZO7&pn9{d'
            'g7cOEv-bvYhdAocFsSuAq3XWtMKC!g8nk+GaP!XAfXS3xHPXM!si6ymE)35ut0sf^XrBYz2!rH+E$UtNY4=>>?vdvj85BzuN7JSe'
            '{BUa`aqy)We7-*b?X~nPM+IptN8GzbK0L)cts_hiaqy+~SqVUL`sKys)zcSGp1u+r*wr!cbf34|Z1W-Xos<QeOMwenZ<I*v^~`J}'
            'b6C$FaM)o>jy44PBF2Oub26R-7rf5rs$576Iu|y@{<0~lD~BMjtCDG&luBDd=CvMG8qgbj?4x{ta(sMjV+#<PMlUcrKArDI*QQdy'
            'ctud?cX-VWj*jnOP5{%Jb_NJJK{YL)GI^qwJ@p`c8<5dFX9<N8+bB=@yQegxO&KkkGyu*}{}2^GR{~@t!T|M^fmBS3-h%LD03t%U'
            'LEgUFf}lJUuhB76h9_iOA;Agq5R@S%x;O08x~|hI*`RV679U+^P&NSUMOg&~s6}D?(dgLdaHDV;p}XuxB*2F=?)ACyR4|>7$d?M!'
            'j~wjvIIv<NBn{!owCiDCwQh=MZp2;SALnQoBQBp_5_5(*TCjg3>d3CYzbBNlNJd%=$ph9H5CEMR1Q&Nk(+=RK-4hU8O%}7xRw4g0'
            'Ny)$H49nOBCRABXtAuo5yI}INa_f51(C&O-463Zx7)C9N9$S7WDh&q>)tO-IgqmUe-fw@V-ha*91{D!sixYQJ&Gz?}MJvCC`fcaR'
            'v#}&Xvw;Ks$efn8vkAdw!N8nT=!dnKUlH_nbuYphOQrEmH`Jh8AOiMz;ItGD=C#*cgYcKzhdgQ)Z?j!|z4z@FfA=MsMbMH?ay%@Y'
            'EIoW_6Gg4=Y-|K<8nkFkGJhW;L`2FsgS9OvnMX}W1@?1%VsJeKKn;uZ!1MQpp5FSrIK1|rd;nXcLVpLx?YCl2L7JHU?y#c~D#IkO'
            'qlEHKYcvfqiWqD4g14ERX{rQ!nLL4$>|z|$)nf1<P&RLs+;JLHT%8VcYhoCrP{2AlrK52YKpAJ%WP^bvb=;g;H#jT`c@|ADP9-HB'
            'h6b$Cs-{tPQAZc@gXJB56Z9lYDvVU-yp=}IDt&go(Nj<g->6w<=Nz0wdj@({`Q%KKiws@D;drE@UPle`z7a1oI$fo0T=b+4TBx+e'
            'vn`JXg^C(82~re_h^yJK??i;sIYQY2ttFweSy@E&i99gWW3cf`5uyKBDSg-Eh=Y>#9P@_t23C*Up&KPQY#-?(P$<2>+*ER;6OQ#Y'
            'Vizr+MKn(HOit8H%6QaY`R7lF1Wl3!!U<R}^J}k-FqqfkGMUBWV793#P1$-=<7NPHXB@zsJP52Kml%%;CI<?|gmjl<Zfe-<2Rm2>'
            'se%tVU~5sTCXERyEz4yzml0EAJH?<p&gSHJIiZZI(D%G9X#vMlX`wKFtR6x!1v!~UJHBT_vr&B9vwx$Sm%=D~>95pWAoVoZpP@s`'
            'ht?@n4vFnCG!^EIp1aIio6+mE{@Z(OnwLzUR+xMi{U8A|=XKi+{6@aoL!ZcEVYN4){fp&;+v(ig<O+80m&u37&1fNdlu@An(B7U}'
            '^S!E5Snni_X3=VBjI%@ONCxSSVES&YL4(Gu=bU_`qSLz#I$CIHDD1}DiV#=(f!KRuxwr1OdU=~h04;XBXA@4DxlZ!u`1362aoS_C'
            'j;kyY+kDkGSZB%fpvrwL?pxp|lFY)UqSnrmQ3C8?Q{$YO6x=%n2B7pn7FG-`p4Lstz7e~Dh)aph#=c=5Y4+Xdg2Tm2S<ITTX4@yg'
            'v+73l=I>$GPx{Wc4vx-_s>3mcO)|EI5i@(JtDL*rnopL%yHx@||1#fZ(K?x_By45xHq6Tn9E3_?_{j{;ZP}b#f%y%r&}|taHd(Sz'
            '*PLr&gL#JG9UI9BmvDvUl<Vx4tgpgI2W3f&85DByobmt}5X=}gp!dt<Vws}Km2O9PX*Cc5o5a{4?TAs?$|<U;C)pzjq^9H)>Xm7i'
            '0NdR6--tE&Z^@9Hw`jKUN@|jLksTJq$t)&6wdGc&ex&cr^Yk^{vp=Fc%MJTWvq!RuM>k~ZN;GI593D95-qc4r_XifF()zG+v|}DE'
            'nUF2==**ZzdK*jUUCZC@#z-E<I^qzSZ{WFV+UbYpE1WxDAF=N5#|+8ey412z*S{8Uia%&)^}^ZHOY4QA*PGS8vgdc2rRMj<<KF35'
            'WmDlNt<w=^T&T9XzA<9sGfp+~FQK^W0ln-02Xf2u`2Qqs+3luB(<e`DR3pig+u^SzeB%svsuIwy+9YuQMiw29mURMrcAVlpHTzlg'
            'dHq4We)UG&$el|GSY=1+RH<0A)ybH6X5Yq}_8Bww96JX-rokZI4vn^d`Y3a2d3nGp!EaQSx3mLE<ti*<;+TX&!#HD`hS)IEztKe0'
            'hRxQZmc7VR*D@NZ3TV*@h11z~g%ef2u^CY(ccW|Bq_h$tJbD~nn?UJRV9q6HjOuAt)t*BYwgsmGDeK*gQ2+1${-3~jd5!OS=9?Y9'
            '>6>rfQrdW-HG||YtFsx?)@1plZKRT~B|qGKOtuVKcaHb@;hmb-(71o5MI5ar7tzLb?Yn=kF_JYMZA@i<z;P*Eo#ZUYqE#;Qf?Q|W'
            '6LzzNL$QgWQ0(n2!t+M4knB{9JEEYR#H>X(6ww`)5!9}86*oQ^_%QO8l;^${u+D(7Su3S5i&mJ}3@zOZ2n`S<ZmL4({r%1YT$VM_'
            'Vjy2iU}wkYjW*i?12gVGP{mp>1}EnVz7w1(`Ahg#qKiO<nRU=yVOEqDz(NHpV`0?9_Ikk(G4%`l>pT2wgnvB@oLre4kvAbc(YG*I'
            'L=offxy8P;6ZKA*1Zkx1w;2}<yT+Q<N}*c})_>5{Cq(zx*GuuOG=<+&{F1vF{Pl4SzEx)Q6IIvR5&nw)neVtVK;o%6k8LiEH$Y>z'
            ';OX>Oy_)@T;Jj=b2YtI^PeYOzatX_XBrl;Mfe47IkACq9jJaykXp0%cC+`(?IyruDypQMCHMY9O?@rOHK_wB@!nOZqw)P)=n~nnb'
            '%_$N{zQ#u9%RAlMD-S{T65c%^+`D`9IJ_75cxbbAqmNa)b*)ePkG`FUbB8I%s(n@LTYJpYz~jdKE24&Y^!hb=6r-rgQ3@1dK%v2m'
            'TbLtsldqIY0W-VCDP^P}Ku;qKszCtg7ad2Ga4_aW#5yd<RfqG;u*HS%u%vvCbs_SF<YYOo48kB0f<3UK%RHHLwL`Ue<(UO!ciZSr'
            ';`yP77mIkt;mi4=(bTM>BBHx0qYF+(dBG|e@i^K%uB!nBO0M%uG$!%Na}O+C32!c#(oVL(U<Jnd$w0kvLljN-WCrX8dl{MSf+vz`'
            '#}!LpeHHiq0XX+e%%k-MGVt4cr6RkQoIk`=bOvXyS(W=NTjb~FJwW^({o+RA^6v2X_-o_qBKR5h_^Y`5r;#8BpN2^EcfN9ecFS$8'
            'a?o0o#`EuCYU1KiHh;a0=K0kKFw45J@z)~SyhhKQkvKKqo6l!|y8E@DLZvmHA4r)#5-%dWcoQ#e$n+EIqkb+DASv6&gnyxUO&qB4'
            ';$>YSf~ud5loossGKzZreV&52{0!LbWEP5FkPXIK01wXVRSZDj5l8z^^4~whzq~&~@dy(#*`R4kXgs<HP=V$L$}6!i#4FE$8a`JK'
            'uhhezLh)4ExUGee^Z1Am`&Vcsyw>!27t&&Bs@sPbHtm8JE-~DNfdQT=BK`^g@yBl-$j8A+cufBTxq$w=<|EpsQ~lA!4$Fl1;J;G~'
            'EF3)We|9sI<R3bB(uoXDiIRB{;d!)03FS3wJ)WYHLrycJ$i4|Bf)a{1vA*+)Pj5*$Etd->3@8u15BZ0!^XHA;w-eaN;>J1cspQ@i'
            'Ae~!yN{y-=9BZ=RX3&fweoxGs4=h`{-&YeF+{A8$Dx7IBUCc2MriD2XZj<Flw^^P!;Lf{Q^Y(>99q_F%P(_Q~Vuq7$8|atf1vv!('
            'LW}Aux1}RE+9=LssC45>3Ig1}grnR84O4h?3e=4#=|D@j4~?r|{(>QQM*be`En}ev7|7-Mr95mG5i&gL=p1ljf!VBH*D15$Weg`g'
            '-uH#~;0zD^8c58&<3TqmJF#XmY)|0;Kb(A`?$fJz^)IohgzfI2qpag7wTVZyZpyA|L*H&|Y6GaFwbsr|Lt^fF6q~I1JdRCnm2In>'
            'qlKgqV@pS(TlCI!GYa(brw9Lp+f$+>jZUs}5nNxteLHKP5C8b~?et`~yVl*^1vdh5P8|utU#~q%KUQ>)N%zd5AB4KD;=u;tsnI=E'
            'i^C^Ll;#&`EQQm|)e(}VVApz&=7L@5yAIDd#O3)KUN@cg-hFqIf1KnMzm`MkQyz5GgtZ;!{MWEaei+j@1tXKvqjiD$W`2{cblgQs'
            'K@9o@LTFt_>g6NsyVi_LB|Hi8mmht>+Tu2Mt1WK2R&CjRquSEEOzmuRa{t`9DME%ou)?{w%&W;dXKNqc;jV5}E8~;{c=)4!O|@i5'
            '6kQ;I2ThU8W1sE&=LDm9<sA%|o(2_zalB#`p-Py7+m#YUkF2YH72r_E4uLpxWi%b~%P5P|ZJCr41Yv&QtA5G8K}i`8w7M+5;>)6%'
            'c>CEhFXK$sHX@SNl%os#tUPk*S!tC*bwilC(YvoB?tg&szR0T|Q2(xzXk}zBT*#zZQs-6jtz$xl{mhecB?bddYPII08sy7a8b<{)'
            'cR6LG=46tgbSR8uWU-lCi@C{YS|pTR*Kj}3Dp$Fl>$r*#l^tY|F?)RT2)XgdC|-LZ@N8<}a3#Bh-0<zC@>Ok@YO;*L5>QOj*hNsY'
            's}o&LwhI`4i(0aj>MQHCCpWg)D(C>Xp>d~}%JGMaQM@TZC7T2i(Fow4fricG0L5sHDO7V{P)lFe!}hodoZHLL9iD`HK5}iFNlCZ4'
            'WI$_ZBq0XZS5nQ|@856-?BWn4>Lg82)e?!|n&xB|0Qa&@$|!}ccafwcaZRAVD0V{=hliM<49<ZM^-3T^3UG?yp`|eCsnz+`n7=*x'
            'lf|Juv1s7rr(IvP7Nr7ixOc68?=WrD7CJ+{c6;LD%eS7T*xNG~K=0al1MRn0ocq{%-|nmGZ1)jB`Y_@Vy36!^SJv&Lz72)&K*NJI'
            'x>EPeVTvA`FzF82??0sz?%Crs>>+Ks@%Fb*>45K?vuUN#9WSXd&%U5<td?E)%r1ETNlUgLBsqQ13*TLVjN7dP)Mkl;7j1qZ^K%0k'
            'CeT3MBF~Z!csH9^0mGKiNn5FWk3b#f$xzr*La+zfabuLLI7_kw`VZ%Ew2eUI9tMWFze$GbOwgF|$H?r|#bPLz9o&?fieoIBdTGRV'
            'Yc_-;y7MCW3#DvdMcXV|afrBZw*nAPV=Z^$OuAkeWNV;lTzPKpHP4`Ie7xQ8TC}HixWI5*g*t}q-pYX;N<PR1@A<IfLxd#oy2XZ2'
            '5#vzbl>uJ$wn_@Tk}fG>=A3&k0%s<bS=>Wi2QAvHrnZr`>`k$61}Xga)8}jvwD(5O+^}krfOo_V*31DQ<cc}^KfnB=i%Nl&$?H=&'
            'Y_qS%#ToWpFWN6iM|Ss&oMwIrhu=+@Qc)Umes7ki-WpD-b`3ok5~$`$p`Z2LUBZ9AQv;JTmm67`l&MsGF;X)ro9KaM`-e5+=A?a('
            '9aQy%ETpNTxge~muM4?W(cc}1=Eqw&YwOY<;hC-L*lyUAilfQ{J9Q){b|dqHDrDHB*_!C7LU&g?aP>#Ww%p<WZIy1O@`loWv+oU+'
            'e74>jsyG&EyR=oxuvy;-(?o}k!naWcIG5J=a#oYu80B-9EQ9j><K4!Z6wILe^3}6HJbUrzIXn9i_oBT4%{*I4sl;m_X?Xb@`jrh7'
            'U1%;Z)F$1zRI8B=q8y56tOlRwF)QsUyERbA=g)sa5%|rU*ElL1cQGV~e7EOx=@s5rSy!AJKa?4W<K}`09z;?a4vtvA334=I1=w$&'
            '?}e60=MGGLBsIug(bu#ve3|Z-ee0wSzxCO+O><r{4quX2uh`eH=wu_LWSP9E)AVQgxhFXua{nJHkAKDxb@WO}Uei$0Vw5_;jVd<O'
            'neJSw+v}vqVT;xyLH_4DDdLGFmA-`8ZeR8aSwAsW!eN@M6T0`c;C$IGl${cpG%E}Xy`uO7yVp_&VKdBBSn2}QfTmjo;#zUlRO)=B'
            '_6jAv&XpIxI~iwz@g_Lm+2dCol$TUEjqaHfhv6c5J*H5s)%e<iy=x2S=z;*O<2GNp7X!mo$CtQ+ux1(+IK_0qQqSpHf9Kv2=FFv@'
            'b6w@k>el&zv76|^X_<4bdDC}F4A(38yT@1AtC$ed;!X0EaqkFkCUoP!h)MpUjN%C;dJq1P00l~c`pNscGt9xP^go{%qxi{OmZau3'
            'W9U~IcXC!Ot?@p6vAGH9ke;B8&h^tYGvAs@+CJx)Zui>IZ&3lC9IQz0rV7;MG`FaQ*G{S+N?k)5)+}xVT~+F|GUm_v+_Qd#_TX)B'
            '@+NJ0`G(s7rE$aWflj%+hYM31`}Hs1zj^lN`P0d-KR$c?^V3(8pI$zB`uz2+H+X=TR@2>=d_AT<u#|b4nxs_G-JSWO*gbSa?M2Ys'
            'sHt{(M_>E9pmi%C-9V4Bj>}U@!)g0HE+g@SI`l`~5#C7cd!Q<Aa%ARKW2|wS_G{G}B-G_KZh1STQ^;R-471(7j1B8%xF<T!!cneh'
            'LD|<$${^ipjPHL@YmIX=;X)L8Y^n{0cso1gk?s}2Sv2@?etn8A^n2#JKf2u<>;Z9`SqPN%saq}8wJ{Fc-C+a~m+>|d$tn@Re=pKR'
            'OyPu4n*|O4yhElIIZOzAbI-tDUS#oVgt1k4WK?iIJVd!Zi?Z=`t3}vW?pfHZpCl+4GdPyl%Ue{&Ma!mq3oj3>b<k3jL8qE`XymjT'
            'It%pI;M@aBMuDTdR!j8i28UmjpM!>H3Sma?T-nSE)h*=qwUN~f$hH9Ig=-gap=xo?nStiL52pUwnt>+KSQ{R{?}A5PJ@8mGY=M)n'
            '{m}Sh3s~c{GG=qV7bP2`e#p+teFuzfz}L<@USN(7KuBKA_NhzggqaL+qr@5B%W0qBRHElQl1*ihI1+@JWzgt>Erm^!gn=&VK@kPo'
            '4AswBvCz+SMK;nWXrqWQmJfQ4@jl1Hdmc*l5dce4n~0MK@Sn`<AcvqXRKUQ9Q;x(il~v*^3szqo9f?zTd1~>>`dWoH6#1ze%t7P*'
            'ZkUVh3D%#7>&@N3<he10V=qwBh?E3YahaqBo9(@VH#A+H2Xu2KGlhzI?wi+~*i30KsLbMuzczhT2J2{pwETE<IXb2rQ}l~ybo`y^'
            'bvl@h=q{-=nZ7^0f0qjLqT(ZF@Rq0mB@4Pl>XadFlJ}Fvn!ar7%e$w?8wYHJpQ30cJ8RFf&D+v9o!q31KQ_aO1DpM3TDNtRfot27'
            '8i{=gdO28KUeB8~-SZ?E?%(=`a8E?ALLF&6S-}V@)b*4iS|#iw|4=9!%Yfde)i(N&AO~Ed)+i~Xb@XAV_P?DXWh`zjoH)m>DN<?;'
            '7_R3_W$-DH1<G`QuI!h@c`Gj1$u!jtxNYSCIk4kW8as*;ciXK#ZZOdke7i?zXzJ+*+-_blJm|=u;A3zcg%BzIFjwAQ{BUPmtv)8o'
            'gCnC#(3;ZRA*WZ0GT3>ES)<N`B23Ke<b9zBiiH(?0V2oZQKmZMkuN<|xZnu@YVJL}*41`;SS1j$B0+Zs8)bHVh2KDrDpa^mE{bTg'
            ')UomUHRsEqk3GXq5?TjNXrCPJ+@W*h{<%{aJ4EJ+6}dJFD`zM3J{-%yz_g}qTp*RR6MEVK+aEA@&`PxsdWL4NOcNtBHcww8!frdZ'
            '4%G0V6$6Ky_PEWihfRdsxXppva1;wSZZU#pq*9xUb)l&6r8a+Rmvou$v6rQ{>)4F-*u$vX<uzY9ZZo7##J|Nfjyo7nZw8in%Bw2f'
            'tMSz2Cw_}0miI-kjR{7|vYeC$N#TlE#HOmmU$RW#sW-{H4CEs!14YKcbw|%Ml>RO_q{QmFJLaXYt&uaCP}E>?ct%$Q&(Cpng6r1Q'
            '?1F2zzq=9MKZtXVqV(BiURURb?It+q85#K0rZ+byZ9pB|XI>iNnAksMv1~f5!AMHtPIrKjW3O??UAHW>yU+?I8#fW2p%q0oHo~M_'
            '9hwpOHNK;&jZZ@vg=Dm<_V4Z^7f+Aezl4BJ>;s2AkR|BHF1W?=>2q)I@->%Q@M=tEbj$90gz!*lj_uPZbcS@UcrxM!`AS0n!NMgd'
            '6162`NG6O~U*NzN!IK0p`d&rzBKnK?S|6TOyh2-10*nHA3>xAYou70Jv9%0240IGHAO6Pc6)7K%on8;|uimXin52Wthfnqd;t;GC'
            '9<ha2u_d?MT5NG-Qzg034J2Al?SHHg1D(k)7in}sjPTbe7Rkiz1i7`$O2jb?)V!vl%&21K4H(<9oGIU?=od5FN(Wzaag>V|E!=?z'
            'Yoviw8iO7utBUX>OUPNZ0FlFE7n033b<uF@+zED5qRX(2^?P|R!vV&*T4H=M^yC~FbKZ5+MG55jP@LeM;wG!z8qArR6q+=?<^q1u'
            'tY-NYT~7W1*tTI^&Cn|`XUfs*1c$dd)qFctAXP8YAUqFP!{4eRFS9t8M;oRWhM)Tzm`K~uxkc~;xtc4W??sMRVj~Txl84e0?=sf('
            'A<4=J1LtNj3JsV+DPFOj!gQy;skcqbsA49y^E_HcAentgBZXUhH7$y(vR}24t|?FFb#P$#F4`!&23%aktZ|T@8|oEmw?8C>jjhoJ'
            '9VIY_kjjXMYDgpAr{z(_HRv*t4m&xos8k53(xEx&vnBa`81ppIH<av|LKXMJqB^d+-R(0{XQT3gndhA1c05BIVlJ7I!}3s+%va~e'
            'S;GVPJP1iAW0M{&aT1amIn<>8Z<A-O-*0EsF>%ds+wTD2IPx0-%@LfvXjXB$x3zYMOdnUTd+ky5;o#M|SM4vD`R%|BNVKQGzMPdy'
            'gZzdov`8>9xa=9xrj9<Dz5YFFS?|$wnXl!nIsJ3Q>d!fjj+i;QBSsc}h3<DnqDHQzFE`W}KoWqRnAZkrf|H0z1`v~rd|d;9c<3ir'
            'ab^>&k0n}GZ!CS9OoFN?m^>EO(HuH~9yq9GF}PS`?6Cbnro%hxY4_WyI)fyG^ABgvy{OOe0Oh-9d+uu|17YBG50!Pj-&_D*AE1%#'
            'pA&lh(;ZI_B6POzhqs#HH-YW6UK$q!8q=moUVGvi8VKJ!l+o!$i^?JryYKvcpuK$2?tyaiQ_~Q1_zAn2g91Y#)f^L^nKLaoc;l7v'
            '(<y1HYcSWPslD-*)K@|8(dPsvpVoBL2i4-?nxbB)d$g{I7`M^M+HkaKT^r=72c5?GTfkwRi$d+oPl6&t8AqKy09_DlCGw<ZZh~x<'
            'K83Qovn~B1-tl7o?kKD5ip~7_oq(EIrT2=F>4c}}u?w7>-CILrDx=SrJz%Yp`yorCEozU;#Dc5M%5IS^fQ`4o$m6uIuad<7o0Ow='
            'B}w)7sp(_{*50NSrgp(mQMnyJTHXsIaSIh{;P+!ND|cyAgO;a)S*t@K9JJgK+)5tZaNs%}*ma~@+F!*?5O9;$S66D~4Y*MOtauso'
            '=pl_qap_@1_z(cDqBgHuwLe>G6DVV=QM_eQu`)7^|K5|dIT2fjX`5PliR7$~t+WhmZ?enq9R)@qc{zjAU{RS&lSj$wR8X4oRt2p>'
            'H@za8*T&7yn#*>X6TWT=mxL6*e0-Xd!6lq(ex)T|d+Gh)j>oVV(H=t<Mwy?`>&~wd?HYs6I%RaleC>9mLxe-VuH%yfp;O{Q1nBzK'
            'Vi1){xiN~597D4gsadtv#%LCijJyP+z3M@4J(&^Y<UUq5vV*jNkg3FddkfSvG=m<Eg`dqc6p=cIdizr-PW%2DTh{AHsb74oy`h*<'
            'H$kgclj&Ao+it#$D)rI^gnl`tr#v}8<H1K$+Nx_Y|G+_1s)^ev2<@aUs55tNg1tEr3gchv^<|#SVsvVkU%C@(wgL-X*zQ{m+G7{?'
            'BqsW}2*mEl!|ryvLlXCeBF^jC3QosKOleEz_Sf#loa2-Dj!WPnF<P%lNqXm=d54nw9re@QEnvr(?ld(X><^vX8#3A5(yd}CZx>P5'
            '-N+50c0a<R4bu1$bBf?ad1*X=eIW4#*o3OsicOBa(KjYj)xEbSA0JP)K-}tSOdj9sNHS)4*#H*<4&*(M@0q+1*NXJh*?Ek>xlmo+'
            '9GA_(%@Ji-7q)(SVBJlnF7_*8vQU;7A*YOg;%hOb1V0U=uf;lfkIE$uz~h`d7|zDl61u}0Hiyz8r(11b;rKZw<spq;UE5~|<V}=L'
            'oI{+Du;KV=HDsV7=lf&L6nWro>S>ykOYB#+ttd-h&s1JfOp&}TurGl;lEhlKFyfp+i#j76lX(2-#fz735HA4JF((`<0>=$0$1Foh'
            '1-p6`^WtQ=*4hX_@0z^Q#3PvfxLm?a=zv2uU|>pJzQ3@g%O-F=bYGMDYQ9fMPpdF2SXcrTqSJU50lNjgnC9~>`s;}+3=)9$5)R4S'
            'g771L22{wYA}q8qm?b4nD>?OWv;YLWqWp=MNko<3r8!y!G!m<k$_GGTz?lV1W3T@%lu4`PczESs|M+MCN0Ul2jIVeeIJFi0q(c;a'
            'S875WbkTv%KYH;*M1)tktiCBQE_bRo(zQ`91W;DP4xw&QQ{xVWJ|&uW?2e`04=_WSpm+;85{A*Iz|yix(p1ElK!=h<F<#Hmk6<op'
            'TyLH|JzY&_-iagQeM@uq4k?=4@D%qJTSg&hIo==&s@vCj_Ta@VqD+!Hjq}wqq70g__0S|Sr|YoHnAI=E%Z<D)JD-w}RdC#}G+gzM'
            'q-R`z2y6vk?OxJ~pn-;V8>MoND+x}ujNQ_iqNnjP!l&XP2N=T;^EBp5?=b^3?!gsa1KvdFeyPYS%s8gT#F3F1iDm`+hPw1Bj^>nP'
            '8ZozA=2z4qQErk9DB<w4h2{R2k8or^3DnwaPIUsCJcIc%vTm9ot&Hi8>F^8FbZ8{3@&j(}Dt4g~c~?~RdT*P%K=f3ToKmTO*l+b#'
            'Z`se?;RbL=pF3LiciTRH&C7kqonvVV1(u(!7Yy21;Txf-_kv7cMxw5)K^T%;rZ6E%0UK|gms`4`h?4%Sqd7(@`r5!`VyzHvX_%2W'
            '$@y2@E^DRD^BH*B$&w6jb|VVBoz#h|w=NuOg{k@RRz_aC{T))16p($`9e!>^o^#OoC&ina!}zUX=iV_oG3Z8nf2~mT;ctjVH@YW7'
            '(&_%9by9My<RXbNOMC@~PJxE`G37&UrYHgyY9BUH{eI}p4aVP0gLGtGZ#VVdYbTA-jX$38GMbhu%X4#?&9skqyXX-d7zn1v$xaLl'
            '!m9<{?dQvO*h<=neKzK*ug`)O`2FeqJa`n$qg6sFA1D!_c~=5pL*hXtU<>MPU9M|5??x(JgVBYCwEWi#*#CPh+utab?T<|$l9(kc'
            'z`VbQvi)ZcWxGWb+sy%Ne2B1m5cS>a7R1sDAhQ^drZ6)fpF7O#QHH6A*64ksJajnT$-u^2Mwd0$Ir><z&^z#bp=Ey$wfYR9R=+g{'
            'q9gv{uNeFA8{!^pjQOj@J1|n)tHtlkrE1fXmi9a}wASGs9eG<T-v4b86QMbV|KRY6ugN*_Hw95t$DcoVLVfazsOBN{Iif7iB7XCq'
            'G|a-vpY*p0x2TT)9%k`BILty${Shz6pt)#ThhHQde<6C@__#tX=0l*y;@OMWk6wtEFCP8*nRxQ_56_-IdG^Y>QH|6_!0V+T7Jooj'
            '=Z(m-XbV__X?rl}^0NnO+%TTMV%UajmL~f_>!Ooea-YapGR5hs%U}BKD@s^d{);kx*ftMjl_T-|(VM5QwDn-9Js-3O0n#Nvw^9Pb'
            'EM!wMB%(cNTGA~hky6N$CO#Kwo?SGS6U@#TrwhDNmP|F2Ml?+&zzWR}(wK%ODayEs7m!eTPH8c5EFO-vqbF@05+YkX@U9otoLZN@'
            'Hi9VQ2is{{CXgFLY~+F%{u}$1<6UtzjWdD|RF~%ZW@w026R8%NXdfS2y>|;tGb|u>AUURq?dqjeD>YcoJ@(<hH@x6KXRMK_Uf<IP'
            '<02v_Fv%pw?srLpv6vf<(GaAEL>z&S+hC%Uc??^O74S*g#A?`To|}TBZs<(U=M;tc%A<9P+XW+9TK$#Lcuf4Cy?Ffe`O}}CzJ4aY'
            'fBMJg&;I=Km3tXLj{{+9Is)dvNnrZJJS!p|ZnP~pxf^U}kd8yqX<;S>A)p~%NYeMkU!&ugix#%()i(`aV=DKdc!{0Kvl#~FfRssR'
            ')E218DlhAl)z>ZHPO|pj5MJcz*xxE3$LC(9JVia9I;-1$SS|$n>&M3Q*qMC)M$tXZ<hQMf!Klm5_+KwNs5{8#a|QN%zF?k@3*d3I'
            '1n3v(=neMzrW=X_sJch1;P`SnDh8<QzWOLP_TS>V-tDd4<)MBXfAjx7PQ|~$PxzyplK)3K70*)iC9<4uwBO~6e2{OnUy}BT%A_9>'
            'UoU`*C~u($G|g-DdZSDFv>ljbITTMqv5L2$)**v{g}$C-ofX63P}g7$HP>KGQx(sp!X*@6DK&L)3s7W$Vo0(q;kZ-!{{x#$%uN'
        ),
    },
    'youtube_data': {
        'sha256': '048809136491091e9650d1783a6708cf1bc59c4a719983c19ccb1fb85b4a9f69',
        'encoding': 'zlib+b85',
        'data': (
            'c-q~4*>>Ybvfw+vB8T_(6;P%qmF@EMuuS`^CH1w&wRK75ZnvyPfkaY53k29$q++S{5p&MG&o9m=Of0zqpr~@YUdHFRL;$%)M#h#A'
            'k;CEet!%1zmdTMQWt{95dA1fm7tL8Slj10@V{!22MBIEO=0%ptd7T!y$coE!J{}BCui{cBVzw6hUq^K*^CT)(tD>x%JgwJJH809E'
            'zZ|Elbv_%3{b$ilTE<zlf=Vg1q(8-ZGT7gb)<si8w^7{G#WKbM(K4=TS#mjXlg1(zv!bZ0x{Oz1F0)LexsdVvO5{Z?;nTGUDk%r='
            'vto{2r7M}Gx!luDo61p@r>m8$$E)>wu_#JWUr8Z9#Pizepq3x%L46h1BCUj6&Sa9neAB!xpm<S9(`glost_>@r@kts6!Vxy2IXGA'
            'Iv%9URVJ4*uW3&39_AD$qFCtC#VVDRppNE6lO-Z4(?u;{%vh*e=LBszZ%TkG6v<(9Fwh_=a0CFCB8zJ&Zm%Q&BS~w3I<y93UZj}}'
            'Cj%jN#Th;?i=@e5*z!h}wB8i}w372u)?!f>OK}J-%|uhlGQqV6xD?e`=CE)ua2$JGL2E)3w>S^z38r)-pVNe8S&3C#RRl`JnrpeH'
            '<@vD6(s^0~VgW$j&zdw#qIs4AK;DmOY_AF!EsXg}maqsowiRGbU5yac;9nTV3R+LAnj0_UTipRS@V<(1Fdt<k%d#jVT)6jQ&w1aJ'
            '*?aNw-RYSCypC5eB3MZnPMoh{fOEORrJ<=|UEB=Ss>or9XGOB65xjf*0_p;)NPq$&#4W5w4G_mx#Q!{f{c7*<^d|y26sM+W7BU$@'
            'y?G2k6_-VEnaOyS^7L)b6+jRnI?tL!(i|5Jpin9Q(a5T%#<2h3_9_KjM+}Goji?|(ImUH_qU9P^pp2IiHcdG0tULTNn|y+^U_`*h'
            'g9c!w@?=%OkX=-Ss_9}qq9HV^tcVj{k7kyo)fK{z*A9?Fk&W?SI2;ZLouX&~`>vExBw!N&Wq|U4v9u4W!9cycimR(Eo$1HFSFjT5'
            'H;{oV{ky32uTtt?b-I+?72vDHC*6^L9#NS@W_8RzKNh*FvVz8;UtQx3_BA*?J2-oH8vS(e;@$D7n2I}^`4Es;mMLxGY&a6b1R$3c'
            'D|`{HvUo19iY$>Oek$cnDsLlPyA(*>kYP6jq$`UXd=^7kyJoR~O@Y;Mr{T**45&k&p<vPwg^X~$*3iWLVDReo+m{C~PX2m)6#aB^'
            'bo@FxeEs6x%U3wabME3MO=J<J)J6>(PVz{rd6^QHQQN$5GKu0EpXXPwBr;RA^pmQsn}z8BHsn+xORuGUniorgfq9WMC6n&7Dt`aX'
            '{!{NAkQ&`+9bZ<|0IbtxQLfbh61j+*td6oczii@5$wg@NQG5dsLSCf`5&^<laf`6qZsIxc9l)=K+D@yex`L&R%m84QB`Kl?kDXYW'
            '0>$8BFgQ9nJ$(Js@!K8{Z-o9A8v1~%bA&7Gy(M+8U(m+15UONc&aa}P1bm@#ylgPp=&ES&l?8hh0ou`PEGY9!C`Irt)f^BD=bGEf'
            'Rw4C{n=9p9=5wc@Uh7h>WL(?#%*oATlETj$oIx4q*Bv+={&4W>)$xliGCqh{7C1b41DqBR6|Mlu$RZyNm+^<F3!$Zq^T?ZkUYifm'
            'Bj)vQ;uL5w6JX%BCDLdu4E>{lQ3%}J6=2dx@G_2JGa;#j4ZQ>&4zz`tKN1*d4)`VUN`=<q>HhvMSCrLW60gOqnO}nx0%RJv26W48'
            '*-+w2H;pYrdnj2kr^X>(%3Oj(1+*>&=E?)q4gM(Q3Yb@Y#p5BkRRRY;Lr#!GV}HU5XCO*k%ksGq3P3!oB=G=X7bHdrU_t2;B@E;<'
            '4TwI#`H1n42Y-6^`s@G}9~djpO1;b#fn_)mPm$te@fsgbYPs}2(9(Nv9GK#p`@zxi9}nKWIEzk?58fXB5FH-8coDrgd3kacy*Yjx'
            '9Uc4(y#h=PLjD{<iPwJyynOxk==d$7V=&bF9ns}b<4-9wc@x88(?g6h(=bE~_(6i01rweNl7)A74F;SdEdfC=OE1$|nxQO~z~0Fd'
            'Tz7!Y3W$jyLW2@5t`H&=R|1NgE%p!;vK*1rRu!3q_MxVxPGe@)C=^!6P6Tg;O0`6kdoI46ekGP^-qdD5x;_ZRFe>21GY}jPVOzdF'
            'L*X275c(FxeLzegs3U5k%s?%OK#niR@T8g+w<$3V3BnDtd@PJ(Ruw?BIQ`-E+q3BC-P?n+lh?1JACBL>Jvlu)IgDN&`~_C)5ZC4R'
            '`-6cd4<|>_n}f5n<F~KiODV^|H&(#zf^zt`cZcV{-0oi7eYrpS^6UG5hgFD9PG2MMu<iHOrL7GOya~Ra1mC~;7WPy1-7m=(;ZgYg'
            '+4t%tkyLv7L#W>|`1tKh^A?y>0c(kG;mt2&{Q7-(8h-!p!vSChfNb}-`wv<>1mOTSS_&HsS(e1T2f)Pz40to*9>0mRMyfRof8#L>'
            'fGi1?4?s7XoAD3}l>v1firw#EeC31*DJqO0(Shu_1JGKemmu9GJ17zEEYqrj!SCRfd36k%k;$l?CABw3;gA-N+CeU?n;D99q2>!D'
            'L;}_37Kfh!_{XRxfQEolwNesz55eG3Jy>g5pJaFvaE%gzPg?GUrbt)@l6aZs0JVh4Gf2~@p%?)lt81NN{97+$^yBf*(VyNS`FnHl'
            '_Tc5FtrmQrhQG`L&=~$%7uPcXXFQ+Fs*3o5^}ui%od4}VeyM&L?p%BkB1t4H=~1u()|5b-iJ-*^B8m7X=yJdUr}U?TI;S-Vq7?#W'
            '7i#QQSMdtq<>)PyCY%DOE3r1zZWtwkbV2P@4Qp?q0W=M1&w?5laRnlfudRV5c?q3q&5T!4CgYAy_CsB-7&}xutecVgzfm(A3D(S%'
            '23LvqF!1TO<MH^r_lQ!&OqWenTiqK<g9vhsh*8xE==x@7AY1{2067A2NH1Y)Xh_quQxN}*fq0-A5~xDJ)Sf(f0s`8>;o0#K{2YdE'
            '+nX?ETs5;Gi<h$`7R!lP4DXiX%d%)z!PD@5+G=qm?C{iXCW}c6V?gx|;D17p9%`}#wq!KfX<Lg`i_c5l)xi;%uBwgD<dtsIncrx}'
            'xF|50@Ut)kMnIJK>XP-p&q45A15DuV-q5Vdp&kOt2<;7`#UV5b(wnh(8>ba4;!n6%ym!=QRn6m4E#j<Ni$Gh7K*B-u0?0S>>&-k!'
            'EYec#GNcUEb-IF1LC6OJ4MNWqF-h2+bGE&qRVJ;`7>LLe8XSPlL*h$&1C=;#XV}5;=hyGf-u>YiL`x8BflwjZh;?!O+yfi*cUh*{'
            'GG4}6I#yeYaI^)ivC%TLm4TWWxR^<Af}t`|lA!t({HrVsviuRYh1u0}pc^tWtxo?4P?efU&$Ad9=3*UTaiWR=P{q!k`vVZJUAb7`'
            'a8#9a&fKq9^enigh-A`?*-jP{f}%Puv<{EG08bpD0Wl$~D&TU?R^wd|xN4wy@nv41bX{3;S2`=w3q+bqr6FjBGcdJaeL=ASoJD~i'
            '=RC5dLfbg853&vb3A~i>t0oAIku#(Ep)Ci*y}W8buq%MiRH#8466_AXQy5KIV-%n(>IUt0gcKlYCNdZ{^<ww;!!R7fSb`zV5<0)T'
            'cVFOe&hPFo!t?zLe$A~To(%mlG6xjLnh|yfhUQuCB>+wXWGcSi-|w3Xi)m>NlsHWy#rTE~HS=_PGay5NDh+Z+f_VL4YhQI+gAxkH'
            'psfT$(tL5+Ks?xrgbMMCbeY!fTeO+@FT-xr?qI3iNb1tthQZR*tD-)%A}!kWigK1FK-RW(hr1TYR3vFv5sZ|829?vk0qB~xi-!RD'
            '_2y;4ZWo{$Cz~1|>dQaBn~G--VM--SAWckvxVg=UX%Mn!2M33Sc+f^3iL-PmflBhWc7SPSA?w?sypGV43eflUj3&R%i`!g}3slqB'
            'pqU<$dU<wydKMl3<;{zOSEO$`hF_(F5SC6vKY!p;lT(tPtDQPVh9Pd`I<}`$vV`e@-U))sM?e+;5U>xKq+@YfWa9t+pa1>802VnO'
            'R>3L)wBxHfE>m%hXFX6r)=&jFg>5jMLo++C^dlq6j;Wca;^1(s&0iaK9n1%n6w}+KoTFU)k;k5{YP9_%FwnFbi~k~`3_?oLBmxD5'
            '0-k`gKns=l514&<lfoGAmDOx3(R_rdZkp8C8ppSwK7zjIR1zSvDvQ5EchK5(nrCRLBGAH^1ujdPrR66L{zW<~W8h`*bEl`v9sF=i'
            '!!)CB(Fqvqe>Hj5T*FMEUR)6bozeU#0j2n3(v;1z;nn<vL6YeXgs6DEfadd9%;IYi%$hPz;_DP-hf7(4m<0uZ*GN&vSAcs-T83uK'
            'cE`hTPe2vQVTrC`jWPgg*5n}CAa?#pZ%&SOpcfe8yF*ZE1RinC;wpm@p!6nj&Ikj7Sdz&dD3%nLWh2({N04lE5s8+iBC`W%ZB|6a'
            '%UCoS!i#!`3B{LL42UE?!V5I9!4}#(dUd)7i~a$`jiPMw#sD@PixXh5CA2JbO&qpGxFp3x-D_B108uVG`$ID~1(IvKBLbz7faU?}'
            'J`5z(u(?djBxUFT8OC$#ycfkSjW(^9bfAnd62ThrcaO=0nwVWIkEA+V0Z|gRRJ=eC<6r@g#4Lq%hwAf24HW<cnrHJIzr-JLE&5>K'
            '(?{s!5Ar&b`BJU{F|f@mIuTY)#h@9dglnLqF4K=#GOa=`1d4O~cZHQ>KD;9N6ebl}i7-NmFwiISEb95@1XM7qI!g0ZQ(N^Bod=98'
            '`$m=s^$HK#LG!!aofl}2cg@gjQ6~)nxmc;}S7^(xnv9hTY;!DLk#25uc=sF|WZ9U`ofV3ZDhf-EWncy?M}6mJ4F}-`WXW=3Px<UO'
            'oFkAtnkAI5Q+CSu$0rjKP<uY7N%z}TTt#ffts-rThf<_CIK|kmqb2!nGkA&*_)Z6I?#KIw6)b|qdAs!s6r^3w+O6Pphov84*5_VR'
            '7j3b?S!PdKFF^_~6S9#3rM&?ZWGHq}(peEbLm-`H<k}6jh8DUy34(-sIUlM{lHtXqrC4l+&<3gO*TKj(b|JnHi=hbacx<Fq>`?zZ'
            ';eFWds|`bU1PZNbu*G$E6@9e5AXqav)SHbp(!?w7zV6iN#c-?*l2I1uCfX&at?&V4X-lnm`UUUo@)@;vx=HhX%s$W$|I!wqHPkJQ'
            'wqSx!=`XsY)TX;c7eQ-ej{%s(^bX?tko*j0GR6f#JCZkVL+cYLECF!^>GglIJc2A^9_K}#;!&n9!dUTH4ph=Avgpc(D3*C4N=`iM'
            '58`O|lx$3Ae_VTvW|dVChT6^pJ@DMNw(ix3zj0YtXdw%R|7lG7+sGeqJ%<HvCV`ty06c#EmOEp9J@-~Jh{X=cmpj5=NDx?d)MD<0'
            'qAmu!`Pxeg(pgMiYv?EFO91LT%{g8M@l73Mu`!EA`5aGUl{_lmYl{6It(U|uNq%@w8{j>;wUHCn?bVu1{89}J4V+}0{;(qV67&#L'
            '%!W*q*b170(!7yuujG?|J!W1&$GREM)VUx^8@E>liQj-8M&w{4`P43zUh;v*=p2DWOB<#J9f>OzrKN*D!9sf$bp(Vy`fBP_&5SL)'
            'rSjk~UtIr>uF#=m(YCnZ@$s~G%Y_{_y8;OKxINYohrNMAMj^kR4%1|$yhBIIVs`|nm1fll`5pNNpj9qSrrgt5evtE~mVu>#pvZT&'
            '>E|C2?8qOe!-ob|$@VyyKUz}17#g*QlpC$rh=A3<YpWBo;-@L?j5LOHjYd~-XHe_EmG>Kj3hddj;-Z5H%VR4xb<!I&fL?UuD*Cj2'
            'CT)%O*NzJsqaM%qF9z-YtY`<N&dNr1O0=XJZ0UU1l4U%(NEKozsHJyJ!(1%vRPPmX&<Vyj>zSr~Y;t=jCHfR6Z&X9xH`R;Nzx{;>'
            'S^2|vQ54$f47x)cNZ!D9R0HNad+&5xaEVB(GJSIEZ58XfXtYiC>99iYOjM&&AG57CbK7bn5b)<f$uB;g>DI7D@|+q#&IT>m-xpnO'
            '9Mfk!{_%KnWw@jWG?qs^pNi=*tF!tYaMdH3H0bXQ$Wx*Bk*+4}9uNVY>WN~`K6Z27Ft@-=Lvip&5Pw{$4q5AM0sMKfkmQiZBhyYx'
            'ISL8LQI)G5HO2HOWY-Ke0zqE;5*BkQ%`DLcG+tHo5g`cem+P@O&{6<9yXW+cr4%Je)WwPo1jM>4KDU}lat)~g^;flb1kErQDR80l'
            'pv)bXg&XGlVnqK}fVJvC5q(B9I-s$^!HrMXVjWKZFxq~C^2*a0$1C)L3$#q_me6v}z&kVPp^7lQHc<MhyFst0SVVC~4${$P65Kn8'
            'c2DCtNrJv%dAbqx&^KAD@p#Mp8FZhM)7QJ`8E1X|KtQP259v2oWUy(e{kV+}bzr@@mDaHUZcr;4WDRpznW751W9DOGwS#xwoPGWB'
            '=~t&9+SBo}a>E-_Lz?;!c}o|(fw?w1)UeqHb%Z9T(b91=Pz>c|Lv9IeF93J^oiilth~AO@g`P5-d=Zs~=C)=OHMct|wz(e^@NoN6'
            'G4?z`BCSXOx-_C3gAgWJu!0Q$?*PwX()(P9TpZh_*p=KZro0~81=VY>s9J+|QT5s@s^&ujeIkDU&DZ<;;tMsTC*rGb=uvn5M0|sf'
            'svWD%Zt~I(+{gi=;kHrDIG=)2`Qm4!-UidYor<lVswpd6xy17(Jo!QQS@O^|F&adQmFq0B3XCE0-GV`TCI4ToyICJ#?WN1UzkB2e'
            'EI_>~aLH@*2G#}5DN|BN^0!s;$_hixP(v-z!FUG{pPJl(#fdMiB5O8@nLU(LwjOs6xG=>c7M6YZsn^%7^a!Ya2HJOo-g4ys0>G||'
            'Wjd!2ELn}j+1ZQT_%<#<p#-X2HmiCU<FIzCcp-mt8xPs~#32sEpx}Y1D9R5V3wKQz6$XW|lOGS8bbS~r`7m-cRKlNjQDRl9Z9Kmw'
            '>Ufpbdr0B-O1VIf@KE{HRA^1WkPY}<Hn8qVk?$a{-mQ=`aU1~0YvmfLF+Sk-3eW985GVIVY7+ag;yH^;mj%Ac<d_v8J`+h~CUFBJ'
            'R4eqoLlnhc0k@uOn5$W$42AKGupA7PwVD_YnPSy&UhGo<D}-L~(20(x^A>l)e>lhv<tmnpQi@@)@*M9CvVGPV1zA+uAu{wVDaa8p'
            'CRMK+&k?Cm6k#|~6b0yLZT+mu3B`pux;uU!o6kTZ@q1uigc}Num)GzwQc{3VHANX^1bZ8GwBni`TQ#*JbvFb=z@wx65ucez{h$`n'
            'Jga4h)XdTRvVZT)5aZp{^fZbplOB)K_yHoJ#aY=&|C@N1r~hbV4%ozDlpOhhva#`r4RotcgCQ_DfD(o7I<3vi^{Or|%XoDKE1jg5'
            'c%)_|eRQ#&*TRVpVSASjlC3cC*lEq60qYe{f4^^3HqtvVS{8+L7C5kvXZyRLHn@#C%~9M3>l?roVig;lComTb|0$X(Ta)IHVmNT&'
            'pdn4$X$kQZBzpKa$M9FCr5r7(On5sLS4wQesg6b_&0_4dhA~3KDedg`YM=mf+E*~PMfw2=`Zeme$O@^ct7aMJyC^23aMT)H4x_G~'
            'D`#5mnoO~=&8naSZ45b53vb7w7Hq0RH^;CdUq1ULSPbtp4&49pVSoS5;KBWH42qub^>!sca6^uL3h4oPjFRg*`sS9y^S>SJ{x#nH'
            'xWD^9qVetp`cy?j7st*g-|Tme{aX8UPq>4N?sP}+lLv7>MsGcbBB1%;5xWH_PMvY&-oz7y9h%{UO;Ah(ulOonQ*aH+d&rn|<Q3Az'
            'fP%SA%UVdm|1Q%TnTI5yP|PV)HdFLDBhD>)ww8`$Ft$aehP#toC}Rxg;ME6d14wDLBLeHgsVxHJ(^Ibjl*$0KtIbSKTWo{Z{_t2A'
            'J?Kl=u$kkK%aU)60yeD^)22T9(N;;Z;21<k#NMFo+iB?Lxu%pC;)|(x+GpkSV9;^nWV1YwxBxpuVmm}U01SrhJJmRh%xuy1gp)bD'
            '?z$eJ!J?%Y6AnZ%?jjwlJVq-5=v!=W4RkZ0lIp^)M*6V3rwfY?W9Wg(uZJC~We%RwD)}-cov`TeR<uT%34?4J=%DOzQ=^P;Arp={'
            '@KdP^K?s<K3dMN8?WAZM|IXQsopo{NXqjivG&fDvZGgCPtFEeJ7mi}PFDy-|9!pcOBJp7#lApL|97McxkZ2+{FTmL1Ur3&`Faatb'
            '-`m}?Tt#Xmv>kD?M5HZ!pzSi|-{fX6hQfCR8Dm#z5`_1T`PObS-6HgO`^b$(J1c3HE3;2Vg5X+V!hyJ&r|DE3W0C)1J^j-5Y~@3z'
            'Yy3Av&Zq>?BIf|wH?L1m{z6pw&(KnFTYU!Qh8+5g@c(&K7B}A4XfE_znGj#4K7gh{5mako@oSYeN>~wy`!*ScqW>RdS?C~SmG2@i'
            'NS`I&t6NyYWt_;6f}QbWvz(z+hI+y==#DQB&W=y1rCD6DV3ic}1|!*#S%PAvBAGD_NL&*qrHl`?li~t<QHsQKYOBwwa;MS`lq;NN'
            '%3RrtYz8f>Dd&JYxd}0%ZfXYKc(d-%RTVJ125i7<5XXt3UZd(_1E=r%GzNn3gOyD0Y>AAppFY~sLq+Vk&UMiXO;zYM9i7;f-w$Cc'
            'KkeyJjr7yH)zj<iTYuX!0E4X&aR`Pxc8I-q1i@w*I5WP!5KkOHQcu?-@ieqxwxvVcXI(^Om@|D@zXqNhWp&W2-GZeT**?;?GiQ)@'
            '9&JNsEj(fmI$UWuYpi$g*})GlPhK6JynXIvBZ${!D$rdu-laIDDkWdziu`ba!gd=)3AmMc0NPI}!9Yfp(Q7nmswQi+|MK1?XddvD'
            'Jeb*mxu@lS(WlvOF&z0&BKStLb)*N4oJy~eTEJEncGX52LU|*wjILyHqFu?N8&}6H3{`{zGSsEM+6dWhL1Bl=1VY0$)h*sUR7odB'
            'q0p8n!!Q=8y){3=_HmSQ8Kd{|Wuzwo?JnaFUEJFrZSN1{H+M!`mQQc@2i8y*aVp<`4yQ5_jz-MCcA3g24)ZGU*r)s)rz}b=Bl1nZ'
            'Cp;K6mWju<x5<s&_@WKU=}0>=M~he)&K?NlHqyciOm+t9-v(Pxn)4GmQjzVF^ht_a)Y#DK7o_?iCfQ*(E-$GZ9pYG9NhcuDZ6-t`'
            '8acV3S#qRUQ{_rAYJ@6v7<naf$~jAZvF^A%GL#F*6qM!^GDAr|lmLWtGs>(|=coLm#kvf196NzIA5t}Yl_r|OxWj67%N-8FZMm=^'
            '8%hh<`aGIVO{PH0Q>fK9V5Z+KPqf-JQQdy~bd+9xhRyqq%n`dFpV7M6RE3{ozrylHJ#IFg^Y%yVWqZ=RY4>^-167&<y>VY}-Nw%9'
            'yR>KEStitotqWkKWfwc>(WW8UiEv>3<Z)hZ9cQF8ncChQ2B4#%2iEn-^&O)e8@>Kx{k~%xj=Z$L*Lizj7w28?s0-tJh3dXR6oVi-'
            'HCy#-@Wes0M<5Dk#zNEiMDWgh6yiMC=9VNM#!-oZY9{NrFp-CgI`O`goN`1pVcx<sn|Ao9DwBC&nhM3Y?#vvE2-Ne8FoqD31K^zl'
            'y!&W%hvo0RNQDPCL(hij$!FzJzpb0XZVa)irxqNYZQ}K$_exJtckkvgm6NFhf)N7oEiu7r`fPth@H@lar>lDOkhtb;I^XBdEE2Gh'
            '8l7X<!Lhq-sNf0OA5qsLzsfY<yk|pil?~4L<#Xusjh@7?$6jo}>rH0{+RnA-Dz7JTKCyVF_i{wX1+beuSKd}Q5mOCQ#`QJ~&ekib'
            'qL^*EiLjprahEw7;WG3(R3ihp5MWt89tk&nA$sX-h>!!N>qepj0`6IiU~`Yv)DqC?>c$L*WpT?#KkQswi7Sd@6qpIJ+EW=ID-qa}'
            '<na^oxKzi#BA(H?1|Dg!!wS_8$)tt}SlRtud-E((nI&W!#00qd#8wMi${BXEgDm={+_6n%yDzF!SYfMIyw5$h)qd0Mpy`%$vtlBt'
            'QHda^ivFf>*YmWPZgs+fFP@m?`1qT2pc7|+@EE$mhsv2>r$ctVi=AR_646#+_mf-2OElW5W4;~KUL7yoz^kFYoFu*FduO3Oo^>YU'
            'B$AvsJKJ*_)|moBx1y8!bVEffVJf02S#e6WW6+C;_5!B>M6Jh}jF(DvLkFtR=05h4l=68e&f>LGvQ9jF9QBmMM6DAK6<wUc)U9ol'
            ';dHJ*wB6-#a=W${M}2B7Xea$^2ifTR3;mh)SYL^QCP@oh%o|iYo9x8jkE#H)J)L4U)J6{d5$e3Nt$NNEZy)$+X(x(msExoMPx+f$'
            '?V;SYlgInaxf6HO-A+2}L&w)x;guOhlrZ)CHG*rOAh=o3<kE!f9VvD6C=rz0+YRUYqh$9Em*J_&pxB|kKQQM}EtVKJ{%l{I904a='
            '0Xr(l`7h3_LX0_+$RBXN7MBeGJD<yN#OVTcbQY}&>2@jy3>h=f=L_w9#;x0Nx<a=>O2Cu>rwowt>>(A=d{t*|)oxk#486U8?m_Z{'
            '$9l{zid(%Z1T`z4<M;?R5ycC#<9U851CSPdk$eOIb7YvAa+$S~l+(<_ZSo7Kc+o4c52NAu??svi6~ZwIs8$&MEAqOVX4Vtbd-1JT'
            'OrK3@9326joA$L>9jIlOxG!d_QTpw0q~<#dA1`k|N_cL0&%<(S*Y%B(Wml#QeG|^-Is5!ku5XJn+2vzDn=vi0H&U=onPgi6BL2EE'
            'iFz@CGi<=m63rV~XxWU>l$|=De;vx!rwW_eiQQlreLzP<?LZG~klY%vVdQM*7jCl05~I+DLT`||40NS%T&=PcnhU+aPyCO!6&4!L'
            '0lteOO9D?xqIOXd$C6|n;YbW0f9yOo7E!+~7ZRCMB!%%oP<Uf2YjujHT1~DZpioXIv-Y2&<<@EaU7v5I+HsAIW*kpDqYjhJzBUL='
            '6*Q?G1BQP^PqQ!f_b2=N!_b2nw>GXnrum}i`cQivs<Ou2fa**=;@qBssuEYaRM0K{>lMprndW%DL(|pxOH9_mV1Dj`yK6R6NSzh8'
            'BM?C^kpVyls`ZfU4f;X1M#a9JVx4YtPsjVF%y(j+QvvUf`4l`t?Bv!iTnszsF2pXjiGD%>)mM*LQ{_>%wN;i<KCH6lIcl{f>R6k%'
            '7I&;r&=PqDdaF8yI?D9#JUg1vqV~R^;ek&ml<m&;+?9yb{4YuRD-ziw8R;@A)wvO?Hx?z8gJHsDFi#7#&DRYK9ySGz27MF|eIL<F'
            'oi5j*gy+&bttJDWo0x~AEI<xZ9_Vjk%B-vc8d74Vn0YcYmW2uBfyIkl>N4gAO-BAWC+ot%p3&186KC94vHEk)*2E`7bbyOV*oy`*'
            '2v@{C(6pZ83YTedHQn5lhW>$!ql@7Y=Kry&mULK&8mvN|JNm1gr|9^Xf`&c>Pe-)h(3Xd~qJu)AB3df7vKRl4vI?dnGNu#z=^vDY'
            's9d@l9}pQVO-tjJrBT5b`rD`cA0UQ9ffmj?MXfz@IN^yo9}yo<2@t3_S>$Zx9&&eqOY&jbXsgGMhwx3Mlrd#(_09(AX+@PWN+mbe'
            't~qE+w8l@Z;v8aCzmFQqyI7-gVgVj^QsbW(y1;XNPP)=|`_<!*Z@6JC_C9&S^LqGz#!y&a(f9;0mi><xsVqH<8{IGBhL!7{u#~Cd'
            '1cRf=p_OmamK%}rG-R&Rc5~G&`<&idzm;U9UxdBpWzBf-9ugx`>Wfem|Ix=cPW-kFN|fX-K}FaUe9j)DwisxCleXmKlX(hBi})TW'
            '3aR6$S(k$Qy7XHRu_+Nc8yQrdWgxeC#UU*tbyrB0UgnthF`#^4xp?v7C1yuTK}KbAeu@(pw-Wd>aye9Zg}%i`JbU&#I>RR&`^){`'
            ';a<_#*@&n7kwc#pFM@{>mr%s>vZyfjfpm#ICd=sOf}u!H_kTAQFO_{+-J@akGj0k7<%B3Txv>u^A3(8)XC(z5Cq5GZXroUv{sDyE'
            'tb&IR3v?@(8QrJn?GJWC#2u2@z<P}<{3rBMp*Ty+0anz2+A0s&ltS%=fOs%@IDERK8MBuTHSZkMXgZIfvprHio+bDuV6;X#(zWX+'
            'PV8+AjE#W1Ya8F>8BY7!4uLlDQ^#GLE(}7ltw-UQg6$CKT*EEp*nMp&bME03^i8;--><o-NX=}2q{c(Rv@ioRb*sE_o-VrursU12'
            'a^kyEZ9sz_nNfxIE)B=K#PtmKu9AX=JALh`;Su4(ABax;%azM*?1wV{w22gf#(}sxln8yQ%{uQ?D7%jehbGgqx=#yj%XCKoMp|oK'
            ')^X{k`?PvnFjsADX#xk?tjLJk^2hYArDFN0?XK5%-Pc6PWM(M-U10m+21h|PVkKC$dum@8Hi3~hz<A7?k@za?)bZ}h(pCJ|J@wjm'
            'j_Hc6_apI*>!Ifm1Vz?N-ULv{m=TzBayD>I^;sENLtwuHR_n@6&OMt&#NBuH+{n|@9S561uPY68eQt|ULO$;!XV!CkUR+;fbNd=v'
            'IFRBg)%qOoEgWaX<<k{T_`D7Mi?DB&%1On6Y`a-%tGyx@5445afHxmSds=g3ofrc?)bYvmsjcsztg6j(>cjWmv2>dy$uCdjhmkHT'
            'bI{gsI)SM*>E#Iuoc;#z@+gt)Fcn{lCr?DMYwPZb;LESY6Y<|2ZsXs9*T49^ST<sk-GQ$ojMLR}y4?pLw%3NTGZ|GoXpQp=uX!BE'
            'm(~%T<SV)s4JPYb#`nk1p215GZO>^TF1&JIK*=w$x&mkC<5O&=g{YevdddyqNPBGAn`v(AYC}V@dIk?2z;qv`UVbb6W8c`0e|e&A'
            'L?hd#TD><c+RC()o9eo^73wPV<d?ly#MF>k<bHu`{hg{aj1rXs-nnku7bDS2p+=&IWuvGvbdM=Zl#DCM;p~pK#}@7U`hoq8Ar})D'
            '4bIzFGLnJ~KU_6R*VgH@;Ipel8OOp0)u_5yMLDr%+9xBO9wqWFrL0Ve4Qdf3YiThqHeRK<X#1&ojLXVJ8(9~MY#Z5ETOg;A;GHHk'
            'B6Q4E1lQwScm3E&hx1Ok{!-e-qG#wGcM&V3plot}(MD}0M9~3Auh2mAIM+>Fc#3tW6gPLFsfaT`Uup~aL`7{FJj?P`Ee0Yior0of'
            'iW6VLMJ!Xi5Klu}L-p3sFaNDJ*O5rr!L~5mvdFYwY0R9I$C;?Rp+lulc-Yt)<;9@yT<{_MPz|`bv9=&etA<K!xIt$2vw(VV8$B$E'
            'D{0@yg&rZljf3qgWhbOrk@ud3tE(P90<2+Yterb*H7Q3_=qzKa?BlxZvX0IrxQ2gVBphK8iVTRo>Zz}h){VLDz<X=mTfc&BOsIV_'
            'ectI!bEVEyS_tHcRbCcI)j&VOK7xZO`hb7PAv93-b>baic(IMmVZ~p^;?yZS-OZ^=GYBw7OSuyKX2(=r&z!6%u4hmx!jxWy1%AEZ'
            'xM?be?@osgpEFHS4jVdWVbCn+rzFGWcs#z?a8Ga8IToYcLBg*d+LzC4A@`)F&EIk1e7^q7`Jx4fKB++C&_?OtVUz0SBRnYb&2Mt<'
            'w$<p{bqhXHs8z9yUu(<hu9~G1U{s1KWhJy}wV8?1eZ_Q<uyw{LD76)aNUj+uh+iXheK1*qIf;)x=emm7-j&5OnPuvZO|{x+iM`B<'
            '8J+d%Junt0It`}^xj;I(Z1BFo*7+QH8+Ed5Y<G^*#;9%1AK}2WJP}d{be9Nv7vO15pgzI~HG=D7IeRISs(#K!Ya}ZC3op#!kB>cQ'
            'yfBw(d5!-D=Z}7a`XI8s>8xoJ&m^};r<x;6w-F7X-^#G=A3f?OcPV*5BC|RUkkm~b>CB^nFdPp<tb>U14FQt%R|OBeu>n_-&|}2F'
            'V+QW~_Vldr1;-t@fVk*A^sx$ux);$%u_}l!F>y4y<B{zQM>yG>_l6JX`n@9D|9YK4jnAFa=V<fMuN$z`A@8Hksiez%Mm_0T+RSI<'
            '9T<20>3lnc2ft227$btA{svI&WT$=50Pg5xgUybwq@uNyBQZ3VVEw{3FYn)XS`7OKW)&rL(dv3{IR#apT#|8kQ&HcP<-u)hqHT-9'
            '{ThhAIf>lBTNxB+GTKpLqaQ-&B7v>9nD}+Ib;UirrZeyM606V+<y<`NbDthOZtIw?dNzO>BulC8fpo>EF^UW*&OP0D#;k@k>R>9e'
            '>D!Sj%Y=T5{G%ffYYld*RLliGQ7+a(hkDm*@6iprvfSta>CO0}xnho%0SBGSVuzUc4|G8cmrhIfDo!pEnJZuY9ljXe8JFoQycaPr'
            '6s61ENI~CpullQVV?ok9Z{^frO<z>kMcDvF_;?B2JWJ9NmjZAb?=Ay6xRe#<?kIt)r^A-bm2Rf6c!TpM=f0M?%FY}bHhFL(N(Z%Z'
            '?n6pC0xCIOKBuN27V$O6UyC%Wkw%vo2ZQ%ZC==Np&-Xgc@Op-!9iw7y-T?iq;C77aQnk$ktDB_}Z1f<@%O=C?C}%MyNX6SOiY#gL'
            '2;6HjekCyYzDp3SGY~ZBPMEni3_3Sb3^zn?wc<ra?Jue)Q$RhtPFU#*FBFwgS&&5r)3`HZuV4)GxO6?a7E&fvgsDIxU9absI4e+d'
            'zv{t@GQm8!^(Q78^BmLo=69U>UBBUwPhZ8*6VudLt1r5B%=kuvK!b)OPl#bxfS@8Bslsp3uxDZm{DfS(SEez?!9Cyl>@Th?$Vpke'
            'SI7ko@l|a*)VO2uvXs#&Y{*Z$|Ba+m&tfxeZJ%jdb~0q$IB+@4h~50u)cXAVh!uAn+f@D<6lilwrGwNIpTbQ=wR{I<^-rfLI@7D-'
            'ygHs`aIjVQLAJ*tAnfvv45+QieKx2e=U8o7L0hplgRW&=F}+SaWMvKdRr@w2{Kh~0fG-vKVU|>>*8fUc3(g;+!j&J+ZDAv>IV}^n'
            '@TmL+TU!2?g37wLsMvBBPJ}gqj`HHv#fG`Kks!<h$ihg9pz<<&dwlx(#ZSjatxM`KwY=gFP}g3mqFZiVWu`2Tw8kOkRHk1#NCA`B'
            'R1ICN{9MH5va>4?7EIcCHWDbqRG~7~vZFMJ7!TcCW@6L)*bAF}z&T-eE}!hZwx}D)?_|O7LoK}aH|1yfL>1N_X3LPgF~`F;Epyvc'
            'Oiabd++9UNgEn;OivEpOJs{n&<J}3Jd>5`Jvmrsr;PF6eCzj~J`%i(D=M}I)je4~9)F0iq`MOHlHOAxj5w}E{mD+warn2UDJviG@'
            'n48>h+qWuNi$VJ;CAvn*W=m*?nqf)>@6=PNXfzNPOMEX$?Euj7hVG9Q>%DG=Yza51>#*0<o>Q0uRrvH7;tXHYJ<TBh*D2=1;+qA^'
            '>T^U5Qq$IYtR<spUfocJT)tIc)Z?Mh%A1LIOJuirxeDKP0W=#>dWLsf0N#Thb9nlb2(Dy|<~>|Xl|($mkeJ-2+<4DD(Os5!JJlK^'
            '4uIm{<`sH9FDdyWTAJCgewEfLbR<vWYOJg@$5;_>DIwFby;EdKmsxOG+$bw`d0w)>__t(?RsM)#y;0La+Gv(}HQkhRBy?1BGHrOR'
            'Y^^=exsu21oc`F$7m~%eF3civ;@}S)p`kZ%#7oPI&{@y+el|00hlx7DjE{#N(<nWA^_M_J^l~$xbkWZU7Y0km*7U0|{?Cl+HWIiU'
            'cc3?c;7u@6cebN<KOr~Wi(>Gzf64SvM&>Z3i$)u!rp2Ak<m@TpBc_M8EPF5BwmEK{jkS6?^U)Uz_s$3-lf2>gBpv@1$kIuj|Co&4'
            'Hk&tJAFwr(H&0~Nq#62&=!AcuiRS4->NHtw8BwV+Ju``Yr%V_+pQdBJZ~`+`nnb(ytfeUh#axZp_fg%D*z?ITExQ#E2O`S2U@|?C'
            '<YWU@Yz8R2;D_c-chtL<qJo$mWi~hE&4%I^TMmiis*-ru2phkE)1RIloV`1ZemZ#Z?)Vf1)IX$Is!k0xr0@90{r}%qE%(v2AoODi'
            '*^zp`cy;@5h^qE|aOc>AU{J5`b=!zw3Oi43t{a1i%96(qUNRjYGZ8keR~{7m@RjrEY(!+0pju%rWd~Nj-8fVS+tF@#O?x-EQAxdl'
            '*3&_&Q)--YQoZ}O8Dt4XIB1Dy8*JcMcB8m4Pv<v%;Z1ygXx&0dy6G!;;Rf8BPaUn}j#1X$q76c{dbl6-mc!$C8$M-f;5m+c{W%gV'
            'BWH21a}795P~D^;_QYY4#j{;x8oNg&X5M#S!HV(Q&yw3i8&Xzf@prU+5n=>*nK%xBO_g-%TErQMqXdVd-|JdsQPhfqK4glbx{+T='
            'dHU=rudY2Rv-fgMQ{FrH>Fv8W2jb_09}iAX_I_r?_jO$?DR%E>f7tcv(YFloefF5zh~Qt$u+wUy{c0jQGSaFva$I`jxGGTr4(>A='
            'IKJXpERj$Ez6S`2a4cSO2#-qkPNS!`=n^Wr$w&j0&U4xjrQl1tY6GQ+vAS~(_zy?$Q&SpY=mxrVg<->V1qlft#GVLi_%_AdYe0in'
            'DfU#JurOO2smh<%lv%{Uxn6$b;2&_Npi6<d?3f!+!oHeyCFm`;Gid41=EImq6|}H~uX~5pDJ6Z^(s&o;?4-PlaUXR&zbetxzZkyz'
            '`QYuz#9LrN?FyNsc;J|%BDjO<_u+G~1|d1yyOi^Eo*Li-wxPvJT_UhX^-&$J<CT=|K;48=6#GsW>~*(~bDn9oY;RQqIei1Tcev-4'
            '^qV&Va4UBL*1&^|*b^e0t))H!CDU(RnWVk#dP?JOx7e`5a{YSr?!j7}t?bnHzInOs++{(|>pr_F7>LQrH3g!2fEe{mcjV4t)Hk%y'
            'zwoF!hqf;stA3aqI#5wD^##z54aOLwbz#Sf$(#ER+ZWApCJO($bmJPSub{+u^FVub5$A6uf>Pa&p?fdy3EiVe);u~qLjNhgxJ}b2'
            'gmS_;_Ps&3t;^e@MeLaM-a>cqVebfsUO1($xak5mlZsxI8?-YJ{XZ!unkx'
        ),
    },
    'viral_prediction': {
        'sha256': '18870ba7aca2895774557ef7e2c3542fb89fd10476f857a2b858de31367800cd',
        'encoding': 'zlib+b85',
        'data': (
            'c-rM!+in|25`E`a^v;Wh2}#jj_+jal0VS3b0kRZG@*0L944RxK+2P!FdPXv><-hN#zRX>=vx{IKmWU!vPjyvw-A+~0p66YQQt@RZ'
            'm@0T7RUubAVh=Ls(KO3NC|89{6Sm|^L^2U9N;h)F)RyBNTkc2mSAixDYT}v9_Q{g*BxLh>uusdPT#A5~MH+K@AH-Z0A~$5S(daTQ'
            '#4=5HZ2pSzFqA~YqiE04^;+Z#j98kPMRsxVe)^!Ml{%|5&Uh{rRP=WuD@KWBHjPp`h?2=lFj+9E`H0V%$Ym^&0`DMenkEy5Ev(oI'
            'mgQ+!n&W1?*p6S1XqB&u62z6nSQOhdRQh<^GzxJ%CLVZH>a&3nFIXs6G8CFAB&K;G!coskzmw9Dd=P9aa+p75%e2@sXiaJtLK!Qw'
            '6hdX3MMA5sx58%PLFBB=qLf3IG-p}4jHKEMQfyyrWwOD+G*QfVw2G(VG0k^cz-;7s-e{etF$;n<9F&V7ASuul3}6v0NR13l$O}=x'
            'Y>vV`PZ<6G3p5;!?5m`VvpwUAC7D~wU@9!&zbqV$Zf;L+KimZWIKB9AcEe8C(}>~YH9g}^m={9#D9vag$RfTHWNVSrt6V%t@fhUd'
            'pQVI3y@}z#GdS>po|z{TRj$`^C22df<+QlwGBU4gPlF2&_h4}tjozNUJN<BR8@xZi41PVoxxKpn6kK0@Bqd+YtGB;iUA(>ea2ve4'
            'KK=Rj{OS^K=d+jX&6dP&epS7^zPh+Lzx*Zmcz*fz>LXsgn2$!IP^_!)3RYW~UqnHSIN^z3<Xk2}xL&|^v$r_NyPT8V9hnTrZ2E?+'
            '5eEhP7rTTj7TO$+AWt6^1UWu;)gNBX!n<cDCk>YOnuA=on9M|+6??;+KDj{DBu27dbJea{rDalh_cMEl@tBzDG_!Pu855BbTIW%Z'
            'V^j2VT)Vc*(+Iye@aYPE{|?(t*rQCs^l`ytQou^Xe)=;kJX6SwfXaqYzUPuR40z#<|5w&0qp_2=*aA7@8zYVTPBMFXVh5cgk`3SJ'
            '8W!;I_Yw#Y{$EA6p$>=xsr(0`oOVZ&ig?>OcOI=)8`d(467J8<1}En$5^|vnt3L-)WV3`PW@p1uIqH0+t8a8THmt|a&vbabGJ^>q'
            'e>?*K`D1hX`D|{Z21qUd#f>n~g>(K_{eBIU1&E;_&gSe#K-OnJmWl6ob(s!w8nYkRs|mQh8c*1iel<h2W%wVgC`9K^r#ENF^0VOm'
            '+3l}aZyQ*&6=kkFp{cdr)bQ{Y&(-NkccU#uQ<EXhjqM`U+gP7!>e8gSdmxK6cIjOEBJwm)d@TYM^Gx`rbx{*WU59!ml`IiXk09WR'
            '=Q-ckJHuk1i4(oc7*(@s6YHe`(oiJm9SrGGIi6$q*RB|~Om=X!1F6~ahP@EezqvG}%%ayOu6ijt3?Go7<MSmv-jsGnobC}vRvYs?'
            'MpSFj?(qj8^W2p%+ue!4c|iqe9Bl9BZ{fEz-w#wVOA)Mdu6vDzwSJ8=-Ckl+9qub1B!&~Nv$_jj-3~<iotjQ#Iy8PaStJX^YR#F?'
            'Bi#rwrJGeW_N<9XV-k^?`gUw(o8Unc?98JWwVQj^%PRycmLDW%?@!N;O~Op^mwJEt+u2O>xz@f@?CHqW(4z@Ea;G#;XuNZ|Qb!B+'
            'WP0!*ftm3%o6Qb~gJER)V(?JVX9_plejPxM-a1Jcl$9Z?+i*pbfujBz^F}NSIz9Egnp-7vM(r}KC^8*D<TDIT6$*CoBat+sfP%+='
            'fe7ome~jYPC7rStCjQ;A!M1)slNe&|7L`ncT(wmOh%4H~{m8<OJ^>ZCgDJJBvkm)A+`~G7lY7paVGxN`Gwp1)*Jb1GeuAIsJ@F4F'
            '43SCvYGsL2PVw4g<`|MSof;L~Y9?(YAGda*IOxsrU|r9epH-R^GATt*o|<7=5J{+uLYt{Qm?O|CG-0!*wzWy~Ys>D*jd_iut~A-Q'
            'M8E1XN+D6LxQaO**Fr6tb@}OD;XmW3u>#z_fgTMFbVS`Xg6O`&PWSO}V&GN|n}t69o*>XMcvafh@7#)}X!Fo|1lg*nsP)Q(z0_TY'
            'Q5?=EFHiu~9aonTr}^1>aG|r%9IB=}G%@7Kb1?|&Ivl+^nz3F#=!0paVA=lVFnL+ZDC}L41UxS!bs}Gb5Br8j{D3ZJN|bt4<efLD'
            '+lsJ--h2;yP;^F-FlrdK6N4$1>$8~B7>U1A(w~HYe;fn9lWi_k8a)8=T+PuDbD|@vX*Rm`fa&5)zv=_*Pvmr^?|+7`eW*Z?P4qSZ'
            'WH>ESw0_}&Z8Vami3!K_ajI27&s2+N&TDy@7$N<8sELN=Os;mCLUZd}VHrrI?NR!O;}AN}9DrIpK5~~TM+`l)hQ#jO$&rRxv}h?<'
            '9ChwyYT#@{m$53m)j`A2U~mR0HHN)qo%8@0zj3Y8zxA<=DE<uUS77J&)p>K$%Yv~s;*dro#~1xPPc~FOpFF(Hsl1qQ@!(GTaB!_`'
            'Un&V`kN+AqYgC8yGEVGuW3_zO-gDocy=pM3HN3Ox-M5;C9v|vy#xto>z<3OB84t3eqkW4JMz}U3JIA8OGIxu*b?92AQS8`XPMDYO'
            'y!tea%?_T|-bSoqU;v_4vVFNjObNo2<Gnz?q2w1dhy(TVP+zl9Aax?-n<Dd-*s+#mD~<V2JtrU7-_f#mV(Y*Z*ydx;8m5}?EPl`>'
            'NUB5<m}H%}mKFBZPM~^}tYxT2;fO9vb&EH#(Pv{Y!7s1hH8z39dB$2|DB9c2pKPV!$xzzWy@g?6Y)&1E*We%XM6OYIOi47PCr3{`'
            '4Gy}*6loUmn3JLC!e4%=JRC|R5cd0qH~(Ap{T7T22EucWW<tr0Opx3>LT?!I;Ycw-ftJndh#CrFy7xcQR*0mLu8o9_jEC~*P_wWX'
            'JLryTTQt72IQi}ow?EWQdRSbSj8@Fm^6r{_R6RI!GoZ7_by)Y!{#zqR#cYg3>vaY#I}WC^Lr*0N$#&(c+`%^;&dxKhO@D_nle0~k'
            'mr+qhB}*f|l#$H6)^N~0P-jMHrnj9Y`$b?U_9qDYCE~?QyIqny-GATde!DXs83|zB+tZSL-`<2Q5P3CRG(AxD#P?sls_Z12D_9yL'
            '(P=cMg3rE{HnqCiV@%k5Y<|40+-$2Imios&^`Sd{bt|KuJj&P`M&UQ~4bR++5589NS-Mx2%%i)GfJg^V-7YcRZ>(B7`ZZFzuVQ!b'
            'Tiv$#Pc6S4##P@}Vs=AK5iAraNeJCg=IB7>YcvyzzCKO(Ub3Yibd>uvuN7$5gl7+6d#tbO?bSdS$$cmpRqL%>Em8ccGdPV4zD2-G'
            'R-)6?c<uH21DBwcQ14S(-j++jZnkM&pe2enX)cSHE2^hENv8)e>>{0>bg}WEWDLPrKzU@lG$|SS+LEn#4l1Gpl%R7BMFiXNt&A8{'
            'iDD6x^{tu4-&Sl{<{TNiD<f22i62(y$8mj0Pha%zgre`1p^-ZcAu&7s?VQzahx!sOeBD5S>hPxtkM;#!N&!pKF9?vIZ?BKZj0tIX'
            'yn&(fx=P7Cj&}O#@|@kAeZqY7%lXCI^J{mq*3etaGOk75^@3s2C9?ka7VUUw&=I7fv<5(G8vA=$)h4w)CJ5_?21OTLgJO%JjiQQ1'
            '@6OA{zJdO-p-W@+xjoe}pTXhwt%KKVVz9G6cMsB6!)1F*wcOY>Jg-u{9DLEVqb?yNq19#91W^<RH2R^A=7G)-H`%n9*h`Nm&t<r`'
            '7-(gQo129i`#T6x$6M9pHVxFS>DoB$rKa|Ew$go1{nVh@U{atjmGr=}sQyg@MyW^N{j-KyS*%)9jM;w}{ToA|miq'
        ),
    },
    'approval': {
        'sha256': '8e6a6e6d92d0b51fe6ce55d06c96ec76680530d63e2335bf2d6961293a8f8357',
        'encoding': 'zlib+b85',
        'data': (
            'c-p;NTXWmSl78o}m;*lutfF{ocPpt=bm>$n5}k0PNM1=!)~j-nATT5mf&dp6%d|@WeZTIW0R|u`Ih)=1!IA(>qo*%l-v*xNT?A>C'
            '#$k}CZB~@0OP%UGDB~<ut7;RZr@?lcXP<F<8I*e1>;3lAM|J<x<WBu^bNS(Ftj7O2xx2sXogVzxJGZMhRZ*(BR)M-3zdye{zaOij'
            ')LT`qf>N#U5}g+6{N4Td7MHZD3Z3I}p@Oiif+X3gLhH0gw+x3mQbAtE3p_AX?}MU<)1{gfVIFVGsW7dhv-`Z#GZm+0#$*>&8uH%g'
            'wLMFz;Cr%EZFG>z1xsGV1$YfMTfEY<lsyoYWx+-(h%f2!R>E8}Z*i$kPqE=emV;8d6KSe2tC9#C`>Yinq{oKpp4Q<VX;$J%Wd+_N'
            'fe(lf?gc8Z=5xq|8G#z^WJ8X+bXw*+C=r`2h&d~Z=8@rcHyfx~Sr+sg-WBy`xlSXUPpdLc<Z7goc+O&h(1S44kjeE;%@w4wtTuQT'
            'Eyuo7m8E!4f7W?VZ@?@levb9yG}r%K#kr1V%Fl8&TLf`JfBap|;<Tt1i#UuyWf}!#fUCV(1QJQMCDPMv5`=n{CE#q<hj#O9lZo0g'
            'HM1I+)n=o=gh^GzpNZp@&h?;2yfI&oD=lgw4Uzb&?KX*x#_xVQKmFJ5zE|0T|I}Yd*<qHJv^{J^cA_JQ`-jQ(<>dNDb^iYS?ai;}'
            'SL*&oUEEyXkFW2Kn7-Km7;@P)!pn7-<x$aWVrpJDEmi?ofqICOC|+u~i{Z<%(VOqcR()frrT$VvoIw@EnOZ=)_~UaNY54<Un}sVK'
            'uBZ0pet!m1r;sTmRfx5k7cC&*mdT$KDy;IH<X{*as3OC1YfSL9z<A7{2da6MrgZWC{%)Y4Z@R>mMHh*(a28Z$w!!j{T{=#|RvgtH'
            'xTT$DRne>E1yd=>O(2N^n@UTp3WGe~5m{&m-t`#dq@e4pT+v0+zpA!L7DUBBZL4__7b{&bQ?wOqj=U+=<ztqwfhzguI0UYFp4VeB'
            'Os9*gta3e_s(7=_ax!#yCOc5k>zQkSrWO3e{`q^6rLvrMQ^tS<bI1M-#p3_3S*qo^ZBP;c``CN@+3OkW`r&Fk{TT*#r_Pk`K@_28'
            'cq6|^y@B#X@buf~4BW$za{5Cg?)7`UHvkoqr(LONrQpcr_+oN5xw)=&JrLjl@I`S@Umd}90=mq&1%hIQ1rcEr)nQzaOU7lPlZEm('
            'S)|EGsyxKcO-7DfusLVs0qHWq<*X{VRS9aqL9!{~m=v>!mwmP|(5rorq|7f2)^KCmrkc@C9z6Sc@6&7ua^hx-SOZ)Ih8btW72d05'
            'Cv0Hx7U-6O`wS+>At`|~2$bWQd6I=|;5H}%vc;glask^lI2>t_J*^^;VL~Lo#+F?SANHWoi}BqZuw(lE>il|Y#XtRd{1NJHHbP2U'
            '!hJ+bi(Eroc{T^`h04K3H4h5j17I;r;&3-2&IpH7!eB%kV&0$&89jcBDh_&6ia0w`w}KCrc-W}eVB;1+R8!~}_O8djj&G-qInpna'
            '%keFl<de5V2=ShY<8!ORO|TQhr5JBt&*)1E1x!wE$MDOd1QQ!lOf2?j9wjA8t{~7Na51mq#Gn&4Sfw>#1A(N0z?IIa6FES!_73&P'
            '_K*Y>Gh42*Lc8?h6lPgSo`5$wQNH#Pkb7Lk;Y!W1gLNlEAcM_06A?>HDJftmvUv}%tswQCUtcOpO7JdrDvrMJuy_F=e(<c}c%L3T'
            '5RGDn9q-dy?DLli+mSJI_8<tbQ161I0D|3)|KkIMd^x=xUqe$<gs02NJxq;Zq^I!DD|NWyWN&J$mCS<m9(e<Kq?19}%e5$9xI^xW'
            'y_ASM#nfU{4{&sIJNa>PeSS4P|8P0E`I8I^v`pazTFlyE2=D4edVzE#NGkZ^)CibTDZhcXC=ScMI{hdAG7_?$H#Un3&SrL`vRVZc'
            'X^gRofeK*+9$I{>5iyX6Je^eOMwcfe^*|O-d>W{eMvJ)oG#n0}aZ$3NY3tqt?tOY5GQo4CbDM3sQ-&a5swge=-dtaOq&%=>;1Z(@'
            'Tt9SN&@lJ|qNBlPm<-I)f)$y5q_?z|f-52>yT<|37BMMxYLgeBTf_g%+&on}06RAMQ;Q^6);_=q79fJ`-%xw)pTCJEfSvtQ?0jHw'
            'u5r=+0mk;^J0sQ(`67$Cp&$Wj*cjhXjE~ooTsU6TR*H=lRl<Snl(Q{tncV&xcZW=B#qtK`lsUNLfI}!Mc^@T<Mu^r(`6nC2h>hX_'
            'i24cs^YBSD-EdA$RaZz$UHYDE&g)ZHczW*3#b$xM!F1w<LrR`BZrHcA3aN2a@058Oaz-BcFZyv`IJL@}O4L15ZDwmEbh&C2O3$|l'
            '$SMaj-E}tbPbo{;7mVZ|9l^BLP<oXj@g$c4sD#1R*qycXX^S<#7RzhUS3d}Wceq4$kmd0*PO(MK@~1^LFXJ-Ng;y`r@CkJs`uREG'
            'ln{*L5<ues*CUsc<}%BImbKK*k(Y@{?qe*nW%ezDU<#u5J>NW{%Dv}|#K9iFw4mZ3NJH)KJvm_4=r{blQ5T%WOQxA`j4KPK3eqAP'
            'P;OFEL}k`3(gbdn;>c*9!Knc&?^~tv0twkGX}4MLYH)95`!yt_m(}x<igJ!;>ZxAf@sr-jNT#{&Xh*`T-(2ZC&+RL7w7&)C`2z;('
            '3E=q$Zn<x2yX~m$RPx-j$r7aIo>C$0(|;pG)tvJJ{H)W0cU6)oAITPK+>rdgUpEd``ioPm2eb^jL-G9K{@v+66xZuR3))7grg9w0'
            '%YA+^;g09^hdK?xr0-Sb;`AS0zdytagudTz^Bmf)-IPjCKb!x)$G6knpf1h)cg=PE#rgHk_2lCGO8xsCsz81vg&n^@9g1MH*1Ljh'
            'gkp<Sy6Bq>qG=NrnG{P1d8nRa!24HQ<Z~~qt_*|ZG1wJqN9k%&-)Cw@Xt`dZdIz2r#3kOWDB$HJ4x}{@9<e}?H={WD7#Gw(<We@*'
            'RN?DL5QPixxLBLAj|@71M5X+rBtDH=1sdcOC&?*aaB4L-#EL2+U=s;=7*(5X(V#k{Ilv<Y^$CI^jN>ztoRAQwBtq^y;%~hT5Hzy9'
            'IP<*$MUjz*Iv1J@kT;w`Ut*Gbx_`lB3~Mh~2Q`3v>M;BnV<sZ7m$ng`2hO@Y3BQjq(##BK!Y`+e#2i5t018Dw&21zLS5>+uaDt+9'
            'KM6MTC>WV1hg7Qi-+%YrzkLsB&EXqu@DetL)t2<>Go_Zq%>8`<*>)S3D&WFjBmZT;AE*QMfh#(j<R9o9+J1%UmT*w{vW36FfXd9Y'
            'PH59it6WjxYx8Snz-+G5>fx9K@^#BbGNqSWEd$$4F*Hr7T>mpP-6o>6aLn^f&D?t4>AG{n@!jNVEVg0Rnfjyqo&}{Zrkr=yNZAy`'
            ')NuF}WtSf3IeWcn*EZz`lCGG0EJlpMJZS{J)1~tDa4EhUkN^?PPdSA)I|PKRgQX@jLfTdVLP{>oC7a<E4;B3q;oBI1AWGV?LA2YQ'
            'YfLhyjX3-87v4NZ(Anr6_Jh5^-PaIyl~<?%iRRz(XXVfJI)PYIT<g+)?%OokGC}EJw-wlMP=jtKjee>9)OWL)PD*9z4U8{P=7A^d'
            '<+eBOZGSJ(=yE<JP*rhA#iu&4R8PpjmzCOO>$fO_f*qa=!eFiDG0=RUXj~*30cQ<_&0_^D2C8o~GN@tu;k}L87fPbZi9-L+ZppBb'
            'Mu9pZiYLJS(C|aML8yect*L(pm*ATST6!5LQJf!T*x=dWsBgD%z*0>2rP-8e>#)CfOD*S~w<jy-<{uW3^k|T(36Yl8U521=<R6IV'
            'p*AiIw_QV>{kMno-Zc#XRvxdv$pG5&<jl`u<bWWX279x6DaI(o?N_b!t$j&4%Zxiw*WYh0_lI#IvHLY^TtKuie&oM7Wn5XYgH@?{'
            'l~l;$gD3`~G-QD_-Z2UA+Z+x|6M^8`Ztz>02#oU6W?8#bbI7?HZ!RC)_<>EE7;|vH4Yg*!2lnc`)T9K`(w}Qe2c{BZUdaktT|=7|'
            ')I$ayU8h<{JcX5nwoU>DxE3Z=WHV-Ni7_djb>PxY+EhARWxxuAdnY=nYpzH)Rawu>YUR{HD_vyNq;D!B+6)RI`r*_`cHxvdi4|2>'
            '9Z-<V{dTeE%qp?s8nD(X+H8QvJxPe0o1D9l8+BLl4MNdYsrT^7)@qzG<vXW8fgL_{nV}xo!Q`&dV827bVV73dT}l0ghu%(F67Y<<'
            '%YoeyErzN1cx2Db=*gErMMfAc!E=w9=Xhj4$0sqPm$$?6e}UsRTmw*oWfJTF&g$f;#n6dm=%oLAqPAHA9jyZ-gs9$Ph$ac@SiK3p'
            'cJSKskIeP^4n+6&1Y{<eZUg7O9wJkVn+rJV<sz&3IKJe9!lc=L+cKzAjw-a!GQtiVlBPJ*%A2T19$V0X49_%Pd8mdrtjQQY$AP(Q'
            'M+)*b$35NvkNDUC`!Atf(`u%)*|w25fkk<n=w`@R560rcq1Ozkn#L1NGN8Q|2OO=)Lw6ob|K?MtN%XqibZn>Nru01bWLhujA6tFU'
            '^4Ln=QHxaFT%Uhb<Le*CS636N`q?B?sGCojK<n$QK>dm=YjhwjSa?q-19dX|J&RLe$P7<D^`D38Rze4nfzhU%I#j5gkgf$WrRJMM'
            'nFbQJ6q&_mt#%diyZAI%qi(3#P$f)$10Ex4(}EOS8hWMN|8PpzLIsXKM#EF9yUvtqpqx--4}x?KjmX_2BZvB+$$V$1z%q5gztjBL'
            '+mC2++(y$W6~oJnf{+A+vlC14gbz5u<ar(yOZ_-I!sAStx!^0uOl;-eo~s3b!QYrk<<oi|*V=xxM8l{*JWx+k7b@Ku(J!hZNCX5-'
            '@3A?U;||aj5=c35z(YS(R3Rn$iz=Zs-mpe+#38#r4+VBm`$yF5*-6JqHv0&`({e6r<Q_$0n4ptjOfShY8XlP(!a$kiTFRFh-!|u?'
            'UCjZZKB*;}mpn@^hw7Ykln6a=zN|FOTk2C3x&EEAakGPWc!Z4W>_jl+gef1M)HAO^B!16lcbpj`Q}YY`(i{TX`S$UC&*@vR298Yl'
            'on8fR4+^F;4^^4R%T*~8yFjw5_Wb`3P5%GNo<KWqc769aUC((;ea-SJO+ZnAZXIvHBm_hNg$i{SQQ!_YI*EtvcOAo)-nW%|-jVf|'
            'p`p4-Lvq|ubIrc0LPW+mjmaC}ZZVYIO&yk5Q8J#Ub-xYuG94+S)IUd9P&*_Xj6wq`0<lRv5b8Bp*e91q)v`*flm!vD%WhNoyipfs'
            'jQhj*-3=U}4$(YllQe^Ds2E5iTw7q_c}vkr>fgw~#GR)xO4TRT5)A+-99TGUT7wmgvZAbH_<Dl$f0-A2LuJpQ8lHYRzMNcLPDZx<'
            'CuT<_J|~FlB(9@n9nPbebap6!y8+)WgLAZkIb!*NKyAXi1a@w`#La0hM0r;G;b?xe!EhkH+eB9){Gp|H(#43sBD~Nd@j6{@^jeXf'
            '@3Oq$`Ne)N>G#!|ww&&z4l?_eNXgaBHxxT$e7OWfadEcdCfTzlb;NOpG|Srh{b7<l>Kv^7?X1Otz5cBLvDh~m0Rxt;nsXbBecg0('
            '=p`qy@}G_{_%QmH?_mtXVZWn>g=fJPMtJw({QBpK1&191vH#?5!=XYAm108*8mfu#ARI>VAvh_rGzS3b6rn7&y(gGcmWXw{Jp^{J'
            'z#N`nF^^3V6zpdA)ZhPquO0ue*8WVL*b$*yHXf@JNuD`;dJU0vipHLs-@+q!{M^Pmd-uV%GpPP(V*J5p=$K?dCrk0@!jpkc{gz-s'
            '=O#^hhIsqr1mEW_JzhI)j`X$LGBp<?Zaob4?|>$?WtzVZJrmGa-kvfZrrD!!4;l}vGNjI026RC7?YpfUE&{NSe^QdwY04ZqpN<u-'
            '_qZ(@Q`*!yoSC<^kDwndZVvU;>{0c-lA%>9$ae_Y2BkbfU8&5c9&*>PW)4yXWs~u<xVp#p>e>c{Bu|Iw7qN$vXX`Ze$0b}UxA7ml'
            'FsDYW#rV#YshL6dyR>9n%(wY>p;dEh`W+CLNfz2=(@WG0m!M_^XXfInAcU&l@s<dG+rhQwe14(s?$58rtOU33!Kc~C3XWGAb&1gw'
            'Va<^sI-_`gb$3Ij3~WvlQOYGQ)|{|b%Z$$~%PgPghcpj#XPhp#Nnre!tF6;hY50BSsbf|xSI)Nz^0fv2-V{0^GZe)d;kBy*IR?mQ'
            'qaC+4X9DDio%6Q=m=TuOhwTnGUn%N36rJuOz3;(mK~-H-Fs{ZjTtZ<cb$bV_<?2ZvMMvN3b}fZ`ujjt1=AzXrtzBzxPS63r_PpR2'
            'ymU$uq5a~vJ7vgzlLFC29eJJ90Wxl!SJGnYgeXZIw3?gD6cK(v6%-19_$y-J4naSlVybs^*5nGa)yK9z{Edz4*W1zut)i%cbRBoH'
            's+BH+t%P14o`B5ri&j8q0)<^SA+#DgkPN=2DU(_JIKDtSGWoBOWssB*!M`QFA`>n)drA7j^KI^so;_JE>Rb*u!J9s_sa})i4b?}J'
            'v6<`>Wv3zaNA&nc{pw&um0Cmvr`I?4s`<Xnw2%b%Ms3{V)`}s=*jYEVn{)%P+8j1^&h|Rb)2Z6Ox<GtQyG@PQC|FQcG^!<2E}-GS'
            'EghZKD)F>_sY;|wz1t-|UNjyX2)}}Q0^J?R+uJ;6vdZ-|0~s-8_zV|G+&Tql-^EGN!D6tRD!thJZSLAO?+bza)dujb`AV^Lz5r9d'
            'o!?*lG`^J6OTy|v{WiJ(>E^?|^JPR_)(uiV1%*1Nu$>ZaY8av>g_ch^#d8SrlMHrUp8~S3BK5ERW=PE58IGt4*5q_vkh-;Ys55KH'
            'b7l@t(MPXxV8;2(mA~R5$K`&Dnn@IyQOKIL;~0Mj5Y*Emd4+Ts`I{n1IsV=>CMl6|KV0Hr)ZRC;NBiAxZ4K|he%$^nRQkL1hiKeq'
            'uTwd3Is=0VeWJw&CHH!otrp<R?MCOZrrl_3-MZ(v3%2Fap5tq+42xdf5cqPhQq?oSzo45jxxN@*jei-dpU*#DPFfYR)63uu`cE_+'
            'KEhf5g+7J>6w%?)z6A{uo!-S>g<RXo!3x>leehcD(k)i)WBbLbV-DWPwV^}L{O^j@BceN4jJ{UgKAV6mE>3-athYmrPQ$gvYV-ZI'
            'pfw#lo&rXuT-%zHuI)m$thR}M5ZtE!Kh@BBk)<d$kQsz}#FWU{`5<=P^=(cwQ}h=eWGd`8pX_$(YBGerOtzxzF*OOs7Glvn+LGA_'
            'm&WpqgN#^cTI77$Ur2^gA0^p8&g%{81LJ4*r1GKjz4d0y@oCzrFAuc_p+`5=+#t#Jd;bGEZ&}L'
        ),
    },
    'google_ai': {
        'sha256': '6a2cc8b2916cf3485b8fb85ca1b4b4ad217c00dec1f962cecdc219a2c7bed49e',
        'encoding': 'zlib+b85',
        'data': (
            'c-ri}d4JnDmMHqaKLwg~-&m}YWM`>#x6|r+DqBfu;w59tRbB1!P?SX3Y)Pb=r0ghe-_QQe0)PM)SxF`L&D`JAZ>A%Y00<l$oDFAj'
            '9OuxF;wT*Z^QOrBbk_3k{Ui`?<M?JCh;Cmr-Uf>>3dNp)9}C|VNxaH}L`>sEWVc}|mP!1NV4Ssgc7FNkS26ghe<4oJjxLXTqW8D{'
            '#o%J6Rr%k}j{yu3@ED7DvVzYAcR{iiQJe*%IG%~oDx7B`%u*5GM`CyrM1I&loQFY_HQMdAI~0C2*}3(T$vB<_6ETEgf_Z26hGyJ)'
            '(SFsM&i(XucPPSW9!5c13~v2QEc{6T!-{aRjMFq6!I!gGG+=GBD8AphzYWK?g2qZ=dV!cGL7<i?#(pIHakld3^R-9<e|*~%jS+MW'
            'f~9WV%EBPoIe&k$H-P^?e}4pPncM^!yo?uO6ok<YJV_#$^k^+;#Bn%*b@_8K^`jPmxQUAsjj;<4!Z|EJ_&duan1tiZhZ)1`)g+{!'
            ';@)55%5pe{Jnys9o><QPF#_4o4t92gc-9tUm~<AT!WUx#rD&u9Jj+^PGzpeLgwtpW^>e(2=eY1-;zIxDZQKhUo*)DuxCzq?w(UDH'
            'j-zRKvw}(HKoI|2#hDMoP5sq8OJRr?Z7~VcaSVH%0yLNYD4d7TG!=KDFNQp+HiN%Chp`*H(EtL#JD6)mpI?akFuR5E;6DIh@L@R*'
            '$FQleFY!HggI6!xVi}HSxYkJkoe?e4EF3cqizQ4CaUNQJ-=+=3mhd!%Rs@CK#h`!E9rTXmGUfs7Vgl`7wPE2|g0tQeBOkvJPp9x3'
            'p3x}dFd464IwOGgjJo`>E$;kzI3eHz#8!Yr2r8J$a+Qg64ZA<i@FW6abx)5(8Z7)sK7jszYKwV%lgdK^=y(ShlL$Y{g2ggJ=)|-w'
            'ev*Y#7zw)kxh+yZ0^oiORKFle;NXt<?BN-}1B{S`i6naMlO6GFzl}hRQ$m$ASo#Ufyr}^VlZ^ZYbQ{M&7GQf2nv!hjU0v_&oXXP-'
            '$hk^0Jlp`e7972Kut4ndqcwnzXFdX?KuiwSbT*UBqz*t}e(s(3-}GTOx^KJv(~E&Pd-JA$*zX>Tw`XT>f#LwYIyyVNJn5Ydx`Y1N'
            'DdKh@&U@XXlb&$i`RmA^iAj8utmgj0cibA<^!fkow8RM=i~doHkO}z2-)7k|J=og=`gKDfYEOc@JxQnhWw<wq$LSv9JPc7d1VH!#'
            '1il9c4DqtTbPh;hQ^-fD{tHfEhlpbn5TGznuowjsJ&GowwTF(#a{%MO^j5fAu-!cW77=5St`b1C0FQNJ%Qc2KxD>YyAEn6-fgG;^'
            'A@StReIRp4;b1(#MxX}SdKskcZx3bDc!Z?rwZ$}+eu_AH)IS_th%`o|PBVX<N)$p0xg%(qi+DU^f(euZjvXQ0#nC%BJJ4eehr5ly'
            'hry#6P-t%sM`k`m+8iu_=s38Ng+F)1zk~xshW>>U5Uv*V^ESM>bwmTsL5l$C;tp&F74=E@{NdU8dGC;r_bzSI6cy_k0lk?Xzy#sI'
            'BSHZA7&hV0c<C+Gj~O#X+<N$zs10oRD#~~}PtOK${1$M|CXAUle)oNU55UkCXVWPhZQwySE9e;k;{Y-NV@VO<1u>QwsGxrbn32x{'
            'h!C-9N#%jd?Ar_pzLF5hfx%Kd`)hI9gVQW7dZ$MhOwLiDP#<c4Q2dt;@X$k1VB(nv0r7z#%<LNYa0uJQoQVLnn-a$W5~H>o5e0}N'
            '@LnI7>O2jQ%O%0s&%!urZwGt1bI;q~vux1<Di;h9FbLW?5l{FKFl(Wx;YHvlFvA;QI<O<~b!MDf9vFGvQW}I7%s4K;2nPnI$%6-z'
            'RuXVaW^tdkC?XFy!`Aa59QlC1kgk?^ju54gS$cRRhp&lo(!oJS2JlOPGN{?W_{fs7;6t`OWzb(wUE%@!1=ycy_yJZ61Ho3s2?~1h'
            'Gm6vEGy<Pa?;9<6n9^6Yg26%DP)A_78o^fG=0o8OUh7|C83XD7s>4TtrCeknYQ;oCfKGtmFb@}q?l_v`FM%Hh%>E=?o5bbt0h}z5'
            'E7lJ6MIj6ZbZ|l>PNV{NE5p;X2qzJck4(Jx#~~a-k%f!E*&cX!Dq1`(fL;$-J<^mLSRZa22;{wE@4WXG1oHD=hf+Y~BMFD$@a*(W'
            '|Lx^@_ebENN4+=Q%i{qo0wo3#PFHY#65!^*g6Y^Eh}Ci)`;)Y(A&Eaxj$mN{<;f=en*cE(;Uj+tr#y)fis{ZM$nJw6Qs)HdEE1}8'
            'PlAzX%_Ihi3%V1)+B=Tp>=51Zyy*%_kLM{-!H*&w7g2F%M?M?*Y4GZ$`s)_X>pUE(KmU=&k@_8{>Q@3=6)*H(f%?UWuKvR9;fWxH'
            '#lO@vCL|@Pr%h@J$364;=f^mb9hZKFvr+xt!=F1l--s_MjC>=00_6y>86>u~NP!NHVW;SHr2(NY%q$!cO$Cz7#JWXn4@C(;cGwQW'
            'W*BO^is+z2zZ((;;7uA6R*UI8TrLsaV0DOGKFIQ{Sf%kQ0X{Om^`jfsLrA3f2K5HO{uBpHK~N4zu>xY>{=zWW`Q|{J6JnswWn*0T'
            '@bK*Nba1iv{`~CUdni8u#>q2TQej|O^aBSZ>3(qsKtRqR`b3UF0br$V$c)B|Vo&0<M?JknA}9)1aQIdzw*ZlW0h3iE!YK;F2tfoV'
            'aIdT_4iRoRb2`BnNa;Z8Cos=J@dAuOdZg%i1GqH|Huq6M=k3P6uqaS~!s!%FJdib5?9ct9-nomDo`PD;+Eogily^s|_{aj<JKf{s'
            'vtK+3D(^oo&jwxZ{7lN%9q|SkDM2|Wz3FPS2x)IqCEdYVN!OK-hbaNxgAPNoK}e)?vcfY@f||d^0ihdcPt@&p1v&=&br%NrNXVZi'
            'i>U5tT8N`*Zea?K_dwPHR6`Te<>(cldk82C{YJ1`i}uc&^Ip#z^n2&tQ5O~u45WY9gQYh}`*ftxg0y~xG%gK4ikY9mgb#ox0#{DL'
            'Ip7?OnS?VKRsh+`=+PRLv$?PJCWi;iK-GeM0SOG6uA({g6E^|M7Gbgofa{KdYt2^kIdq?sqLtvVfE&=`CjJO&yBg{y08|Zd2w{vG'
            '&sLG_*LV_u1Nib(jd$zM<{=Q$g#fCy_Sa!?$-Z3jB3Oq}+u3pX0Q&c$F^-q(1eshgaTQ%a9X8(ciNB&mLqZ`l7seMUQoaV1pVKzZ'
            'W7d!ntSH51K(Ny>Xx%910qMR+jXVWGfmHKFkoiE1eIj-#>Npag&%3{PC%vP7*BhL9#s)L<%klaG5dHGP`?-63*}H(p4-5y;g5moM'
            've-CH;Y^Tb#!GzQk;&yYp2H0B6(g;e1pi#YR!p2Gj{tH5qW=V&z|O5m^v3QuU3`dUAAT**1{^03&%+vqc!Zf9{?t3^dH}EsK8eqE'
            'c6P=flco}#y8ZV=_<N}GH_m|!!G!4#_kic%IOl$d2lM_GP{tRxC2oP6au>!(Q6_C<zB@axBjRZSCt80Mr$DCkm)^i~QrNxK1U3&z'
            '9e3OV@kpp{;@)RFg#hhxMEV$YKP2T$Na_S?4q6O`!G~ZBbjoA&^@wNzWhJf@#knnFJ}2G+)C3KGG$2~j$h=5PN7MjS9f@kjMmaqp'
            ')C=SSxZ_l8qKCX%{4YVkR&IoDLgQ{4tushg0g%Bk!4p08KhAx%Dq!GnfLT&XLwP#&pm5StJn2A8{aZNqK*^&CP@M#b2S}fLQZ!~c'
            '7Rd{cFF#%QGciL>R~Rk4oul1~iOgul+y<D?+E4u%kfb*7SK=3Ie;$P&$G+(PdfMk^M5`u2aY*83aD;D&rUgWb+Ibrw8ed!zx*+)z'
            'grA@N`6^)H{XT(H2-{6!4B+9IWJtsmX%lOrR=PkF+XN+77zTyLmKJQ;d@F6YUqf=~eL#+N2nTQ^S3p)bR&#-;f_y$FngH@Cl01;Y'
            '=c|+~lJ{_!TVqhf0F%)Mf-G<zf<O$7Lgb}sJYO-A1AZm#or~V#d2ir#kNe#Vlh(=e1w6qy5Dm%h-hoyFq>1fr9`HWzUGG=NmBeig'
            '2(cOkM9Xx?U-==ngPy;hT@EgP>@iC%_6K$LpZu}&v;WX<o9gu#$q1<5ApR@*6Hci8;qlq!(VOG$c~76I{!w*DP-?s^o&`~<<K^Hd'
            ';0o`0r+G*JBkCIIG6=@E9%^*hr~CIyXnyg&*E{^lo}kqnE#z@j?t$vq8Gi%Fx#T<^?7>m*bkOe}dmyO2=^vw{;Yc><?5Ky{Y4@ZD'
            'L%_=<XyY~mw`e5Je_nN4fA?D-_gjCy&VPFC*7d`F^VN&T|K-5pIy^h>{^-g5{;7BTzHYPiiF6nR0cAjMi2~qlvrto)hCEv?+IDV$'
            '9w8wOXF#oh3G99+ehq-W+=L(JCZTd<?_F%vog9c=jaa*)aTlz85v2H$3B0keY}ajyqu?e;4$R?lfiNX?T}d?nnmt$zvKfDp;N*~)'
            '`twP6gIs_o(KbigRVHfz#~fA^?y6=a<{h4Ne+5Y|T7kF&=N;spC7i2u>eG2djBnor8E`;jNCps$`v%Q4#0U1&SGwad=t1-hWll{9'
            'H=U*8zQ_&Gpv;e236@po4k2pHRst6_z!)KjhsFRWeg!@-fDecxl&w(my?dd7^dvCHAO^?t1wbud>Z2B34nasl#mb+b2U9IRbeUkG'
            '#K)EdVB9fq*gK@=iodgxvC5R`LSL$Z$dTU0_mPw*hDoqYKvdWpf>uxeGMQ(+3}pZkH;Krq3lwSlX-ZR0AZ^hs6_C;({{7<YRPzSD'
            'q)Sot8tWA)4l@t@8Q<8Tc||HFO)?q^yyX#iTS>MOy4phfGZJbb{HafBGzdF4w@A3d$WPXaOM$YNi!1cQ^U`^orJjfCiZcV`N7_q#'
            'z(Z*1zk<}m>ntU2PJx=>Pmkb>Gy_LN(a-bX6l9*)NP{`(yEx<l8b_KUAC(@6e~|##6xR4n-m#`7572+|tK7FE>gl1IOIbLFHckOL'
            ';6L_@d_wR#|D&RjnjN&uK>G+9)>GH+AtwrL)qS3q@31G|HsrsBz6?+93je|eBa9fj|1Wd_ts+}%Smzh21Z@(k;VjJ!F_W<sA2y5+'
            'xOF4a+kCKmoAMz6chLHpfXc=vtu4;*Fbt&)o*4w+7~z3|gS7%VnlyFPpW8}SHs)tcEypA}Pu5&Y>vD1kI4FD5IL0+|fJS_WYz@(k'
            'lMkK69vLRb9^NM9-@w1Dm0qaR-PFghmHJaOjImNl_MO8RhzKZY;`B^WAGEvRL9bfvgDvdRsXNGV)y92?z@XkUOx!{fbs!Hc1S~W%'
            '4)sA1|Ern;c1zPf^`J|?@`*5}@@NtPq@a=r=-m_I8#u-O&B8xG1&prvMGLRQ?*LLoC^e?eUo2WrC%E&V;q=HDBA-aeJQh?5{BNoN'
            'B%8{=9{J46hdD4F|ElO2P{%AG+Nc&}E|H0<rgR|9jX!g##BK;wQh)ZOJm)bsc_1(VIyF$VC;BHzoZ>}kJBaSW1W*wr3v8tZ+JXY{'
            'fYu2Eoc&q5BKT~>AETWYRU1NYMdLuY2<FUJi6i1A-NKJnz{+4f0U0Iq5m2-ZnW`E9+B=R5y9bA<;VK;(d0<hJZBJKo5DK#KZG&l8'
            'QJE&itxAVB_^bNtv(~GZAekkgpu)zmuz~8UR#9*L1ojl(`-`U7ZSTVWc)Nl`rQOyp8S56oJT!GuV@D(<qb-nfi2}Ra(uTnp&5h81'
            'r3XZfDi~q@pM@PAB-6By)e_P<&m#qS0jQXgUU)JfnuBN+-k@y*gbuWH@GhcYMEi(0SUfQ)BhmG(w)wL_#aD}nGq8cTeBnjlMwy74'
            '$h#fkPZ99)-i8Jw&y%(os`cZ=|1y=xXCqvyCYk0cuM~zzSCu@RST&l0D%4#ib%0nnkJ*Ht$;*lhz_)y{rgcYy;uBcA#Ac9f>AqyF'
            'UxS*B8|>o5^}y6L%5gyNzJKSVEtkZKlxT7TkI~GfBZtDG&;#R$509=}e?F!TnpC1ium)YHd^`@sgPPGFlE>E|SKf_R89ddqcyxGl'
            'gV%QStglq(Yigq)-9ZztYTk4aG6qzPS4@zIceAvWzq4Sf2Bw`~Hws|(<v8$`8Y>z@0Q1Sj$Ne`ElBr`tLas1m3p=WbAJO_IZUP_&'
            '@K~awKS6tZ63l=;@Pzq<;?|ib7N=HC>8TUV6Q^j*HJ@shtPd>Ops1_l71SX|0ag-3Pw5KPeM#6<4?`&q+D#>evuF$vGbzY7b1M={'
            '@(L|F3Xm41tQyBs3)G+GQEc*BCaa%E9qH;UzCpJ-wVz6zqC0)sknt^udrDa>^JtfD(3M}LXs{AKH1&V<$FceLkSvIFU<6ReOiW@R'
            'eSyo?$NWl)=_ua{r(&q}ZurCyr-;Ig=n4aoh^i54r;JpQSVd`&?akxyELWKE(w}@k$w9@zLo-Elgd=@<{VYGu`u3lE852LfwJBwO'
            '=54DX2`IQnRR{TCOM<1gWtFUeCgEs!HX9>2a6uGANWGXs+TV5`-Tyg`pL6)sQPUF47_`QThrbcq8|p`eI8%}<yORldduXePV(fIo'
            '^#Q%L&}R=X*^@is%C~ft?T__cA9+T!e<x4QyKOQhloU?2RC<o{fX||5hK7L!U*H}z*E9<jk8RCK9_3S7@uRvkV~uiT&7PH-{K0{}'
            'WhLnD8vMAS?WOLn*b}=_3pSoNg>5*6DZufWgzl~+ZKrg`$4a}!tso>S-Ff9RG_>zB@Emh`4@M_L`Enf_f_mByD1*2E5r<Ku*i$Q`'
            'k9r9Tss=D`gZa`!pR=4v*Z@?gqG<irM5|e}r;@FkEym&e1|<V!<P=DZ+RoRwsp1F+_R~k%yHVkb{~EQ<+i-$2V^a)~t<egN?`g^$'
            '&i&#yQs(t?j;SZSg;iI~=$tEes>*y+i}_8tDi19(ak6IiuFt`bI2O_w%Ntq7Gmv)`#y%KmBy}2pX6Q#K2@I{T4$+=386O!$G#eDl'
            'IG)S!5q6$q;w0VpeT;+fQ#8>60SjC5R!Vvf>D%P|elhrfBvAUqAor)?qy)e|uTR~v4_zgo*r00o%Mh=q*bCeeRM6QGB#%_yki(Sc'
            '=>NPI1=)R^%!Hgq7~PSjy+j5I5_k^ZW`S-MIsIV0tkm2qZ~|sHBca>24H5HCL<29rm!Q4htv;^y5ESVe01LD=Bk5ofe01AIrCrYa'
            'DI#Xv*q3p8$2K@A2eSA5oFc5_SzA0*5+vi(Bpf|fOwry&b*Et4W+ery*>K}4hAcqGfkCW%4f(i;Z{**A8FT+*h_--OB1jaFvWnPY'
            'l1?iGtNydn$fZPgb=c{j4htX^$Gd6lka^xUY_@<&nPHJzk=<eYjdo50^=%*)#-MmnH=l&B<+Y}|2}Q@Mg^w;)=m@?ThhnzkSd)+K'
            '8dK6{vRDF?{pZ(UKPOgL7qAd`HcXa_;odN#|4o(?4koB_8@?bpnxX)uSK@BU0d(jYsC-FRQHbGR6zw)fr;H1b2GM#Hq{}#>kfR8Y'
            '5*iu)*V60y9yBx1f7pqhV!*x;Fs05uxfU-M@B{IL-H7Jl4OxSvTI8>?c;V{^urJOZyX0r@4SIh=7c!?9xZ|h>7lZT5!@(uUqu4AD'
            '2J<)=Ot*V`es+0!#H}QXuno@B(FD5fV1PawPL`$mQSanO?rR19=-1u1APwVd{|2#b?5W@ZC0wBg!?^Kn69XR6nX`%TZHf&42SDBw'
            'cBo9+X}C?;(FKr_j>E2mj%naDz2-XPU`GfMq#ehBk1WGZ8DzF8{K4=a;5j)5>dKD)HZCj-anyr}Lra`~<?D=IXLYY(vDDkRNO1~l'
            '*YhYmH|N8kftLKkcw_k!9`Jph`OF-JAm$t}<c#O+8vy@nJY$i8oh{*|BhRE~tWCL=a4)=vx1$BLZwQ?g(LE3t=+ye>%EtgdDW={9'
            'QUF3x{;6XkJyY!<U6>uCTW%8N4ed0HQ`Hou6Qm!h!(zY+$-(!8u$|}M?|b|E`-Hfi=lgk^5)SJ)uO^3iHFcC%QwDj-FKBNiwUqDw'
            'yDx>e_(!?LAO)@#ff=hxQsOq0uo5m5EA`cY^Q;Ow-|v5&oqqNuR`h><0ldG52l6JTrNYL94jWV9{?FUr8xd<nMqE+7zt8^*;{8To'
            ';D3_s)P?BL1%M0`juvd3#hAT{ki522i6hUy4AvKMWdCb{q9FJlUH-jyJUHncAZjeaQuHE4I+8P;`1R~v6a9kMD<kqi52R4cGa?f+'
            'bWirt2#L~iXo|)a=z4S@P_}6-*^?ekI(YXu4rf^?3f}du*oGdd(+jdvOOCq51`RHnE=fS07rY72j6HZ}?13fbpbvMAY~xYfgHu9E'
            'wNQfeYq?Nd@<%_9g3(KU6943IxZ3|~R>_zF*qKL-ex4+6!ORGNeHv0ETq{^T+m{FPg$XVCnGb-_u_f^*4xx^>1?}o$(0g~>J3Z<B'
            'T~Z+<UKZ5Q7~mx^cq~B&Tor*^IE#xQi_kfD7Y9BN?RH!I3;%Z)ZDF%;Jx2$Lk40zv3cyn;7}1EAaNcF0cFcjac`%eKwgddaZ@p~2'
            'cs6Q<*_KpN;im-ovsVVlUwSX`x1oeA{4(w-J=06ax52`<>_@N3k5fS~q;x8<Ssec0uM-McU>KDS9<Am1t5M5;`QLs$ywh($qX6CT'
            'Ch_l3+P9RFrihoAHile->bG0E#OI@EinRZI>-qlkS5Je9G*tzsR|-!17G*L{nu3<`O{ZBwu!>;B%Y}6c+|B=oFA+O@4M$=bqN#!@'
            ')Qz9uuyDqR@7?)H>aRnP62!ai`|feS5{iEhlDNgjU^*vZY9$pqkmeTjY|+6eXkp2dt;oUfzuw;tk1x?E&`_?(!H4mDMWM79d#myS'
            'xlQepv%>H#(W;l<U?N{ldXw8FH)_8Gbr2nN5TS}K+NVgQ*CJ+n&5VSGwJ*9C$6axQVUQnd_#Se1808cAdY$04dGZzP?=#l(<WiW0'
            'gN%O@d>S=*3hKZyR#CR%U{nzBzTB({{QsS1oQq2T)o1-kL1Ba2pd#N@yDp87OfUtcp?^B)onQ10F`*S(P7jC}y%xR{>)B_JY_Tf&'
            '(})t~_}cnT@sirYig&tgX)&T$#^e6QK)cb~mLzpCaO<aD6h|L{DDs^aCp7Xx<I*;FbH}ZK0qvpa%2xHVZh5eI>2iRBc}<6OXmwSz'
            '4_=GE!d6h4kMipYzm`h~Q%~EHIr7v~w&snCp-2`6H3r6GN~J2pG&|d(108`~?=i3k%nWub6U-oz5g0e5Uek^ga^-4vco|qQ^x?|u'
            '_vUNdH4HGt3#~X6jrZ{FK)i6<;tZWURTjG`tR5Xa+4U1qHC<82)O0l$(=bUhit)t|&%xRIi#&k=XOUU>$XCTOT;ff>a<7{DQHlrS'
            'i{AhM3mo*kF8Z2u2mPNZO+eY!*{m=cHtP&#&61ex%Fq1h{djr&&i+J2*9wzWvr=4D;~tt94~?>&r>Pnq$y<lUX5G5^gjTAC>**Vd'
            'MwLpYKl?^sT{JezmC#{l(7kx)Q8y!`^)}8HF$L>uDWSM=l0|S7h{`<0ahx?xF<y>D>+$-@VWst)bmOeP*=RMNdE|6~NrCL|^XAKj'
            'cVXhgAw_ZmK*%<<R2dPfmSi=|6Dz<B0zX<UtJ{?fJi1wqMrug1Kp3bapXaKqel1a<SyCR#pOM(0xa0dEX*E#6Z8EKMwM|TrO{^`t'
            'Bg_OLI)B9*ZrfmnM2FKv5Z(-a>Yu*rpT6~udp~3Pl}!lrP<5qYnB6LF3H00gAb?sUI@$FeH(=@4na`FJsyqu{X+~R4sd(hR!fdvi'
            'NcEw9sPAkT`RgtqbeqS`1mO#RrJ9(0;g3`jqA&c7DkAn!b-|%_QcB_|zkq;;GDn{Dl>fx=<VeYSnEF&_4VGbdlE$3({^PO_3a7`)'
            '#YP%q>L2wdzuc<wIF_3gk4U*iHSU(%)u3OwVKoMo+f^b#xmndd7OWE%FEXs&6|Bgz^ls!tmYvE>=Wx>fn<u?+rIIul^iJNB)bvs+'
            '6E+LNxWfh;Xvy8WtWFf8b}768g?kp54PC$<xk<X5QY}bMR>_-FOqgo=A)*mX;j2o33d%@>Ex)LM7`#({5)<@52Un{c%u+j|ikFfG'
            '$87rH(XH3#XpE9Vr#<Jl&<;6V%G7}*tDmYLtPfJJl3PbpG+Y_1#G%?>-rqMT94aw3X478yQ8>jo5vdqG|MKp<Ig+0C-}Ej99{R*%'
            'IHuuml#?}2&8Uq@aGY}Q!Qj~Y>Fn|xar^npw$hcoX&B5Wjd=8rCPnf!m7{;bsrM)xXDwFfsE3%x@ocpu-!072%Lx}!m4G2k%rKf%'
            'I>X{1;!XejVxa65<2VH(6OIq848uBa4x;jq9T(H#-mnb}VnIJ<!8#psuuGOD;RuyfO6V}WL4uYg4Q+L&Ku)pCgY3FPaZe%fDaH(B'
            'lqR0gj#c&%QYcJJb{kJr=2DdbTT>r#7odY*$paT)P~p(X!%L{DriP;~fu4d9OP%1A4l33g<Jvx-yxQ{qX-CfpBc(APlE;URqDB(`'
            'z7)SOESRPeNwzqr(L9_5Vwaojm6G^tszTAA9fel#8DRAI%;8W=+!*+@W$npi+*UhX0wy<{J;%LjJ-gOn$-EAnm8U}>l}{$m$M#<R'
            'c1s6VJLdo_;IMmocG^Ge9*e`X6F4|W2b^`0vhCK`@7NH>;j4MQg&!E-72$u_jb|7@!F_iR#8o&+ui*iO-R`O%9=adzKDObUC`#kc'
            '(Geaez%e119+wg`Dw`k^d$u#ttp#6Xv|8FMcz-Kp9?a90khCKm$z{fVvNm$gUS-M>jY2|+GM&*L?uT&7R)`_Mb8<O>=W!s4NJaTa'
            'b>csqlYNY=P%fQSrc>juT-<4MlNrfmtXGXgF^14Vi_*b+aN-#V4$=$K;Wfa6E7&xL_R1lS-+<1pdgRGm5=O|or?Ou>mKUek2`x{?'
            '%so_})&<ASE_7-z^#8X2`{>pzfiFMu735zvR>J4QUIjjT+^Gtes|A{UpIr+<yP1Zj3`$cdN6!(y)}B^%Rx_58@47UCohG9q-ldG)'
            'pG8qN>cPf;`5s96C5KB^5jr14oVme62Fl*1D&yt0A1Dmsp(KaiMrD1ZS*<m=+kPpE@3$wWBMuC*6E_)}nHFJ`w4kUUj9s8s!@{M;'
            'kiMV9OYt$k&6Um%t-s1>`Na~g_ml%YNLra;-X>a!ce+}vUZu!pbV{o7&d{~u5J$pPBuLdx`ltO9SZ}(BXEL>=e}l1mnK(N={#C~%'
            'EfdggleHw^AY5Rb8kMr0Dle2b&j^0P`QP8ha~=#f40yziN@kS<b9@_&XF8|}&nG2N_t(2ArkRvk+)+lOVb#nqRt;VCDVsa*706mP'
            'jfPDJjKu$imMWQc8~$#nB--s&Hr9$l3Ej&=QZ8C|wqf*71YT^5yw;J)k8+8Hz{#tgs!3z3znK%h?J0Z-UXOLhaSH;zQ(`s5F!U;}'
            ')C{j(89SceW_85XqH0>Wsf+Rz)wSXl^A#%jfOp>ZqsiPG8%NCVy$XYwpOKnr#Z{Th!|}=9JVJGE>E_X&8c4v2fW<uuX|m?S9aYX@'
            '1g5iiz!jO4%4nP|IU){1l{!H8z#BEr5h`TOAY}*G<x*QZYz8x-!>LCSMrytv$>Gv01lltSIIZ3u=v5KsmWiiR&Rho~`^^m}8dyab'
            'ECs^4j3?9!^VD+y6a_pO7z0RuZJ7bez4G%?3GU*R^k$GVhI%8+n1HA5QKtkj?@Lu;7SNZB-E_sj5Ks-ChHIv$t1?bpsS&SrJM-)L'
            'Yi$Y4Q-!H~U0xpi!g;&szFIxJF3K>aIGF0|@A_U@b+2r5FOHh2zDqCkD#WR9AM0VUM>+h)fJE_S2_iZYhuh0POM;^vmBVeVhs7Rs'
            'P65z+8<&CNG;<OL1DKd-$k=ud6&%EK<-e{=Ioza7Vj());@JHFzF^#&vQ4C684uViSK0OoHtZ`L>)OnGNb$!YLCiu*xZEK%=PGYf'
            'JZUtIVkQg<utq(Nfv23_Jbt5r*woW&O487*ARn$ZcsNz#mjs96D%zMcI+_%$mD(-L;^ew7uUmC}V2)_)DdmaB?_-!5{V>%n+ARiZ'
            'nJ6$caLMTOsZ;sU<03wBTLube3FM6_kQtsWHsD{yQ6zXJ6{0CifR|Gnt`HlMo?BTtIB{bX%ek-=;*{`IGc<G64m-KxPNw{MI~BeW'
            '{-&zr;S^ZPGrnP*F3-;>pzgjMrm@NeYsUIzbMF@D5U!=(>{&5%j&FuyoIkAWp(HMY9rB~=`sPX^EB4{%X2)Fg#6-eI>~<k%W+{JH'
            'ZLoixC`^0&pweHOnWdX~)=Os<VbRUVF(pt(b617U1#a$E?_!k+!i=%378{{`-K(nJx%gmq%%2*4Oyo8MbSWqp4Rs4s5-{-(0|A%}'
            ';Ax{DcBHDQ&nao$jFqj*Ac5cqNKvH)^IV{j&qEI~&N<EXP|(s+RR_iBmmhb#JVvpFr%7EQ-eDbNK@`mNu@A)>_Hb?(4ZkCA>^YXU'
            'OTsw@W>r-zkWSf)Ii~FPQ6q%`lh-J}&ck9vIh9u#`8RXuF~%|n<=6*FjgU!H*iUjXPEPx(v>`~gv8%EM*{^LVY^Q%#Ym+wpRrBen'
            'A6()$?NsXj&FcIFs;^G!ur2eI++iaRuDZ=cT}?>>l`2~~OR&ZdAYG}<*}0AEO<doat@>g?psq-Dq;Pc|&qU*)FpSD_RIe#2;i0^k'
            'N(A2*jiPffN6&rCzf^=8!@c|g%c{#!4O}f`ExB}shQ|DQ7Nq!EhDG`qyJz{9$_rA&6NNS&h`m5h(%Xi5BJ-YVyB;8=;;sjvH^zP@'
            '(Lp9iD|paKkZQ?5rhn9iJ-WoTIlH^T9~i!k-pXnQbTOX)k>?v>2-siArGn-Lsla{0F0ZXyn5ZQeKXPJXFXr0a*_>dmv_#83*3LeL'
            '(KN2NkMW%H_DMkXj2hgpQ#QYSim6Vee#X`<<14QCODC_C*C3#F2p1XvWLIC%#s0(tp@vQCRW^0NG%Pa>bRba_bhCkd=hKaMJRWS*'
            '7{zWv?$5KKZ()tkPsz4pZaz$(v0l$g%wIm5wn^R}V^z)ktS13gxf{xj*?YpPU?>v=hPto<+Tu|i9UMyobP`~(ym$&fb*1=mFrV`s'
            's;n<f!UA7OlJQz!9&eYtf2SS{h~?@^+?0upstFrP7wqKx)yytvpsg$RY7;Tm`fKQ<jF!#JLQ%5(QC>o%$hb&%Lw3tC?_-DR9kr>}'
            'XM;MU(*)<V3Zik0t_mGzl}%fJa$L7!1TTkoqrs1ub{cAkbkJcRAPq+M2~34PPr=zx?i<DFbj;k<m7gkeFdl=?B>ca@xH|fmY5uc8'
            '0a79UOR}JI8)oG^_9~MVHQzFoC??oLgX}m?lGQTf&Uh@7fT(0qjehc0a_7s+p;gyf70+^@8Yje%r*p{RbvJ{KYBzd_puiV_b&d%`'
            'o4VU3F?NG6{&lWpwJa`0p8D%jw@g(}*jrV{+w%1v-K-U_|2ACxsXI@ZWff@a*_$__F+9IKRW6E$XQ!t<<y3j0%ODO_nPf$9-8SbA'
            'HLvvKJ>F|7;J4;<@8nLe?qTg4J|=m#VJGpU+@ByLbCzd2Mwku<l4urg{K@8tI;c+dp9W*h+&Kb)I)Yye8LEhh01PIa$Sqqfu!JQt'
            '3r;eMzTRz}73sf{;|POE`90LbsYV#F0r$DiCYIfp@B>l$z&2VE**8UXP3BFDM(zQMPi=N!Su9;MS+cZLSD9<ox6S(?)1IBXo}zDO'
            '_(V9Ldu@J}1dJ-{G{W>s!}dD2_2QQ>I~aXXTYlwz>ZXmV3wKZ(<`inc@{va^?$NEB9Nwzw(xwG(jx3B;fqf%|x@3nJd$oVPQyk4k'
            'yz6?qcuBfYj+0cp&h|AbW)EqUph68hxxGO)#Ec?{nPo8Y)l=ueS@<7hf;}$$hbQ1qk9G%l%Uvy>s_Jbv4E!4|^78}^5un9lnXOe4'
            'I8VPJp-cVcx<%hAYs|;`Oyni9vK4}Sc|g<=r{CkO^?43k2EcT^9$e*DaP{UYf3_!-_)e`=rh8_0?>~NdckkZ$`PsR5-W~M3<Nis1'
            ';GMreL6M5i8`(nX9=NLAQK{WTwZn3mltgls-KuA&XP}#4xCKdU?1K^mC!}%%vdu;B9E&|;{!0otm-im>?Vx|s17b}XEyG0+V^U1n'
            'uluGq_?7zk(}2P-robQO>V5z8=k9U;$b;VRd4Mn?UC*n?v&mfy1>&TCanV10i`^>No@{@4`tJ1X7Yx<W*B4x^6;~i#y*L4M7Gqr+'
            ')<m&pe3*$5s8F*Y*-fdyB9<@??}C={&!BEtjdgI|J-z6|tUUhayZ-4Bh1#0XYYGcmt~m>?zys>^_|`1yzKHLr>O9MHKvVc+ws+oi'
            'kB@)s9=<Dnj}Ar^>NKmK9;v4q1)9bHm9tiV+8fuz(%~vg`*HtW&wG!dGv}xIq^!cH-Tp`aYQObou6gSH*1CRpw%>eIeuh@t#t)(U'
            '+XxdFUtz-^Ky3XNqHmnYL-?;z;3q-yuMaJx8(0p#1#O_uZ|S#J2c2v9=eLXRYR7I|{qS$sjUNtZ+-2zT2PkNrIgi_Mum1DjeoKFI'
            'cCWwNFvWD%GK#pGYjSnH%nOaxXN1(8XFIR0QCI9+Ghu1vq%%Uff||K0h3av!J0)GZQcX%3C6unC49(qKkDa??fYC+oqzi}X5b)_&'
            'P|{*d?oT`|cN-wr{U!$Bx8i9F<k=e;&4YE@o0yUc3(qIv%}tQx3EL^BJLtKntx>{Jj9ij=r&0_GU`*yR<q|kVBF>pYY$}9KsRCr?'
            '<EgB7CYZp;+;?ePmZ#6uywvToPVjD2NUvu+#h8c_tb%2g`E1C#pTsOML?OCy@x)?YCpf|w7m;GB2yTF!PO5gl#K}x&;*!Ne_xR3j'
            'Zw!kQ=_&zjgzJ(5DN>9GvwXzatxgD^*XxqeHb$Hc8RTNjCqlWyRS6I1RSH%yZjXs?mp7wqnKx;C?Ko8!UpJIZRqBSU(ZT8lEi;TA'
            '=>R;DHDSm;NJBgL(2<o^OzR=Pl~<x>BUUz%Gy{VX4%fuUy=kzzo}Xmw&WXRMIG82Q*q@M?poVW&&LDrQ^}zf>5J84tGK@3x_+DNX'
            ';7r@}XT#MOr349_*USB9P4RO71uflKBKoFuaR`tOsjO$D$HEq?2q`h;qsCH2*8AUf3Kl@jd5lJfoPd?alUeFLDFm4BYc>}J8fzJy'
            '4lLj$jh0I>#qZqvsW3_^ZJf2FxOw@_rzfS^X=!H9|4d_|B884}Uhv`8U!_?EXrbv76JxiAVn3Ao_^!NqqA(i|WA#!GBe=W?h%#$='
            '&nerwl|_xJ#D|@y+nkhe43p%F@AseM|GdEedCAyD9{mx}G!`v4uN@Fid4!iBfVWg=1g`)pd~u8IW5GvFT#K~D!RQK1RK1x5{sdiX'
            '@#Z}UCR^c*l&|zxnVpq}9&wql(sN*Mx8W$vjC_@wAqb2)l%6G9KqM#e)^jOaD|ds<cvY2RNd;R*>&C)QW|&AQpETDmRZopN>Oh##'
            'PlU=}V-+|0C7tS0Z%Mu|YfdS^1`72~wKg1*aGvtKV24#SU(IN^rE;^;H=00HQ@o-9Nya0A9Sf7iAZ##gt`3y_DUOIU%S>G@DThoj'
            '7%*A4x+M7Tv3ztC%>6Zsd{lHRQlT%$2w05RZiKR`ri((yPlLgGF*F7mid#TEEZyMaEgnvEf@5MK2>pRVoGjf0V~{O;4h^5=bRI?='
            'Opahx{m_W@;#um5Db~`Z5+9LvT9<6Atz#VN=8=|Sq$x{%_IS9pil5`ZJnz)t$2Q+n0&vdx9NDFwTNfHsh!e6!Dc+XKcmk(Fv~vvj'
            'GM?d|zWNh1p7UQ@-8Z1^I0ecH-;#sK@04$wO4hc*@mu&G8vAX`DN6V2a-1s|_S!99x=Hg<Ls#%=ijrzks^pGEpL9t*03I*eKpwql'
            '#saHe(0x&W!e1u5mHii5O6xBnn6U@;o`KK|C(;mb3Okd37TULEYqAmJAXVQi7om~Wdhz7qg}^utkQJ-5tmBc5p;YVI;jHrM)K*@K'
            'p+<&=(>7<H0A-OO`vZaYH|GaKNp*f`{5JUxRMFqs@UQ!WoBrS!5+07y*H}zLmtCC*P3T=4HKAqWnQK|gRNT3w*cEF(s5mDSw+*x>'
            'lZfb+W$gl{;z(}YNO{rh0+6_(bmIIcfTI8lc#<Np{+EN5pzp1nF2G_dc;pD6GXzj@x_n+F8XGa7!e(9t6FG)kviELvV2jB#%5Gd+'
            'Sz+a!CYsmhSe`S)<XRww?U!`j0nu?_&AGO_M~V(~LH#zp1$o#Ux<vu!tOK`I#QOwm28o96%H+53r<wFbmJj73o0u~Jxduq*C~nWu'
            '=~`d;H%7?w*2Fv5YPUdt)3rE0J9~G*`3QU@xO<j$NYdWDe46sQ{_^}7xE0WdT$k5LD-xA@FwJ6|C4BcDQ)#~Dge(-=uQPQP0@hg-'
            'gh8uR*SwhW)Oo;U;dHdy9!33m-s6FIfCrCNT6w)sH&N6l3Z}_vk@}~+#EP}R0Q<lF_FK@FyO?woemW}Pv3@4BAp85<ezfj$tSMum'
            ')(Ck~rCG7`4cVccu11ZyzZgw?u{aQog+BG-yYlCVc5sWdA@`-&XyZ*Ee!SWOKx^g%7B&IK7gq=J_+5X6Ym4{(7_Tc@sGgvd%sEWZ'
            '%K%f+uTXG9>GS;&)e*@MFdRb(2jd2-_VJa*t1o}0K^$kv)AQo#6u!=WI1QOPYn6=;v{#}l!@M;;y1Y<p6PJTS!2=wK)*r=Wg(OED'
            'T@0F{^(R5)?(f4?g==4;Qycm=k~1aCq#p$(oWS%GN|9{JD(5J!qLqaz`z2K45k{uce6oUbAf5xpUm#dXygQ*%8PE0)_V;OO(qR8A'
            '&5TB;pqDO20O0*U%XThPj{agf59GGwImBgZaX|`7i?)hQdoX3&vWVA&;+HU*#P_L8TJu30{)X8{T<DOJ*I>qv$%gzIRDDnbO4c9z'
            'JN(^;zjZ4HJRp0%Yi&PO{-X`mzTtF%K!;<0?>J7qZgdmOgVb?dbTH*18Wt0m!`6ye^85fsG`MLy1FW5E*~Jxmd*a0_U}R7D+-iw;'
            '4I-Jn416L5w}fO8M8*a_ZHgC7@yrw*yp*@LBO8Bb_{75Vw9|McTKukk9|SXah7Jt>kN6`r{iAZnl{jI2>De>=Dax>ve@VBm{+N1Q'
            '_#O{6!GSWgK;~}D4QnZF{=EHc5l85!Wj}=f-#{0!?hR(zu5&`hnFNvPPTerqDW_pQlC!X$ZJNZNH_t%g5|9q`x7L99i-`>4FM6Ux'
            '7g~SPW&oyO@O?B^RuZoY*IP@pf`h#)a|2Nh2Gr(}B+DB`F$(4}mPyItCZ?2NN}#DXqi8sZ!kgQyrHCTY2<<y-ETkihIc|3R&ee<j'
            'P}Di0TVF8RKk0;C*U=y%<xcZ>?9Z+C)*)ce&ZoGG&5CKrHXS_Ij3t!&jQ6d2KmzWzTdiK@tP7X|i%*%#9>%x1#XZz)`M<+HYDVxR'
            '*tg$79zjw^+yLt3DL|Pa7~^;`3ISKlZ7dVUXYMDZ|IniYdu<bGa-XInRTFL1E{(CP&LI}LH6f9eGO|lexYGg10gI&|f9TM-P@RNH'
            'z~zX)6LG4xPi2Bh{8TwA$5cjsyh;)@;-Q27*pH;bGY@>qCU)!3r!D+k7DQ4q+|$ng+CSg$ipSr%uY7yyCzKs+2D57LAE~Uo5q!Yl'
            'uXsj(O6eKa)8URNxB!KpPQwrE#LmBAO;e{0?H*z>_bkIn!+msg_cp_m{I19iezIDCP>;}rFGeY7L!O_G!%&qnr9$yp=Q$|`zhUO1'
            'l8-zCE6oqWV%ZeDPIYDNy@6dgj>og2`!4`IPBE!^JmwPb<C(Y%eR0NF!~b>WVP%*mXD{+m;G737Y9+55r0n`Oj)DI~@#^N59XMgP'
            'c{_GNNru@%H(Gkq!X!pI$YO=oSW?We45Ta-KH;0XIn`5FA|o|A4x?2rGy)J4yuX%}?8Cs&ahZdoB=5mWGtUDVVm@WKG}Yug&&cet'
            'jOX(X{QADh3uLQ#=M_9M6K_EOwuwr7X$oM*t!X1r3};emEkDzgec?0mr|}ul5xSXdk-xNu^QIF?zRl%Py$88IW4h{cM~*Lv1UiNp'
            'Gb=Cd6Hf&B%u{p?KC>m^N~5u{5I|Wh_C;{i-v=@V&n5Z+&!RsN2i5Oi&e1L1tv74Vr#ltKW;_5roPSD2t4}a-WAvT@z?K0k)W3iy'
            '15g&kV*gs&-Gf@9o{zGMV4r3U=?_+hZ-wuv?O0n_>e)rj5-qJ_!xI16(b!Wx$T#e%FF3UgMF^-;7*i}72d_xm!`H6`5tm?6GZL(x'
            't_nqiGK3+G5MhSUAxD483B;U7>Y>|iJZ&njDxC+xvZ0TlYaN~s25tBj0O%nLb*+17jk2{mA0Xy<9;ZP=HFbAta9*25KMe(}_{%)U'
            '?s4zvZSTA)!!8Hq6LpoDb<vCO&~_J8R`%5N4?XOkA71tc-j8qq-@)L6pU!(1Kb;+;+vSVW+ru+>bN0(ATEC#dv+u>X;?=(9JO6>x'
            'I3_-zInR4cv4MHRf#CHn2K|#RmJUL$z`jy!<&oJ1aue+UjhU2&^Y{$Y^Qo??AeoPzN8v*FBV-NIwwiGnX?d-ua*VR#O%WxW)C$L)'
            '`y-6tn1~@J30R<?2zyo!+oGQ_?MvC(fft-0{7}#!6{l8pb1EYT;Y5HSz$sbLWUP_|r%M6>vNg0B8}5`UB^G;)p_yjklhv4#^U?Q-'
            'zmlM;W~;}gM5_cUw2S~k<p}|g=oVQjZz+5ZhKmV!xs0!jS53aIP`+4(O^yZstvIc;;)C4pO{2P~rn<1u&Zc0z4Sbn-GI<xl!#vBj'
            'eH?X&G0Y4_O=*rjcv4%@LsZ0ZZjJygKPrq!w!V&>Ah|c%wx@ilR8UHjkBaC^Jtn%CC?6^`l;TN~dHaLUwVUQ!&XXk3!;UbkPq4am'
            'rXBXYSBJ4O&Vl0C@z_&3o^o7=^K0+CcPs|opHBLxZM1T(#_<1<SO?h*1Gp#=!eo`K7AsiL4Tf#$oF3(1O7wOTnD-;YEFf6x@)PoY'
            '7`Z=ew+>d?J%wuMzffV41f{xgLS3hrDC|=*vJ?{cD2(I`PF2WV=(<`Wn#(Ll<&&MKpLuvJzVV|K3`?snM%q2*<b)7cW0qsE5@*Ae'
            'CVN!bLB{z(**emghcP3K`AIR**xNoryZ%4>+KN9UJqmL2RSrL{?SsTY!Ji$K2qtww$;GvH>M+2q5$7e@;&JMmv|+!c#L*}q0p@LU'
            'w~S7%rIcpKvb-cBdnTGpslJC270Af2QzbuY<V*d#AV)r1;!{Nd6;@T*|I2j!$~-aGdhW<%Y^$DDt5D_Eio2m9+21Q0XmrbF{`CR_'
            'XeqaJPU-pi41h<V@*>Rj%auiJp73;S4T{_}c83JCjtNK|9at{#p)>F(v99WZMvhn!X11Bz=tOnozb6e--U4(niVohdXw5IZd$M7$'
            '`M#1|+Fu7jDi0{!?n9+%O~@0js;RsXsGmn(3(-S~Gp!kibI5LN0!lze4eHoL-U?Xj#zx~fo&yXc54D1MeLH5U+F|*r=0?1fNA-YY'
            'cr`^%)h$bgU3og;9BF?Or*3UB7>(31PgN1vums7i^PFZC(U{dVF+Utu&R53y!lA0DRoXF%))Zd4n|f!PdS_4d=0d2o-=0+Tw$r#;'
            'eXUoVLj~`zJY!7Bg^K1n4;3PLR?`N5s{HBvlqMj@hNuAQ6`&`TwK4lm(WqT9XdURzq(`ni!`D0IFk&52O1qj%<JSC{9?Mmfxfts|'
            '1{2CHhjfQqygr0B6(m-s#*!4p6@S2sk98Hnz0$3&>2_pY*`^u<&|$x3$g0DBT{5)D`($&@Rg~FQw(!w57ENroDeD$BaI1`NvTN;?'
            'SM_I9uO~0QiUK2N$dcXNl3&7KJL1)TotrPiy9#L6tVUPLFk5Z0<ZI<!u0MI3_+xpT&GO`zSl{QZ?f0y!oD=v{*I)>QNzj_k`8x3T'
            'S_H@>OeB$WI95o7O;NpI<>a|vnZi+;n|`@nD?`}H<;8$<*N+19=SD36ty8oPN~Hw!75hu$7yfq^>B?Ctg{x9}X6^o8u&9Ch_H*w-'
            'S0R#BDRQcUViFpzkzi^XgvldVQrT0{*?7wVFra-Aqtyh>fn@zNEZXX>vMlo&B6eBUt*7F63nxQj$+SuS@{JmbsY6QGAd;<MKPKUm'
            '^;8-~sf;}1?-ESflZN9^01uft#q?Fw)TYKX(UOKs4L{+ghHvca2{Dc5)?M;&@2zOD^)cD+lc=#)Bk#pu<N)<w(&<4TXd@p*V-#i~'
            '9G3@Lkje}CX%w$?7gS{F1sSy3%Uko|LTe+j%W_g+yhYiL#2{-LOR`lwB@+O<f|k|p^bY6v&HP(9bq7M?Hks(eB%ZG*yDE*oTQ&Ob'
            '_2b?H4E~6YE(Ul(4&JnwMde}^n-`<0JFRE2)p{oWES&kgp76;9@fYNp7PzJpV@=tnHD#4+%H*1=$vD0G?RPV~@9KvCD}Mh|z(oCQ'
            'VEylh3?VVFAYnx+XhTrsFVovN+tR%838cH)I$la`^GW9LdgH{C_*eI#Cu|X)xUEmkm#r3?Pk!x8ZGG(EX^rivk|uR}@>4}k>NMo1'
            '%9_+^ip72f^MmeYJuiQ)U_i;gX1%yB-BBoWQAv?2+3i`JjMMCQ%9BO=B3VwVWyx<r&YN*!tER~FG*x7ltr2FP5JsEAHrs5<%~*)J'
            'mA9Yrw%yBaG2<t|x>I|{UNf#5LGKg}zUeCd_fe*Lq4H@i_FYFj+xcvo&lZ&ZCb`eo(0{U;Y%wL45-R6onI$*rcmIWKJ}3`ch*+)t'
            'ZLl};LRA$v`(?xa)7`N`9%j|wD%pt(;4YZ^HrZLr3bK|*OLPrys2p_s^iZ*K>b3b&KW(V3nW9Yxi&mr>%bPibWfCty0~VB%C(_2Y'
            'Pk`la*lcMy--O2YrvZFb->^UNjX2`MaO6Ej+N8_?j;RiWk=R1&bM;Ol+Z*jFT-nj=xW$?<8D~{;vxCn=#u6Hv*!Wau5adfF|C5lE'
            '(7L`u8BNKuXS$S2qh6<qak$m+r0B^iEYVqFYOK0N)laAA6Qk99#T6YeWzS3ZH9SE}3dV!M*C)PE$wn9cN9QT)Ho8}N3R?R3*UII^'
            'a^{~Pm)CAgPIPv*B_4YE=7o3Fq@}FVltw8j-LPxOXstY=9H)Y{Q^GGCotvW!MyZ?GX*<%<dUCRKI`JEZVN841MTX~{PK*{u+NqnJ'
            '?-V~I*Er?r*0ekf*jOf4pt6dj;Y61DX2&;}6BhQC_wpm4B`Ftp^eeLEx6*rmNq*u!B~Pn1o~d-YY}%8RJmh+1Y4&28hA<6`c=ie}'
            'BHTN`XR?i9ZnI<#SS<jgSjj#KX5&D7j1x36Oje1|$%%IZG4t0G%my7MZqXGeKY5tYq+Elos0mU!oMNDFvBH~LVyP9Q5w+u=$~s!v'
            'k}hIv926>w16rxE+2x9L@KAv(c`Z_Mz9Jx|_+~=&L4gniAKO-yHf@>k(Qz`RoXh$y)2vBDzA;>(v3&n&TS9p(PAN^J9z1n_+Fr)X'
            '1~)Dy3fbDDMJ2SQ9WH79^i<`JXNn}fcUj$GAC?2tiFRA<=ZOUJUr^;iGa@?_>MA4Ty1Z3kZAuqJxmMVGuQSh64hA(zEAG3U=X#eE'
            'T|T!o#Zt~`1XCxTSuo0f=4-2lOsD#B2~PH(X4wJsMwMXF-;xt2SEE9zVCH-r-nUU=*KoG(N&rR|OmLHcv%yIt81-oqffdWSPenhW'
            'ZzC_!eR1?o=9h5WM%qP=(<vFni^)E0Pd53rLTVhonTL&|*E>dfL1fSL677%`HLSUbOq(zZ)^;wd1Dzp7e%EY4&S6pfPyayl{`lgV'
            '3|bE6lv#xegaG>SwtB?=@G4`B6|?XzAuYb)t5fUYPBCNkEe1nnX?aj88X~iwS#MRawcK#>QKCuG(&QycezAkpjMwmJV$TE6gF8+Q'
            ')r22mWQ?e)uHr`BnehydiXBL-QwVS2J7tR~<vTb`R>0K%_siD!)=yFj?_y+AV>FqolSrat!|0}M@cu;Hto&qRq8jIJsll{;vlfkk'
            'YS^imut(;)Fv9G(RXNPAP}So@<5?3<XC&9Vw<lh@Um*j=IaL(k@}-=!JNQ6~g-+O=iWlPFzDCw=vTW0b70aQ}11eyUa~33?TJXeL'
            '27td9E|7o}m+AyPW(|^B9>KIu_6efAgr$dt!?A3*#Xjl>!i*%`QSuDUw(>$oe^aGLWQEVcpm*|qaPisva4Tj;`mdSr0?6!Y5@w_#'
            '%fcy`5s!`l@;sw_3cl{_B7}E&LhwDaggOU?U|xZFMkxnDeA5*$zEh0E7)nX}yEGJ?yrF~%)vRE^oc#<MUjL|f!DWDluT{=Fm5sU5'
            '4+RTHQR0u)2$q?xxCnD>DH`M3I2==cl65L>VYR6W6#WHtl3^tVbYxu^9SMU}AOo?QlD1+P3?^%)KKwz>azVkRLfX+NM3|#xFbO#p'
            '2mw%YCSJP4FC`SFQX{gup;6Dn%pO<J52g;5#)uZd8i<@6`Ub<^!UOSu!~G$_JKA72$C5AbOqOUhMt2`w$2Ds#su?uYIdc>&?L>B^'
            'soVr(lq)_8xxF+#?Js;7M=nf8V%fh$jw3h~jv-cO8-p|TZ>pSPy*FRyB~~f~0M)8UnT>(2)bHyZ3m$0W(;%>mgj0zS6a-N6*F#|l'
            'kL#AzK-s?X!f>lX6H*0hR#nBOS~v8#5XX@F((1a`)HW|?8^xYlpQjd4<v_Z*ysg%8oYMZAQ~=f0P3M1?@b+={x+3UBf+Y$Ub5`2s'
            'uZ6A~%7hK8AuU2Kqr31ww#$RKRo<(%bhmM9jkCkVN}_DB)asM9<8=i#U9NuHVnXIyqoxpz%C&^(GQb+Xw%(9#XO(j%`&3plA-nK1'
            'ef>&<f%$_(Hau3>sDSD`Wm;tpqpBnE3FUE#<bF!!`C=j6LK&e_FmrwV+`6*6l{Ixgd7e*K-`!&7TNmd2bTC$z@ZDxoHMoZMx8kmE'
            'Xj=B@kTu73!c-H_ta8D>Yu#!}APy8tQxPrN1^_Y`XhX^_vK4AmLeYmYpI5b@=$5Yv$b?&V75<c!^N&Y&=gIhgI{oERS*EH~EpnF%'
            't8rbfgJ;(}6;G9Bgm<%LG2C^pcXldytHo38`t*vo?pPin#<3$f*tNfnwk3Y7e-~S+3ioE+c+HvRuga?PvV1jzf-^4y=QfYWrdsq)'
            'v2&p~{cSiN=^|@Y=q(y}&i5o~(y4943`i*jp}Mb&h3e&)WB(@6EE%f~RurHoJ&^6XvO*o^aA;_iRF%~(rDgq=lG}P%&d|533s>e4'
            ')8!h;)i~F3M}8DZyVJr?G4x%>&~pJusWcn{#-m|S0;+$+BxxCgc(bM~hyiCjZMAeZ0M6K2s1GR-+M$S2&VZ>GYw&8#UCHL&s4Iyl'
            'eaDPINuCPG6d)OolW=5(fz!)Iv0&rV0ihc*sxm^VdQo;g?A-qC3`nhi&`Z7A#k073{rH=4lQnh7AOY<N28DO|hoM1nQH8c$zCqF@'
            'Y!Hq3lCbbS|6ULZiPaK=M`7e$CRCGMnHif=jMurVM%k5#XH|}>x_Wj+RP15CAWWP~mLbX_31YF#y?@9CxpP&)hzjPIRcdm3s5C5j'
            'WPH<~qq3PFvRtpjze)xye8^aH3)@DQ8@ZUK4UUvhwlsWhk5mY1r-D~?TVzl9kM2cfW4Pu5=B3Bh6$6n)KMcr1D@VYV_Ua5orM4CI'
            'iD`L?DN!Q}Jso-cxLtWC*^aNb(kjDHhD^cJMGuDtnQ`XXMcqmWQna0)8vz2;qZBA0@3&><8aCAs=g~N#-gOGQVN6;You@%CL2gW&'
            'Lfy2_wZbGhSUHV)ag4Uvzw;pb$>Z`nF^mWGB=Td@rV0d`qVeliFM#N$<F;3T21%n3Kb8-N#N4i}VZOEMYdn=Cui<bJ5Hz8@KR^4a'
            '|6_m9JHo$yJU%;ocMQ4%mqR%054xxumckP>Y^}9)9*ikHIyI@yJy{CdC^$L|Z%ngl`TkC$JSWE3qV{vT241=FkU?P4Dxk1Nwsfl)'
            '=@x@=Au^D5Z9d-Q>}s7nS)^(I{1KPQ<(Y@P#v$J%;a&}E%B~J|vFJH)MtlPto6Dc!l_{kQ#K%~&a8G3M(p$=ZXYjAaGQKQni}x};'
            'T0EVGV_-!<$?3-Vg}7NkLriaCj+oxYtNDaXl4u(YvsB#;iD#GuPWdk2nkX}~K<Ax6_!C_UL||@plriE(n5imd3$jv@kVBH$J&d2l'
            'X!MK|IUsyaPsyi*dy;@m{RK!AnA-#9jvfWkc)proY@Q{yJRLJ{+9G&ASaic1q{qAzLjgN-yyrew71eCfkwppo)|<TKKnWW;JB<#='
            'rB%w{Q0E30Y8dps5xjI+U{3B~_kADrXaGE*2CyF_So|GKS`@sdw&oWNgY`FngmZLqq5S_`z8r}p8qHNG7m*Se&NrM$6b#Di;Q-^('
            'W6l-|;wBy%L=e&$=>fSGu?^97Fd}Uf8k9k=2jB<v%?HOG7BmX_S8r*7c9<$IZ33IWJiwRB!@=cw?+9sYzB9^hLN%+)Q_`a^!Uf1y'
            '>!oyq^0DRt7n<GU<YKiwd0r03pj*$>5_0-uatAr_x98}12Gf@IgeRw7V<mr3yYQ$C_7_y=B>y@P1K8i$9Pn=)Cv8K{(HjKRk<Y``'
            'gG#f;)lw+Q&EvIN5fRQJG~1m0qd_k%Nr>Mo214$Z9%UCrh7QE7AvJ6dQ-OYHnoj0T_RjRCQ4@5jH0Tf|8~e-7pYTT#-4v=knZ=y$'
            'QK2eUKGJ2Mpxf&_Lp|Q?z!MVrS-j!7v5VP1F#Re_hBEl`?bHw^F*I*=jagS4VyMszT2MhKl(kv%E+86FHqI)uG3;H3{ufAK0!S~@'
            '-xtS(2{Dkwx1#Y}eEY4i;Oe^ge82s@_znka^GKMjT|C!Em&g07FruW@1~lkhp@t1SQTOSj^9HrIt>lc3+Xw=|c#a;WlBM$>-qfd*'
            'x$0p-AFpTdTIEv`!k4?!{c?A@U*}TyX)h<8tF1maojJqY*%vVr*tQz#d43KhG}Ou4Fk;z7SdR9o!69`4B6Z`LaUvUrl<rEN@<RJm'
            '=dNTGrTU&5Whu%HjA9pCVTf`d<(aPLvY4VY=twbfD9=QLk3iEfk|G<RNuY&5@s%M?8I4&ij=_Oh8I@;ou-9T}Y<a`%Z18CumrCeO'
            'hOq5xW+yzO$|A8tE=g7>U3`W@Eu19vw=5MZ%fPeL@gvs9#Uegogbr7DkItpF!<5rS;zf2UGgXcN`><>>U^Q9YRT5;Gdosw-G1jn`'
            'p`}BSsg~%<b0wrjA0&eQ;o$gJanbvaOU_mXf1t>4-UC75^g;<_RLVfeNRM1t>WaC~mA_Gr4XN%puAVptR^%f>&%#%57se~b1yBiA'
            'VT5u&M@lpeuP{zGPp(Rwxsfe-Q$M02MoMyK5iaxR@l9JV4~x!YZa&;QlDrFL3l(Q#PK7Lr`BxK%i>#96`H)9?NDxD@&H<nD*K*OM'
            'l1?fjAyl5e%Cn`jYu0}F<^Hq1m-{cY6G%?%siumwfoV^wd!QWwp0?Lm1!|d^1HIJVBz046>e~dXe7nB0V@V(%^ZO4U^q2~$#HX9Z'
            'TJ#bHPR)w+LX%Kqn(Abv`f#b5Rq%~%%5-0JPkOe)r-ZIzL;1I4@oi<+fe%~^&fZ_NWj<pvSO)Tt?CDb60GZJmgXLxad^|dutQ+ch'
            '7MXM|1C-xoEZ=R)6%kKRADjB4gz6C&Ds#bif+X{z*sIRxD9tE}<DhDyej-ZfjK@1UkJeXP)wf}6UNzi|)hBey86!LNPF`wGIj)tQ'
            '_3~GpN;fk*sg72YI_zhW%Om=iA(W^eO(lQuKQ<?;_Ajb@+N_X1I^{=&@juB;*ofwn@*(m7%j3W1j!dB-UL`REox%-TIkj2EwV3je'
            'FI{vPt<;rAX`tmCZ)r9TQ{OqiWUK0|-d1#1E7u!PaIkTBC`zYD4a#s(JM5wRDZFO8R>k|8G;0nY+6n+0f<yW36ItGjSW-UMqX2WJ'
            'Ki!T?xt5wCACAUrl=v`PU3<Us8j}gprmM>0)>yBlA=jLpcAmGU^tfYPv^|Y6<|Q?~F%8amOs>=_$n*d@>x%4s94ZU&t!ZwCqdmQ}'
            '!nJJS!cUqE=(vS=@%LIheaTvym$^~be5)Q!8impVOmUmyMMbi=lnSDjGT>m5_W2#Ul#`r{HxBZVe4+(tTbDG|NUIhZFE_vNJK?PK'
            '7&QPbe)s+un1fw7$k0fdd(ywS=%2pbW?#1{B>Gi1f^Oa7G$h9a%gYJ`-38gW8-&&F6B$r*ZzkPNd7*p>zmzaf%vHSXV3?H2?X~t&'
            'Df=W=u_aOE%o5Mabhny>I1~{uUwf)@d{Z`ncWUZ%K{cPJ9-Os`Jg9Y^K;<A%J7lOj4789B$&Lnf$v?Fs{9KM)6RVo*nBp7KhX9Zx'
            'eRsHHe_NiGYHo2|m)rJ<UV801uT8;Uh{;RA&|wAV=VEyEYrR9(V`Xe(MZXYPZ@rY^$42<<Cl{G9=1_{{W(zSUxKQ?m0s$=<8Y&{O'
            'S$*i#UN9GHbL&l|_cb{d$!vW=Bb{?YHebtL<q_FRVHO==tA=UJz(Ak;Elc`Ee~IF3i`m0UlwbWPcZzC#gwu724wt9A2vUk9`()y6'
            'pJ6*ta;M%p*|Z}7)zxMm1Zao<eZGvZ>}4I>ZdM-JIA;cC*dh0Pg<Y45^D`*+m!;g*6r3QL6K%QEMVg)eQ9bUXyK&V#Q%buc=en`E'
            'DVm@iD!BKQJE60&V?jv4A6tCL6iyoTV0(M{bMN%H|4x9=bJRc3Mzk!r31P$u5S1)!i?^%0Aex6W3j2W%;l-$*F8moso;$T4m@=65'
            'QdNjT9FdsOKkpx(^p5&yftXNairl2f>ajL&MYVpinZ;}x?{men%JeL`h*ovb^5iSs%fV00S{c;+pCh>WOi8u!|Hq@ms@V*KQ)Xfk'
            '-y|!_&2Z~umH|vVLdI10pp<$<_N+fejy^sq#=T%xQw(4bp!fMREELn;cmz!+$=hJQWL~=~85d}byY8dtuF=gk{GBIJ<Do9|a-Ej~'
            'bN=r?#23BupL^%(7!Ud<y|c>!mq96uvVW<$%d4D9xqGENiT_pI+?TpHS!VMJiQ*^$H_A)f*MQA0{_(C6{=XWQWhJOq=&<up{SWj$'
            '4SN04pS#EXBM&g|{pEn5SD`5VMLavG`BJHEEv)p74JeV@+>cPzT{^zyby%8Lvsw|;Y^0KO({;~IA$OR(FSOG>OToW3p+(907xWT$'
            '=aYs1O2J<iF#SBUy|scuwvX)ufA{~m(WRz(P1$UliwpA4tz`wLjI`LJ^su65>1gBCN9nk4vhHlD5U6I-lz&ZyEl_K&Y-JhtiGBI^'
            'l?^L0E85viEP<wYQ^D+&3=eg9fPC^<<!JV;1}#V$Aki_Y<ehK4D^nhQH{TIgzI0;<qfPjEb<lctt?c#w72?|U^-i5MP`qIwvE;Ri'
            'yr~Ife(<k$X)kxLA3-qk$^N#RBQD5IPCi$)gtDGZR~B(J4rU?a26}yJ%RZXxot>xTTUg%Cb~38t?0gOR>PyH~HS*LaQ=#9lx!B~V'
            '%xnx|WHY8hCgU0k^_djuQ>oEo`AXsUY;%pCRECaud0elCZKFm_oouzRuC-+0%G6InOeWEIFzC~62}5?TL3aV=CRhYk{Gl@rXIUsN'
            'e?7W9zdSza9`)Ofkp%1aAaenUzj!j9+_kGEy-%dsl;YROx9bBtLrLe7i{451bkIM%_$tYH0uEK<x_+FGi~Q9nrp(Buf%TMuq@{+h'
            '*1unzoyrtmdw!DmYx16orxeblg4?7USQ;;=P(~{56Q4as!i>pkoKi%$FNVV*&O%JFVyxR1=V+%+MWrvCs<0?atgt_exLKj^nBfde'
            'm%VZ|Iz^-WyPTv^8pf->*KkI0$`x6q$F>YlDvrjtbyV{fJ<;3jxZOw`SY6|X16-i{!*A)gjrMmxxbO@J8z+l`wio)6ng%E(n4+g`'
            '^dw(4p1IZ+RpBXgKyHM@wL(%{X>L<E)D7nW1HX*FuN;@m_94bgw4S+nmNWQ6JKylY*kDUQ5s)&U(PK1^Z&rAdEuj{$pznX+i6uOf'
            'M?Ew7;2Nj)ofJ&a(;d3~rBkewwQ(B8=V$)Xj~4zSOx51m*V=U8jWVt{lPmrfeE8bBv#!6Z*fg6dPsd*;WEYZR0SFy10xX_GM2+w+'
            'eUqYGJJSYBlg&Izoinm^8UVWSZ7`nUPuEDx#~lsoBD!w|7y?C65qpx-<gcYm=Y{26${uqxdAtY2G|i^7DZM@r!v_bWkjXd3aUO>('
            '`ld4ZoeU9y{;>A>9Fw%o@tlew&wI$#)>FKk#B&o*sbeDob+U<a*Mz*+Bm`Xc2#n`0d<E!GDbG?NVVi<mAvxgY-BM|cc;!)$!vBR$'
            'f%TEmnvD@EYC3Jt!-;y{qeLz%ZC+uwtk4r~Bm+)(w5s@$x;e^3u)?2XLabRDV&pZA`G@4u2rIe6_!O;<Tv3Z_a~|BlNUmWP{H2$i'
            'R*7~Ir%TTi`Efsc&OlMiGe+nYHXgyln6jZ9b3B73B3d*`2vcA0xAMMm^9uY3&Mw}`&d~Kao{QZFobKZ;S%L-dr0R2B$lX%PbpL4|'
            ')dCcJiau|L<V(#{M*0wIm*<>rd7maACJm}{UoTP63ZhjdU6e0{^%HkH%UWRtYWfxk)MUPb1lLWn5M7(dK8DMaNfgDi{jZcj@z#I-'
            '{VN8S&Wx$^4Lp^}5~g-L4@Yetr7^-V>D%DLB)kD-qTz1LbJ?Jnq0u@EQunhGUSj^dB*P*uDVfHqlDBP0sYx;cRZtHas($)T{n`jP'
            '*5WJxVrlUPRkpSO$c^D^h0*v+6h1;uHC*Z1Nk(w6fpFXcIh+S}VgyPufb=Q^wXDGjWDjIZstMnC_5JrRUcrgOZLmoIPeX<8(MWcb'
            '7r!sBPKJWZ`yPp$4dr%=YK%&+@@-RCp<qk$JIpxg`=+DE=1Lh}T~+Z$`kkvLPlkib<_zh|0a>lwzPgT(VsNgDRt~FEeR%0=gwfT&'
            'bnSTbjbNa#Y43Ig1kUr9sIOw45;OQrMrE40lI6!YMKk#&nu6%W0XeKbw^WpS9@@hpIG}o=6<<d<DJ6Q3Fkp}D1bc|uduT#z$y_$l'
            'MGaXxN5*q_52Z53-2AKTHcrBiEI8f*=AxzBH#}xw9+&6GO;+8|&neJcl?goyaTKJJr{!U<EEFB0o0KEg{W%_twYW#0QW?I*QBakh'
            'J4074oGJ-SR}uUFa}@v)VBx)8-Wg2p0v|)G!h{2<J^Iq_YYNnkd1+*UVo{amsshM{<R|DY!)1WEirQ+o<qzYexx+&8ALZsHgEKlR'
            'N&`i!l6Q#C+a%Ab={l_3|7On9Y>^zUHVlMDN@>1vji_#Grkod40#BB-wm`QrN2qP<s-#NB*6fw{MF8{2f<^H0HEacDPXKkPlynLZ'
            'xs~}N5Q#zoI@Ds~i$?9<xG0dJYjw$CTF=M|jxY?5Wl-vu49xm4iE>wdV}>1oV!4dNC?mt7qQl1f{9#>->OhW~n2hq2J~u3*ZY0%g'
            '+X#)iaXv9rqh_pf(~9vbB0DM$P_dD_A@YM1kn$LIGuPc18?vFhiX^CtyeORIO#v9@-xSPVLpADyGJp;E3=`)dFM^8|JgjoG)zv%-'
            'ktF4>o9~+#Y2t2*dMWoWCTZFxCZ;?}O;dM^$eE`H<Cf-FmWU+<Bit-Q_P3g_>CSvoj<l-`Gb(?>62_<woBg0TtWD|78QNcPjj=Y)'
            '(JjP06$CVWu*$Ejqm%@oQ-<iC1l#VxXTmXV?VIyn&l~i6=iX6w(8V%S{llKouR8h>O<CH_X~UkYvNV(?brReJ30-k;ahgH*r<4Ba'
            'QUAQDs?w~3YzCA(l^KpF$eL6{lTwYUrOe!kU9zq#$8N!yTx@o!nYp?q%O7!-9(fn9ANu7mG+list!Q6DCcE!rO|BBPhWczJs>N4O'
            ';NJ&HD`%-FI>r$iG(q@Y06-_HR_L3k$StH|4?TN^KyGklLFI!m<-pl5r)0jG$YM*x<J%ys25XFU#>0{XD67L@<Ss*`B5BI7^n~ni'
            '%GZVhH?Y<>W=~1j*@nE6xK0hq?us?Qa1`IaSk413jQ3S65p>Eu6ggzlA<lS2nq_UWu&wfrZL~_&CLGJt>U64=w{O2?0DH50Ke<Vz'
            '_$mv0c514R@S<F!GPM1^yizwWzlMhKx=a#Qt{yx#I0yckX&~G_(N(7+(N$sisq;`C@A0?jx9EYt{8-GYgyAOD>E&v|sd<IPZ7%$6'
            'Z%qZVlpj@Xd+CWiZjKN7SF<uyk~1RFHnnTg$no7}Ctsn0YDfKON{Ds%ABhFZ^oTd%PZ(cq(`CV>rm-pM7K&~5Sc2?N)uS=nmIaNh'
            'z@3AtARTEkw~mfgf~?w+4ev6IctBNstS%!FCnK`6<EY=w(#ciz%T|#Z={C+5@q#@kP^XsXRwX?AR(9Mbhs-pLfc(RNEgn(uh@!!z'
            'bGpq!+O-afA*+QlMVtrYAiN7wA!7@H5G3%o^lPP?G0cKBiSJXfC(<z>8W0z(+7iQGAXTR7J{viM3LHZPCKin}fC(HQpZ(&UcYpCP'
            'tfK1;&O9yE6ms}g>{bC$FqN6Z>xAEOuPeGSGGXhTJn5zEhl%G=8(PC_iB`)4>)K5zzWf;=;i#$c9TK{HEU)ytYZslyq$KBBxMT1e'
            'r?}mY!F|51MG}U-qGXuW;`d^S+|PVOjU{>F-YqvlUYdUD9>_H<!U(W2y2);>O+e^k(p0$v>Km9IRr?IMSUdLR(=abVZ#$rmR`Z!B'
            'gY{(ozTW}%+wkVLbr;W9i@?m?f@z2p=zlC9&@ciDHCGHjK@Wl?z8Chapj(*DgFVioP1Z{?m1h2pvB6E_9CMDRdEiH@B~VeRV`^Vx'
            '+1{8Y-5>*hxI?eqIQm2sso$Ck7;m>ZcVXfK<p&OyER!Jny}JVs4$u1odT0-du0Z4#j(?`)G9c*VW^NL-fi6{eRZcx5^`@oXm{pIA'
            'vAL<W1edDiZ3h^Z7^uLo^xCnZ!s1Atxme{r`~DLlR+Si17P6)%5+sL;yIi6dl+0K9y#2kAN|N*d031pq6Pc>X7Z)eME=06ii~yFn'
            '2k}y7CQ~!lIqb14uD@m}CRw1GK?!Mn>=TYD-4<7ir^I41ZPUG!xkjsuoVeI8#5d}6(L{cYSA*3kz%!DBV-io~$_IEFQ_A&&=^#Uu'
            'nCOBFHeQ||r+ed&w=N~UM&2U_d|@%eOME#Utw=t_1U1wE8S<M*ogv3Wp%Nlg;*aTQs0a7Bt|Yrls@w0<+lFQKG=0I{x^e%xB0$xQ'
            'E8lKCu8qj-)owkqS}QQX`@29^fY14n^HyQ7pI}7XMfa$8*b}=WEWsPC04?xB>DC{SWn=;f5$@ux@Cq#=IvZ+Yxp!K)Gb4k_U9k2G'
            '2=oy?7ZC<w$Z90k@C%Eu^$t&Ze=j(M+D@IO=VQh0m@Wx;qUXAts&<1+4O}H2k}B`VVGhNs>vlf6;uR`8^&58o@zE)A8e<+bLg*WP'
            '+vkU3_*Qj4QN6_;UykWFhj>%`OE4M=bW2J_W0-%FlitHL?t@W#m4the-Ng$(jD}KjR9Wb+uJbId+JSRS*Nf=)W6&M7l#=h23AtK_'
            '1X+hbLK~Q65Ja+)6c(j&H|LPIoKZIBk=M*wr^YuE$d@<9DHU~7Zw@wB{;_WU<)?IQUh_-59o3}}V26x}VngT(4-175U=#CZ@EU6p'
            'Bw@K)0!|;>!k6$LOlb^YK6tvu52sZNqP9PoG@z?JvfQ){(L*?!1N>%HG!9}PytSs!4=X%>97fY(d->F@jM|YXCZ}0@(lx!@c+#<='
            '%n&{*voEV&L!&YSC;vq5n5HXqGpyoG65Y{3S&>Q*+`<LbkQ`C6!m&&hi75anBj>#jG=@GK<V9=c1o+nL3Z#<jEy{Xqe_(J3{KR^5'
            'ZI7yx!?;=iR5hXla`H!{=c8IwY5(O5g=^Hw+GW{5S%}48aG`{!BliF$q2Bv0Mw?1iTU$q^l#Ua?eIjQg@|jD7G<Lr2o%B!p9*pSy'
            'eAYkgU1*l){5@FSt`jQzuo}<s-@{u!AtBRwhc`s<@0%b>LTYmyOnm(J4EP=X+l>QVe$hDu)CrU1Hl|M9RTfA5vj-~l+@EllzP}7I'
            '{&~3=!HVb9dVUiIBVP@51X~R1DRl-e2#T_76hz^O3Q_o}4}XE0MAP!2#Uu!+Ll=-NiTp&3)tyt|627`X{PSrgZ~bwy`oQa+V~+$B'
            'EZIFw$X(#?z_X(OzquNPYP6p*?e!w`fs<433qYCi?T9~(f6PJK3g{CTtGPe*vBtH#qY^UfEx`B_g0J=JxO@8cvip|aW=U&s?(BSX'
            'Adcekip}6ka7Nr<yWxR&{!+|-`q&iDUbRLcS!sd0p~dC>;fZL#?=hEw3KHOu_>qh5$$Kn$iKx_p?%w|XzWL&p{?Xtkd_xZpf9jr|'
            '_Kp!{o)sp8vmoQ)2W1fA4;-GI4tjrs6`j57opNk6^nUT2J|s==ii!~~d*qX67nsRbfZkX1qG`B3$>aB5sq9Y}p%z!F`@@gD0xvfG'
            '6RE+LOw4Be+!fhJd33BgP|-3kMo^keStuyGH8S_N-5M+AhdS-6m+jH3my>|}9PxQoIwB%@yzSJaN%P)$G7tiBj$$$Z-+TpnCWv<='
            'I?M13MziI3;bpOR@83xjg9isfA(f95)I%eAr*JONl8p`^^R#20ff{c*InaGL$!<HvgTx;G_(jc$e@Xmhs*fn3pfAq*Z{F<v(*3!Y'
            'pH?YY8z)|d2T2Duj5@#pxTv!jwqcb+OnM`Wx{Y81P#Ry({WaeEvZIhpFiqhL!4kCX{Em&CL)qAagd+oS0aT=+iyr{h7)?~DY)j`l'
            'S`(<v(<xO&z|7Mzm5!k~*upfnB|uXKD-Cw>t%fuFG{Kk`gYx`&c!Lu4rj`s5#6Jfi>4@*-I@(JRLmJLs`aqChyrQDt5h4Kv<cu7I'
            '_;is8?2uRLEOh<pr#}3@gA}%jm#AtY76vu*HCAJZv6!Z{09enTT<iNIKGv0s(DSz(hk5zmA-)%5ZkWJoXJNMdgb|;I&a?gJFVV?!'
            '$zDMljzonqNMY(8p7ilv8h!+2Y{gc_y(o@aXj6v<fTv{nl@=ZLlmmPN=u0GjPZ2gGzTp0;1%+7}%%@F>e^C%jsJ6{?$_W6`dcsI4'
            'MO3=5knwDkem%?zh)5aIJMg{imVKvXQ8~0rQ{|Z80|=w&8h~%xVakw04xsM{a_0kmR*}NgWgT(|Iv$du^w}%Y7*u})+@2Bq3op=5'
            'v_6rMqoC|@p!%I2^ikg2!nEeRAY2EzNhQ2sH9GA-WrH+^V;`o<D{XVKM%Dv)7V;+`gQVUPQ;bl1`Qp)Z%x7?4DOg{ZbZVS+np%Dy'
            '^6KC~cE(BbBGF}we^uyQc|}n~uSuM)F}+bdU!@etE}_M&i#8lxHzliZO;BE$KvRhba^9v^q;bS*BK|5~zO>q@{Vg=><R>#Pdt|(Q'
            '`I2xR|GqkSd5st5is0$ze>!;fT)si-W?ZGKkK0&#EnU5HXBH+H<mgsQ>^%BE+-ojshg516NjjwoCQa!1{=th^*SQQp?o0@Pik^Md'
            '>E-^x%a_;g)&8|nm4<dg`Pe)Z5Yn2SiI;i>^(m{$IEzn=fb|@7_E9F=p$RYq{4?4Nh7J7nsys~L+|wu~TkLcyNwQ_pG22tx9K{~h'
            '#|!Zo@`%Hy@(tJ%Tr#MZB{ygldwam4Y#~k`3yXB*QR(V&DXW44L79SKXFW@mlY{91#`U8g3iWMMz+PIoOfpCY|M(o5;S?$m%dc~}'
            'Sf(H{Q(79DQEPP$?cWL1137I6bH7<r+~pB~ZS=>2l_<h^=c?+$>2i))bNRv&I(k`_Hsqf=m|b4sfWEG8Z(f3mRW|)x;Ap8hJ3aoD'
            'jiTyqt}VLi2AHB3rq~Q?{ZR;O7?EQbTkqJVOP0!8EGS4J4OFca!^exXLfEci-B)w^s=%cg^eu@UhlKs5W2~)Ya>X{+I#i6dN7{|p'
            'CfQUMJ<<^C<dr;A6m>B!m%>@w1m{NLG=BI?=c@gOAFkcs((hyxyL(_)LysDAR9DK8YAg!w29(t^<^#!%3)WVvzN>7a@3HNsdI{<~'
            'JEjB7K}};!p~rtbV(DV;`REiuUHG)gC7zt2@;9QE<(tQ9RE<}erv=cvI2`MOS|!^^C8eJG(aj1yp~prot}5F}W$y@6IuN~?go&!s'
            'E$ewX#yKJ`KnpWZROOIGm!7RIOLJvToAy|D!h;vzN0qic6(Y1sW?_1Zb>dVB&c@IJOJ|piM!Q2-`Emh0#R@wFZ(XZX14C$FV15dz'
            'j&f3qrK$w!%cyw}CRn+o40k|8Ebp+1XlStlzYT0sNeqHnGBTW|$oYb$AE?XW?)&%WXFqq34^*&N(cFd88y13&hN$ZLD<A?RtPBN>'
            'cXUY`zHohi-ut=V`vn$Bckv-ty@LkfbS>+VD!@l-MbJ}vB6H~1P>1CZ?bd$ELpTmC_K;HPL}1P6cHuU;LT~QYN)W@X*kgM;TFlu='
            'AdGHdh|DhiL`|`n%WggyyysSQSV#SE_#4vfDON>mVyPnO(kzCyTlJ6?QLzLWSlxi{Xt9sxZEH;G+$ya}{q0(BK}97xv6ul$HTaKI'
            'rH?}~-;>$otO(r=ETKXtkRd`&mZdV9t0JvZUb~}QQ=vjZC`OY|Vica$YstxO*s7$8RaA{qQ+1^_ereFXc;}s;T@HE|SG1h#stcoH'
            'z!HsU7gwpMV<Jn5=sM|7NfMRBk9SE{$p(C!h98iFS5Q&4UnuLK8?M8`%!Y<6O*z}RE*sO#3n(!ylh-Y!n~e<-eNzYh+oMIW1_{WZ'
            'EFjonk^>s|56L44y|a()^2n|u=P-`v97H8=u1H^#cUPb6I4y^2qZ5`{>f5@7<;^VBEZ~dj&Qp%R<!thGp1}5oe<wx1Dg!mT30Eg;'
            'Hp<_|6QHFTCkDH6x|c`&Gv~Tm4RZi60s3Tc72gA~ArrL4^6*HRe{az8XoHO}z_AU`Gd%N3Z}<UvGr*tp$Kzv7*5~r$$4yBrjr$Hz'
            'p_Q$E<84UQ-GR~YV5g=eLcIZ<h#ashX)&L~5%A^7$|+U?QOGHULF1!U<y{yZ#W%1aX;$>PNM#~Zb;-K(8JRhh`AE%6<-i&_QpeJ7'
            'J|T^ZBT0Y)2EfKCt*+t=J731BTsmx3x|K^#_i29Rg+eGg-`yWL^txP!h<BCDgxAFiBX*lSyHZulQno~W>w?C$;zdj`!)QUAxCdcK'
            'n%f$>lWU5cY|E%JbK4S1%`(v*x~XAdt@)o;TOtkI0Dl$I^_gUfEPdt^t8E3pq#hN|5$QGLRe$M<qs)o?r@bt8B2j{X3~(bz7!i*a'
            '!!0L*l~68s-T?Qdl-Bc%v)bCIAueT8kFz<HJo0i@ohMzCBK>Q@xf9XPlO5;^%0XLP00J7$co?Y2S<AvC4fa450Ld+)V10fhWb#uA'
            '+K;ms4skdWH^CTPo<G7q0HrVoTnCe2h7?2!^+tEBAo25g_xPl1JMbxUk4CC1WxcxITqmqpIG{djr8%~GT<bi@@lil+@-z_mhn;Sv'
            'tkQq0ADZb+Dr$z*GUbXGSPp$FYtuS%U28?HF-R-0L{xCKD#TYz=B4RM>CfHyJsP&k>SIIVD5DqM@edi&!@XfijTknyBJKZw-Cb>O'
            '8#faE?q5NuA1oD;6Zeo@YeNN`T2726PGKj9GYmtbv|D+hH&?55qqzR>H*e&UORg;YQXFvk&?YOA%N@QAXP$Y6TciuT#>m`@caLAa'
            '#9zDVQpn0J-w1ViAG9}6|AK~GVdZdbVBBz6BH}DNC1H_mjyZO^_9!wzqTNQIXGRpi?$s`-eY@8!XA*dE-d*=DvJ++0=~llkwi`J&'
            'DE|0vl$->l#o=SM5_73Vs%^>&>sHu7oarX}l4LzR#C-vFWY{%8(c?WPLG-UWJ-awMga7{0d*A!eyEs*EdcVFuz5IAyO^?y|XvssK'
            'M-bwEb^N0vOboGw8F|*;2{}ltgT{$TFVEhCpbyTEZu~o|;8_x>!Kbdz!N4ag6b=T~KZj|2V`jsa*n%6K2D+Uhjhp+%BQJ1j#*X!{'
            'eyo&7iV-`x!)BtzoOZrLgc#oQ73jh95gF~6dkakfs}ZGRhDP^zZwU8WJI&lSVJ}1YI{aF|+|GB^Zdo3UNquTM&81(vYK%@M_S!=)'
            'Za{VptNeHV{%xDutCI%4A~xu8VTg_C>R|u%wb7&M1#K)z-oHKW4v<cg-r1bUeM=i!&6!(qus56WMc^o%)Ytom7XNJgR+Xo2tL}^$'
            'E%H^r$dkUggaj#WZ3%|_hWT;-C~ym3_bH&Wzu$iq776-Xx3=BSa{ItaYgo$w%RRCjSBtFXX$UmPNKYyye6kWcYV*E1X-g6cZzvx&'
            'QaY&KY?vkqQGtnZA3-zEVka;gRPVzZ%yoA5wtsy3p?7w1`tb}7?|;7jF;apuqbd%ehsD!LO?Ns!G<FcV89qmIF9K=?c*YYraB#>N'
            'V)(ZjFH(!FVapIR819tsBts?~c_*j;CQvc&`3*yv+0V0CHq}e!eQF3>pW*T?e^h@dRx1t4b~MGD@JOpx)+?U{#hFS1M>i34s{1+$'
            'LzA2KU&@q-)h1AOeO3CI>OF*BFDq82kIXGUp_tQ%i{5I;F$rzr>VuO%o#*7WVz<<W8M2vD5SKHa<fMdE$O50`(<(n)-DR3)3w5eP'
            'FyOOK%8-LGrqHO39Yd=l+Br<>@+))C$))X}QD7mupE*JheM+r8H3(<+KztgjMaL%2$-RTmyouCtSOYK~u;t-l<_5nKne#0u25EB0'
            '7HcOG!unlkS8A7y^1v&xqoNb&7OpEKadq+Tr~`+Q1S{!3SsS*br!+tbBRI)bKd1~Dc~L4HeFP&w12oON>j)!hE~Lk#1(FYfp89;L'
            '*vbaKOF7VgpZjIJm=&ORz=awa_U8oVZ9ksY{xQNh&*9>H$91b6^|OQxZ{c^5)G+KTso5jxFskkf6wPj`qwfp>FIUzIkESXdn-&w{'
            'BZUuhvd;1+I~<}^AtaEvyI_N>fP32BiFpRh4b83yE!=;w4K(e;)`QDt@(hEpv-NaqB7~g54g%uJ-=1JD_JpNu9)*|@{Vv6f@p&25'
            'WE?xpriAIvYkI&Cr}Cqy0(=cWkB1ZRZ%_0F*bYEvC(zlVV*hr)FWi0$4axo}KJAq?>5D8!{cy2{?eN|<{~$#5>fY%@B;TXoXA!;e'
            'TwF2E?T(iaA+Wfm>sA%xBQ@ze32ZGB{Ru9Gok(^juc-$$&X!5W32E=$w?4=?@N{a_IFbx2I9GbECt@=TYhKKnaj)!(r6>tc<de4Y'
            'XLu*bliV7vvC~csGIaWL(JNTpXQ_PkVO<T*5k`OYim)i~-YH1nE05H$Zl4aDAeugFvZrEBo)TY83k<&WnBPw|YfZaw(ZeXDQq64Y'
            'wi=RRM{DB1)c~~hsNr9*TiULJ;U==&M(`S?*KmO^zG8xe$x72kxtPv0Y?d;`F|Qb{`lakQ$%4RcFz>#o@70>5_@%9Aqx_oPa>uzS'
            '>S%rKl>Wk{DVU}O(+5uES7phelw8|(snMZVl4QWmcGXLk1lR4lN#I1taN>J(=Fs;SsNhlq-%C{!hBRN?rBid6E<?=vynx%*!rQl-'
            'k3T%1=k2Qw0yShj-Mf<>YgF}YBcM3m7DG^bZqnpMNwX~{mT24idoL-c<ch35(i+FEGczbLl_KtT&frd!6rcs8^1P>xQ&^g)aU+ol'
            'H8;-gQYUMs*?nWlOsd}+NOoT&J7gMr%&rqo;!uVQ<oe*HM}Kv*+*!GL%Y=Z+*gTp>DA&+F{sl$Tp|`g{JA7@%5L5ZsTo8RLjeNs!'
            'B8?B?u@+Of_|b5}_q^Qu>4y-Zdu-Z6V6r&+^k})ra|9u;#0F)^8DtO#=0#!78Kg%W;+=&1g!!i=c#NXG*(Rqnp|120rYARO`#V5='
            '3*Rd<gK}i~<7pc4Vz#}vx<gP7Gc5u{t!y!<(t8j0PSt<zI<oaOR_>OK>2FY%2qr>ZYj`#Zo&?k;j&(Ke9&@QRrA3T}Uw4I>YIdMH'
            'vn#i)Yco;APSGE(-EX6}PtlypQ?X`l{P-%I8*i6snylM2GwiYxswZZ)aZDpkMs0Vd8G^R$J31JcnUs`XQo{t_;~oF{k&&!^K`D0P'
            'H%g2Z!G|#MdkDm2URhNx!--&qU#eU+MyLQUvV_>xYKPwWu&c%lm5rs7m67@%hN4O&3c1aLx~=;5%PyhZqPkRbgdr64f{xlph#V-O'
            'FZhil90>-%!!6QoIQ`9^Fm<RM0xy)uPc~e+FL0F#j9kX3Ts=k<a4jYQA_mcRQEu1LO@HxRxpha5Odt5$-6?S`OULf2eR#Wu6SFBN'
            'rz{c1D#~x}viZlW$dUOQmoxSo#b9P6)ar0nCE8_qKcPfTsa|$$7wmy)mW*^ojwJug;Om8yXtg5eOp}ilQv@c4^C@!7uo?LX4w&&$'
            '-{ILVcI1=R0R;ptGs1+!vix{*;+n|nId0KHsmm5E2~^*0FQ5j4ZXr|j_PIoYKa}r$0d32zf$f%Cr#|^6XnXe)J+WGHGPu3K9xun~'
            'vG8O20Q>XLXMN0A`S$$iU#Fig`{%uj-qHCl?~GIi%=I>dnM}-KAaM9pPO?WBc9U~M?nKN4i=`R)ExMZbjNM{B0A1WqZZS#^qEQX-'
            'Q%?-6g|cmO)nU(8OmnYvzEegyyQw?LyFUd5?+BGXTnEtppgNKRH7t^Giq=YM2W5Tq>ErpOgy{-?A<w4Fv<B-aN3ugyJQBU$%>@A`'
            'Qa@eg4z^zq6$5wZ2C#7kppB?*X{K#p!>lB(6@zAMUDTU4jq52wCtg}T-F`QsrAlLGvr#v<7z3l4cY2xYHJpm`)&V7sUFXQsv(yM+'
            'nrZb|A{9*eD_|#tuD8`qHX3G22DO)v_Pr|o(0>PLkBYU5aDOJcz0yc^Zel*=%A*K-JCR{kiYZcZR?X^j3PpRlb)J~5wp+IQ=utLN'
            'I2u7$hQ(@>=0UO;NjI1jOQg}wL5%=ae^{)G39l%%Dkhk^0!KF2lU9%=M%HaN{-KuJej|(c&s}Nef4Im5o%7i8`n_SimimU^1S}<7'
            'Q6Svv_jm94yB;22RWd%%qRzARVtM#~y`Weu?lSroK8MXw3Xeo6)HodiHg(<(n)_`t^*ftqcf=v_41o9MS8QP?b#M*w#Y3i3Hzley'
            'p>9oz9@tnT3CDxUaIMVhujEWOnuHO*XcAPRycEV@>D@Y5x(j>tPSq{Iua!;txOdjOyy(4G+SLIysD8y^h!_5hD7_h~P%zSnEjFo5'
            '+SLu25W=t>7c=JmWMJ_u{b~=|JYB=Cu+A{!ss0*l$n{uUci_t;hz=gNq5y9X?et-twG(Uu&eJ=}^ca2kjR3i6(48S{6X~=JLbFC)'
            '^1+sm0v3Jz;Z=b9^g!oju^=6sKU>R!(RJV@M5`94_GsG`^7m(pIa#J@2h$2M?TO@192a_-m^?8*efzwx#O#e|hYyG!{d|j=$=@It'
            'I!&7A#T>blNt=>W1HtgJ91s@2T=h%rsYY&Ds7C)UpYv4}jbxVl+=6W;dOg<@&Op$hg;O%Q!VTg){k<Uep#?_TCq&FOTHLOe+4u(U'
            'OVmF8O3}R_KSlBbT@0Cx@F7%RN?LWkH6azk?TSLE<(l&zsfpCd$?P^ABkzVr!$;@On+y)iB|P|TeSwJ-j>-B9|DI;U<syfjJVky`'
            '2`pabyz%O}p00C3i@^FjWVA?qmWDQwQ$yL=E#g4%Jue8Ew`1+GLRnxL^(@=$c!6Kec+BXXt2z!9O*ua1JHB^xQ$h&73mvF?XJj8#'
            'F@mK68nBw^)eOW@-I^~y|N6dnhQ!Il?G;U$g(0izvY23`{Y2s34Z2*8B&K@3)Z;+^MumN(L2pO=_BdUplWC@I^kfPbGSDh-vRk#7'
            'CdC*JgH^hQt8lvVwtTV3*r^QX?mU~1n|=ac*#jIc;Pd6t`RT#Nv>Aj7+EefJjHmevhh^Q$$_ZLwv1M}{U_|d`1iE=lSZ@lrx<Fx&'
            '4Oc_KIjz#YW}{>h^d~{D;NLbP=0BPWT9N'
        ),
    },
    'provider_adapters': {
        'sha256': 'fedefdfc9b523bfbbd34256d74bb5c83160b0683f01b86bbbdbd16d100754c8f',
        'encoding': 'zlib+b85',
        'data': (
            'c-q}uTX);GvVPaEz-AvTo=TLHboF$(b=GklCGn|aXUlRg+uMtVNXSCPi^_`~*-QWZ%nZOgCCiD^?i@YX5=jgOgTc%<g8>c0I9((%'
            '47k^0P9$6rB$9jm>y$^XaN$xI$NfbV-bhzOjJtf1h$v>=4>KMM_Je74I={aAosHieow4_)ug{N%Z1~&J+4!u}Z~b(Bor!=YGr``5'
            ';k74N6sCz_GG^lAjHht|opwhbUbA=stn`>4y24}M?S2RB$;e4%!a}%XB8kpIE`%oo!Qxq%dM=v?#wS2C37HeZK!Q}TFkqa$K6>+J'
            'I2xXeSw570pWV)+GsCWV#}n5wkv<1j@+X`b53WUz1!2PeOv8jL4*VOG5s>Z1LUgXKX&5$_*1_@d>96*|;o<4|$=Lqq`RVw;9>JgS'
            '(do$*3hE?GW-_>DGZDeK(;#MP;E6b9GKovVM^VT)vDg9gWgKIRxmY5fC#Pe^116#<jP^PxL_Hxa5{obbDp&l5OAm#x{{|?DD|RcB'
            '8Au@J0SxdCq#Fqyqfi2urorK!C!$U){e`y#ao>m-WM;9GFeff<;ExP&+%b|$9S69crXJ9WQx9edi3K4{L^wsFen-hp4Fm_^h0qVo'
            '&y3&5Fk;;)hnbH0Fat2p5sMSPjES8^1hWt=SrYLeb|N_uQLpm><~bS+Jzyj_=98G7r^oXS#^bYm=3*4~5SU)FKqR+eG^bIr>rh}P'
            'pmEnTl%ov(5DqHb74?P^yrL&jemxe4)Wz*A^h5{rF!i*aipY^MG?M|lf?niwY18<u#Zt)|RGJi!JH#6gmMC?PiNIZiGDuJtNT}(;'
            '3%MKj*dm>HGM)+i8c8i@0XhM~1$o8L3-@69itAR#FpSPL3VmkV(=<sVVcQI55k>($g3@x-(zw&n*Jg?DW#9h*4Nz@eP<c>%k+sZz'
            '_Za@YgE>{tEqF5X<Rp9S1N`i$J665|vYXuwgU?~E&?7PJ(Irrq(7O?~9vNNB6w}sZNV$6*v(x!zj~#|E7wn84X94$V{zNPaL*#|w'
            'oK3?BRtwB<sBM|m`7k>D<>>WrWS<QWM?+AKeRfa%8P`$xr?JPn>Pyd5Y#5FgrtZ|^k)WH!Pn$bVm<EY0U8Bb;H_~KgC*fQKMn;9-'
            ';XHs&3ui`j$`@={#z>-aEdt9E5`h&Bc_f_B_d!#d)|X6W%y7Xte8DHulgZM`3@P_+aS=glcxEc2IQbFjX?aDc$UGs9COQ$Df~Ln{'
            'xksnxZ{LB<?4d%3VD=-cb2vIZJF{~k!Xx8hNrp3dJ+uEzIWk};;$wnirCay`(Gp8A))P=5EgOA@gw6~!;#>}{7fBz-`0V-a#Hc)2'
            'nh$vVf1K%HLz7JW#Me#erM~zyw*I^o!2tifu)gm<|L+HRDxUAZ*q!wGi%EaF_@U00@F&7`!4$;n7&5MfF9X@npX@(>sOt#_VN9+?'
            '-+xhklxz_`6~poqfg}24-e8gV<hp$@*`PWB_;_&oOK?V>S?4%2zqFoeOy~4#M!fs#XmoUPcy!!jE+}>E3s;&(tC)3@K8yzjaPy)M'
            '=4%pesQ{ZB=OaiS5;pk^S_{>VD{HuivgnUT{ezh=k`s|eF!rJ=wPt@oCm@B}=SlxIk_kv0P3i#>VyX=rr_mfNJi7@ccu%Wy@UQdH'
            '&>o+Uem**TXa90~bVv)V+tOXm!_XIA#sYObhShwS%ZZg|mZAP@_#~BYMF$OR0#Ne9IANZg3vl@WmRw+8Sqp$yh=<tW>B*a;x96jS'
            'R{-c<58oV|ACG|s7!V&Y8Un2dM6ZX<_%)Y7oFD-g{F(<76~$IDffg900s3LM+=m$?uHpfZ0i_i154|L)A<_cu(K$RmJ%9b?_y7b1'
            'm~wb*|1umMy*WB0C}jf*8lt4sJeP+5tLlLr9O-e-r4NGycq3UX=SnMO^dp<6;Nt@yoCp{^SRl+$v>wHLdIxmDOb5&jucOrCmVsWY'
            '<BDnS=rYr7m(X1|ySN81ChBQ-ZeNQ87SldiCf0w+T<IixiY>z!DS9zMv88ST)0Df0KnR2p8TcS@AwDibLXVu?a4!{rI*{B*z<L4f'
            'dB2mj-s!QO95><*1rhOEiSL~~c0Y#!?^QkfYGo1GCi0vJ2LL_zo(#SMa)V|{!2+;GXnp`S_NP1neFaRH2rC;g{m1TNnCiMSw?J6A'
            '7)^RqP6?+|i@kg?M>JM;Z)6XHNyxsUBhFZc%fxhH2C|Q#WC0IM4RpAHFP1RnF~9csQaTcZk`eqzv?72|hzwfYuU15}4Da_dL00il'
            '7cbWI!9;9BqeACQqf_*<$Y7scI2QO}h{j?&iC_g?>|RnM2V0c7wMYr#u9?Zo0nHAk@*0ekdzlGQ(`|~dBAHK<XIYll^3IchQ~506'
            '5nBR?>)xZnS$_aP?rJW~mHA^KYOrX&ykk6pwd+cwM!PEZ(y!vVRp*g9F^*CI!Ufw#bAC&|R(kQfLIp0x>L=BXyW6bvvpi1QDQ0J&'
            '<*~zjzM#ofb|8bhIsaC+avHN%7@gWo5^p%RqB{c>VJj0>*Vc=Y_%0i3i9TRE!QSct)hAI=L%Ckm*G9docBP-^%jB{?@A$z-3f`4@'
            'fd&ox0xsN#mC{kI)wJ62TrL*Et=n&<9r{Os%jrUGejBs0Mh6QjsA2gJxI@kdE~4@vik>%DnxXkD^cSZ%MWN=muxXks(j+5Qw5IsW'
            '*we%6=?gZ>inJux0=qefco($EwK)>bLP;wKZ@XDEX{Cu{S~3n1j`F0dp;Rv9DX33V#x=3(v``ORoKsF)wXLbAlezBmA)%`~WvQz%'
            'Z6oy7V7OZwrP-L2e2pqPPdXuy_%oG&hw~IkH~(htWnLq{gw7p(3y!bhL-MB-5FzsLhXZQu0@|I0;Qi7FL_8B5(>VJ|%Q0DDPeOO8'
            'f5R}i{)&SDneKnTYZkrfECezx;v{D58+{3ks>ko72TGLC9UU^OJP?4Ag294w_#J@7+Ry1(`b#^@`oH3OKWDwyZVc-m8XEp7hNl0U'
            'Ie{cv?p3HQ_!6^7#G>NX!hX9&6Hp+rNZ646fyY?a{@$=7err#bARaOSnjGe`K=)Sl3C$l+x^6qLLV6j!PI>ud^9)+`4eLMM2dKS;'
            'nDU8<cHWLo|7jn-I~d#Nqhk<^F-wv~yf+x|g|sk;rJz8^qc_h64n#%+xU}#DCEW}~VLUoGIXfI3y&7&who`O#2bJI(XsMNmiAfvV'
            'P-X)yq`yeCIs7ke5$PxP)0Qh~3pHBxGG7LImDQ{-jiZqOYNh1!k19bHLud)mf1f?ueg0h&^_&8`2S<IBAWtNw@KK_ykSM0{aiT4D'
            '28K3fjwg5k!7F9AgAgN(9QzQWQLAA+g2_@5RZvk5C5TJ~&@>EM4heE<s*MJWmu1nJwJ@L-MghONaT#A;UMR{6OF{I6t8}AD?leq<'
            'TARJnGDkYyD)9PndNv0BD|%m@(agZz8wV-ay-411og11OuLQ)&5xXx~SHZ^0SY!8$LuFC=<K=?DCtoZ)=@7*M8ZTp2ZCUt$`-VR*'
            'lngKCKh~OD!xu3G6=dQr?u{rUtZzW58WFx%tSNK=OKf#n;-#>upe0JO>>vB=`R;Djrbb+X(-@@zn(rZb@u~4{JpRD$Rjbt=ev7(#'
            'R?Le%pmDh}N_HOeH~ZPNfnLlga|AlE3u>XqfEto9e7P~~&2w*$1UiHvn_=-LyaVmZSO*8vD=<#G*#YsxaBT3Y9Q_|kUk#jMeRa2-'
            'ru!U(qx@LiDKd_GJ5F<vYe<?2uq?867-}{xcG;qL$;vA)yT|BXiv?ek<=hM;S`2){nonUgxRsLkMC@3%{KjZA%sRrXjDWTdCfu1@'
            'taT&r!|3S!!RU9~V@==&7;kT>&;V|F-ex<%YU>g6RIv7C0_L2@^O%A(UUc!FXx3}_@vq0WR}slmSooTuCIs8G(WrDbY8_J33YcBa'
            ';AUxA{5=+Td$U=a|1W1dD@KrTmie$L{mV^aLmcbv;!~sl*7W~VhXWL;mNtGN;_DW>!sV1ik@bQ9l?24Jg8ggT3yvp2M1z~}2AXU<'
            'xGy?b4FJl6=^WfsAFE14P-1I~xM1Br>IS#6g-==Ec&pH%=61F-^p)+wCo=a{nh#-lJwa=&dwU6iGWP^gw`y6`82nX^VvVLNgRd3J'
            'e7x(RL9Fo_6?p$upmSCERN!-Eq19DTXn-*Ht2&fdck?GHbz;zVjEr&rln0LJW+hd)9!!-j$2C?S8R|8Xd~i_Z&Cw;L7TF|>AM$wB'
            'jraf1aDh6t)ef|Z8*LB^s>;Kg$A7WQ)QXZj&5C^2MH94Iii!G$w)!)RBi$^|npJ~QQ4q>?k|uf+KTw;%5H{iteXzXcORL7!g_iOq'
            '+Jn~AbM3ia#)8?B3RLO6Hpf)dzjSeS<28<{T+y<1vTA+e>ML<8<?5iX!m-fi7;Ze%Twd1fm!4pPL1eu=87J!M*X@^dJsFMu@9{Y|'
            '!b!piMc?gia66{DNNE(4u3cQ#-G?UiC=ZhB>C^a<vfh6=Z=whKutRZS%t5S_1LD)JZg{bKX@1@Ary8+UC!{h3E$-*(<F#r{-1zKE'
            'glgn|R25&6#9bAbQmCQBBG3*S3!2FfQ9)krhgz!dI-e92HFIh8q30<s%6Mh<fdP*qQYc-nbu8yXmwQl+JFtx<3zS!bm4-W)Jyi+j'
            'rVe;|OsSyzzly|-6t|f<cdxKWpD3OTK_!94478UD%@_D+2zHA^5kg-(aCw327)z<k6<J_D=piy9z(!jY<9g+5R8}UacSLZlZ~p}i'
            'b7n7H5hr2}%TrZhq0E4n81tz2MCKpSN*D8O{!ESTgb0ZMnv#oZ8;ZFy!Gt~nKGa6|L;#{rwV94gGE0o5cI&)<qZa*ACn&H2eY;k='
            'TK$oTA?8z{K+6!G$Abu#h`||%^4dg4g#}i4)JE43>ejLE<@KI4c{@J=JA4ktJwl7UZ#=Vh4ffIZ_B<Y3@2f>@yxb|b&}z$lv2Q3{'
            'Ss5?yOPsBK8qnpJKMjbwZvJ_orBhY9ECqD0<fu`H1=D9K0b9PrqW^H3sJu{kjh^;KK=gZqyFY`QXT}<c)ATB*&Z2*?kbeIxo#ffM'
            '{?7uvFK(DB;M954<{VDOw;<^jvuOO*f1h#RKP3nJhMX1B07p6PKmTq%yX!yuej<~;9|mE6;rKN|ZRup}(-Hwz7j%s@nf8Au?*u&^'
            '04sqR%0<_s`%-a4J!;p_Rl@eH4x$FVi1fq2OL+@4`2e%f#4$Gv+%$625}WIe<{%8P>5NYHB>n1Uj?c6ffB*nH`*T+!<isEGzN#0a'
            'GgWptr=R9p3RLK>#G!Vy9)N;%<6ldtClwT^e8Q)rz?$=`KtcrB`7m1Os(lE&+j#gv6q$KdJM*b_=0`V_82@$XZ<7O7zz^)g7fI7s'
            'Pcr4m$Bn?aMNbU^$HU*o_QC6e598s8PAz4JzN$4ez1pd-wtRYC2M1Fqac=0Ut(2(?2BF9Foluwb=0$P&bvnA)RuBix>)AgfNU<{>'
            'sf1q^{E&-oJrh{*N19sRA5t+pn+v~m8zQLBETn-=2hIv;LqpBxs=BkVT}=?+>+K5PK5LVqUU8K3&w(1SoEIOx*pi3{5R$ojV0uG@'
            '9?CA(=DC6I$B|U$6X{TcK9!fBNhOefRfz4Xr(nDQyY8eu9@TQQ(|46~8ligX&C%#=oSnV3*oU0%?JNY%D4#E&qd@w60DM;<N*PPm'
            'jf4}@(QdF6SNbaOqOyfNrk!``q3@wXy~SnSwW$IEI>xQSl6+N5JkXpYT6NS=Wj`?<0Ub}L5G~_QmyLe?3hM*ZLQ+d6%UMqOBtpC)'
            '+-mTVO(*7_F3Vs#o3Ew=YAt=%)rv(~|L_FQIoY*Ck8ci+k6#@e{;aTQ8{jFqH(iijug|X3`<4oIv!UQ7$E7}(KnLMrXdoYuDpmQ@'
            'H&Bz-<Cp|W%K782Nw9=kVL?2gpzP3_5|=)cED*BpcA5q8`m|V}cqzo09R!D!P`0cs$8hwdWDOdIuRM(wxJJay>rP^9Q+M57;MJ&r'
            '&%CxlryU1j5`q_%a~66&U-Hs%9%E%zXlJhxs9^sG!@zaNBkD-C8UWIQE2ik>J{q&`Gqe*)x7DthW>vBiQ0e*RPWBqZQ@wq=&z?2+'
            'Ec85T)2|g|Ur$KYW><b~n4rR{=ikcSWWT=4N{LVHdnwv%cf_TwHNEJ}lEpUli5A++06n^F+gJ<eafk+I!sq#RoqitYreH(sby?Ka'
            '<~4ku$rJsB=GAO@!b%rNA67(3*B^C~plvne%3qwMa^yS#)WNbG2+Soe1ePv+-Z)y-N-sZqkdHfwQU~8c;OaRkFQvyW`SNtCYvu-?'
            'l(fpz)hpY8GO@x7s070%s>Ml|qP@Cit}08p+;e^68-7UTryDNF=(QhA>9kj{R^)V{e_vJ--<h(mA<^cFvahT4id{2OKRwz;tuZ<3'
            'Rs`q-(;c=lIF+nV=_DK8`j#304*ngyKL6M7c>I32$4&u1&!yU%SfkihOK#9OfW_c(7}<@02jBB(?uNI3ZGeD}CLJhxfxL4#yB3a!'
            'Jjwp|?DRxO<5rbmeNm(PF05!{d{1IfKhdV<4BBPu+FfT};lcIR^&wPEYowqsH%j458%N<BSV8RFtv92nsK#~a$dUP3K%>;z!$zlU'
            'cofeyqm$Emgw0XufI>}(%GwM_ySXaMOq;6=C$~+)b`cR{U6gM=lQ>0>Y0V5by%S!7&PT0Km#_0ardb1FrB3%69eyDbTQjqWsN1Y;'
            'O&2%Ndf47-J+bjsMPM{Mugj|nWa`pr0O+}OCA6KzMYO0;SJs?-EJV}@#^}s%nFDoBHv=4X8iiQ95cfM1Nw1`d_+7sM8S3o|3=9vx'
            'b^*_L|5!W1-Pvd|>fCI2+DKBr$p9E{FyZw2hY62)1YF$xsIU(^|NZs(==}Ko!Rw>)GJl$oF8f+NO4*WNDfVBBG)qw-^JGbp4OxxH'
            '5ZIVodHyI;Fxc=`y*2C8`Jv`^wT`Iuoi~F8kN9F1@mTcB??~ytmw`Nf-#>o#ynpk&<@iIX`r*~vVgLP$9%eNHdI?)`Y5_NuT>Xl('
            '$}3oiHJ#2G7ugV%f>E!1Q)LGIc)=a8t`tGyJDm>cXhf{uwxY5kzWkU~GY~=jVmDV#rvlUW_%1TMYZI-jEV~qQe7RTPijVo*K_W3p'
            'rAL(<GHLAZK1D*sS(EOa0}%1#7f+?MCW>0Mm%U`WEwy?N7qD%@YHfII%UGyZpYi$eQ10z>^zk=@Mg`<(i;FLevf@%P;<O}(y?g!G'
            '4vl>0a#dW(i+13u(Mr0hCP+W~(H5-fO2#acRTwG)pX;>f+1~mC+O$iT878~(GA_MmccyQSi+fcB^m!$B%>uHP4!b3DYG&K5)UjzL'
            'A(5<7%Y_g0Y86wo{h>P}j&`>P=EeKPGVL!hwiO_!$U%2s)%Pr;Qr^XDg3Z?AxV8J5IlJ3bw{10)(yr%Lm2KtHwfy@A@|ER=rI5-G'
            '-<Oz!HI`Fa8J`7fx7w)uOs#m5<nUBwhf8&i>OEl+vjZ-^=_y$*1ip>Kv}`)eX4|VQ%k`ze`%_@{R@`1FXyKltFyUZUAT-NIjtU)V'
            '%)CSm?0k;f?mR+w*6&ItCA9oNao;Btk+;4A;nT76L*<1v(E0xZfK`E1Ybj=VqDV&jo9h5uwu!ezU7NAMMf<CP8nqHzVX>!%UYn4X'
            'jxKY6yB=$00?@jHqi$5lG#=hi%7O%lQ&nW()jH2j-ewM(_N>!{VgCm+lU7s'
        ),
    },
    'narration_presets': {
        'sha256': 'e20513c4b0498c302ef1dc00560e424a09bc306ba8dcd504158b89f2be2034b1',
        'encoding': 'zlib+b85',
        'data': (
            'c-oy<ZExE~68_F#F=bI$3MBQ@0S6nogU?n{UsAiU+!RF-$SZOsZN<0ll2(+!_uKE8S&||pIW3wGt;L<4otI~xncXl9Z&j*e%9ORL'
            'G<im@+$GwblcJ~GM4Fa1PpMpMvU+7)X?Ij4mBVtZ6Z53;J2qBFymt}=zx?nkUD8kY)7kWaE^n{t_U?fmez~JJ_tPo8xtYGdo(1E-'
            '@Gtlvr>iVJaS-KMvZGXOG&!9)O=Z5)u+dH^yjm-pXCZA(xyE+VpCVK0I4G?$iMAaj#!;4+;#;hB0nc;0D|NzBD%<55j&~abXPMiP'
            'SfVIWRxhh0*lST`siHNc%A6kG=*$}XNP{?!s#Iq<m?9XYqEcH#&qU|+YRVR=MWSB~X=zsL(gkdT54Xu&sZ~t*lH6JqCR>pyC-jpt'
            'mj8xeN>vuaM7A$O@+1yU+=$k!Be43(=q;Zx5~uhtC>D_-K}=#D83*INS5cl8$_uH;=FX9=GKXY7u)Nhym&6%xJssE6tDD))-7SvD'
            '*~e_<ZS{ZG2C?Ls3sh0q{7EIWR&Fh4nhZZptjbzYoX?*m@-d($9yZ%pM3uDFRtGDqB&x`XKY^Xi0mv35<1ua5CR%fTEr*w}JX`0s'
            'tQBInfCB*x_PDRn&=*no#IOp24|g|LQ<_cRPp=-}HQmev+Bdf!rne8%>p#PEFcsS##m`5APUNj}I4`kC_YN=xJUGCQt2|#Ndalgm'
            '?exc++nf2r!)z|c&mKqKwIDUt=C+P;wMdL(1isa&$qYfh1{1ymJCCw?e)YrU{pHoe^!{e{aC0?#M3kf4#-b!ZEXG^>IzTJE6=SG@'
            'vy-@M03MZ)#gae)z#e_H&`G`xu#CS*RWVl|A;|EV^9Tq5uUx`Z+DhzepfLAfbA#a{Em5tGlDvvh>1IhPsFA1{B#e94QWPJ4`O^)o'
            ')IN!>DhlL#5`<wG2!YM#%c`ubp3ljog<v6rq@>;jUS`rpzL7afKoER;PFxl**7rU@-nIz(4ZvJOw&fTU%Y>C(<%Qw3hGh?PIL-^P'
            'k%TcWsB(}N_y`Arbbdz=&2onbL7me{!)_UEttoNNh%yNpWZ4=-Ju2JSnRpVM#0R~mF9G3S!>H#J-emA0MbMsfIP!~cK^fI2Ui?vU'
            'fSd1^$x3Ht!3%G#dV<HhX6Z&1x~v>8T_sxC=KH;JDq$WsFPt?~u}Vs{(v}6@W%)MI@k(=G_Qz%>Q7Ny#F*a>J-&ucN_uGtFTZt+Z'
            'XV!e4W|2-*?B)N1;w;y#<(}SUN5kRG(z_hK?@xFRg4@gc`%C7Exo?M^h;qA`ciIg$`W0E-=pAc15*qDA5``*bq3grOHJIKG`;ytJ'
            'LdfP(9*coG8~;4(N(c@l89vYZrElpMAgDyK#`%`ZIZ}Whh1)fNw+3O#)_;f0oeFdUg8!q^&bLMKT}a!Gx~)kP?y0`-Fx=N59%jMJ'
            'LDK7axwQP?(a~os0AQR~sCedgraPv8;c@N^OVkO{IXQ54?kQg-?Dn8t?0}su>DZwhz3Slc6?c?WD+j#CBTzTDImJmJ04FIa?CVSw'
            'Q)u0vM*Hrx<*uf<C`+3odvMrfn$L)OInA_=-Q-Z~Jv&cu{+v86Y0w)jjBgLk+@mvXS9RWMDm_prEyyuVD1<4#pt^>kw<%$$g}#_+'
            'maAPQ$e!;BrT)a_e{rbP^8-|Pdf3|)CAs6&W7dFW<Eo$q@`(A$8e#En{10^2P5TKPh<Q#IXw-jH=pp6qg18R9=W_qPrUD$zHHm|{'
            '(Q+yev?=Gb6>Z%n#nm_2OAb2)<1L<7g^-))zi!b;f9Ot`2zS8~jm(JbyNj$TO14G$d0WOl=4GCuX@<UpiFL0Lf&fWmHFsOkR<x06'
            'ZK2nShaU_AKz)dI+u|o|=b5adr*jG;7>tl`A4dWc^lXmizHDUyfJJNR)K5Dv&07zTYv`T>EkINLqM(aWT^UjeJ#2f07#;VOuk7v!'
            '?XyCw4XFhe`#;A2{Ei$fZjz<oDrz&~D8Xb`mp2tE(&roo6g?ugJ1We!h8cStu7pp(mkGUNX7zW2da#UCl8(C7$|QE6abLiwa&eU@'
            '&*d1|*~hwSWaqj{6_Wj)XPUIYWQ=$%6{HVTZl&8}u&y`5jcr~fIKJfDPst2N2|9dofe4BhZ1O@Qt{TEfITz_+v5QN7&cQ@YGW-Dm'
            'n0||Tz^Y?VFzGu!4Ht!!GYaEz2RbqoX)T;*Vv(tsF(Zy{J#NDo;634)9N1O-A4vH5ZD}}RhZ<}5CUor`LtU>&jh8s7D(j0G6BdYA'
            'tybdpQC`v<s2;a>z&fQb6gc=z3iKqnuTc`XIPG+4+y+CIwi^v#W(jBAT+~R6(KP?}9e_%)S}WSAC*?-8sj*X`u#SAKkpo>Rm%>UV'
            '6Z%luRYj-@ZXnfXy)&Q@zes9>J6xt{VJ!Yk!2bst2OGGF`(gdRC)R~Vm8#D}#1K&0fF4z%EJ2fElRoE`KRNSxLYE5#R!q#H)}>Xe'
            'l&APStLWFecc){B(LuVLXOng;aJ~NHFRm@0O@uk+r*v}WYuC{Tm|*&-FB>gQ3Cr*;-E)KV)GeblDk~`#tL1dmbce!a_k0P$;W=**'
            '0`^&zMMjt-98n1ObXYs|JZi?ar2>h)Fs=EuQ$yMZ1xnf(zBLc{S;69ZY(9T=OuL2HsBye<%?(~<J8DEvOmH%Uj=nbSh)C0d3sV3Z'
            'nLcxbtbmjaD2{4R+P*mnU+8^e<Da#LFnqDF!Wps;)YTy&ZV04y3K-0cy`Z)F3{V5!!lG2|8u<1tI!NU{XPCjSr*AHQe*Z8(oW(JU'
            'w;4Mc)aUP}zw)qNRz;!*M`Qdj2m<V-<6A3@|0P%Ma}Sr08=+&+s|N81e3kihPH37I<xV#6FbOfw$@RU=HMdv*m#?v`5-7#a)WNvl'
            'KFSv!y@0?`dq74<22XGUPhsA98#LUfz5H)Ac0LNBV)N0B_q<pAjR6x?@I=$6MzJUS0&||v>Fi5=!kL_K_VM|wd$;9|!k(iLn9+9m'
            '=749svO%>2QB_Hb!JoHmOp5klhT*UkZV`!~_#-E{Lk>Df9d`ZRi_1e(`|I!d-@3cgF}eVoc=n}L3YW!4?_g~QhPcXRt)?1GFg#D5'
            '^n71L-j{(S_tl7c1o4_nPv91tXb(50PcGx$d+s91qmAICLE(t|JUXGn#Ync^i%I^Tqxj7Zfnde$jSs`eDwp@adVuY{BKikC&(u6X'
            '`=-uMeNc(P6uafA3-jrJXLq+_SNf+(Kb}&)_#JFP2e-8b^76v@nuiz;1zY}7O%Ep!GR0t6mx|QG>l5X`@L;uGu&o6yIgaZ-;}RqK'
            'ppr^Y`Ms?V_hq<1d(2Y3(^=jBQ-0NA1~X_8e8e~Q`2g3z^g?eE+{QA<Fgyk)`Dggtfc$S2>yhcpur>{810>l3Z@KU*zrv#J?{W*H'
            '4hk1O>@68Irt}-uN5MevThD7=eQD=6ulpN^e?0*b<L5T$d((Rg!99f03=_D$ILd|th!G12V6NK$_Lz9S^tM0u3%Aqy{b^`AjcYs$'
            'd_3{Aa*)42ZJLezsnw5PnHpXSF{-Bbscw-q_xN3Nc}I@s0?8NHrq|rAjROtr#{T`fZh|8W@<}ProkA73<NHQgGf<<rZE4b*`u*%{'
            't?Y@haM;WG=6(%qM=j=W0+avoMbKo`9_Zpq(;2?>r)EDT_HnYG5cd}F?@orlilV=q{{wyfDXiJk{d&@ZdH*#ITTi|l3%wqgIy|Yb'
            'JD2u0bXGc+Jom~_f*t%HK~fHW'
        ),
    },
    'source_strategy': {
        'sha256': '359e9355c0fa739c228e07ed448a8e92f8fd1ea867cf80ff2a8626f7db37be46',
        'encoding': 'zlib+b85',
        'data': (
            'c-qx{Yj@j5vfy|AiXP+xiG-viZ<0G#j&crdF%GX~$t%g3@zv^3Alal4f&d2(%ao%1?YAD?jcx#xDNnNJtj<g<0q9qCb-k<F^St-B'
            'K_<n%==H=jh?6*+29d~;BArSRWSPvem5g%{#51u@lPUZi#*1E-r$H_k8?lapIEE+vR_h<XeiWC#jxNNzv*QmZLoxj4=;HFC)oc7|'
            'y^`}Jl_I|lGqFl$MTEagu?n(WrXo&qIZ2YG_`O)I<xJps;o>$I^E6orFVHjm6wYMQlApsY&peSQ!b^)7CL2VXEX=ytWek08CQ@Y6'
            'G+gJ-qsz++uPg4t{FcTJrg;%WEm-EPnC4*;3$siaPm>wUBZ;Gp2w?;fB?-Xo=KbjX@MLs49K)iAH`oAyAuex&90r+55$3IMwN3zR'
            '6I@~<I)N{88AZ^?ZLtbsfpAM=Sq_#JVV*H$)<K$w(;^B|wc>88Gx5bFgzd<2jG3H<8BP^?#x*Fogt@p2GM)jzxl8-knfhW;1ZfcG'
            'QgZX4HI2e`kCugyi{&SPex4N5TbaQO*FiYj`&W_VL2sJGS+RoEiNX63Y_~?75T`0r-p+S})8oPA+4)C}FaSyDD8Ea@TBfiiIUy4)'
            'Y7WrO4qGh&KdH=$D9`qI^5YWi`oCweG=a!@hts_S1d%~jq_7p=nMdct*JtnE4Ns4q&Pr?@Ba-T_%FYm6yaT-P0D{-Qo?Q%EtxKG)'
            'X^!X#Yak@UorSgHqGJSjn9hU;-B0Ch63t}l(Z{r`S(wUcK1TSCGns}mqv;9p4yG5b0U2%+z>x&knTg)dVx^y7+$L$B^~G5lE<!+M'
            '5frm9!7<mcIGKKe)dP4{ceo6i8%}k$iGx)*#TA90WbZD4`Avg0A@g?tni%FN*5O)4VN9ziGCAw3o-XKwh=q(LU?MibO@u=N8wRtw'
            'Jpb_e^27Py<m4kPE(-vkXNTejrj$gV<amwvFkS!(XE&<DnT$i+{aZXL^d636mY72?k>tb(g<uKCPaw2m{cj|oS-5w$mhoV;r}`Si'
            '`E8o4VQoO5qCP@64pKZZvoU}mNjG{GG6J|G&ETV;Av}>&_zfsV274_eQjQp2?}=BvG=bv?_^Xd(FDusTD3lh6q5m!%e4tIR2RTCr'
            '8z@Qxzrf7<VgM)TPLUb}uY#q-)0!s5;?~+UqK~7~H)_2C-se(uZWc)beT{?g1}1mIeT~aax$zYZn@7oAX8NPvU1U(x0Mx=ggn!ij'
            '2Wvp?PXL7JZ2+{ah~eLOfq2k*e|9o@{ZY~ERu8cqS^?kyhBIC@tZH!l+voxh`Pu2o$G*7X13PAbcl;X>M0dd^gO+)b#yH2DOzp}T'
            'xBci8=-mxr*G*95$ymQV`Xz`mdGiwK|1B<o4jUi|OqJ+uKxorfGsAO@^O+=RjEMLtly~D){;PnaH{<@XEm1eM{WrXLd~LxDW@w-W'
            'XnNb5Bn31^2*ZYGc=c7=v8G=goIn={fJ<tCr}X=M|Hj984uPO<criqA8P-`zg3M{WONwYFj?Y8{M>j(*bA!EId78$1*Ee05>D_HO'
            'y~SnA&oBpAn(4}HEUrOJVX7&Rk=HmPkFWVdmw7FK5<BGI>6Gvb#&^bPegdeuE@1KimRx-kqzeiBr%26VU_dG)19mYF7y50<A;K6Y'
            '7|xJPxN}BCSi#qLyf+BB(*yuucrom2Fx%X8k?!KBJ>8I=6zeDnX7ua2m_%WAOTTAw3OER`!VN~whMjbQjN(b2;heKPL1u!}>$g15'
            'YmtC49?uIvT{#{LWZo&tKY)_7maNrM&k)K{IMIJ1gW=}5x;zAIQ*ZR&06F;YJs>}Sw+?dbK!5fg{%rA^z5$DR2mE-t9G7d(Zi5#;'
            '{5Uou+qfLP&vX%CM*IS3pX#JT)Z%#f%izPw<@j`Pem=Myot=)~pARpFmtA^WeR|Q=T$QMNweN1rZ?!Hi2bUi%#=i|tJ`69!k$9jc'
            'p8O0vKSaC=A~+L($Dd@BtntCv=9%!Sa&|o))YFF*KM6tnn9sv$h|K|_0Y01uVZ^Vim;zlqwpu8ajgLnlAzY4MpN!s*FNWtMnxj|e'
            '&mMoy5EvehFMk~(k%Mn5oW^UtKR<gtytn}BO@beG*x52xS2XnsQ2XXkXc-Dm_>71G9_?9j7lSY;gOy^MD6W!th7ubxzc<v=`(Fna'
            'JhVr0@K|#rga;pJMOl$}z_S`J!H0RC@ebgz^>|+IkT(Ov$cg;BLIcaCY4SVt3$r0$BZ5VeC`i9tR3cGI%CSQ@^(6Qad(=Gwy1zt8'
            'mFI{YI=_=LX7X~EFui24i+9Emwhf+bf%~EG5+y*5Nq`z>#Uu~&2sXM~HmvN+p|#n7AHkaONbeO8-2iroWEbT65pW1(?g?%WOW7#S'
            '6UjuHmyzg1a?ngK0(Wi9jP}cGcu));4nt*+4@KuPEr6zYp5HCMs_pxh@9Ma4sOW?KAzs{d9zpN3qx!))1WFqIPNFJ;Xqf;o3Q#&W'
            'ux>4d05#3S2&msn;kkVR&QHQre+SA1FzrPO98YY5H8ctCy>b@z5)gC${7J(f5g=%o0{#Btq*1VX#*Eb=t8dBUQyBUG76+O>RFH0Z'
            '_0*E!;Am!ZIXHhayc{2YP|W3G_!^M^$Nkp3!9T}mA1>d20K`B0aC!+(58&zOR4sozIDK<69KZf`aE_$z#XcRzUqGHgQ5^WV$O0ff'
            'DBs+HR-D|S5I!%Ws5b}J0E!9-Te(Q)C_18o)qC?wyiTHEvUi+<oQ0h!MS<n7Spvd19Dhy5+8G{g=H40W*mG-e)P6yE0r>~S3+$`|'
            '3K)JG-ot-yN!-YRN%_hq05WD2Swf&x8h5F>)3iZi!HL}g1C-E1DtpQT0lfiBU9s>2k6G}+=SD09qU1_esFPuLv6Qn6Y1K^TfEfMO'
            'FTlTs=kL!)r<dd5@tfiJ)yGR>C<ps5z8BxpKP`Z_=>2&=(A$MxXbHyekfTiC!Jj->YbNIkuw!eyF^f+fWI<gaXA3!=Y;vRtIGH2S'
            '3>Y6-A~5Dd7EV0x7~q(q?gqO$b=E`pX4|M}Tl;M8IA)v-V-%@j!)_*kX^1v|AlCfx6Jm{^`TYFHPl!C~kDI=zjF_Y?txduyP`HCk'
            'Xco@rAkxB2SgkdG@5xz6qAzgrUJr<!q#m|l0ye(8-7wE5n+Njn@^qmD+}F^ce-WYpKon&Ox;{M(=hPG*^uuh7Z9D#<AtGrI!cqGL'
            'o}MQ8FQ|GA(=<stb8i`J;$W#jI4Pn6z!c0vpo!SmqsQGN*kDlVJ_9Me<HJ8=d`hns4LH&QdtdFoZ@ujoYn0MqNb0$d+wu6v1NQOw'
            'hu7!R*YS#c-ut`fb4xNDCTkfhJo3_s=ZheNapGAd%k{ZK@Y_BLPMs3<RG$i3w4$GF%yig~>H>Ce*;xgjJNry<L{Bt!_{=%5+OAvN'
            '%Fp~8z>Cil9#yHe{rP;gmWwf*-L9C=`4#=aFyRODm{fi`K(la~n-g>n6UdAbxkp+?LBF!ii<AsB)65fxND@M*<8-6PczP~0cqfeK'
            'NmopQDbVbhe}hOJ#=mQW5uYYlYet3x`B|n@L`@obnxJi0M9Bh<9_~FT#mH6QL<My6UcxyA+By$_&!AizvUTQ$S>zbj6=i*%6*aoi'
            'L|aSIb0FOrO^)r{Y7IcLxg%bm=!G|E7|zJjJYWzD)PxGQOrD-y!k?gY9pWss5!QxPvQ(B9A{N9fN>hJ^rkgB5b61ulDFR})DA>NS'
            '%V9@*Q93iCEj7TmAR_?r<PR+YLK8oDZBCvoqX1qrcIpq@6OO0Fr7J?h$^_z@)s03m5Q?VIQRVGjIHUPh9^XpzIW#;EvN2&Gys8MC'
            '^p0@1;hn-rM!tHtdT+M2lt>kwc8{98>N7z553fsl!8QRK2emoCLPexnrVH_nh=Tjgp;$_}&eVChIDXrkr;1(V>82dOIZ{Z5cRP;M'
            'UVffR5c-Go4_TbitTnY*10mCu1AuE71n6|WS<4Q5?DxlGM4IvV5%>Z;c=W91M9R3AR;%8l-VE$h{oe2u><NB=5-u|w(#|tfy>ay@'
            'i#<I0{e{dso_YtI0G?i5)Aw_-G?7Lh#=3R6NaQWkF|G%C<hR^c`NDJ9gWcdrX$2mgWn{B@?j@RDyX>Rk1ABGV&-ip)SLVh~{pODO'
            ';8dHa$7LHC)fb}<&(;a^duPIFa5pxFNZ6D5roaZQf}HonJk(9pgdC*SBRX^g2Tf5t>C8du&O7$Fepmc2ANR(Yj3Grfws$UJtt8=e'
            '#cx4W@bCT(gPWczc+FyNv04Qwl1r`sg$qy=v(9ajt4A3b&gyJ<_&HC}A4jYpO2`Gwjg2)2*&k>u37%!7D8wF>$z1EBiXoR)EfBWI'
            '{lVVFOuQQYa&|sc9~>a9(iT5bu5PL`Wk(WPzaqH^K?&o&CJfWQ7{$t{j{V&PpMVFVTgql`lO$L68eRdPN7Rvs&w@6@B1JDk>!evW'
            '`3yin27qkHhQoT=JeZ{66s<!pC)CXxrf25UK+x3|kwG0^@7v0DvL4f1>l|VwgLvD!Y%S&Vb_}XznpJvKuVymN!hAEHuCoT7W`UmN'
            '<N7z~m4~PHx9nEkB+Hi~TP7(wsy9IthxgM!DB&IC26z<76s?D|U=ze)>Q&emlC3a9fg=EEuZFQ*LRaU65dJH$GnObn-;t=cR>Y@#'
            '>iJv7HC~ViFq?=)T>5Cs(bce53=lk+b&yPw?3p0hwm3#GK7L2yO3_lr0-sn5aTUFOSe3mZ=j$%=tGp|SeEB{zCmV5)U>B=S1Ue1M'
            'YN|C5hnQ_eOhG-+fNBFXOv&jaXGpQ2ul@eM;eY&@)xd&h9;B$B@e4I$R@&(m(Z8<qI*0%|IF5O5={qckH1q~9p6E-zwR&=RLUn!L'
            '!5FA~_4gG-JE&8D62^bf)a+U9Nd0-37$_^SH=DX><!uXzA^Y1_3WHm!cWMi<Vb+FAVr&C+`JPysLcS)s+M!6?0V0s<K<v@DyLVS1'
            'V)mb3qGxJw8=3d)?b7_&-oc-NOEpcSkr+0*-0IA=fnI8A=cw1_c>n44HP=Xl(L}){i~v$vhy5#zY>Wisx}CklXYY_mcvml*|Ex<Q'
            'uQ42n8`?AP0|Wh(q|gkcNfi*&g>z00{L(&)JUWQ9h@MdFj-u$G<bdX0$R1*iRG1|?DmVshHlNVCu#(DG4|2h6a_9FAhsWiiB{qOv'
            'hyErYCM_jV%pc(VUtQz>Tbi%%nib<QLaTMisXL?~FMhD-KJ|^hUmzqMW&Fwm@=pXy^e=o0(w7Z>3xZNAI^Vtc!B@-;(YhQ?&-SUl'
            'Kc2lU`|eTSuo(h31KzSLM6Tq$CBIi^Ty_Q$f#~#bGHhfkrE?oJj(|(7WnV+kwD|c*9Q=L10dfahAvgChL`*#BJ`Vf)^T$l}1Z;T{'
            'rZWtn3GSnS7dZ`=AUa0|jt3STE0g)LaW4DMpUaOp6_`eFj}zK5kNqw45c`ME91*r9c<F3uI7MX?=tLTYz{#R}*j*TNzZr#UdyRu6'
            '(X>FMn1#BbIRT1iF^4s5%D1)Nf{g9Xmk!tJiv5b(i!A_j-@Dwh2_+f<^FTd3iU4FTMH*fbG#D0>1kcn|#&YT4fTgOc3B1@gf&H(T'
            '0PYN-0L^4otd>RW?0tPRp1HaWOPV>Xix9xtWEH1}jI_EJ4Q$rg4&6*S29Src^FcjF9PjUMfz%gvYMI1EwhDlbEQ&ZP76srlcYD}B'
            'm_LFbo$h7vOCakQ@VFE~ly8FjP%MgRRKWUv*bRx5P|EUn$a>5#a3BwWOvPychXnuHZ-)rG6eM}z`^Z+)6-x0)dqFY8kOFuVrW*Yf'
            'hkvj=tt=raV?8kE&cQxf?eV$q`cr9!-l1VNmZ?-xad@eJwcgI*`;bLJ+>VAXID}VbmdAG-tV35leQY@ly%PHJ6gTe*6}mVD`3k@a'
            '9Cxh4z%W#Fk|a@`<DaSldMfIyk?Tju%X#RG{Vr$<NMQtR95=C<9EEW{{NEo&=fevTh;j-#(z2W%1p_Nnewdr6IPQ_rT7#AQ&4LYz'
            '2Nc|Khpqwk8c(DOaUpXEhTQ5U>>xjUUy&dFJ!bjieiMUw@l^~8IaCh2qUa-_4F>KSPVr)YE0;WY@n^9~=*gYM2E|*4V;$_qvYup&'
            ')&QC%E&%mp1f(Vg)Yy@|F_JvV(P&Z61kQstIkfHW+SiX-tyV$IG}jQ=a*zUnS{6|R51Pb@zmV+z^k-82S&<dMjGKtR{}P6cU?*2F'
            '^g@CJ1?pF7)BdKtax_fd4)Im9wX7myB|lAzTqDssO;+o?;LQNe9xPxN%pUw3oXJ5*paS8F`O`rg#aZH5iXD}_2-g23Rep|K?J4@X'
            '3|A5GTv*Y<1U&^9G8IOv5Y1i;l{CqbgVwQdM<vCLxY{x{>KXj%t=xdd|K{ur<AnzAN8`7{kN!(1{%oZuiy>NNgw}J{>*_Yjj969q'
            '*s<d8qJwdGgHfDb>eKA#T>qa+7Nu3H6fBAb#uJP=sGLdJ7p(-q-;KyHn0*Q}loh6=GhzEfifEzC8O$^i|Mx%lGjv?*c=stqQlWz$'
            'd3;=~Ch$=*XPYJBE?U-%4b>zB8Om<>_%oUFWMg}#aFY%wkECOs?(K>H-1q(6^(N@Dv~^oOZs$5Qp99~oT9{R1#F=UsUe}&^*BaA}'
            '5!tl|U&oHPsUpfO%TbLSFj6{=DP+X)9IGeay78SVB&K4AQ{R4BBE@j8deA8UIMFLzHF=PQgPdX+#2?K4*fCcZZC@2(G|R+Ib6(qx'
            'a+kg1jJC|l&rmkfS7(>Ms{AvPrZ{uVd$+Pk={NR*1e4^G5>?qbh*^AY22X!WR!^W#)ggs@vE9CiIS~@0Vb}x4L8X-Y#KHM0CkG{I'
            'H_PXSgApkc$T%+RIY0Jiv=J508VCFw?HF~`fKg@>^)J5F@s(E(3uM3b)}X=K5Pro4xVs~^;@c_7`qp^xFPT{l>9i<+b5!w@`km>f'
            '=~V3U^VGTX4L=|pQ_O{YTM6f@ssQ^~)bcEh`+&Z^zAAas^|cAyDDir%SRHnLtx2qIQj*lF%{&ifZvsu1hejn-j%LN@#OaxxCy628'
            'ccINuzrlpFpir@6(_0xJS4ud@CA%f1kq=bFd2Q1q#FV~p-FLzOXen~0D7~{8Mlp4sF6HJ3<eSNikL980xbR|m;kwI<Iy%~hY8L+Z'
            '^%*(9ZIstQXR+r0sEj2KYBJWV>#H_B1Z8FqBo~2krNw4Zlrl5S6rHF6+GAj7w3%*-WhR$RBHUBh<BIu!3H0ILjvPqqD4K0(c33+R'
            '&wz>r9qbsuQeY&7yI#;{!st*uv|oyL|98L(cMIEE{%T+Qk6x|&X63@hBtwj!IeDG*0Mpn$Z!~EyB>A?*BE&o&U=dr4X!ibX6>o3l'
            'Di~uXBj+a^SW>6X!{i95L*dmJt!H};^B^x!!rhW>W__f<P|GgS@yMiaa+U~i_z;&EE<*y0GEt<Ury+Dei(Tq40PRGXg|&1NKu>Wo'
            'd_8(M5~I_L!KrvVI3Jvk;8=<d0Wa{q#*7k6B!Ssle<6p~d58Ttx$EfE!+w!Z{XTGdj*03x2_XbO@6Z$*sHK!JSHLf-`~lN!Br_Fy'
            'Pro?ohrvM-4@eI&kD5MoFw*|qd12GsOG;i%B`}dFSYSpOX7C27t}QXW4P)8EU|STbNm1wUUVxX1TC#d$`mA&FbObWc^t8WqwqZ8b'
            'a#Y<V@5nt3K|zlA)DYprdK0|2H7D=zNsX#WAh3GkG~RI~JobsK9oM}**7+xmnqGMhk^ASel%dKumWpSmR=fgAPX#D2#aq?3pnPXL'
            'QrW3;0ee{*TZ>61b2l2x;I=im8@S0I_)lT|wzicI{jJAhdk5uLz8xH;qXim+4|tvaUl2p^zAS%%Tkuyz3^Wvu=&;BFh)`Fi{;?%D'
            'wk#&0vShm^sGTVUom@)W#5~L(Qfk+w%4s33iwjRET_c%`JS{-m!iEib!c?vz6ZbM@*Fz}?_Ba<XB3!|aaaCm*jIsy@86k9y8>TZ8'
            'QWjZBLc@lPb9*I=6t@F~$0E!*UO!a<n>e6yO~QKiUHwT*ITYy@i&_e5H4CQ<N$Si!<m>TDZX_y*Xo#XxJs_H3mw!tiMQ0+Hzz1av'
            '&yc5gsv?*FsBL}gBnqc!Y_3g#<(g&zfB*)adn(lylK`lrD|&N%wSV2!rmfn$gX>2}E~Li@1v`J*p;)vDnv7;ql3OU*6*hULR52Y#'
            'Hco?hnRQiW^Egcsrf3KfgTNy|cPa&<iYZ|CWUC^pmBPw_#zpm!3c<%xH6hk2pz=&Aqh#-fGcdBm3dPl>b-F5aHV)?$zfYLNzX3Ze'
            ';$S`}Uegy??S_QrRFA5%oUy(Gj0}<;96CFdWlsU(^c1GR=Y#5_z(h(bTxl}EfUCQdur?;gS)0+EH0Z7{Y1uRjHDMQtEx77@6C2bj'
            '2pqNm-Vz$x+rn}9N=e5pEtNq;3SczyGa^NbIhhzA*|GQ8_w76`wN#7)x2A+4m(^ydS{-n3nf>V=i2mxk2Ojx5ad5c*VrEJ5G)XOi'
            'Ke&P-web#xX<Ny4jslQ*;JsEbYNOP0qYa=?!a{gmQHyI{6}J6}g81lNV{#33q!tArQh7cGSbpV81rEK_V=Fg^cD8H?(5ik-mNT!~'
            '<(#toON-3*?)b}$PTXq!z1>qKJP$XaMajmmS#}Y?2Ny-Kpu$+%Y{CKxavrWwgDIL;)O#RapPXF`j|H%4;t1H^CnT|4z6bF}74Bj;'
            '1^bSeL11$EjUIG7C6k(F2%nO0N}aJT!}UQB_30D_0#kK*DB-J;WXeySfp#5Z%@~yxhFPkrSW%3<;qlcYc7-YSK_$=l>@rUZ$^n5D'
            'Es63>I3JFRV)a|E&rUCgr<dcC!Reb1gEvlHkhu5Z!b74yWjR+p>6Ym04S@)T*{$e&xbUfT1@0Js>Q!x09s_NxvdDDXt>2{uxiY+e'
            '5KXdxuYr4uTp5-egD#k2wUuxda%~#nJ#7#31}lj)Iu>dYfDZ}rUNi*-@rEM@42=7p%KZ28w0C*#0EM5}K)Ec^B}k|?IP&z1;J{Y9'
            'fP>3_kur+_Xcr)CEpYY05|moOgNsdY-~S>MUI9*UNYgBglml=T&|<lm#y_a|T;&ukuv87E#2cvw)h|^=#GJ3;=QZ$n5dER$E$7^W'
            '{uWaS5`4~r7aHo4NIG#Kw|Hi{CYhO1Jn&l(+Bo|I`_lQF$QpPG8S97ZRFaQ~D*(s{a9pcNRb0n}07mH+h2v6>%y_Y&9}1YN21+J@'
            'u%YoD%M-Drx^^Y+45L*LMlCccs+x&1;|eJ>4^^5DS@|id^ii0pYEKlNKTE^8svlGqkjt>QV7j~uus|H30U)=E5jL5vC~$E&%fE_%'
            '!-r*_tKe9!U|{G;zjZ!5JszGTD1I5e87s(GhhB5}@g0DFjT9ZwJfaj%vfRMDBVG2#e5;QMeu2s}7h`d>2d!A@Pdh|iyQ0)2i~#nV'
            'Dm*na$4@EsN`dI2q(BZGZA4UCPqD#qQqn`6d0f{GZY+r{aq9OGVGgNk6!tZ(FclX33GC=fVJ?KgrU~gz?TU|N2Bgvey^xITQ;LS-'
            'a$sT3De6I1Vht>l5JuyRQ7=?!E*=DHPr8M~vYb4ijcUc;j+)&IRac|2IZny!OzfHYAb@(?Il?4gph+F$9GzO4OpU=jp~+M%(=)Vv'
            'px&ZXn<;GF5=j*n7MLp4#;Y{4Rh<l|ubg#iQz*3^wQTInQ7tN^;%bJFu5gS%O7SqyCL8tS+ov2eH9f_3R6f<T!5;L?9&FKD`Ayli'
            'G^ToT>^MC#NfFII)l=QzZhfkGwMyADn=6@<JyyIgUo-aCwYRi|BTpAIRaJJnWHiWjgP}p1XL_rf3a8$fq|f0$--FIRx>Ti8nW<`e'
            'Sn>aov)97i8)AXb45%;{7@Qf}I@4UT*V8pkxSD~JMG7C@fvk-Th|Am{P18<l><)ES<&gH);dH5Lcp^dHB!#Da-*`B+*ipKlYb7LK'
            'l90e_`-&=%KOy=28UDSyO@8)oP$h1uu!0rrBaqq}QLQ8-mgGUDK$7FVj3eME5yuVUBTa7bggikWy@i}oNF@cKS6iY@0XAm|qTvGe'
            '1cd%Ff701-O_{g5-M&PaX&xXFRIE+TD7%`SYCe_?)>$g@Vp{AaTxxPj9-B?F1azqS5*Za_s(6)!#VP}`%9}Xzo*O*{>-WGd#uvkr'
            ';cH}<uRda?C<bh<g8P^3TI0YzzyUzcWsnAGI4k4%N<%Sbx>r%bF(casW(S1ojm^lp5qDyO!TjBK&fTM3{hH;-+q{~_F-X$hY^;+;'
            's`i$qbFsx=qN1VfeLE089aYDGC)JRqCvBz9Yz}%4)y9wLz>ZKNoCUDoD5nD~+(93W^q`(vv17R6))Q>{X$Um=c}l0bC4)7%d|KtH'
            'Q1JS}ftQ)&PnA@y1}HU6{D&U25`gXDMm`NB-R#lCLn@=6638A9NcfD?lkqnsNKNLzN3$Tyxb<@d37&!fWiKWs8xJyihv8^DBH2-O'
            '*K4E-vkb^#EHl)DK<O3~&q3WH!IK!Hj+7$+RkgzzD=5isLdyQ3wmR-MO#Rw<B2i3{Hd{K|Lfc$2fAl4pUjdbR^u<r2xtd=ko9Rga'
            'GmtB>O6+T=nCrbwk`^GJplYOY>K>X$rRZ^BW<c5NRt%3(&IAol3hWY~zd(A+-g`T2!3{kL=T#c=Hz(46w`JB=$)pr>w^X=0FzRWD'
            '?3gBm386M}9jJ5DG)p%s1m?>k`EzI3<Prl;jc1l4;CUZ30upCqi0+|HvsNr~3p_QFf^^hkkNg>$Y_tO0O93lC1z6*b%^-_}ie~Hg'
            'Hyp^w;`ivnsfnoiBlVW+@BxP`U-z~Cfgg;Kl<e734YBa!;p@=_Qb-Il@4K;zH9D+Z%NE}?YP{p5R*rw?yw&0}8iEo{$c8F+Q~#hI'
            '<}tP^Z`qB_W4lQu;=sjoyPiN}JVA|s*M00nJGBeK?$Dx67~!I+l&eO4Bi<{+8t$^*8Zx;|b@}tKEWVzr;_>Z@sl)0Il?l66IeM}-'
            'w)@#Mr28Z)CD~IfbY#aD8aul#X^p6ATiZ)RXHev~aELPFF6%pS_Ksd$!nSS*t3Ifh78;AV;#ui9>aP+aZBP%)ezH9g4iC8_#CN2u'
            'F`$S<a9t$IqBH&o?zV6BH;S`xChI$w-G%v-vhme0)$QZ+TV92MomK10)6CEVx#!$|0CU#?J!_QOlOi*dx8QA*`C2I0wu0z&a@na`'
            '1f0MRZpEab+E=YBOHOO9-fsUmo};6$MiajpG`W*7P^o-ng{`TuUyAn-M~zV+?gajn$iSL%{^!O8Dt#5Z#_YQygV`3*taz!y`9`O&'
            'LF0QjycmgB!?!1+k7wtilf=PB1RLa+vv4&Fx>Yf7tI2_aWMSo1;m$62s4tB;G_1z1C#b5q>qxVDdG#x26Rz#H<xW9M`qz33tOGBG'
            ';r+x9uGZ|<=!yl>Jo>&L=5m#Fs2<DW%EWeD+r#K9W_Irh>CCp?wkIg<gw8ciNSRXgU=f#bSJ|X!2oq@v|C?;QLt{&5`hbExt-ro1'
            '*pp0&%2Q6iSA_pMmu-sStcp6dPwXM*3GdC=jhbxQg=h6!+7SR4PTljWZ9^BrstH3@tfCRAc8OoKyCRdQxunChE4RDXS?JKu@)7Sj'
            'o6F0GLB05>vvt+)-C5S!K0YUSQ?t>U%Yx0FB?Od_zb>0T)9Su0iK`w-b*_>+0&}TYY03rF2XUP$3g$t?PQ4<|f;rv3hQ!nO@6P#t'
            'B5`$Hx}9}DF!f?e?tc*HGP(|EaC)rjx5H7Q3(~BL#C%WHw&2tx?K=U7<c`&~3%M`Pj1JAXGcBoX(1w$OUaxL`sun_~fl^6#s?0ps'
            'rc5PMv4R4C=?WHg?WDp~%+0*D&9(NfsytN0GK2qxFqM0ZawqeiXSyce!B+cpl^>VG@q|s1tkYPuEA859RNq+n*~5uOpQ?n=HLSjC'
            'vBz$e88)G)!}Gc&59qo?yIUgxzN2mP+?16Zr|)Xs?#k!hJ(hNgK>J5v%VicWfuc|!kEI2^1hZNkMV{5K!`TYp>aGS^6o_hbWuP|Y'
            'oJ!@j9N&O+TXjVvhm@`i{6at8IzzI7qq}=i2-As=c6+au(#lOsQ#s5~w<s;-<qUUoTU0mZOJT9z7HdlhD;MQ_<3@KyFp|8U!jT3W'
            'nbWiAJeV=t&YQLm^Yl4m1@EOS@_-yK9S$4xx7lRyPA5Y5r{woiY_BieI!Lr^x5K9vYqzC|W3X=?E`u!G0LtREE!isE09Fwl6PScV'
            '<bjIwU}XZt;t9*l&h&P(YT7zg4XtX5c};CXEHm!x+QwCu+Gy#lj==`zl{5}Hr&g8u>dWyG+wlC!I0@Z<iSJF&Y`A-iYlS=wyeILX'
            '!Te}VO<>UppjA%>sd=5I%YVKaEy_U2DqfTZ1l#ym=9f#(Qz;{h9HT%Lht&6KM{8HLeR)SM%B5?kpyPZKq;#hSb?<Jd!Kca;$*#gj'
            'xHgX~?&$hTeRch%iEYV|)Kkn*%24x!lU{6!Y*oNPhbg<WYSVZ`+qP`~+=B@}Y@e~};reN;t{OXZ&ZxWhj_LAgl&+q4HYC`_bi{C?'
            'ZvXCKd&aC94qAO;c-%I&aG-MtY}r5yOxr{Q_1-GHj1DCAl#q;SW;hF}J9YQ{tuln3lyaUqE)(K_CHt^TTH!1nP_=i4BzHU)^^f={'
            's_q|Ys`k{kPKZ4?v+zEQDOYwP<9m*MP-(RE0p>kV!W908{yupxiGA61sA^hQ!<KJ%8_;|kG`H3c=hVHN2lsjw=qg0Sc9G+D?3tN^'
            'TMy3Ik|7+f6~^IJa)7UuAdPVDXk2AMDyzz5k<vX#N4shB%~H!&8v(&j4e=~Nts#?mg?QNv#+P5tpmu;#)uS6~b-6OJ<$|$L)|8)d'
            'oK&jXo-PhfsUvnh@XikP*X^7f3*z;{<=i_JxGBFbnP?F&<Kzz6?iD%vZLhyx<CVL<YrdobzENz{tn%94eAV;Q8XE5wIb1h^E^JPn'
            'N6b-GFOP0is8&+Q&cZjlvS(mS!P-g<cib?_$WnDBe8E71hJusPZ^JUz(7l#7PN=e?OdX4cu{+QK(AknYk7=(Ic2qpL8`)m^U&)Ds'
            '_l=ow^kAOL)Us`0Y_z)j$oB0xKDoYlRpf5MB!CU{7<I8YM-@xFIqB8#<m?|hky|CgV7fNtqB%E6l(lPE=jAx)m}bl2ekz_F`Ra|B'
            '7b&G}p!-i5a@c~SjW^owSKdZJbpZ2ZotYcjl>f=PRq0<vD9dCVy0W`0Ky7ZP=OjA4DSw%D$8lkxZhgIrs&5Ikc=jk&rAs$hcj^0>'
            'J)I>1?06=Y)j{xXg(m6yXf6AyElt;tLU+65t@?0{F*wKiZAnh?eIur;g&djRTv491Em$~NUbHXO74%IH58r;<oGQ^3-+rq;bCm>|'
            '=ocE|H*T)$Qf}z3Ar6c}$0UB^i$CzxcGN(=Un_}%Ul9PI1LAJ#Q}gUAW%&n!&}07r`glB~UXLECeWTot(NYxhvJ6S!KYDPc1fRUz'
            'BHObN5wtg75+g9h34g<?iRgrjf^B6~ng}-bG$tog##Rq?uvDtD3eVR!+l=+WHPyz@)J`NjFV;Hv2F`Ow`K@$~Smvn6u&X)mCRG?D'
            '1CuAzrFhHOq1|NrRwVB3mg)>?%34zX307KQih<YODRIdhdknJ%IBm-G_40;iGPmnsZqT6ct~?bkT~<G$OMckSPEHSj)%t<%E9!>n'
            'FlN$$VmXzdXwKVK?Ap%V-ZgoB9^9jPi`_Ja58Hi~IULWUpvt4uX^?A0mA3X-o<k&HoHf5q04!4sY$Ac*!rzL#+!Ulw@nDGCjMM$9'
            'w9Hrx82F@X$E;oPmM!a?y9`4|ZsQGX)b7#jUhN=kk+HS;8r=h>r>qimu^rS*W?V-+!69A+q>1nVN0vpDpRg&kiKuvLOUO}A`;yta'
            'vdu>NjZj0i^)lM}TE_AU`?3mON>!)y!CV!Yx+U1^@73t^*bZ8%ozh;<DoJgLT-G@@qL&It(gSvYM&NazJ-Zp+r0A=hVN45C;ZJ-J'
            '@yb&!N2u}{Dm})@Bms$Tftej9!A_}MWo!-79USgRNx-`KwsZ<dHKQ`5+`*Mc=v=?!?%AjkQuTzT(Af|=UR@8=X2OhBm$%`{j?9O?'
            'owjFHHq>uIDXJlI#76BVBf1q$L@z4Q{S*}epX$4gI5=V}{VG3Iu`bjdUW<!;KVnF%<?}%8$E>rA<y2kj8^IOhoJov<n`mD_NpWha'
            'NEcMb`X1SsOnZ!VH1j=TMXO+KqPMhC&_*F#P778{r~+eIe0^J45ej#^57(U%Fl99nzxy<+_kI5v)kXg70zJPX_w3CBQXAGmEJ6S2'
            '09ViCl#1ZPub460Gr<u4Uvlfue3k2rTcT~Bk35ZA3g}+XLp0=MPgDg|(1NFGNdRimyT=Fa;8CLshpgB}7h2|*Yeykr)glTtF;~2E'
            'Wt7^TfL2+4lQChLGP$<SUReznbvta{%ho_%fuO6xj+wqrfNw|Sd~^kiRY=+&DTo->iQ9Hd*|}J{TRD-jwtc@=`E-Y9;<ET?JtN4i'
            'Ol38SNQnl8v5X2i^2E2|KmKmTiTpD7ZxmE|;_cw$>ENvxU0@uWxVS_E_<xNqM(C2YY*@Y>HCO9y?!8rZuO!<9{0t~8p^+7X+}1J3'
            'ZQr*`mdw2itl+-~JGg;;+1lI=KW@W0Zi_5#`;S-v#OjON@x5VA^YYH&wz4Zj%Y*$F--D(lUUbC)b@}d9yE3`g$vwJ}Fc?B?pB!DB'
            'iVOQ(f5$hZ(yFa~1!!$#Wj%UuKGs#nacQvY^|zFGUY+H8x-Z)PX_}>TJuXfN1cjwLg1#HKjUscy9<KwsCMCQN%?QpAb$c{}yWRG$'
            '4iA33_8<HGzSHqLyC^=H%`u(qQ;@c+rL)_{s@GO@rv35kZQBLwACxR}^x!mfE?9tN%Xl1a!ZFrxJW^n?KQ|ZX)izCCX3*5Ae#HV~'
            'DPL?*ZrF8WRdBji&SKznj!UDYKuHs`CiRsO_&o?MN`i8}_OvTux3a0v&pWK98`4-&fMTEb;p3YRCxg?$tI^5m+%u(pRSJVHxT?0#'
            '8=k%yo}7$I5C`iJd*PC6oKf;|@P1J1X<O*0?zqyyyTN~Z+dzB8(Gk~zv7sD9m9X`>gsphMB|Zw#d4Pc)ZMQNRsJSd1T4%~x>N1yj'
            '4NS$xQrll4zQd$AZ17m=TV+vJIyV`eyY>DzLBK}QmmAunG~*BRlJ9J0GWX6-hv&c_%<ZRn@F@#7(%00Wf<^6WjHgBY%X(cU^tOhG'
            'x`jh4Yi?L?uL{kE)p!meI*T#*8aOw*@i1R6BVtrtQgaS!l_Z<)ZYAI3oJuCk+q-K>v>`U&gtptis}i%r)8X3_*p@^4oi1(WxOxvk'
            '?pa0!I+L_#0|YI8MkU_5;_Q6%U!&8(33Pm+gDARg$IUe&=)RQl@-BekytkB5xRT~3ukW<M93-aP&Su<Y04v)Jj(3|_#p}VF^U(<$'
            '@4nkX)%Q-2r_dUyT{>8Z$i94KPhHTFM6g~xHIT%^c!#;c>Sp00tS2tUD$h_sM_oR`x2TX)5S7K97jM!PyZQm^xUh!^7W0RKGX1?n'
            '=l5Ye4bl0(iB<Sp+TVh5Agy6ZV<f5w_v4Xp<KvI&=xx6yyG+-S*qj2o_2$lNluDeL(IHd!>`lWJ=;0uNl4<97_-1r+JUUlqxuDI('
            '(+l_zW%FrQ+2&AT=aj*yG_q|Icb$Tk-v5a`PI)KN;=3vishO<YP*iBMTJt{1@=~4CnhjO^xx=7jDt%i130$8wmAl(g5#HUKv%7K-'
            'pmw0t0?dqrZN8OpaWBqJ2Or^Y-1#qQ1sCSH%}SZn@#nM|C0x3BZc_-|Y-_}0MOe2fck_@>k?db>O5NOi7ZPn`TJ1D$UV7=jRw8n9'
            '&bQxoCpogh;#^VR9iSS3S5vzNm0+>5qe@T4>(+L}z&6KM)nsHK2-!1xQXyVdo#WC9NqTCjcKB0WyTr>s#j@<f#V0G)yOD!$@a6(c'
            '`x2|dZMh}!1<}EVM<>B<Y!h`|B&w|Y1Kn#yH&7@(lsY4Ck$&CzJAwip>bFR;2c?uyieF^qVlyFGbG~4eROwlz__YesC5i2?M{fm*'
            '>J7$8C7Hu42DZds6Wt!Lw@2mKdGtVLz{fz29<Wo(U|Si;m^OJFedBvgw5>mcSznIC#}B^^5Imw|2WPG1y?<z>BIw|-hA=pee|bVE'
            '{f45>uG&<A6)nzq__Zpj<I!xwKq@+L4AriTRI6xyPf4}gJtU8yE4!%;Iwx^<<d{@Sd}zDhS5C3<jJSCC9$l5jZG$q8x^n{ST+D&w'
            'k+FG&H7HA#yht!ThWN+DuY(@i5jI+2{969}Kg3+Wiv'
        ),
    },
}

def _load_local_pipeline_modules():
    # 1) Zaten import edilebiliyorsa (yerel .venv, repo Colab'da klonlanmis) dogrudan kullan.
    try:
        import local_pipeline
        modules = {}
        for _name in _LOCAL_PIPELINE_MODULE_NAMES:
            modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
        print('local_pipeline modulleri dogrudan import edildi:', list(modules))
        return modules
    except Exception:
        pass

    # 2) Drive'da veya proje kokunun bir ustunde daha once birakilmis bir kopya var mi?
    search_roots = []
    try:
        search_roots.append(Path(DRIVE_PROJECT_ROOT))
    except NameError:
        pass
    search_roots += [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in search_roots:
        candidate_dir = Path(root) / 'local_pipeline'
        if all((candidate_dir / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(candidate_dir.parent))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri onceden birakilmis kopyadan yuklendi:', candidate_dir)
                return modules
            except Exception:
                pass

    # 3) PROJECT_REPO_URL verilmisse git clone dene.
    if PROJECT_REPO_URL:
        clone_dir = Path('/content/youtube_automation_repo') if os.path.isdir('/content') else Path.cwd() / '_youtube_automation_repo'
        if not (clone_dir / 'local_pipeline' / (_LOCAL_PIPELINE_MODULE_NAMES[0] + '.py')).is_file():
            subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO_URL, str(clone_dir)], check=False)
        if all((clone_dir / 'local_pipeline' / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(clone_dir))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri PROJECT_REPO_URL klonundan yuklendi:', clone_dir)
                return modules
            except Exception:
                pass

    # 4) Self-contained fallback: bu hucreye gomulu, build sirasinda
    # local_pipeline/<modul>.py'den birebir uretilmis base64 kopyalari, SHA-256
    # hash'lerini dogrulayarak diske yazip import et.
    fallback_dir = Path(tempfile.mkdtemp(prefix='youtube_automation_bootstrap_'))
    pkg_dir = fallback_dir / 'local_pipeline'
    pkg_dir.mkdir(parents=True, exist_ok=True)
    (pkg_dir / '__init__.py').write_text('', encoding='utf-8')
    for _name, _entry in _LOCAL_PIPELINE_PAYLOADS.items():
        if _entry.get('encoding') != 'zlib+b85':
            raise RuntimeError(f"Desteklenmeyen gomulu payload formati: {_entry.get('encoding')!r}")
        _source_bytes = zlib.decompress(base64.b85decode(_entry['data']))
        _actual_digest = hashlib.sha256(_source_bytes).hexdigest()
        if _actual_digest != _entry['sha256']:
            raise RuntimeError(
                f"Gomulu local_pipeline.{_name} payloadi bozuk/tutarsiz: "
                f"beklenen sha256={_entry['sha256']!r}, hesaplanan={_actual_digest!r}. "
                "Bu notebook yeniden uretilmeli (nb_builder ile)."
            )
        (pkg_dir / f'{_name}.py').write_text(_source_bytes.decode('utf-8'), encoding='utf-8')
    sys.path.insert(0, str(fallback_dir))
    modules = {}
    for _name in _LOCAL_PIPELINE_MODULE_NAMES:
        modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
    print('local_pipeline modulleri: repo/Drive/PROJECT_REPO_URL bulunamadi; gomulu (self-contained, hash dogrulanmis) kopyalar kullanildi.')
    return modules

_local_pipeline_modules = _load_local_pipeline_modules()
video_analysis = _local_pipeline_modules['video_analysis']
script_generation = _local_pipeline_modules['script_generation']
tts = _local_pipeline_modules['tts']
render_utils = _local_pipeline_modules['render_utils']
youtube_data = _local_pipeline_modules['youtube_data']
viral_prediction = _local_pipeline_modules['viral_prediction']
approval = _local_pipeline_modules['approval']
google_ai = _local_pipeline_modules['google_ai']
provider_adapters = _local_pipeline_modules['provider_adapters']
source_strategy = _local_pipeline_modules['source_strategy']



In [ ]:
paths = youtube_data.resolve_project_paths(
    is_colab=IS_COLAB, use_google_drive=USE_GOOGLE_DRIVE,
    project_root=PROJECT_ROOT, drive_project_root=DRIVE_PROJECT_ROOT,
)
RESULTS_DIR = paths['results_dir']
YOUTUBE_DATA_DIR = paths['youtube_data_dir']
YOUTUBE_CACHE_DIR = paths['cache_dir']
RUNTIME_DIR = paths['base'] / '.local_runtime'
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
RENDER_DIR = RESULTS_DIR / 'render'

SOURCE_ASSESSMENT_PATH = RESULTS_DIR / 'source_assessment.json'
PROCESSING_PLAN_PATH = RESULTS_DIR / 'processing_plan.json'
HOT_MOMENTS_PATH = RESULTS_DIR / 'hot_moments.json'
MOMENT_SELECTION_PATH = RESULTS_DIR / 'moment_selection.json'
SELECTED_SCRIPT_PATH = RESULTS_DIR / 'selected_script.json'
SELECTED_VOICEOVER_PATH = RESULTS_DIR / 'selected_voiceover.json'
RENDER_MANIFEST_PATH = RENDER_DIR / 'render_manifest.json'
NORMALIZED_VIDEOS_PATH = YOUTUBE_DATA_DIR / 'normalized_videos.csv'
COLLECTION_MANIFEST_PATH = YOUTUBE_DATA_DIR / 'collection_manifest.json'
VIRAL_PREDICTION_PATH = RESULTS_DIR / 'viral_prediction.json'
GOOGLE_MANIFEST_PATH = RESULTS_DIR / google_ai.profile_scoped_filename(
    'google_capability_manifest.json', GOOGLE_CREDENTIAL_PROFILE)
GOOGLE_QUOTA_LEDGER_PATH = RESULTS_DIR / google_ai.profile_scoped_filename(
    'google_quota_usage.json', GOOGLE_CREDENTIAL_PROFILE)

print('Sonuc dizini:', RESULTS_DIR)
print('Drive mount edildi mi:', paths['drive_mounted'])


def find_artifact(filename, primary_dir):
    candidates = [
        Path(primary_dir) / filename, RESULTS_DIR / filename,
        Path('results') / filename, Path('/content/results') / filename,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    return None


# --- Faz 0 durumu ve ORTAK faz kapisi ------------------------------------
# Burada tanimlanir (Faz 0 hucrelerinde degil) ki alt faz hucreleri, kullanici
# Faz 0'i hic calistirmamis olsa bile NameError yerine TEMIZ bir "plan yok"
# mesaji uretsin. Faz 0 ZORUNLUDUR: plan onaylanmadan hicbir alt faz calismaz.
source_assessment = None
processing_plan = None
plan_gates = None


def require_plan_phase(phase):
    """Alt faz hucrelerinin ortak kapisi. Onaylanmis plan yoksa ya da secilen mod
    bu fazi kapatiyorsa faz TEMIZ durur; hicbir istemci kurulmaz, hicbir API
    cagrisi yapilmaz."""
    _plan = globals().get('processing_plan')
    allowed, reason = source_strategy.plan_allows_phase(
        _plan if isinstance(_plan, dict) else None, phase,
    )
    if not allowed:
        record_phase(phase, 'skipped', None, f'Faz 0 kapisi: {reason}')
        print(f'[{phase}] Faz 0 isleme plani calistirmiyor -> {reason}')
    return allowed


def show_review_prompt(kind, artifact, label):
    """Inceleme ciktisi: artefaktin SHA-256'sini yazdirir. Onay hucresine bu
    degeri yapistirmak, icerigi gercekten gordugunun kaydidir."""
    info = approval.describe_for_review(kind, artifact)
    print('=' * 72)
    print(f'{label} -- INCELEME')
    print('=' * 72)
    print('artifact_status:', info['artifact_status'])
    if info['content_sha256']:
        print('Icerik SHA-256:', info['content_sha256'])
    if not info['reviewable']:
        print()
        print('!!! FINAL ONAY ENGELLI !!!')
        print(info['blocker'])
        if info.get('blocking_conditions'):
            print('Saglanmayan kosullar:')
            for condition in info['blocking_conditions']:
                print('  -', condition)
        return info
    print('Onay hucresine yukaridaki SHA-256 degerini yapistir.')
    return info


## Google yetenek kesfi, kota defteri ve ROTA GOSTERIMI (calistirmadan once)

`client.models.list()` bir modelin **sifir olmayan ucretsiz kotasi oldugunu kanitlamaz**; bu yuzden
kesif sonucu acik bir allowlist ile kesistirilir ve kotasi 0 olan modeller (or. `gemini-2.5-pro`,
goruntu modelleri) otomatik rotaya **hic girmez**. Manifest `results/google_capability_manifest.json`
dosyasina yazilir ve TTL suresince yeniden kullanilir -- her calistirmada tum katalog yoklanmaz.

Kota defteri (`results/google_quota_usage.json`) **YEREL TAHMINDIR**, sunucunun yetkili kotasi
degildir; gunluk sayaclar dokumante edilen **Pasifik gece yarisi** sifirlanmasina gore doner.


In [ ]:
google_ai.assert_no_account_quota_rotation()  # politika ihlali varsa burada durur

GOOGLE_CREDENTIAL = google_ai.resolve_profile_secret(
    'google_api', GOOGLE_CREDENTIAL_PROFILE, GOOGLE_CREDENTIAL_PROFILES, get_secret)
google_quota_ledger = google_ai.QuotaLedger(
    GOOGLE_QUOTA_LEDGER_PATH, credential_profile=GOOGLE_CREDENTIAL_PROFILE)

google_manifest = google_ai.load_or_build_capability_manifest(
    RESULTS_DIR,
    client_factory=None,
    credential=GOOGLE_CREDENTIAL,
    credential_profile=GOOGLE_CREDENTIAL_PROFILE,
    run_connection_test=RUN_GOOGLE_CONNECTION_TEST,
) if GOOGLE_CREDENTIAL.configured else None

print('Gemini kimlik bilgisi:', google_ai.describe_credential_state(GOOGLE_CREDENTIAL),
      '| manuel profil:', GOOGLE_CREDENTIAL_PROFILE, '| otomatik profil gecisi: KAPALI')
if google_manifest is None:
    print('Google yetenek kesfi ATLANDI (anahtar yok).')
    print(google_ai.credential_help_text('google_api'))
else:
    print('Kesif dogrulandi mi:', google_manifest['discovery_verified'],
          '| onbellek:', google_manifest.get('cache'))
    if google_manifest.get('discovery_error'):
        print('Kesif hatasi (sanitize):', google_manifest['discovery_error'])
    print('Kullanilabilir modeller:', ', '.join(google_manifest['usable_models']) or '(yok)')
    if google_manifest['excluded_zero_quota']:
        print('0 kota nedeniyle DISLANAN:', ', '.join(google_manifest['excluded_zero_quota']))
    if google_manifest.get('connection_test'):
        print('Baglanti testi:', google_manifest['connection_test']['ok'],
              '| test edilen:', google_manifest['connection_test']['tested_models'])


def show_google_route(task):
    """Calistirmadan ONCE secilen rotayi yazdirir. Gizli deger ICERMEZ."""
    selection = google_ai.select_google_route(
        task, mode=GOOGLE_ROUTE_MODE, overrides=GOOGLE_TASK_MODEL_OVERRIDES,
        manifest=google_manifest, ledger=google_quota_ledger,
        enable_model_fallback=ENABLE_GOOGLE_MODEL_FALLBACK,
        max_model_attempts=MAX_GOOGLE_MODEL_ATTEMPTS,
    )
    print(google_ai.format_route_line(selection, GOOGLE_CREDENTIAL,
                                       raw_media=ALLOW_RAW_MEDIA_TO_FREE_TIER))
    if selection['skipped']:
        for skip in selection['skipped']:
            print(f"  atlandi: {skip['model']} -> {skip['reason']}")
    print()
    return selection


for _task in ('hotmoment_review', 'script_generation', 'bulk_classification',
              'grounded_research', 'tts', 'embeddings'):
    show_google_route(_task)

print('Capraz-saglayici (opsiyonel) durum:',
      provider_adapters.describe_provider_states(get_secret))


## Faz 1 — Video Analizi / Hot-Moment Tespiti (`RUN_VIDEO_ANALYSIS`)

Videoyu Drive'daki sabit `source_videos/` klasorune yukle. Dosya adi girmen gerekmez; klasordeki en yeni desteklenen video otomatik secilir. Gerekirse `SOURCE_VIDEO_PATH` ile acik bir dosya yolu vererek otomatik secimi gecersiz kilabilirsin.

In [ ]:
SOURCE_VIDEO_DIR = paths['source_videos_dir']
SOURCE_VIDEO_PATH = ''  # Bos birak: SOURCE_VIDEO_DIR icindeki en yeni video otomatik secilir.
CLIP_START_SEC = 0
CLIP_DURATION_SEC = 240
WHISPER_MODEL_SIZE = 'small'  # 'tiny' | 'small' | 'large-v3-turbo' (GPU onerilir)
SCENE_DETECTOR = 'content'
TOP_N_HOT_MOMENTS = 5


## Faz 0 — Kaynak İşleme Stratejisi (ZORUNLU)

Kaynak video seçildikten sonra **hiçbir alt faz, onaylanmış bir işleme planı olmadan çalışmaz.**
Bu hücre kaynağı ölçer ve iki mod için **yerel** uygunluk skoru üretir; hiçbir API çağrısı yapmaz.

| Mod | Ne yapar | API |
|---|---|---|
| `direct_clip_series` | Birden fazla hot moment → birden fazla Short. **Orijinal ses korunur**, mevcut dinamik aktif-kelime altyazı ve render hattı kullanılır. | script/TTS **çağrılmaz** |
| `narrated_theory` | Seçilen momentler **görsel kanıt/B-roll** olur; senin yazdığın teori/yorum açısına göre mevcut Gemini rotasından yeni script üretilir, script → TTS → render onay zinciri aynen işler. | Gemini script + TTS |

**Öneri bir seçim değildir.** `assess_source()` her zaman `selected_mode=None` ve
`auto_selected=False` döner; modu Faz 0a'da sen seçersin, Faz 0b'de okursun, Faz 0c'de
gösterilen SHA-256 ile onaylarsın. Kaynağı, modu, momentleri, teori açısını, çıktı sayısını,
hedef süreyi veya render ayarını değiştirmek onayı **otomatik geçersiz kılar**.


In [ ]:
source_assessment = None
ASSESSED_SOURCE_PATH = ''

try:
    ASSESSED_SOURCE_PATH = str(video_analysis.resolve_source_video(
        SOURCE_VIDEO_DIR, SOURCE_VIDEO_PATH,
    ))
    print('Degerlendirilen kaynak video:', ASSESSED_SOURCE_PATH)
    _ffmpeg_bin, _ffprobe_bin = render_utils.resolve_ffmpeg_binaries()

    # Varsa mevcut analiz sinyalleri okunur; YOKSA Faz 0 yine calisir ve skorlar
    # yalnizca kaynak sinyallerine dayanir (uydurma sinyal uretilmez).
    _existing_hot_moments = None
    _hm_path = find_artifact('hot_moments.json', RESULTS_DIR)
    if _hm_path is not None:
        with open(_hm_path, 'r', encoding='utf-8') as _f:
            _candidate = json.load(_f)
        if not video_analysis.validate_hot_moments_schema(_candidate):
            _existing_hot_moments = _candidate

    source_assessment = source_strategy.assess_source(
        ASSESSED_SOURCE_PATH, _ffmpeg_bin, _ffprobe_bin,
        hot_moments_data=_existing_hot_moments,
        google_credential_configured=GOOGLE_CREDENTIAL.configured,
    )
    youtube_data.write_json(SOURCE_ASSESSMENT_PATH, source_assessment)
    print(source_strategy.format_assessment_panel(source_assessment))
    record_phase('source_assessment', 'completed', source_assessment['status'],
                 f"oneri={source_assessment['recommended_mode']} (secim YAPILMADI)")
except Exception as exc:
    record_phase('source_assessment', 'failed', None, f'{type(exc).__name__}: {exc}')
    print('Kaynak degerlendirmesi basarisiz:', type(exc).__name__, exc)


### Faz 0a — Modu SEÇ ve işleme planını oluştur

Aşağıdaki formu doldur. `PROCESSING_MODE` boş bırakılamaz ve varsayılanı yoktur —
öneriyi kabul etsen bile modu **açıkça** yazman gerekir.

- `direct_clip_series`: `THEORY_ANGLE` boş kalmalı, `PLAN_OUTPUT_COUNT` kaç Short istediğindir.
- `narrated_theory`: `THEORY_ANGLE` en az 20 karakter olmalı (script bu açıya göre üretilir),
  `PLAN_OUTPUT_COUNT = 1`.
- `PLAN_MOMENT_RANKS` boş bırakılırsa Faz 1'in en iyi adayları kullanılır.


In [ ]:
PROCESSING_MODE = ''  # @param ["", "direct_clip_series", "narrated_theory"]
PLAN_SELECTED_BY = ''  # @param {type:"string"}
PLAN_OUTPUT_COUNT = 0  # @param {type:"integer"}  (0 = degerlendirmenin onerdigi sayi)
PLAN_TARGET_DURATION_SEC = 60  # @param {type:"integer"}
PLAN_MOMENT_RANKS_TEXT = ''  # @param {type:"string"}  ornek: "1,3,4" (bos = Faz 1 en iyileri)
THEORY_ANGLE = ''  # @param {type:"string"}  yalnizca narrated_theory icin
PLAN_NOTES = ''  # @param {type:"string"}
CONFIRM_PROCESSING_PLAN = False  # @param {type:"boolean"}


In [ ]:
if not CONFIRM_PROCESSING_PLAN:
    record_phase('processing_plan', 'skipped', None, 'CONFIRM_PROCESSING_PLAN=False')
    print('Isleme plani olusturulmadi. Faz 0 panelini oku, modu sec, sonra bu hucreyi calistir.')
elif not isinstance(globals().get('source_assessment'), dict):
    record_phase('processing_plan', 'failed', None, 'source_assessment.json yok')
    print("Duzeltme: once Faz 0 degerlendirme hucresini calistir.")
else:
    try:
        _plan_ranks = [int(_p.strip()) for _p in PLAN_MOMENT_RANKS_TEXT.split(',') if _p.strip()]
        # Render/anlatim ayarlari da KARAR blogunun parcasidir: degisirlerse onay duser.
# Render/anlatim ayarlari KARAR blogunun parcasidir: sonradan degisirlerse
        # hem onay hash'i hem de render oncesi drift kontrolu bunu yakalar.
        # Bu hucreler notebook'ta ASAGIDA tanimli oldugu icin globals()'tan okunur.
        _g = globals()
        _plan_render_config = {
            'render_mode': _g.get('RENDER_MODE', 'blurred_background'),
            'preview_mode': _g.get('PREVIEW_MODE', True),
            'caption_style': _g.get('CAPTION_STYLE', 'active_word_yellow'),
            'caption_theme': _g.get('CONTENT_THEME', 'auto'),
        }
        # Anlatim ayarlari (ton, ses, dil) KARAR blogunda TUTULMAZ: bunlar zaten
        # script metninin ve ses dosyasinin kendi hash-bagli onaylarina yansir,
        # ve Faz 0 notebook'ta o hucrelerden ONCE geldigi icin burada okunsalardi
        # ilk gecistee None kaydedilirlerdi. Faz 0 yalnizca kendi sahip oldugu
        # kararlari (mod, teori acisi, cikti sayisi, hedef sure, render ayari)
        # hash'ler. API cagiranlar (or. Creator Console) isterse acikca gecebilir.
        _plan_narration_config = None

        processing_plan = source_strategy.build_processing_plan(
            source_assessment, PROCESSING_MODE, PLAN_SELECTED_BY,
            output_count=PLAN_OUTPUT_COUNT or None,
            target_duration_sec=PLAN_TARGET_DURATION_SEC or None,
            moment_ranks=_plan_ranks, theory_angle=THEORY_ANGLE,
            render_config=_plan_render_config, narration_config=_plan_narration_config,
            render_root=RENDER_DIR, notes=PLAN_NOTES,
        )
        youtube_data.write_json(PROCESSING_PLAN_PATH, processing_plan)
        print(source_strategy.format_plan_panel(processing_plan))
        record_phase('processing_plan', 'completed', processing_plan['status'],
                     f"mod={processing_plan['decision']['mode']} "
                     f"cikti={processing_plan['decision']['output_count']}")
    except Exception as exc:
        record_phase('processing_plan', 'failed', None, f'{type(exc).__name__}: {exc}')
        print('Isleme plani olusturulamadi:', exc)


### Faz 0b — Planı İNCELE

Planı oku ve aşağıda yazdırılan **SHA-256** değerini kopyala. Bu değer, kararın tam olarak
hangi içeriğe (kaynak, mod, momentler, teori açısı, çıktı sayısı, süre, render ayarı)
bağlandığının kaydıdır.


In [ ]:
if processing_plan is None:
    _found_plan = find_artifact('processing_plan.json', RESULTS_DIR)
    if _found_plan is not None:
        with open(_found_plan, 'r', encoding='utf-8') as _f:
            processing_plan = json.load(_f)
        PROCESSING_PLAN_PATH = _found_plan

if processing_plan is None:
    print('Incelenecek isleme plani yok. Once Faz 0a hucresini calistir.')
    plan_review_info = None
else:
    _plan_errors = source_strategy.validate_processing_plan(
        processing_plan, assessment=source_assessment,
        source_fingerprint=(source_assessment or {}).get('source', {}).get('fingerprint_sha256'),
    )
    _plan_errors += source_strategy.check_config_drift(
        processing_plan,
        render_config={
            'render_mode': globals().get('RENDER_MODE', 'blurred_background'),
            'preview_mode': globals().get('PREVIEW_MODE', True),
            'caption_style': globals().get('CAPTION_STYLE', 'active_word_yellow'),
            'caption_theme': globals().get('CONTENT_THEME', 'auto'),
        },
    )
    if _plan_errors:
        print('PLAN GECERSIZ/BAYAT -> onaylanmamali:')
        for _err in _plan_errors:
            print('  -', _err)
    plan_review_info = show_review_prompt('processing_plan', processing_plan, 'ISLEME PLANI')
    print()
    print(source_strategy.format_plan_panel(processing_plan))


### Faz 0c — Planı ONAYLA (ayrı adım)

`PLAN_REVIEWED_SHA256` alanına Faz 0b'de yazdırılan hash'i yapıştır. Onay yalnızca hash
şu anki kararla eşleşirse verilir; kaynağı/modu/momentleri/teori açısını/render ayarını
sonradan değiştirirsen onay otomatik geçersiz olur ve alt fazlar tekrar durur.


In [ ]:
PLAN_REVIEWED_SHA256 = ''   # inceleme hucresindeki SHA-256'yi buraya yapistir
PLAN_APPROVED_BY = ''       # onayi veren kisi
PLAN_APPROVAL_NOTES = ''

if processing_plan is None:
    print('Onaylanacak isleme plani yok.')
elif not PLAN_REVIEWED_SHA256 or not PLAN_APPROVED_BY:
    print('Onay verilmedi: PLAN_REVIEWED_SHA256 ve PLAN_APPROVED_BY zorunlu.')
    print('Faz 0b inceleme hucresini calistir, gosterilen hash i buraya yapistir.')
else:
    try:
        processing_plan = approval.approve_artifact(
            processing_plan, 'processing_plan', PLAN_REVIEWED_SHA256,
            PLAN_APPROVED_BY, PLAN_APPROVAL_NOTES,
        )
        youtube_data.write_json(PROCESSING_PLAN_PATH, processing_plan)
        record_phase('processing_plan', 'completed', processing_plan['status'],
                     f"onaylayan={PLAN_APPROVED_BY}")
        print('Isleme plani ONAYLANDI ->', PROCESSING_PLAN_PATH)
    except ValueError as exc:
        print('ONAY REDDEDILDI:', exc)

# Onaylanmis plan alt fazlarin kapisidir; PIPELINE_MODE plandan TURETILIR.
plan_gates = None
if isinstance(processing_plan, dict) and processing_plan.get('status') == 'approved':
    PIPELINE_MODE = processing_plan['pipeline_mode']
    plan_gates = source_strategy.describe_phase_plan(processing_plan)
    print()
    print('Turetilen PIPELINE_MODE:', PIPELINE_MODE)
    print('Faz kapilari:')
    for _phase, _gate in plan_gates.items():
        print(f"  {'ACIK ' if _gate['allowed'] else 'KAPALI'} {_phase}"
              + (f" -- {_gate['reason']}" if _gate['reason'] else ''))
else:
    print()
    print('Plan onayli degil -> tum alt fazlar duracak (Faz 0 ZORUNLUDUR).')


In [ ]:
hot_moments_data = None

if not RUN_VIDEO_ANALYSIS:
    record_phase('video_analysis', 'skipped')
elif not require_plan_phase('video_analysis'):
    pass
else:
    try:
        SOURCE_VIDEO_PATH = str(video_analysis.resolve_source_video(
            SOURCE_VIDEO_DIR, SOURCE_VIDEO_PATH,
        ))
        print('Otomatik secilen kaynak video:', SOURCE_VIDEO_PATH)
        ffmpeg_bin, _ffprobe_bin = render_utils.resolve_ffmpeg_binaries()
        hot_moments_data = video_analysis.run_video_analysis(
            ffmpeg_bin, SOURCE_VIDEO_PATH, RUNTIME_DIR,
            clip_start_sec=CLIP_START_SEC, clip_duration_sec=CLIP_DURATION_SEC,
            whisper_model_size=WHISPER_MODEL_SIZE, scene_detector=SCENE_DETECTOR,
            top_n=TOP_N_HOT_MOMENTS,
            preview_dir=RESULTS_DIR / 'moment_previews',
        )
        youtube_data.write_json(HOT_MOMENTS_PATH, hot_moments_data)
        record_phase('video_analysis', 'completed', hot_moments_data['status'],
                     f"{len(hot_moments_data['hot_moments'])} aday -> {HOT_MOMENTS_PATH.name}")
    except Exception as exc:
        record_phase('video_analysis', 'failed', None, f'{type(exc).__name__}: {exc}')


## Faz 1a — Moment Explorer: adayları İZLE

Bu hücre en yüksek skorlu adayları gerçek MP4 preview olarak gösterir. Zamanı, skoru, nedeni ve transcript'i kontrol et. İzlemeden seçim onayı verme.

In [ ]:
moment_explorer_data = hot_moments_data if isinstance(hot_moments_data, dict) else None
if moment_explorer_data is None:
    _found_hot_moments = find_artifact('hot_moments.json', RESULTS_DIR)
    if _found_hot_moments:
        with open(_found_hot_moments, 'r', encoding='utf-8') as _f:
            moment_explorer_data = json.load(_f)

if moment_explorer_data is None:
    print('Moment Explorer: hot_moments.json yok. Once Faz 1 video analizini calistir.')
else:
    _moment_schema_errors = video_analysis.validate_hot_moments_schema(moment_explorer_data)
    if _moment_schema_errors:
        print('Moment Explorer acilamadi; sema hatalari:', _moment_schema_errors)
    else:
        from IPython.display import Video, Markdown, display
        print('Aday sayisi:', len(moment_explorer_data['hot_moments']))
        for _moment in sorted(moment_explorer_data['hot_moments'], key=lambda m: m['rank']):
            _start = _moment.get('source_start_sec', _moment.get('aligned_start_sec'))
            _end = _moment.get('source_end_sec', _moment.get('aligned_end_sec'))
            display(Markdown(
                f"### Aday #{_moment['rank']} — skor {_moment.get('score')} "
                f"— kaynak zamanı {_start:.2f}s–{_end:.2f}s"
            ))
            print('Secilme nedenleri:', _moment.get('reason') or '(yok)')
            print('Transcript:', _moment.get('transcript') or '(konusma tespit edilmedi)')
            _preview = _moment.get('preview_path')
            if _preview and Path(_preview).is_file() and Path(_preview).stat().st_size > 0:
                display(Video(_preview, embed=True, width=560, html_attributes='controls preload=\"metadata\"'))
            else:
                print('PREVIEW YOK:', _preview, _moment.get('preview_error') or '')
            print('-' * 72)


## Faz 1b — Moment kararını KAYDET

Preview'ları izledikten sonra aşağıdaki formu doldur. `CONFIRM_MOMENT_SELECTION=True` olmadan seçim kaydedilmez. Karar `moment_selection.json` içinde analiz ve preview SHA-256 değerlerine bağlanır; video/analiz/preview değişirse seçim geçersiz olur.

In [ ]:
SELECTED_MOMENT_RANK = 1  # @param {type:"integer"}
REJECTED_MOMENT_RANKS_TEXT = ""  # @param {type:"string"}
MOMENT_REVIEWED_BY = ""  # @param {type:"string"}
MOMENT_REVIEW_NOTES = ""  # @param {type:"string"}
CONFIRM_MOMENT_SELECTION = False  # @param {type:"boolean"}
REQUIRE_MOMENT_PREVIEW = True


In [ ]:
moment_selection = None
_selection_source = moment_explorer_data if isinstance(moment_explorer_data, dict) else None
if not require_plan_phase('moment_selection'):
    pass
elif _selection_source is None:
    record_phase('moment_selection', 'failed', None, 'hot_moments.json bulunamadi')
elif not CONFIRM_MOMENT_SELECTION:
    record_phase('moment_selection', 'skipped', None, 'CONFIRM_MOMENT_SELECTION=False')
    print('Moment secimi kaydedilmedi. Previewlari izle; sonra formu doldurup onayi True yap.')
else:
    try:
        _rejected_ranks = [
            int(_part.strip()) for _part in REJECTED_MOMENT_RANKS_TEXT.split(',')
            if _part.strip()
        ]
        moment_selection = video_analysis.build_moment_selection(
            _selection_source, selected_moment_rank=SELECTED_MOMENT_RANK,
            reviewed_by=MOMENT_REVIEWED_BY, rejected_moment_ranks=_rejected_ranks,
            review_notes=MOMENT_REVIEW_NOTES, require_preview=REQUIRE_MOMENT_PREVIEW,
        )
        youtube_data.write_json(MOMENT_SELECTION_PATH, moment_selection)
        record_phase(
            'moment_selection', 'completed', 'approved',
            f"rank={moment_selection['selected_moment_rank']} reviewed_by={moment_selection['reviewed_by']}",
        )
        print('Moment secimi KAYDEDILDI ->', MOMENT_SELECTION_PATH)
        print('Secilen rank:', moment_selection['selected_moment_rank'])
        print('Preview SHA-256:', moment_selection['selected_moment']['preview_sha256'])
    except Exception as exc:
        record_phase('moment_selection', 'failed', None, f'{type(exc).__name__}: {exc}')
        print('Moment secimi kaydedilemedi:', exc)


## Faz 2 — Script Uretimi (`RUN_SCRIPT_GENERATION`)

Bağımlılık: geçerli ve hash-bağlı `moment_selection.json`. Geçerli insan seçimi yoksa ücretli çağrı sayısı **0**;
aksi halde `ALLOW_SCRIPT_PROVIDER_FALLBACK` kapalıyken tam olarak **1**.

Script hücresi manuel bir onay boolean'ına güvenmez. Faz 1b'nin ürettiği seçimi, güncel `hot_moments.json`
ve seçili preview dosyasının SHA-256 değeriyle doğrular. Script onayı aşağıda ayrı bir hücrededir.

In [ ]:
# --- Ucretli saglayici secimi (fan-out YOK) ---
# Varsayilan: Google/Gemini (GOOGLE_API_KEY). OpenAI/Anthropic fallback KAPALI --
# key'leri eksikse hata degil, sadece atlanir/loglanir.
SCRIPT_PROVIDER = 'google'      # 'google' | 'anthropic' | 'openai' (google = kanonik varsayilan)
SCRIPT_MODEL = None             # None = Google Rota A zinciri (3.6-flash -> 3.5-flash -> 3.5-flash-lite -> gemma-4-31b-it);
                                # bir model adi verirsen router SADECE o modele sabitlenir
ALLOW_SCRIPT_PROVIDER_FALLBACK = False  # True: adaylar SIRAYLA denenir, ilk basarida durur

CONTENT_MODE = 'spoiler-free-recap'
TONE = 'gergin ve merak uyandirici'
TARGET_DURATION_SEC = 60

In [ ]:
selected_script = None
moment_selection = moment_selection if isinstance(globals().get('moment_selection'), dict) else None
SELECTED_MOMENT_RANK = None
MOMENT_APPROVED = False
MOMENT_APPROVED_BY = ''

if not RUN_SCRIPT_GENERATION:
    record_phase('script_generation', 'skipped')
elif not require_plan_phase('script_generation'):
    # direct_clip_series bu noktada durur: istemci KURULMAZ, hicbir script API
    # cagrisi yapilmaz.
    pass
else:
    if hot_moments_data is None:
        found_path = find_artifact('hot_moments.json', RESULTS_DIR)
        if found_path is None:
            record_phase('script_generation', 'failed', None, 'hot_moments.json bulunamadi')
            print("Duzeltme: RUN_VIDEO_ANALYSIS=True ile Faz 1'i calistir veya hot_moments.json "
                  "dosyasini RESULTS_DIR altina yerlestir.")
            hot_moments_data = 'MISSING'
        else:
            with open(found_path, 'r', encoding='utf-8') as f:
                hot_moments_data = json.load(f)
            schema_errors = video_analysis.validate_hot_moments_schema(hot_moments_data)
            if schema_errors:
                record_phase('script_generation', 'failed', None,
                             f'hot_moments.json semasi gecersiz: {schema_errors}')
                hot_moments_data = 'MISSING'
            else:
                HOT_MOMENTS_PATH = found_path
                print('Mevcut hot_moments.json bulundu ve dogrulandi:', found_path)

    if hot_moments_data not in (None, 'MISSING'):
        if moment_selection is None:
            _found_selection = find_artifact('moment_selection.json', RESULTS_DIR)
            if _found_selection:
                with open(_found_selection, 'r', encoding='utf-8') as _f:
                    moment_selection = json.load(_f)
                MOMENT_SELECTION_PATH = _found_selection

        if moment_selection is None:
            print('Gecerli moment_selection.json yok -> ucretli script cagrisi yapilmayacak.')
            print('Duzeltme: Faz 1a previewlarini izle ve Faz 1b karar hucresini calistir.')
        else:
            _selection_errors = video_analysis.validate_moment_selection(
                moment_selection, hot_moments_data, require_preview=True)
            if _selection_errors:
                print('Moment secimi GECERSIZ -> ucretli script cagrisi yapilmayacak:')
                for _error in _selection_errors:
                    print('  -', _error)
            else:
                SELECTED_MOMENT_RANK = moment_selection['selected_moment_rank']
                MOMENT_APPROVED = True
                MOMENT_APPROVED_BY = moment_selection['reviewed_by']
                print('Moment secimi dogrulandi: rank=', SELECTED_MOMENT_RANK,
                      '| reviewed_by=', MOMENT_APPROVED_BY)

        # Istemciler yalnizca AN onaylandiysa kurulur; kurulmalari tek basina
        # cagri yapmaz, ama gereksiz kurulumdan da kaciniyoruz.
        openai_client = anthropic_client = google_client = None
        if MOMENT_APPROVED:
            if OPENAI_API_KEY:
                from openai import OpenAI
                openai_client = OpenAI(api_key=OPENAI_API_KEY)
            if ANTHROPIC_API_KEY:
                import anthropic
                anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
            if GOOGLE_CREDENTIAL.configured and RUN_GOOGLE_SCRIPT_GENERATION:
                # Istemci TEK bir kimlik bilgisinden kurulur (hesap havuzu yok).
                google_client = google_ai.build_google_client(GOOGLE_CREDENTIAL)
                show_google_route('script_generation')
            elif not RUN_GOOGLE_SCRIPT_GENERATION:
                print('RUN_GOOGLE_SCRIPT_GENERATION=False -> Google script uretimi atlandi.')

        try:
            selected_script = script_generation.run_script_generation(
                hot_moments_data, HOT_MOMENTS_PATH, selected_moment_rank=SELECTED_MOMENT_RANK,
                openai_client=openai_client, anthropic_client=anthropic_client,
                google_client=google_client,
                content_mode=CONTENT_MODE, tone=TONE, target_duration_sec=TARGET_DURATION_SEC,
                moment_approved=MOMENT_APPROVED, moment_approved_by=MOMENT_APPROVED_BY,
                script_provider=SCRIPT_PROVIDER, script_model=SCRIPT_MODEL,
                allow_provider_fallback=ALLOW_SCRIPT_PROVIDER_FALLBACK,
                google_ledger=google_quota_ledger, google_manifest=google_manifest,
                google_route_mode=GOOGLE_ROUTE_MODE,
                google_task_model_overrides=GOOGLE_TASK_MODEL_OVERRIDES,
                enable_google_model_fallback=ENABLE_GOOGLE_MODEL_FALLBACK,
                max_google_model_attempts=MAX_GOOGLE_MODEL_ATTEMPTS,
                enable_cross_provider_fallback=ENABLE_CROSS_PROVIDER_FALLBACK,
                max_provider_attempts=MAX_PROVIDER_ATTEMPTS,
                cross_provider_secret_getter=get_secret,
            )
            youtube_data.write_json(SELECTED_SCRIPT_PATH, selected_script)
            paid_calls = len([a for a in selected_script.get('provider_attempts', []) if a.get('provider')])
            record_phase('script_generation', 'completed', selected_script['status'],
                         f"ucretli cagri sayisi={paid_calls}, saglayici={selected_script.get('provider')}")
            for attempt in selected_script.get('provider_attempts', []):
                flag = 'OK' if attempt['ok'] else 'BASARISIZ'
                print(f"  deneme: {attempt['provider']}/{attempt['model']} -> {flag}"
                      + (f" ({attempt['error']})" if attempt['error'] else ''))
                for _google_attempt in ((attempt.get('google_route') or {}).get('provider_attempts') or []):
                    _gflag = 'OK' if _google_attempt['ok'] else 'BASARISIZ'
                    print(f"      google: {_google_attempt['model']} -> {_gflag}"
                          + (f" [{_google_attempt.get('error_kind')}] {_google_attempt.get('error')}"
                             if not _google_attempt['ok'] else ''))
        except Exception as exc:
            record_phase('script_generation', 'failed', None, f'{type(exc).__name__}: {exc}')


### Faz 2a — Uretilen scripti INCELE

Asagidaki hucre scripti ve SHA-256'sini yazdirir. **Scripti oku**, sonra bir sonraki onay hucresine
bu SHA-256 degerini yapistir. Onay, uretim cagrisinda degil burada verilir.

In [ ]:
if selected_script is None:
    print('Faz 2 calismadi; incelenecek script yok.')
elif not selected_script.get('script_text'):
    print('Script uretilmedi (sadece prompt paketi). status:', selected_script['status'])
    print('status_reason:', selected_script['status_reason'])
    if selected_script.get('prompt_package'):
        print()
        print('--- PROMPT PAKETI (harici/elle uretim icin) ---')
        print(selected_script['prompt_package']['user_prompt'])
else:
    print('Saglayici/model:', selected_script['provider'], '/', selected_script['model'])
    print('Tahmini maliyet (USD):', selected_script['cost_usd_estimate'])
    print()
    print('--- URETILEN SCRIPT ---')
    print(selected_script['script_text'])
    print()
    script_review_info = show_review_prompt('script', selected_script, 'SCRIPT')


### Faz 2b — Scripti ONAYLA (ayrı adım)

Yukarıdaki scripti okuduktan **sonra** `SCRIPT_REVIEWED_SHA256` alanına inceleme hücresinin yazdırdığı
hash'i yapıştır ve `SCRIPT_APPROVED_BY` doldur. Hash eşleşmezse onay **reddedilir**.

In [ ]:
SCRIPT_REVIEWED_SHA256 = ''   # inceleme hucresindeki SHA-256'yi buraya yapistir
SCRIPT_APPROVED_BY = ''       # ornek: 'insan: birkan'
SCRIPT_APPROVAL_NOTES = ''

if selected_script is None or not selected_script.get('script_text'):
    print('Onaylanacak bir script yok; bu hucre atlandi.')
elif not SCRIPT_REVIEWED_SHA256 or not SCRIPT_APPROVED_BY:
    print('Onay ATLANDI: SCRIPT_REVIEWED_SHA256 ve SCRIPT_APPROVED_BY doldurulmali.')
    print('selected_script.json status olarak kaliyor:', selected_script['status'])
else:
    try:
        selected_script = approval.approve_artifact(
            selected_script, 'script', SCRIPT_REVIEWED_SHA256,
            SCRIPT_APPROVED_BY, SCRIPT_APPROVAL_NOTES,
        )
        youtube_data.write_json(SELECTED_SCRIPT_PATH, selected_script)
        record_phase('script_generation', 'completed', selected_script['status'],
                     f"onaylayan={SCRIPT_APPROVED_BY}")
        print('Script ONAYLANDI ->', SELECTED_SCRIPT_PATH)
    except ValueError as exc:
        print('ONAY REDDEDILDI:', exc)
        print('selected_script.json status olarak kaliyor:', selected_script['status'])


## Faz 3 — TTS / Seslendirme (`RUN_TTS`)

Bagimlilik: **onaylanmis** `selected_script.json`. Script `approved` degilse hicbir ucretli TTS
cagrisi yapilmaz. **Ucretli cagri sayisi:** fallback kapaliyken tam olarak **1**.

Kalite-oncelikli varsayilan: ElevenLabs `eleven_multilingual_v2`. Opsiyonel uretim fallback'i:
OpenAI `tts-1-hd`. gTTS **yalnizca** `ALLOW_DEV_GTTS_FALLBACK = True` iken ve hicbir uretim
saglayicisi basarili olmadiginda calisir; sonucu her zaman `development_placeholder` kalir ve
**asla onaylanamaz**.

In [ ]:
# Varsayilan: mevcut GeminiKey ile Gemini TTS. Ses uretimi zorunlu ana-pipeline adimidir.
# Piper lokal; ElevenLabs/OpenAI ise kalite karsilastirmasi icin alternatiflerdir.
TTS_PROVIDER = 'gemini'              # 'piper' | 'gemini' | 'elevenlabs' | 'openai'
                                     # 'gemini': RUN_GOOGLE_TTS=True olmali; Gemini TTS
                                     # modelleri PREVIEW oldugu icin cikti HER ZAMAN
                                     # review_required kalir ve yedek ses/model onceki
                                     # seslendirmenin onayini ASLA devralmaz.
ALLOW_TTS_PROVIDER_FALLBACK = False   # True: piper basarisiz olursa siradaki uretim saglayicisi denenir
ALLOW_DEV_GTTS_FALLBACK = False       # True: hicbir uretim saglayicisi (piper dahil) calismazsa gTTS
                                       # (SADECE development_placeholder -- ASLA onaylanamaz, bkz. approval.py)

# Piper ayarlari: binary/ses modeli eksikse Faz 3 temiz sekilde 'insufficient_data'
# doner, notebook COKMEZ (bkz. local_pipeline/tts.py call_piper_tts).
PIPER_BINARY = 'piper'                # PATH icindeki komut veya piper binary'sine tam yol
PIPER_VOICE_MODEL_PATH = None         # ör: '/content/piper-voices/tr_TR-fahrettin-medium.onnx' -- yoksa None birak

In [ ]:
selected_voiceover = None

if not RUN_TTS:
    record_phase('tts', 'skipped')
elif not require_plan_phase('tts'):
    # direct_clip_series bu noktada durur: istemci KURULMAZ, hicbir TTS API
    # cagrisi yapilmaz.
    pass
else:
    if selected_script is None:
        found_path = find_artifact('selected_script.json', RESULTS_DIR)
        if found_path is None:
            record_phase('tts', 'failed', None, 'selected_script.json bulunamadi')
            print("Duzeltme: RUN_SCRIPT_GENERATION=True ile Faz 2'yi calistir veya "
                  "selected_script.json dosyasini RESULTS_DIR altina yerlestir.")
        else:
            with open(found_path, 'r', encoding='utf-8') as f:
                selected_script = json.load(f)
            SELECTED_SCRIPT_PATH = found_path
            print('Mevcut selected_script.json bulundu:', found_path,
                  '(status:', selected_script.get('status'), ')')

    if selected_script is not None:
        # Diskten gelen bir onay hala GECERLI mi? Script onaylandiktan sonra
        # degistirildiyse onay gecersizdir ve ucretli TTS calismamalidir.
        script_check = approval.check_approval(selected_script, 'script')
        if selected_script.get('status') == 'approved' and not script_check['valid']:
            print('UYARI:', script_check['reason'])
            print('Bu script onayi gecersiz sayiliyor; uretim TTS calistirilmayacak.')
            selected_script = dict(selected_script)
            selected_script['status'] = 'review_required'

        if selected_script.get('status') != 'approved':
            print('NOT: script onayli degil ->', selected_script.get('status'),
                  '| uretim TTS (piper dahil) ATLANACAK (Faz 2b onay hucresini calistir).')

        # Istemciler/parametreler yalnizca script ONAYLIYSA kurulur. Piper API key
        # istemez; sadece PIPER_BINARY/PIPER_VOICE_MODEL_PATH gecirilir, eksikse
        # tts.run_tts temiz 'insufficient_data' dondurur.
        elevenlabs_client = openai_tts_client = None
        piper_binary_arg = piper_voice_model_arg = None
        if selected_script.get('status') == 'approved':
            if ELEVENLABS_API_KEY:
                from elevenlabs.client import ElevenLabs
                elevenlabs_client = ElevenLabs(api_key=ELEVENLABS_API_KEY)
            if OPENAI_API_KEY:
                from openai import OpenAI
                openai_tts_client = OpenAI(api_key=OPENAI_API_KEY)
            piper_binary_arg = PIPER_BINARY
            piper_voice_model_arg = PIPER_VOICE_MODEL_PATH

        _, ffprobe_bin = render_utils.resolve_ffmpeg_binaries()
        try:
            if TTS_PROVIDER == 'gemini':
                # Gemini TTS: AYRI bir ses-uretim implementasyonu (metin
                # ayristiricisi degil). Ham PCM gecerli bir WAV'a sarilir ve
                # sure/codec/boyut/bos-olmama dogrulamasindan gecer.
                if not RUN_GOOGLE_TTS:
                    raise RuntimeError(
                        "TTS_PROVIDER='gemini' ama RUN_GOOGLE_TTS=False. "
                        "Google TTS komutunu acikca ac veya baska bir saglayici sec."
                    )
                if not GOOGLE_CREDENTIAL.configured:
                    raise RuntimeError('Gemini anahtari yok. ' + google_ai.credential_help_text('google_api'))
                if selected_script.get('status') != 'approved':
                    raise RuntimeError(
                        'Gemini TTS atlandi: script onayli degil '
                        f"(status={selected_script.get('status')!r}); cagri yapilmadi."
                    )
                show_google_route('tts')
                _gemini_tts = google_ai.run_google_tts(
                    selected_script['script_text'], RESULTS_DIR / 'tts' / 'gemini_tts.wav',
                    client=google_ai.build_google_client(GOOGLE_CREDENTIAL),
                    voice=GEMINI_TTS_VOICE, language_code=GEMINI_TTS_LANGUAGE_CODE,
                    narration_direction=GEMINI_TTS_NARRATION_DIRECTION,
                    manifest=google_manifest, ledger=google_quota_ledger,
                    mode=GOOGLE_ROUTE_MODE, overrides=GOOGLE_TASK_MODEL_OVERRIDES,
                    enable_model_fallback=ENABLE_GOOGLE_MODEL_FALLBACK,
                    max_model_attempts=MAX_GOOGLE_MODEL_ATTEMPTS,
                )
                if _gemini_tts.get('status') == 'ok':
                    selected_voiceover = google_ai.build_tts_artifact(
                        _gemini_tts, selected_script['script_text'],
                        script_status=selected_script.get('status', 'review_required'),
                        raw_media_submitted=False,
                    )
                elif ENABLE_CROSS_PROVIDER_FALLBACK:
                    _azure_tts = provider_adapters.run_cross_provider_tts_fallback(
                        selected_script['script_text'], RESULTS_DIR / 'tts' / 'azure_tts.wav',
                        secret_getter=get_secret, enabled=True,
                    )
                    if _azure_tts.get('ok'):
                        _tts_attempts = list(_gemini_tts.get('provider_attempts') or [])
                        _tts_attempts.extend(_azure_tts.get('provider_attempts') or [])
                        selected_voiceover = tts.build_selected_voiceover_artifact(
                            _azure_tts, selected_script['script_text'],
                            selected_script.get('status', 'review_required'), False,
                            ffprobe_bin=ffprobe_bin, language='tr', attempts=_tts_attempts,
                        )
                    else:
                        selected_voiceover = google_ai.build_tts_artifact(
                            _gemini_tts, selected_script['script_text'],
                            script_status=selected_script.get('status', 'review_required'),
                            raw_media_submitted=False,
                            )
                        selected_voiceover['provider_attempts'].extend(
                            _azure_tts.get('provider_attempts') or [])
                else:
                    selected_voiceover = google_ai.build_tts_artifact(
                        _gemini_tts, selected_script['script_text'],
                        script_status=selected_script.get('status', 'review_required'),
                        raw_media_submitted=False,
                    )
            else:
                selected_voiceover = tts.run_tts(
                    selected_script, RESULTS_DIR / 'tts', ffprobe_bin=ffprobe_bin,
                    elevenlabs_client=elevenlabs_client, openai_client=openai_tts_client,
                    piper_binary=piper_binary_arg, piper_voice_model_path=piper_voice_model_arg,
                    tts_provider=TTS_PROVIDER,
                    allow_provider_fallback=ALLOW_TTS_PROVIDER_FALLBACK,
                    allow_dev_gtts_fallback=ALLOW_DEV_GTTS_FALLBACK,
                )
            youtube_data.write_json(SELECTED_VOICEOVER_PATH, selected_voiceover)
            paid = len([a for a in selected_voiceover.get('provider_attempts', [])
                        if a.get('provider') in ('elevenlabs', 'openai')])
            if selected_voiceover.get('audio_validation'):
                print('  ses dogrulamasi:', selected_voiceover['audio_validation']['checks'])
            record_phase('tts', 'completed', selected_voiceover['status'],
                         f"ucretli cagri sayisi={paid}, saglayici={selected_voiceover.get('provider')}")
            for attempt in selected_voiceover.get('provider_attempts', []):
                flag = 'OK' if attempt['ok'] else 'BASARISIZ'
                print(f"  deneme: {attempt['provider']}/{attempt['model']} -> {flag}"
                      + (f" ({attempt['error']})" if attempt['error'] else ''))
        except Exception as exc:
            record_phase('tts', 'failed', None, f'{type(exc).__name__}: {exc}')

### Faz 3a — Seslendirmeyi DINLE

Aşağıdaki hücre sesi çalar ve dosyanın SHA-256'sını yazdırır. **Sesi dinle**, sonra onay hücresine
bu hash'i yapıştır.

In [ ]:
if selected_voiceover is None:
    print('Faz 3 calismadi; incelenecek seslendirme yok.')
elif not selected_voiceover.get('audio_path'):
    print('Seslendirme uretilmedi. status:', selected_voiceover['status'])
    print('status_reason:', selected_voiceover['status_reason'])
else:
    print('Saglayici/model:', selected_voiceover['provider'], '/', selected_voiceover['model'])
    print('Sure (sn):', selected_voiceover.get('duration_sec'))
    print('Tahmini maliyet (USD):', selected_voiceover['cost_usd_estimate'])
    try:
        from IPython.display import Audio, display as _display
        _display(Audio(selected_voiceover['audio_path']))
    except Exception as exc:
        print('Ses oynatici gosterilemedi:', exc)
        print('Dosyayi elle dinle:', selected_voiceover['audio_path'])
    print()
    voiceover_review_info = show_review_prompt('voiceover', selected_voiceover, 'SESLENDIRME')


### Faz 3b — Seslendirmeyi ONAYLA (ayrı adım)

gTTS sonuçları politika gereği **asla onaylanamaz** — onay hücresi bunu reddeder.

In [ ]:
VOICEOVER_REVIEWED_SHA256 = ''   # inceleme hucresindeki SHA-256'yi buraya yapistir
VOICEOVER_APPROVED_BY = ''
VOICEOVER_APPROVAL_NOTES = ''

if selected_voiceover is None or not selected_voiceover.get('audio_path'):
    print('Onaylanacak bir seslendirme yok; bu hucre atlandi.')
elif not VOICEOVER_REVIEWED_SHA256 or not VOICEOVER_APPROVED_BY:
    print('Onay ATLANDI: VOICEOVER_REVIEWED_SHA256 ve VOICEOVER_APPROVED_BY doldurulmali.')
    print('selected_voiceover.json status olarak kaliyor:', selected_voiceover['status'])
else:
    try:
        selected_voiceover = approval.approve_artifact(
            selected_voiceover, 'voiceover', VOICEOVER_REVIEWED_SHA256,
            VOICEOVER_APPROVED_BY, VOICEOVER_APPROVAL_NOTES,
        )
        youtube_data.write_json(SELECTED_VOICEOVER_PATH, selected_voiceover)
        record_phase('tts', 'completed', selected_voiceover['status'],
                     f'onaylayan={VOICEOVER_APPROVED_BY}')
        print('Seslendirme ONAYLANDI ->', SELECTED_VOICEOVER_PATH)
    except ValueError as exc:
        print('ONAY REDDEDILDI:', exc)
        print('selected_voiceover.json status olarak kaliyor:', selected_voiceover['status'])


## Faz 4 — Render Pipeline (`RUN_RENDER`)

Varsayilan `original_audio_subtitles` modunda bagimlilik yalnizca `hot_moments.json` + gecerli
`moment_selection.json` dosyalaridir. Secilen klibin kendi sesi korunur, Whisper ile zaman kodlu
altyazi uretilir ve videoya gomulur; Gemini script veya TTS cagrisi yapilmaz.

Istege bagli `narrated_recap` modu eski `selected_script.json` + `selected_voiceover.json` zincirini kullanir. Render
**hicbir zaman kendini onaylamaz**: cikti her zaman `review_required` (teknik dogrulama
basarisizsa `failed`) olur; onay asagida ayri bir hucrede, videoyu izledikten sonra verilir.

In [ ]:
RENDER_MODE = 'blurred_background'  # 'blurred_background' | 'center_crop'
PREVIEW_MODE = True  # True = hizli/dusuk kalite onizleme; False = uretim kalitesi
SUBTITLE_WHISPER_MODEL_SIZE = 'small'
CAPTION_STYLE = 'active_word_yellow'
CONTENT_THEME = 'auto'  # GOT/HOTD/Targaryen/Westeros -> game_of_thrones -> Cinzel
CAPTION_FONT_NAME = None       # None = tema profili belirler
CAPTION_FONT_SIZE = None       # None = tema profili belirler
CAPTION_ACTIVE_COLOR_ASS = None  # None = tema profili; GOT profili sari
CAPTION_BASE_COLOR_ASS = None    # None = tema profili; GOT profili beyaz
CAPTION_MAX_WORDS = None
CAPTION_MAX_CHARS = None
CAPTION_MARGIN_V = None
SOURCE_VIDEO_PATH_OVERRIDE = ''


In [ ]:
render_manifest = None
moment_selection = moment_selection if isinstance(globals().get('moment_selection'), dict) else None

if not RUN_RENDER:
    record_phase('render', 'skipped')
elif not require_plan_phase('render'):
    pass
else:
    missing = []
    if hot_moments_data in (None, 'MISSING'):
        found = find_artifact('hot_moments.json', RESULTS_DIR)
        if found is None:
            missing.append('hot_moments.json')
        else:
            with open(found, 'r', encoding='utf-8') as f:
                hot_moments_data = json.load(f)
            HOT_MOMENTS_PATH = found

    if PIPELINE_MODE == 'original_audio_subtitles':
        if moment_selection is None:
            found = find_artifact('moment_selection.json', RESULTS_DIR)
            if found is None:
                missing.append('moment_selection.json')
            else:
                with open(found, 'r', encoding='utf-8') as f:
                    moment_selection = json.load(f)
                MOMENT_SELECTION_PATH = found
    elif PIPELINE_MODE == 'narrated_recap':
        if selected_script is None:
            found = find_artifact('selected_script.json', RESULTS_DIR)
            if found is None:
                missing.append('selected_script.json')
            else:
                with open(found, 'r', encoding='utf-8') as f:
                    selected_script = json.load(f)
                SELECTED_SCRIPT_PATH = found
        if selected_voiceover is None:
            found = find_artifact('selected_voiceover.json', RESULTS_DIR)
            if found is None:
                missing.append('selected_voiceover.json')
            else:
                with open(found, 'r', encoding='utf-8') as f:
                    selected_voiceover = json.load(f)
                SELECTED_VOICEOVER_PATH = found
    else:
        missing.append(f'bilinmeyen PIPELINE_MODE={PIPELINE_MODE!r}')

    _config_drift = source_strategy.check_config_drift(
        processing_plan,
        render_config={
            'render_mode': RENDER_MODE, 'preview_mode': PREVIEW_MODE,
            'caption_style': CAPTION_STYLE, 'caption_theme': CONTENT_THEME,
        },
    )
    if _config_drift:
        missing.append('onaylanmis plan ile suanki render ayari uyusmuyor')
        for _drift in _config_drift:
            print('  DRIFT:', _drift)

    if missing:
        record_phase('render', 'failed', None, f'gerekli girdi(ler) bulunamadi: {missing}')
        print('Duzeltme: original_audio_subtitles icin Faz 1 + moment secimini; '
              'narrated_recap icin ek olarak script + TTS fazlarini tamamla.')
    else:
        try:
            if PIPELINE_MODE == 'original_audio_subtitles':
                render_manifest = render_utils.run_original_audio_subtitle_pipeline(
                    hot_moments_data, moment_selection, HOT_MOMENTS_PATH, MOMENT_SELECTION_PATH,
                    render_dir=RENDER_DIR, render_mode=RENDER_MODE, preview_mode=PREVIEW_MODE,
                    subtitle_whisper_model_size=SUBTITLE_WHISPER_MODEL_SIZE,
                    caption_style=CAPTION_STYLE, caption_theme=CONTENT_THEME,
                    caption_font_name=CAPTION_FONT_NAME,
                    caption_font_size=CAPTION_FONT_SIZE,
                    caption_active_color_ass=CAPTION_ACTIVE_COLOR_ASS,
                    caption_base_color_ass=CAPTION_BASE_COLOR_ASS,
                    caption_max_words=CAPTION_MAX_WORDS, caption_max_chars=CAPTION_MAX_CHARS,
                    caption_margin_v=CAPTION_MARGIN_V,
                    source_video_path_override=SOURCE_VIDEO_PATH_OVERRIDE,
                )
            else:
                render_manifest = render_utils.run_render_pipeline(
                    hot_moments_data, selected_script, selected_voiceover,
                    HOT_MOMENTS_PATH, SELECTED_SCRIPT_PATH, SELECTED_VOICEOVER_PATH,
                    render_dir=RENDER_DIR, render_mode=RENDER_MODE, preview_mode=PREVIEW_MODE,
                    source_video_path_override=SOURCE_VIDEO_PATH_OVERRIDE,
                )
            youtube_data.write_json(RENDER_MANIFEST_PATH, render_manifest)
            record_phase('render', 'completed', render_manifest['status'],
                         f"mod={PIPELINE_MODE}, teknik passed={render_manifest['output']['validation']['passed']}")
            print('Pipeline modu:', PIPELINE_MODE)
            print('Altyazi kaynagi:', render_manifest['subtitles']['source'],
                  '| cue:', render_manifest['subtitles']['cue_count'])
            print('Dinamik kelime vurgusu:', render_manifest['subtitles'].get('dynamic_word_highlight'),
                  '| aktif renk:', render_manifest['subtitles'].get('active_word_color'))
            print('Caption tema/font:', render_manifest['config'].get('caption_theme_resolved'),
                  '/', render_manifest['config'].get('caption_font_name'),
                  '| neden:', render_manifest['config'].get('caption_theme_resolution_reason'))
            if render_manifest['approval']['failed_conditions']:
                print('Onay oncesi saglanmayan kosullar:',
                      render_manifest['approval']['failed_conditions'])
            print('HATIRLATMA: hicbir yere yukleme yapilmadi (not_uploaded=%s).'
                  % render_manifest['not_uploaded'])

            # Coklu cikti IZOLASYONU: plan birden fazla Short istiyorsa her cikti
            # KENDI dizinine yazilir; hicbiri digerini ezemez. Ilk yuva tarihsel
            # RENDER_DIR'dir, yani tek ciktida yollar aynen korunur.
            output_slot_report = []
            for _slot in ((processing_plan or {}).get('outputs') or [])[1:]:
                output_slot_report.append({
                    'index': _slot['index'], 'moment_rank': _slot.get('moment_rank'),
                    'output_dir': _slot['output_dir'], 'status': 'insufficient_data',
                    'reason': ('bu cikti icin insan onayli ayri moment secimi yok; '
                               "Faz 1b'yi ilgili rank icin calistirip render'i tekrar calistir"),
                })
            if output_slot_report:
                print()
                print('EK CIKTI YUVALARI (henuz render EDILMEDI, izole dizinler):')
                for _entry in output_slot_report:
                    print(f"  #{_entry['index']} rank={_entry['moment_rank']} "
                          f"-> {_entry['output_dir']} [{_entry['status']}] {_entry['reason']}")
        except Exception as exc:
            record_phase('render', 'failed', None, f'{type(exc).__name__}: {exc}')
            print('Render HATASI:', type(exc).__name__, exc)


### Faz 4a — Render'ı İZLE

Aşağıdaki hücre videoyu oynatır ve SHA-256'sını yazdırır. **Videoyu izle**, sonra onay hücresine
bu hash'i yapıştır. Teknik doğrulama geçmediyse onay hücresi reddeder.

In [ ]:
if render_manifest is None:
    print('Faz 4 calismadi; incelenecek render yok.')
else:
    video_path = render_manifest['output']['final_video_path']
    print('Render dosyasi:', video_path)
    print('Cozunurluk:', render_manifest['config']['render_width'], 'x',
          render_manifest['config']['render_height'], '| onizleme modu:',
          render_manifest['config']['preview_mode'])
    print('Teknik dogrulama:', json.dumps(render_manifest['output']['validation']['checks'], indent=2))
    try:
        from IPython.display import Video, display as _display
        _display(Video(video_path, embed=True, width=320))
    except Exception as exc:
        print('Video oynatici gosterilemedi:', exc)
        print('Dosyayi elle izle:', video_path)
    print()
    render_review_info = show_review_prompt('render', render_manifest, 'RENDER')


### Faz 4b — Render'ı ONAYLA (ayrı adım)

Render onayı yalnızca seçili moda ait **tüm** koşullar sağlandığında verilebilir.
`original_audio_subtitles`: teknik doğrulama + onaylı moment seçimi.
`narrated_recap`: teknik doğrulama + onaylı script + onaylı seslendirme + script/ses hash eşleşmesi.
Herhangi biri eksikse onay reddedilir ve artefakt `review_required` kalır — onaylanmamış girdiler
"onaylı" bir çıktıya dönüştürülemez.

Videoyu izlediğini yine de kaydetmek istersen `RECORD_RENDER_REVIEW_ONLY = True` yap: bu, ayrı bir
`render_review` bloğu yazar ve `status`'ü **değiştirmez**.

In [ ]:
RENDER_REVIEWED_SHA256 = ''   # inceleme hucresindeki SHA-256'yi buraya yapistir
RENDER_APPROVED_BY = ''
RENDER_APPROVAL_NOTES = ''

# True yaparsan SADECE "izledim" kaydi yazilir; status DEGISMEZ.
# Yukari-akis onaylari eksikken videoyu izledigini kaydetmek icin kullan.
RECORD_RENDER_REVIEW_ONLY = False

if render_manifest is None:
    print('Onaylanacak bir render yok; bu hucre atlandi.')
elif not RENDER_REVIEWED_SHA256 or not RENDER_APPROVED_BY:
    print('Onay ATLANDI: RENDER_REVIEWED_SHA256 ve RENDER_APPROVED_BY doldurulmali.')
    print('render_manifest.json status olarak kaliyor:', render_manifest['status'])
elif RECORD_RENDER_REVIEW_ONLY:
    try:
        render_manifest = approval.record_render_review(
            render_manifest, RENDER_REVIEWED_SHA256, RENDER_APPROVED_BY, RENDER_APPROVAL_NOTES,
        )
        youtube_data.write_json(RENDER_MANIFEST_PATH, render_manifest)
        print('Inceleme kaydedildi (render_review). status DEGISMEDI:', render_manifest['status'])
    except ValueError as exc:
        print('INCELEME KAYDI REDDEDILDI:', exc)
else:
    try:
        render_manifest = approval.approve_artifact(
            render_manifest, 'render', RENDER_REVIEWED_SHA256,
            RENDER_APPROVED_BY, RENDER_APPROVAL_NOTES,
        )
        youtube_data.write_json(RENDER_MANIFEST_PATH, render_manifest)
        record_phase('render', 'completed', render_manifest['status'],
                     f'onaylayan={RENDER_APPROVED_BY}')
        print('Render ONAYLANDI ->', RENDER_MANIFEST_PATH)
        print('NOT: onay yalnizca bir kayittir; bu notebook hicbir sey yayinlamaz.')
    except ValueError as exc:
        print('ONAY REDDEDILDI:', exc)
        print('render_manifest.json status olarak kaliyor:', render_manifest['status'])
        print("Sadece izledigini kaydetmek istersen RECORD_RENDER_REVIEW_ONLY = True yap.")


## Araştırma A — YouTube Trend/Kanal Keşfi (`RUN_YOUTUBE_DATA`)

**Ana pipeline'dan bağımsızdır.** `YOUTUBE_DISCOVERY_TERMS` ile yeni videoları araştırır, aday kanalları sıralar ve araştırma veri kümesi üretir. `YOUTUBE_CHANNEL_INPUTS` isteğe bağlı manuel ek listedir. Sonuçlar kaynak video veya render girdisi yapılmaz; bu bölüm read-only'dir.

In [ ]:
YOUTUBE_CHANNEL_INPUTS = [
    # "UCxxxxxxxxxxxxxxxxxxxxxx",
    # "@ChannelHandle",
]

AUTO_DISCOVER_CHANNELS = True
YOUTUBE_DISCOVERY_TERMS = ['AI', 'AI tools', 'generative AI', 'yapay zeka']
DISCOVERY_LOOKBACK_DAYS = 14
DISCOVERY_REGION = 'US'
DISCOVERY_LANGUAGE = 'en'
DISCOVERY_ORDERS = ('date', 'viewCount')
MAX_RESULTS_PER_QUERY = 25
MAX_SEARCH_CALLS_PER_RUN = 8
TOP_DISCOVERED_VIDEOS = 50
TOP_DISCOVERED_CHANNELS = 10
MIN_CHANNEL_MATCHED_VIDEOS = 1
DISCOVERY_SHORTS_ONLY = True
YOUTUBE_DISCOVERY_CACHE_TTL_HOURS = 6

MAX_VIDEOS_PER_CHANNEL = 100
YOUTUBE_CACHE_TTL_HOURS = 12
RUN_YOUTUBE_CONNECTION_TEST = False


In [ ]:
from datetime import datetime, timezone

youtube_collection_summary = {'ran': False}

if not RUN_YOUTUBE_DATA:
    record_phase('youtube_data', 'skipped')
else:
    youtube_started_at = datetime.now(timezone.utc).isoformat()
    youtube_cache = youtube_data.Cache(YOUTUBE_CACHE_DIR)
    youtube_quota = youtube_data.QuotaTracker()

    youtube_client = None
    youtube_connection_test_result = None
    youtube_resolved_channels = []
    youtube_failed_channels = []
    youtube_all_videos = []
    youtube_collection_errors = []
    youtube_discovery = {
        'status': 'insufficient_data', 'status_reason': 'otomatik kesif calistirilmadi',
        'videos': [], 'channels': [], 'errors': [],
        'search_calls_planned': 0, 'search_calls_executed': 0,
    }
    effective_channel_inputs = list(YOUTUBE_CHANNEL_INPUTS)
    youtube_status = 'insufficient_data'
    youtube_status_reason = 'baslatilmadi'

    if not YOUTUBE_DATA_API_KEY:
        youtube_status = 'insufficient_data'
        youtube_status_reason = 'YOUTUBE_DATA_API_KEY yok (Colab Secrets kontrol et)'
        print('YouTube Data API key: missing -- toplama yapilamaz.')
    else:
        print('YouTube Data API key: configured')
        try:
            youtube_client = youtube_data.build_client(YOUTUBE_DATA_API_KEY)
        except Exception as exc:
            youtube_status = 'failed'
            youtube_status_reason = 'istemci olusturulamadi: ' + youtube_data.sanitize_error_text(exc, YOUTUBE_DATA_API_KEY)
            youtube_collection_errors.append({'stage': 'build_client', 'error': youtube_status_reason})
            print(youtube_status_reason)

        if youtube_client is not None and RUN_YOUTUBE_CONNECTION_TEST:
            test_input = YOUTUBE_CHANNEL_INPUTS[0] if YOUTUBE_CHANNEL_INPUTS else '@GoogleDevelopers'
            youtube_connection_test_result = youtube_data.run_connection_test(
                youtube_client, YOUTUBE_DATA_API_KEY, test_input)
            print('Baglanti testi ->', youtube_connection_test_result['status'], 'girdi:', test_input)
            if youtube_connection_test_result['status'] != 'ok':
                print('  Aciklama:', youtube_connection_test_result['explanation'])

        if youtube_client is not None and AUTO_DISCOVER_CHANNELS:
            if not YOUTUBE_DISCOVERY_TERMS:
                youtube_discovery['status_reason'] = 'YOUTUBE_DISCOVERY_TERMS bos'
                print('Otomatik kesif atlandi: YOUTUBE_DISCOVERY_TERMS bos.')
            else:
                print('YouTube otomatik kesif basliyor:', YOUTUBE_DISCOVERY_TERMS)
                youtube_discovery = youtube_data.discover_trending_content(
                    youtube_client, youtube_cache, youtube_quota,
                    terms=YOUTUBE_DISCOVERY_TERMS,
                    lookback_days=DISCOVERY_LOOKBACK_DAYS,
                    region_code=DISCOVERY_REGION,
                    relevance_language=DISCOVERY_LANGUAGE,
                    orders=DISCOVERY_ORDERS,
                    max_results_per_query=MAX_RESULTS_PER_QUERY,
                    max_search_calls=MAX_SEARCH_CALLS_PER_RUN,
                    top_videos=TOP_DISCOVERED_VIDEOS,
                    top_channels=TOP_DISCOVERED_CHANNELS,
                    min_channel_matched_videos=MIN_CHANNEL_MATCHED_VIDEOS,
                    shorts_only=DISCOVERY_SHORTS_ONLY,
                    cache_ttl_hours=YOUTUBE_DISCOVERY_CACHE_TTL_HOURS,
                    api_key=YOUTUBE_DATA_API_KEY,
                )
                youtube_collection_errors.extend(youtube_discovery['errors'])
                discovered_channel_ids = [
                    row['channel_id'] for row in youtube_discovery['channels']
                    if row.get('channel_id')
                ]
                effective_channel_inputs.extend(discovered_channel_ids)
                effective_channel_inputs = list(dict.fromkeys(effective_channel_inputs))
                print('Kesif sonucu:', len(youtube_discovery['videos']), 'video,',
                      len(discovered_channel_ids), 'kanal')

        if youtube_client is not None and not effective_channel_inputs:
            youtube_status = youtube_discovery.get('status', 'insufficient_data')
            if youtube_status == 'review_required':
                youtube_status = 'insufficient_data'
            youtube_status_reason = (
                'manuel kanal yok ve otomatik kesif kanal uretmedi: ' +
                youtube_discovery.get('status_reason', 'bilinmeyen neden')
            )
            print(youtube_status_reason)
        elif youtube_client is not None:
            youtube_resolved_channels, youtube_failed_channels = youtube_data.resolve_channels(
                youtube_client, effective_channel_inputs, api_key_for_sanitization=YOUTUBE_DATA_API_KEY,
            )
            for failed in youtube_failed_channels:
                youtube_collection_errors.append(
                    {'stage': 'resolve_channel', 'input': failed['input'], 'error': failed['error']})
                print('Kanal cozumlenemedi:', failed['input'], '->', failed['error'])

            for channel_record in youtube_resolved_channels:
                rows, err = youtube_data.collect_channel_uploads(
                    youtube_client, youtube_cache, youtube_quota, channel_record,
                    MAX_VIDEOS_PER_CHANNEL, channel_cache_ttl_hours=YOUTUBE_CACHE_TTL_HOURS,
                    video_stats_ttl_hours=YOUTUBE_CACHE_TTL_HOURS, api_key=YOUTUBE_DATA_API_KEY,
                )
                if err:
                    youtube_collection_errors.append(
                        {'stage': 'collect_uploads', 'channel_id': channel_record['channel_id'], 'error': err})
                    print('Toplama hatasi:', channel_record['channel_id'], '->', err)
                youtube_all_videos.extend(rows)

            youtube_all_videos = youtube_data.deduplicate_videos(youtube_all_videos)

            if youtube_resolved_channels and youtube_all_videos:
                youtube_status = 'review_required'
                youtube_status_reason = (f'{len(youtube_resolved_channels)} kanal, '
                                         f'{len(youtube_all_videos)} video toplandi -- insan onayi gerekli')
            elif youtube_resolved_channels and not youtube_all_videos:
                youtube_status = 'insufficient_data'
                youtube_status_reason = 'kanallar cozumlendi ama kullanilabilir video toplanamadi'
            else:
                youtube_status = 'failed'
                youtube_status_reason = 'hicbir kanal cozumlenemedi'

    youtube_completed_at = datetime.now(timezone.utc).isoformat()

    youtube_channels_out = [
        {'input': c['input'], 'channel_id': c['channel_id'], 'title': c['title'],
         'uploads_playlist_id': c['uploads_playlist_id'],
         'subscriber_count': c['subscriber_count'], 'video_count': c['video_count']}
        for c in youtube_resolved_channels
    ]
    youtube_data.write_json(YOUTUBE_DATA_DIR / 'channels.json', youtube_channels_out)
    youtube_data.write_normalized_videos_csv(NORMALIZED_VIDEOS_PATH, youtube_all_videos)
    youtube_data.write_discovered_videos_csv(
        YOUTUBE_DATA_DIR / 'discovered_videos.csv', youtube_discovery.get('videos', []))
    youtube_data.write_discovered_channels_csv(
        YOUTUBE_DATA_DIR / 'discovered_channels.csv', youtube_discovery.get('channels', []))
    youtube_data.write_json(
        YOUTUBE_DATA_DIR / 'discovery_manifest.json',
        {k: v for k, v in youtube_discovery.items() if k not in ('videos', 'channels')},
    )

    youtube_manifest = youtube_data.build_collection_manifest(
        status=youtube_status, status_reason=youtube_status_reason,
        started_at=youtube_started_at, completed_at=youtube_completed_at,
        requested_channel_count=len(effective_channel_inputs),
        resolved_channel_count=len(youtube_resolved_channels),
        failed_channel_count=len(youtube_failed_channels),
        video_count=len(youtube_all_videos),
        date_range=youtube_data.compute_date_range(youtube_all_videos),
        cache_statistics=youtube_cache.stats,
        connection_test_status=(youtube_connection_test_result['status']
                                 if youtube_connection_test_result else None),
        discovery={
            'enabled': AUTO_DISCOVER_CHANNELS,
            'status': youtube_discovery.get('status'),
            'status_reason': youtube_discovery.get('status_reason'),
            'discovered_video_count': len(youtube_discovery.get('videos', [])),
            'discovered_channel_count': len(youtube_discovery.get('channels', [])),
            'search_calls_planned': youtube_discovery.get('search_calls_planned', 0),
            'search_calls_executed': youtube_discovery.get('search_calls_executed', 0),
        },
        output_paths={
            'channels_json': YOUTUBE_DATA_DIR / 'channels.json',
            'normalized_videos_csv': NORMALIZED_VIDEOS_PATH,
            'collection_manifest_json': COLLECTION_MANIFEST_PATH,
            'quota_usage_json': YOUTUBE_DATA_DIR / 'quota_usage.json',
            'errors_json': YOUTUBE_DATA_DIR / 'errors.json',
            'discovered_videos_csv': YOUTUBE_DATA_DIR / 'discovered_videos.csv',
            'discovered_channels_csv': YOUTUBE_DATA_DIR / 'discovered_channels.csv',
            'discovery_manifest_json': YOUTUBE_DATA_DIR / 'discovery_manifest.json',
        },
    )
    youtube_data.write_json(COLLECTION_MANIFEST_PATH, youtube_manifest)
    youtube_data.write_json(YOUTUBE_DATA_DIR / 'quota_usage.json', youtube_quota.to_dict())
    youtube_data.write_json(YOUTUBE_DATA_DIR / 'errors.json', youtube_collection_errors)

    youtube_collection_summary = {
        'ran': True, 'pipeline_role': 'research_sidecar',
        'feeds_main_pipeline': False,
        'status': youtube_status, 'status_reason': youtube_status_reason,
        'resolved_channel_count': len(youtube_resolved_channels),
        'failed_channel_count': len(youtube_failed_channels),
        'video_count': len(youtube_all_videos),
        'discovered_video_count': len(youtube_discovery.get('videos', [])),
        'discovered_channel_count': len(youtube_discovery.get('channels', [])),
        'search_calls_used': youtube_quota.calls.get('search.list', 0),
        'quota_estimated_units': youtube_quota.estimated_total(),
        'cache_stats': youtube_cache.stats,
    }

    # Artefakt temiz yazildi; execution_status=completed. Ama artifact_status
    # 'insufficient_data' ise bu ozette AYRICA gorunur -- "ok" diye gecmez.
    record_phase('youtube_data', 'completed', youtube_status, youtube_status_reason)
    print()
    print('=== YouTube toplama ozeti ===')
    for k, v in youtube_collection_summary.items():
        print(f'  {k}: {v}')


## Araştırma B — Viral Potansiyel Tahmini (`RUN_VIRAL_PREDICTION`)

Bu da ana pipeline'dan bağımsız bir karar-destek deneyidir. Bağımlılık: `normalized_videos.csv` + `collection_manifest.json`. Toplama durumu
`failed`/`insufficient_data` ise veya satir sayisi esigin altindaysa artefakt
`insufficient_data` olur ve kendinden emin tahmin URETILMEZ.

In [ ]:
MIN_HISTORY_ROWS = 20


In [ ]:
import pandas as pd

viral_prediction_artifact = None

if not RUN_VIRAL_PREDICTION:
    record_phase('viral_prediction', 'skipped')
else:
    csv_path = NORMALIZED_VIDEOS_PATH if NORMALIZED_VIDEOS_PATH.is_file() else youtube_data.find_normalized_videos_csv(
        [str(YOUTUBE_DATA_DIR), str(RESULTS_DIR / 'youtube_data'),
         'results/youtube_data', '/content/results/youtube_data'],
    )
    if csv_path is None:
        record_phase('viral_prediction', 'failed', None, 'normalized_videos.csv bulunamadi')
        print("Duzeltme: RUN_YOUTUBE_DATA=True ile Faz 5'i calistir veya bu dosyayi "
              "YOUTUBE_DATA_DIR altina yerlestir.")
    else:
        history_load_status, history_load_reason = 'insufficient_data', ''
        history_df = pd.DataFrame()
        try:
            raw_df = pd.read_csv(csv_path)
        except pd.errors.EmptyDataError:
            raw_df = None
            history_load_reason = f'{csv_path} tamamen bos (basliksiz/0 byte) bir CSV dosyasi'
        required_cols = ['video_id', 'channel_id', 'view_count', 'published_at']
        if raw_df is not None:
            missing_cols = [c for c in required_cols if c not in raw_df.columns]
            if missing_cols:
                history_load_reason = f'gerekli sutunlar eksik: {missing_cols}'
            else:
                candidate_df = raw_df.dropna(subset=['view_count']).copy()
                candidate_df['published_at'] = pd.to_datetime(
                    candidate_df['published_at'], errors='coerce', utc=True)
                candidate_df = candidate_df.sort_values('published_at')
                if len(candidate_df) > 0:
                    history_df = candidate_df
                    history_load_status = 'ok'
                else:
                    history_load_reason = f'{csv_path} gecerli semaya sahip ama kullanilabilir satir yok'

        manifest_path = csv_path.parent / 'collection_manifest.json'
        collection_status = resolved_channel_count = collection_timestamp = None
        if manifest_path.is_file():
            with open(manifest_path, 'r', encoding='utf-8') as f:
                collection_manifest_data = json.load(f)
            collection_status = collection_manifest_data.get('status')
            resolved_channel_count = collection_manifest_data.get('resolved_channel_count')
            collection_timestamp = collection_manifest_data.get('completed_at')

        try:
            viral_prediction_artifact = viral_prediction.build_viral_prediction_artifact(
                history_df, history_load_status, history_load_reason, collection_status,
                min_history_rows=MIN_HISTORY_ROWS, resolved_channel_count=resolved_channel_count,
                collection_timestamp=collection_timestamp,
            )
            youtube_data.write_json(VIRAL_PREDICTION_PATH, viral_prediction_artifact)
            record_phase('viral_prediction', 'completed', viral_prediction_artifact['status'],
                         viral_prediction_artifact['status_reason'])
        except Exception as exc:
            record_phase('viral_prediction', 'failed', None, f'{type(exc).__name__}: {exc}')


## Google yetenek komutlari (her biri BAGIMSIZ acilir/kapanir)

Bu bolumdeki her komut ayri bir `RUN_GOOGLE_*` anahtarina baglidir ve kapaliyken **hicbir cagri
yapmaz**. Hepsi ayni router'i, ayni kota defterini ve ayni hata/fallback politikasini kullanir.
Uretilen her artefakt `results/` altina yazilir ve `review_required` / `experimental` /
`insufficient_data` / `failed` disinda bir durum **alamaz** -- `approved` yalnizca ayri,
hash-tabanli insan onayi adiminda verilir.


In [ ]:
google_capability_artifacts = {}


def _google_client_or_none(command_name):
    if not GOOGLE_CREDENTIAL.configured:
        print(f'{command_name}: Gemini anahtari yok, temiz atlandi.')
        print(' ', google_ai.credential_help_text('google_api'))
        return None
    return google_ai.build_google_client(GOOGLE_CREDENTIAL)


def _run_google_command(command_name, task, capability, runner, input_content,
                        enabled, extra=None):
    """Ortak sarmalayici: anahtar kapaliysa atlar, acikken rotayi gosterir,
    artefakti sozlesmeye uygun sekilde yazar. Cikti ASLA uydurulmaz."""
    if not enabled:
        print(f'{command_name}: KAPALI (atlandi).')
        return None
    client = _google_client_or_none(command_name)
    if client is None:
        return None
    show_google_route(task)
    result = runner(client)
    artifact = google_ai.build_google_artifact(
        result, task=task, capability=capability,
        input_content=input_content, output_content=result.get('output'),
        raw_media_submitted=ALLOW_RAW_MEDIA_TO_FREE_TIER,
        extra=extra,
    )
    google_capability_artifacts[command_name] = artifact
    youtube_data.write_json(RESULTS_DIR / f'google_{command_name}.json', artifact)
    print(f"{command_name}: status={artifact['status']} model={artifact['model']} "
          f"fallback={artifact['fallback_used']}")
    if artifact['status_reason']:
        print('  ', artifact['status_reason'])
    return artifact


# --- 1. Hot-moment nihai incelemesi (Rota A) --------------------------------
GOOGLE_HOTMOMENT_PROMPT = ''  # bos birakirsan hot_moments.json ozetinden uretilir
if RUN_GOOGLE_HOTMOMENT_REVIEW:
    _hm_source = hot_moments_data if isinstance(hot_moments_data, dict) else None
    if _hm_source is None:
        _found = find_artifact('hot_moments.json', RESULTS_DIR)
        if _found:
            with open(_found, 'r', encoding='utf-8') as _f:
                _hm_source = json.load(_f)
    if _hm_source is None:
        print('google_hotmoment_review: hot_moments.json yok, atlandi.')
    else:
        # ALLOW_RAW_MEDIA_TO_FREE_TIER=False iken SADECE yerel olarak turetilmis
        # transcript/skor ozetleri gonderilir; ham video ASLA yuklenmez.
        _hm_prompt = GOOGLE_HOTMOMENT_PROMPT or (
            'Asagidaki yerel olarak uretilmis aday anlari degerlendir ve en guclu 3 tanesini sirala. '
            'SADECE JSON don: {"moments": [{"rank": <int>, "score": <0-100>, "reason": "..."}]}\n\n'
            + json.dumps([{k: m.get(k) for k in ('rank', 'source_start_sec', 'source_end_sec',
                                                   'score', 'reason', 'transcript')}
                          for m in _hm_source['hot_moments'][:10]], ensure_ascii=False)
        )
        _run_google_command(
            'hotmoment_review', 'hotmoment_review', 'generate_content',
            lambda c: google_ai.run_google_hotmoment_review(
                _hm_prompt, client=c, manifest=google_manifest, ledger=google_quota_ledger,
                mode=GOOGLE_ROUTE_MODE, overrides=GOOGLE_TASK_MODEL_OVERRIDES,
                enable_model_fallback=ENABLE_GOOGLE_MODEL_FALLBACK,
                max_model_attempts=MAX_GOOGLE_MODEL_ATTEMPTS),
            _hm_prompt, RUN_GOOGLE_HOTMOMENT_REVIEW)
else:
    print('google_hotmoment_review: KAPALI (atlandi).')

# --- 2. Toplu siniflama / metadata uretimi (Rota B) -------------------------
GOOGLE_BULK_PROMPT = (
    'Asagidaki script icin YouTube Shorts metadata uret. SADECE JSON don: '
    '{"title": "...", "description": "...", "hashtags": ["#..."]}'
)
if RUN_GOOGLE_BULK_CLASSIFICATION:
    _bulk_input = (selected_script or {}).get('script_text') or ''
    if not _bulk_input:
        print('google_bulk_classification: selected_script.script_text yok, atlandi.')
    else:
        _bulk_prompt = f'{GOOGLE_BULK_PROMPT}\n\nScript:\n{_bulk_input}'
        _run_google_command(
            'bulk_classification', 'bulk_classification', 'generate_content',
            lambda c: google_ai.run_google_bulk_classification(
                _bulk_prompt, client=c, manifest=google_manifest, ledger=google_quota_ledger,
                mode=GOOGLE_ROUTE_MODE, overrides=GOOGLE_TASK_MODEL_OVERRIDES,
                enable_model_fallback=ENABLE_GOOGLE_MODEL_FALLBACK,
                max_model_attempts=MAX_GOOGLE_MODEL_ATTEMPTS,
                required_fields=('title', 'description')),
            _bulk_prompt, RUN_GOOGLE_BULK_CLASSIFICATION)
else:
    print('google_bulk_classification: KAPALI (atlandi).')

# --- 3. Arama-temelli arastirma (Rota C, Gemini 2.5 grounding kotasi) -------
GOOGLE_RESEARCH_TOPICS = []   # ornek: ['House of the Dragon 3. sezon']
GOOGLE_RESEARCH_MAX_CANDIDATES = 3
if RUN_GOOGLE_GROUNDED_RESEARCH and ENABLE_GOOGLE_SEARCH_GROUNDING:
    _topics = GOOGLE_RESEARCH_TOPICS[:GOOGLE_RESEARCH_MAX_CANDIDATES]
    if not _topics:
        print('google_grounded_research: GOOGLE_RESEARCH_TOPICS bos, atlandi.')
    else:
        _research_prompt = (
            'Asagidaki konularin SU AN haberlerde/gundemde olup olmadigini arastir ve her biri icin '
            'kisa bir guncellik notu ver. Kaynak baglantilarini belirt.\n\n- '
            + '\n- '.join(_topics)
        )
        _art = _run_google_command(
            'grounded_research', 'grounded_research',
            "generate_content(tools=[google_search])",
            lambda c: google_ai.run_google_grounded_research(
                _research_prompt, client=c, manifest=google_manifest,
                ledger=google_quota_ledger, mode=GOOGLE_ROUTE_MODE,
                overrides=GOOGLE_TASK_MODEL_OVERRIDES,
                enable_model_fallback=ENABLE_GOOGLE_MODEL_FALLBACK,
                max_model_attempts=MAX_GOOGLE_MODEL_ATTEMPTS,
                max_candidates=GOOGLE_RESEARCH_MAX_CANDIDATES),
            _research_prompt, True)
elif RUN_GOOGLE_GROUNDED_RESEARCH:
    print('google_grounded_research: ENABLE_GOOGLE_SEARCH_GROUNDING=False, atlandi.')
else:
    print('google_grounded_research: KAPALI (atlandi).')

# --- 4. Embeddings: duplikasyon / novelty (Rota E) --------------------------
GOOGLE_EMBEDDING_TEXTS = []          # ornek: gecmis script/baslik metinleri
GOOGLE_EMBEDDING_CANDIDATE_TEXT = '' # yeni aday metin
if RUN_GOOGLE_EMBEDDINGS:
    _candidate_text = GOOGLE_EMBEDDING_CANDIDATE_TEXT or (selected_script or {}).get('script_text') or ''
    if not _candidate_text:
        print('google_embeddings: aday metin yok, atlandi.')
    else:
        _all_texts = [_candidate_text] + list(GOOGLE_EMBEDDING_TEXTS)
        _client = _google_client_or_none('google_embeddings')
        _emb = None
        if _client is not None:
            show_google_route('embeddings')
            _emb = google_ai.run_google_embeddings(
                _all_texts, client=_client, manifest=google_manifest,
                ledger=google_quota_ledger, mode=GOOGLE_ROUTE_MODE,
                overrides=GOOGLE_TASK_MODEL_OVERRIDES,
                enable_model_fallback=ENABLE_GOOGLE_MODEL_FALLBACK,
                max_model_attempts=MAX_GOOGLE_MODEL_ATTEMPTS,
            )
        if ((_emb is None or _emb.get('status') != 'ok') and
                ENABLE_CROSS_PROVIDER_FALLBACK):
            _emb = provider_adapters.run_cross_provider_embeddings_fallback(
                _all_texts, secret_getter=get_secret, enabled=True,
                max_provider_attempts=MAX_PROVIDER_ATTEMPTS, allow_local=True)
        if _emb is not None:
            _extra = {'dimension': _emb.get('dimension'),
                      'task_prefix': _emb.get('task_prefix'),
                      'task_prefix_kind': _emb.get('task_prefix_kind'),
                      'input_hashes': _emb.get('input_hashes'),
                      'vector_space': _emb.get('vector_space')}
            if _emb['status'] == 'ok':
                _vectors = _emb['output']
                _novelty = google_ai.novelty_score(_vectors[0], _vectors[1:])
                _extra.update({
                    'novelty_score': _novelty,
                    'duplicate_penalty': google_ai.duplicate_penalty(_novelty),
                    'semantic_duplicates': google_ai.find_semantic_duplicates(
                        _vectors[0], list(zip(
                            [f'previous_{i}' for i in range(len(_vectors) - 1)], _vectors[1:]))),
                    'clusters': google_ai.cluster_by_similarity(
                        list(zip([f'text_{i}' for i in range(len(_vectors))], _vectors))),
                })
                print('  novelty_score:', _novelty,
                      '| duplicate_penalty:', _extra['duplicate_penalty'])
            _art = google_ai.build_google_artifact(
                _emb, task='embeddings', capability='embed_content',
                input_content='\n'.join(_all_texts),
                output_content=None, raw_media_submitted=False, extra=_extra)
            google_capability_artifacts['embeddings'] = _art
            youtube_data.write_json(RESULTS_DIR / 'google_embeddings.json', _art)
            print('google_embeddings: status=' + _art['status'])
else:
    print('google_embeddings: KAPALI (atlandi).')

# --- 5. Antigravity (DENEYSEL, varsayilan KAPALI) --------------------------
GOOGLE_ANTIGRAVITY_PROMPT = 'Bugunun YouTube Shorts trendleri icin kisa bir arastirma notu yaz.'
_antigravity = google_ai.run_antigravity_research_experimental(
    GOOGLE_ANTIGRAVITY_PROMPT,
    enabled=RUN_ANTIGRAVITY_RESEARCH and GOOGLE_CREDENTIAL.configured,
    max_total_tokens=ANTIGRAVITY_MAX_TOTAL_TOKENS,
    client=(google_ai.build_google_client(GOOGLE_CREDENTIAL)
            if (RUN_ANTIGRAVITY_RESEARCH and GOOGLE_CREDENTIAL.configured) else None),
    manifest=google_manifest, ledger=google_quota_ledger,
) if (RUN_ANTIGRAVITY_RESEARCH and GOOGLE_CREDENTIAL.configured) else \
    google_ai.run_antigravity_research_experimental(
        GOOGLE_ANTIGRAVITY_PROMPT, enabled=False,
        max_total_tokens=ANTIGRAVITY_MAX_TOTAL_TOKENS)
_antigravity_artifact = google_ai.build_antigravity_artifact(_antigravity)
google_capability_artifacts['antigravity_research'] = _antigravity_artifact
youtube_data.write_json(RESULTS_DIR / 'google_antigravity_research.json', _antigravity_artifact)
print('antigravity_research: status=' + _antigravity_artifact['status'],
      '| production_ready=' + str(_antigravity_artifact['production_ready']))

# --- Kota defteri ozeti (YEREL TAHMIN) --------------------------------------
google_quota_ledger.save()
print()
print('--- Google kota defteri (YEREL TAHMIN, sunucu yetkilisi degil) ---')
for _model, _record in sorted(google_quota_ledger.data['models'].items()):
    print(f"  {_model}: gunluk {_record['requests_current_day']}/{_record['configured_rpd']} "
          f"(kalan~{_record['estimated_remaining_rpd']}), dakika "
          f"{_record['requests_current_minute']}/{_record['configured_rpm']}, "
          f"devre={_record['circuit_state']}")
print('Defter dosyasi:', GOOGLE_QUOTA_LEDGER_PATH)


## Çalıştırma özeti (execution_status vs artifact_status)

In [ ]:
MAIN_PIPELINE_ORDER = ['source_assessment', 'processing_plan', 'video_analysis',
                       'moment_selection', 'script_generation', 'tts', 'render']
RESEARCH_SIDECAR_ORDER = ['youtube_data', 'viral_prediction']
PHASE_ORDER = MAIN_PIPELINE_ORDER + RESEARCH_SIDECAR_ORDER

def print_status_group(title, names):
    print('=' * 78)
    print(title)
    print(f"{'adim':<20} {'execution':<12} {'artifact':<24} detay")
    print('-' * 78)
    for name in names:
        entry = phase_report.get(name)
        if entry is None:
            print(f"{name:<20} {'-':<12} {'-':<24} hucre calistirilmadi")
            continue
        artifact = entry['artifact_status'] if entry['artifact_status'] is not None else '-'
        detail = (entry['detail'] or '')[:60]
        print(f"{name:<20} {entry['execution_status']:<12} {artifact:<24} {detail}")
    print('=' * 78)

print_status_group('ANA URETIM HATTI: kaynak -> Faz 0 plan -> analiz -> moment -> (script/ses) -> render',
                   MAIN_PIPELINE_ORDER)
if isinstance(globals().get('processing_plan'), dict):
    _pd = processing_plan.get('decision') or {}
    print()
    print(f"Faz 0 plani: mod={_pd.get('mode')} status={processing_plan.get('status')} "
          f"cikti={_pd.get('output_count')} pipeline_mode={processing_plan.get('pipeline_mode')}")
    print('  acik fazlar   :', processing_plan.get('enabled_phases'))
    print('  kapali fazlar :', processing_plan.get('disabled_phases'))
else:
    print()
    print('Faz 0 isleme plani YOK -> ana hatta hicbir faz calistirilamaz.')
print()
print_status_group('BAGIMSIZ ARASTIRMA YAN KOLU: ana pipeline girdilerini degistirmez',
                   RESEARCH_SIDECAR_ORDER)

# Temiz calisip yetersiz veri bildiren fazlari ACIKCA one cikar.
needs_attention = [
    name for name, entry in phase_report.items()
    if entry['execution_status'] == 'completed'
    and entry['artifact_status'] in ('insufficient_data', 'failed', 'development_placeholder')
]
if needs_attention:
    print()
    print('DIKKAT: bu fazlar temiz CALISTI ama artefakt durumu kullanima hazir degil:')
    for name in needs_attention:
        print(f"  - {name}: artifact_status={phase_report[name]['artifact_status']}"
              f" ({phase_report[name]['detail']})")

awaiting_review = [name for name, entry in phase_report.items()
                  if entry['artifact_status'] == 'review_required']
if awaiting_review:
    print()
    print('INSAN INCELEMESI BEKLEYENLER (ilgili inceleme + onay hucrelerini calistir):')
    for name in awaiting_review:
        print(f'  - {name}')

print()
print("Hicbir faz YouTube'a veya baska bir yere yukleme/yayin yapmadi.")
print('production_ready statusu hicbir yerde otomatik atanmaz.')


## Colab yeniden başlatma / kalıcı artefaktlardan devam etme

`USE_GOOGLE_DRIVE = True` iken tüm artefaktlar ve önbellek Drive'da kalır. Colab yeniden başlatılsa
bile bu notebook'u tekrar açıp kurulum + toggle + setup + bootstrap + yol çözümleme hücrelerini
çalıştırman, tamamlanmış fazların toggle'ını `False` yapman ve devam eden fazı çalıştırman yeterlidir.

Onaylar içeriğe SHA-256 ile bağlıdır: bir fazı yeniden çalıştırıp içeriği değiştirirsen, o artefaktın
önceki onayı otomatik olarak geçersiz sayılır (`approval.check_approval`) ve yeniden incelenip
onaylanması gerekir. Faz 3 (TTS), diskten okuduğu script onayının hâlâ geçerli olduğunu ayrıca
doğrular; geçersizse ücretli TTS çalıştırmaz.